<a href="https://colab.research.google.com/github/MWANIKID/Forecasting-Sectoral-Crash-Risk-and-Contagion/blob/main/Forecasting_Tail_Event_Probabilities_Across_Equity_Market_Sectors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================================
# PYTHON PHASE 1.3 — REVIEWER-REVISION HAWKES / COMMON-SHOCK ANALYSIS
#
# Forecasting Sectoral Crash Risk and Contagion:
# Integrating Dynamic Volatility, Extreme-Value Modelling and Graph Deep Learning
#
# INPUT
# -----
# 04_Python_Handoff.zip generated by the final corrected R script.
#
# This is intentionally the FIRST Python script. It does not depend on any
# earlier Python ZIP. It starts directly from the frozen R econometric outputs.
#
# WHAT THIS SCRIPT DOES
# ---------------------
#  1. Validates and imports the final R handoff by CONTENT.
#  2. Audits the corrected R GJR-GARCH / EVT / sector-eligibility outputs.
#  3. Uses the frozen R Crash_Main = 2.5% EVT crash definition.
#  4. Constructs censored time-to-next-crash targets at 1, 5, 10 and 22 days.
#  5. Fits nested discrete-time Hawkes models on TRAIN only:
#       - baseline intensity,
#       - self-exciting Hawkes,
#       - full multivariate Hawkes.
#  6. Selects Hawkes half-life by TRAIN BIC.
#  7. Tunes cross-sector L1 regularization using temporal CV WITHIN TRAIN only.
#  8. Performs block stability selection of directed cross-sector edges.
#  9. Re-fits the stability-selected support without L1 shrinkage.
# 10. Evaluates Validation using TRAIN-fitted coefficients.
# 11. Freezes TRAIN-estimated Hawkes support and coefficients for the dedicated
#     Calibration period and untouched TEST sample.
# 12. Creates split-aware dynamic Hawkes graphs for the later graph DL models.
# 13. Exports baseline forecasts, tables, figures, Excel, model objects and ZIP.
# 14. Automatically downloads the final ZIP in Google Colab.
#
# IMPORTANT CORRECTIONS INCORPORATED
# ----------------------------------
# * No Python-to-Python input dependency: starts from 04_Python_Handoff.zip.
# * R EVT labels are checked for parameter variation and threshold ordering.
# * No arbitrary re-estimation of R crash labels in Python.
# * Time-to-crash outcomes are censored when future sector-day labels are absent.
# * Historical baselines are strictly past-only; no overlapping-horizon leakage.
# * Hawkes multi-horizon survival probabilities use the correct t+1 state:
#       k=1 uses the post-event state at forecast origin t with NO extra decay.
# * Hawkes structure/penalty/support decisions never use Test outcomes.
# * Hawkes structure and coefficients are frozen after Train; Calibration and
#   Test outcomes are never used to update the econometric graph.
# * If stable cross-sector edges do not survive, the script records that result
#   instead of manufacturing a graph.
#
# VERSION: 1.3
# DATE: 2026-09-12
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.optimize import minimize

from sklearn.metrics import average_precision_score, roc_auc_score, mean_pinball_loss
from sklearn.linear_model import QuantileRegressor

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# 0. REPRODUCIBILITY AND CONFIGURATION
# =============================================================================

SEED = 20260901
np.random.seed(SEED)
random.seed(SEED)

INPUT_ZIP = os.getenv(
    "NSE_R_HANDOFF_ZIP",
    "/content/04_Python_Handoff.zip"
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PYTHON_PHASE1_OUTPUT_DIR",
    "/content/Sectoral_Crash_Risk_Contagion_Python_Phase1_v1_3_ReviewerRevision"
))

HORIZONS = [1, 5, 10, 22]
MAX_H = max(HORIZONS)

# Hawkes memory candidates used for TRAIN-only BIC selection.
HALF_LIFE_CANDIDATES = [1, 2, 5, 10, 22]

# L1 grid: mean NLL + lambda * sum(cross-sector alpha).
LAMBDA_GRID = [0.0, 0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50]

# Primary Hawkes stability selection now uses lambda_min (the CV-loss minimizer).
# The more conservative 1-SE penalty is retained as a robustness check only.
STABILITY_REPS_CONSERVATIVE = int(
    os.getenv("NSE_STABILITY_REPS_CONSERVATIVE", "50")
)

# Fallback lower-tail quantile graph. It is activated only when no Hawkes
# cross-sector edge survives the pre-specified 60% stability threshold.
TAIL_QUANTILE = 0.05
TAIL_QUANTILE_ALPHA_GRID = [0.0, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
TAIL_GRAPH_STABILITY_REPS = int(
    os.getenv("NSE_TAIL_GRAPH_STABILITY_REPS", "100")
)
TAIL_GRAPH_STABILITY_THRESHOLD = 0.60
TAIL_BETA_TOL = 1e-5

# Temporal CV inside TRAIN only.
N_TEMPORAL_FOLDS = 3
INITIAL_TRAIN_FRACTION = 0.55
VALIDATION_BLOCK_FRACTION = 0.15

# Stability selection defaults. Environment variables allow a quick diagnostic
# run without modifying the submitted code.
STABILITY_REPS_PRIMARY = int(
    os.getenv("NSE_STABILITY_REPS_PRIMARY", "100")
)
STABILITY_REPS_SENSITIVITY = int(
    os.getenv("NSE_STABILITY_REPS_SENSITIVITY", "50")
)
STABILITY_SUBSAMPLE_FRACTION = 0.70
STABILITY_BLOCK_LENGTH = 60
STABILITY_THRESHOLD = 0.60
STABILITY_THRESHOLDS = [0.60, 0.70, 0.80, 0.90]
COMMON_SHOCK_STABILITY_REPS = int(
    os.getenv("NSE_COMMON_SHOCK_STABILITY_REPS", str(STABILITY_REPS_PRIMARY))
)
EDGE_NUMERIC_TOL = 1e-5

# Optimization.
EPS = 1e-10
MU_BOUNDS = (1e-8, 1.0)
ALPHA_BOUNDS = (0.0, 10.0)
FINAL_MULTISTARTS = int(
    os.getenv("NSE_HAWKES_MULTISTARTS", "5")
)
MAXITER = 3000

# Useful discrete-kernel stability guard:
# B = A / (1 - decay)
MAX_BRANCHING_SPECTRAL_RADIUS = 0.98

# Calibration/Test forecast evidence is reported separately from graph existence.
# This legacy constant is retained for compatibility but is not used for selection.
MIN_VALIDATION_SKILL_HORIZONS = 2

# =============================================================================
# 1. OUTPUT FOLDERS
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_R_Handoff_Extracted"

for d in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR, MODEL_DIR,
    DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase1_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

log("=" * 96)
log("PYTHON PHASE 1 START — DIRECT R HANDOFF")
log(f"Python: {sys.version.split()[0]}")
log(f"Platform: {platform.platform()}")
log(f"Seed: {SEED}")

# =============================================================================
# 2. VALIDATE / UPLOAD FINAL R HANDOFF ZIP
# =============================================================================

REQUIRED_R_HANDOFF_MEMBERS = {
    "sector_forecasting_master.csv.gz",
    "evt_parameter_history.csv.gz",
    "study_metadata.csv.gz",
    "stock_clean_internal_features.csv.gz",
    # Reviewer-revision R output: same GJR-GARCH/EVT construction at market level.
    "market_common_shock_garch_evt.csv.gz",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as zf:
            return {
                Path(name).name
                for name in zf.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_r_handoff(path: Path) -> bool:
    if not path.exists() or path.suffix.lower() != ".zip":
        return False
    return REQUIRED_R_HANDOFF_MEMBERS.issubset(zip_basenames(path))

def explain_rejected_zip(path: Path):
    names = zip_basenames(path)
    missing = sorted(REQUIRED_R_HANDOFF_MEMBERS - names)
    log(
        f"Rejected ZIP '{path.name}'. It is not the final R Python handoff. "
        f"Missing required files: {missing}"
    )

def resolve_r_handoff(configured: str) -> Path:
    """
    In Google Colab, ALWAYS ask the user to upload the final R handoff ZIP.
    The script will not silently use any ZIP already present in /content.

    Outside Colab, it falls back to the explicitly configured path.
    """

    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the FINAL R handoff file:\n"
                "    04_Python_Handoff.zip\n"
            )

            uploaded = files.upload()

            zip_names = [
                name for name in uploaded
                if name.lower().endswith(".zip")
            ]

            if not zip_names:
                print(
                    "\nNo ZIP file was selected. "
                    "Please choose 04_Python_Handoff.zip.\n"
                )
                continue

            valid_uploaded = []

            for name in zip_names:
                candidate = Path("/content") / name

                if is_valid_r_handoff(candidate):
                    valid_uploaded.append(candidate)
                else:
                    explain_rejected_zip(candidate)

            if len(valid_uploaded) == 1:
                chosen = valid_uploaded[0]
                log(
                    f"Validated uploaded R handoff: {chosen}"
                )
                return chosen

            if len(valid_uploaded) > 1:
                print(
                    "\nMore than one valid R handoff ZIP was uploaded. "
                    "Please upload only the single file "
                    "'04_Python_Handoff.zip'.\n"
                )
                continue

            print(
                "\nThe selected ZIP is not the final R handoff.\n"
                "Please choose the file generated by the final corrected R run:\n"
                "    04_Python_Handoff.zip\n"
            )

    except ImportError:
        # Non-Colab execution: use only the explicitly configured path.
        configured_path = Path(configured)

        if configured_path.exists() and is_valid_r_handoff(configured_path):
            log(
                f"Non-Colab execution: using configured R handoff: "
                f"{configured_path}"
            )
            return configured_path

        if configured_path.exists():
            explain_rejected_zip(configured_path)

        raise FileNotFoundError(
            "Not running in Google Colab and the configured R handoff "
            "could not be validated. Set NSE_R_HANDOFF_ZIP to the exact "
            "path of 04_Python_Handoff.zip."
        )

INPUT_ZIP_PATH = resolve_r_handoff(INPUT_ZIP)
log(f"Accepted R handoff: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as zf:
    zf.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' after extraction; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("sector_forecasting_master.csv.gz")
EVT_HISTORY_FILE = find_one("evt_parameter_history.csv.gz")
META_FILE = find_one("study_metadata.csv.gz")
MARKET_COMMON_FILE = find_one("market_common_shock_garch_evt.csv.gz")

# =============================================================================
# 3. LOAD FINAL R OUTPUTS
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
evt_history = pd.read_csv(EVT_HISTORY_FILE, parse_dates=["RefitDate"])
metadata_df = pd.read_csv(META_FILE)
market_common = pd.read_csv(MARKET_COMMON_FILE, parse_dates=["Date"])

metadata = dict(zip(
    metadata_df["Key"].astype(str),
    metadata_df["Value"].astype(str)
))

master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

def to_bool(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "t", "1", "yes", "y"])
    )

master["PrimarySector"] = to_bool(master["PrimarySector"])
master["SectorDayEligible"] = to_bool(master["SectorDayEligible"])

required_master_cols = [
    "Date", "Sector", "Split", "PrimarySector",
    "SectorDayEligible", "SectorReturn_Model",
    "GARCH_Sigma", "StdInnovation",
    "EVT_u", "EVT_scale", "EVT_shape",
    "CrashThreshold_001", "CrashThreshold_0025", "CrashThreshold_005",
    "Crash_001", "Crash_0025", "Crash_005", "Crash_Main",
]
missing_cols = [
    c for c in required_master_cols
    if c not in master.columns
]
if missing_cols:
    raise ValueError(
        f"R handoff is missing required columns: {missing_cols}"
    )

primary_sectors = sorted(
    master.loc[master["PrimarySector"], "Sector"]
    .dropna()
    .unique()
    .tolist()
)

if len(primary_sectors) < 3:
    raise RuntimeError(
        f"Only {len(primary_sectors)} primary sectors found."
    )

log(f"Primary sectors ({len(primary_sectors)}): {primary_sectors}")
log(
    f"R master: {len(master):,} rows, "
    f"{master['Date'].nunique():,} dates, "
    f"{master['Sector'].nunique()} total sectors."
)

# =============================================================================
# 4. AUDIT THE CORRECTED R EVT OUTPUT BEFORE USING CRASH LABELS
# =============================================================================

primary = (
    master[master["PrimarySector"]]
    .copy()
    .sort_values(["Date", "Sector"])
    .reset_index(drop=True)
)

# 4.1 Crash_Main must be exactly the 2.5% EVT label wherever both exist.
both = primary["Crash_Main"].notna() & primary["Crash_0025"].notna()
main_mismatch = int(
    (
        primary.loc[both, "Crash_Main"].astype(float).to_numpy()
        != primary.loc[both, "Crash_0025"].astype(float).to_numpy()
    ).sum()
)
if main_mismatch:
    raise RuntimeError(
        f"Crash_Main differs from Crash_0025 in {main_mismatch} rows."
    )

# 4.2 Threshold ordering.
thr = primary[
    ["CrashThreshold_001", "CrashThreshold_0025", "CrashThreshold_005"]
].dropna()

threshold_order_ok = bool(
    (
        (thr["CrashThreshold_001"] <= thr["CrashThreshold_0025"])
        & (thr["CrashThreshold_0025"] <= thr["CrashThreshold_005"])
    ).all()
)

if not threshold_order_ok:
    raise RuntimeError(
        "EVT threshold ordering 1% <= 2.5% <= 5% failed."
    )

# 4.3 Corrected EVT must show real parameter variation.
shape_nonmissing = primary["EVT_shape"].dropna()
scale_nonmissing = primary["EVT_scale"].dropna()

if len(shape_nonmissing) == 0 or len(scale_nonmissing) == 0:
    raise RuntimeError("No non-missing EVT parameters found.")

shape_sd = float(shape_nonmissing.std())
scale_sd = float(scale_nonmissing.std())

if shape_sd < 1e-6 or scale_sd < 1e-6:
    raise RuntimeError(
        "EVT parameter-variation guard failed. "
        "The handoff may be from the defective pre-correction R run."
    )

# 4.4 All observed crash labels binary.
for c in ["Crash_001", "Crash_0025", "Crash_005", "Crash_Main"]:
    values = primary[c].dropna()
    if not values.isin([0, 1]).all():
        raise RuntimeError(f"{c} is not binary.")

r_audit = pd.DataFrame({
    "Check": [
        "Crash_Main equals Crash_0025",
        "EVT thresholds ordered 1% <= 2.5% <= 5%",
        "EVT shape parameter varies",
        "EVT scale parameter varies",
        "Primary sectors >= 3",
    ],
    "Passed": [
        main_mismatch == 0,
        threshold_order_ok,
        shape_sd >= 1e-6,
        scale_sd >= 1e-6,
        len(primary_sectors) >= 3,
    ],
    "Value": [
        main_mismatch,
        int(threshold_order_ok),
        shape_sd,
        scale_sd,
        len(primary_sectors),
    ],
})
r_audit.to_csv(
    TABLE_DIR / "Table_P01_R_Handoff_Integrity.csv",
    index=False
)

# 4.5 Sector audit.
sector_audit = (
    primary.groupby("Sector", as_index=False)
    .agg(
        Rows=("Date", "size"),
        EligibleDays=("SectorDayEligible", "sum"),
        ObservedCrashLabels=("Crash_Main", lambda x: x.notna().sum()),
        CrashEvents=("Crash_Main", lambda x: np.nansum(x)),
        MedianStocksReturn=("NStocksReturn", "median"),
        MedianConstituentShare=("ConstituentReturnShare", "median"),
        EVTShapeMean=("EVT_shape", "mean"),
        EVTShapeSD=("EVT_shape", "std"),
    )
)
sector_audit["EligibleCoverage"] = (
    sector_audit["EligibleDays"] / sector_audit["Rows"]
)
sector_audit["CrashRate"] = (
    sector_audit["CrashEvents"]
    / sector_audit["ObservedCrashLabels"].replace(0, np.nan)
)
sector_audit.to_csv(
    TABLE_DIR / "Table_P02_Primary_Sector_Audit.csv",
    index=False
)

# =============================================================================
# 5. ALIGN PRIMARY-SECTOR EVENT PANEL
# =============================================================================

sectors = primary_sectors
S = len(sectors)

dates = pd.DatetimeIndex(sorted(primary["Date"].unique()))
T = len(dates)

sector_to_idx = {sector: i for i, sector in enumerate(sectors)}
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

if primary.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found.")

C = np.zeros((T, S), dtype=float)
M = np.zeros((T, S), dtype=bool)

split_map = (
    primary[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .to_dict()
)
splits = np.array(
    [str(split_map[pd.Timestamp(d)]) for d in dates],
    dtype=object
)

for row in primary[
    ["Date", "Sector", "Crash_Main"]
].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]

    if pd.notna(row.Crash_Main):
        C[t, s] = float(row.Crash_Main)
        M[t, s] = True

train_date_mask = splits == "Train"
validation_date_mask = splits == "Validation"   # dedicated Calibration period downstream
test_date_mask = splits == "Test"
trainval_date_mask = train_date_mask | validation_date_mask  # compatibility only; not used for refitting

# Reviewer-revision phase labels from R, when available.
if "RevisionPhase" in primary.columns:
    revision_phase_map = (
        primary[["Date", "RevisionPhase"]]
        .drop_duplicates()
        .set_index("Date")["RevisionPhase"]
        .to_dict()
    )
    revision_phases = np.array(
        [str(revision_phase_map[pd.Timestamp(d)]) for d in dates],
        dtype=object,
    )
else:
    revision_phases = np.where(
        train_date_mask, "Train",
        np.where(validation_date_mask, "Calibration", "Test")
    ).astype(object)

# Align the NSE-wide market tail-event series created by the revised R phase.
market_common = market_common.sort_values("Date").drop_duplicates("Date").copy()
market_event = np.zeros(T, dtype=float)
market_observed = np.zeros(T, dtype=bool)

for row in market_common[["Date", "Crash_Main"]].itertuples(index=False):
    d = pd.Timestamp(row.Date)
    if d not in date_to_idx:
        continue
    t = date_to_idx[d]
    if pd.notna(row.Crash_Main):
        market_event[t] = float(row.Crash_Main)
        market_observed[t] = True

if market_observed.sum() == 0:
    raise RuntimeError("No observed market-wide tail-event labels found in revised R handoff.")

first_market_obs = int(np.where(market_observed)[0][0])
if (~market_observed[first_market_obs:]).any():
    raise RuntimeError(
        "Market-wide tail-event series contains gaps after its first observed label; "
        "common-shock Hawkes control would be ambiguous."
    )

log(
    f"Event panel: {T} dates x {S} sectors. "
    f"Train tail events={int(C[train_date_mask].sum())}; "
    f"Calibration tail events={int(C[validation_date_mask].sum())}; "
    f"Test tail events={int(C[test_date_mask].sum())}; "
    f"market tail events observed={int(market_event[market_observed].sum())}."
)

# =============================================================================
# 6. CENSORED TIME-TO-NEXT-CRASH TARGETS
# =============================================================================

event_observed = np.full((T, S), np.nan)
event_time = np.full((T, S), np.nan)
censor_time = np.full((T, S), np.nan)

target_within = {
    h: np.full((T, S), np.nan, dtype=float)
    for h in HORIZONS
}

for t in range(T):
    for s in range(S):

        # Forecast origin is usable only when current crash state is observed.
        if not M[t, s]:
            continue

        max_available = min(MAX_H, T - 1 - t)
        if max_available <= 0:
            continue

        observed_until = 0
        found_event = False
        event_k = None

        for k in range(1, max_available + 1):
            # Hard stop at Train/Calibration/Test boundaries so no origin is
            # labelled using outcomes from a later experimental phase.
            if splits[t + k] != splits[t]:
                break
            if not M[t + k, s]:
                break

            observed_until = k

            if C[t + k, s] == 1:
                found_event = True
                event_k = k
                break

        if found_event:
            event_observed[t, s] = 1.0
            event_time[t, s] = float(event_k)
            censor_time[t, s] = float(event_k)
        else:
            event_observed[t, s] = 0.0
            censor_time[t, s] = float(observed_until)

        for h in HORIZONS:
            if found_event and event_k <= h:
                target_within[h][t, s] = 1.0
            elif observed_until >= h:
                target_within[h][t, s] = 0.0
            else:
                target_within[h][t, s] = np.nan

target_rows = []

for t, date in enumerate(dates):
    for s, sector in enumerate(sectors):
        row = {
            "Date": date,
            "Sector": sector,
            "Split": splits[t],
            "CurrentCrashObserved": int(M[t, s]),
            "CurrentCrash": C[t, s] if M[t, s] else np.nan,
            "EventObservedWithin22": event_observed[t, s],
            "EventTime": event_time[t, s],
            "CensorTime": censor_time[t, s],
        }

        for h in HORIZONS:
            row[f"CrashWithin_{h}"] = target_within[h][t, s]

        target_rows.append(row)

targets_long = pd.DataFrame(target_rows)

target_summary_rows = []
for split_name in ["Train", "Validation", "Test"]:
    for sector in sectors:
        sub = targets_long[
            (targets_long["Split"] == split_name)
            & (targets_long["Sector"] == sector)
        ]
        for h in HORIZONS:
            y = sub[f"CrashWithin_{h}"].dropna()
            target_summary_rows.append({
                "Split": split_name,
                "Sector": sector,
                "Horizon": h,
                "ValidTargets": len(y),
                "PositiveTargets": int(y.sum()) if len(y) else 0,
                "PositiveRate": float(y.mean()) if len(y) else np.nan,
            })

target_summary = pd.DataFrame(target_summary_rows)
target_summary.to_csv(
    TABLE_DIR / "Table_P03_Time_To_Crash_Target_Summary.csv",
    index=False
)

# =============================================================================
# 7. DISCRETE-TIME HAWKES FUNCTIONS
# =============================================================================
#
# Predictive process:
#
#   H_j,t = d H_j,t-1 + C_j,t-1
#   lambda_i,t = mu_i + sum_j alpha_i,j H_j,t
#   P(C_i,t=1 | F_t-1) = 1 - exp(-lambda_i,t)
#
# alpha[i,j] is directed predictive excitation FROM source j TO receiver i.

def decay_from_half_life(half_life: float) -> float:
    return float(np.exp(-np.log(2.0) / float(half_life)))

def build_pre_event_state(
    events: np.ndarray,
    decay: float,
) -> np.ndarray:
    T_, S_ = events.shape
    H = np.zeros((T_, S_), dtype=float)
    state = np.zeros(S_, dtype=float)

    for t in range(T_):
        H[t] = state
        state = decay * state + events[t]

    return H

def build_post_event_state(
    events: np.ndarray,
    decay: float,
) -> np.ndarray:
    T_, S_ = events.shape
    G = np.zeros((T_, S_), dtype=float)
    state = np.zeros(S_, dtype=float)

    for t in range(T_):
        state = decay * state + events[t]
        G[t] = state

    return G

def receiver_objective(
    theta: np.ndarray,
    X: np.ndarray,
    y: np.ndarray,
    allowed_sources: np.ndarray,
    receiver: int,
    l1_lambda: float,
) -> float:
    mu = float(theta[0])

    alpha = np.zeros(S, dtype=float)
    alpha[allowed_sources] = theta[1:]

    intensity = np.clip(
        mu + X @ alpha,
        EPS,
        50.0,
    )

    probability = np.clip(
        -np.expm1(-intensity),
        EPS,
        1.0 - EPS,
    )

    mean_nll = -np.mean(
        y * np.log(probability)
        + (1.0 - y) * np.log1p(-probability)
    )

    cross_mask = np.arange(S) != receiver
    penalty = (
        float(l1_lambda)
        * alpha[cross_mask].sum()
    )

    return float(mean_nll + penalty)

def fit_receiver(
    X: np.ndarray,
    y: np.ndarray,
    receiver: int,
    l1_lambda: float = 0.0,
    allowed_mask: np.ndarray | None = None,
    n_starts: int = 1,
) -> dict:

    if allowed_mask is None:
        allowed_mask = np.ones(S, dtype=bool)
    else:
        allowed_mask = np.asarray(allowed_mask, dtype=bool).copy()

    allowed_sources = np.where(allowed_mask)[0]

    event_rate = float(
        np.clip(y.mean(), 1e-6, 0.50)
    )
    mu0 = float(
        -np.log(1.0 - event_rate)
    )

    if len(allowed_sources) == 0:
        return {
            "mu": mu0,
            "alpha": np.zeros(S, dtype=float),
            "success": True,
            "objective": np.nan,
            "message": "Baseline-only closed-form MLE",
        }

    rng = np.random.default_rng(
        SEED + 1009 * (receiver + 1) + int(y.sum())
    )

    starts = [
        np.r_[mu0, np.repeat(0.01, len(allowed_sources))]
    ]

    while len(starts) < n_starts:
        starts.append(
            np.r_[
                max(mu0 * rng.uniform(0.60, 1.40), 1e-6),
                rng.uniform(0.0, 0.08, len(allowed_sources)),
            ]
        )

    bounds = (
        [MU_BOUNDS]
        + [ALPHA_BOUNDS] * len(allowed_sources)
    )

    best = None

    for start in starts:
        result = minimize(
            receiver_objective,
            x0=start,
            args=(
                X,
                y,
                allowed_sources,
                receiver,
                l1_lambda,
            ),
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "maxiter": MAXITER,
                "ftol": 1e-12,
                "gtol": 1e-8,
            },
        )

        if (
            best is None
            or (
                np.isfinite(result.fun)
                and result.fun < best.fun
            )
        ):
            best = result

    if best is None or not np.isfinite(best.fun):
        raise RuntimeError(
            f"Hawkes optimization failed for receiver "
            f"{sectors[receiver]}."
        )

    alpha = np.zeros(S, dtype=float)
    alpha[allowed_sources] = best.x[1:]

    return {
        "mu": float(best.x[0]),
        "alpha": alpha,
        "success": bool(best.success),
        "objective": float(best.fun),
        "message": str(best.message),
    }

def fit_network(
    fit_date_mask: np.ndarray,
    half_life: float,
    structure: str,
    l1_lambda: float = 0.0,
    support_mask: np.ndarray | None = None,
    n_starts: int = 1,
) -> dict:

    decay = decay_from_half_life(half_life)
    H_pre = build_pre_event_state(C, decay)

    mu = np.zeros(S, dtype=float)
    A = np.zeros((S, S), dtype=float)
    details = []

    for receiver in range(S):

        valid = (
            fit_date_mask
            & M[:, receiver]
        )

        X = H_pre[valid]
        y = C[valid, receiver].astype(float)

        if len(y) == 0 or y.sum() == 0:
            raise RuntimeError(
                f"No fitting crash events for "
                f"{sectors[receiver]}."
            )

        if structure == "baseline":
            allowed = np.zeros(S, dtype=bool)

        elif structure == "self":
            allowed = np.zeros(S, dtype=bool)
            allowed[receiver] = True

        elif structure in ("full", "sparse"):
            allowed = np.ones(S, dtype=bool)

        elif structure == "support":
            if support_mask is None:
                raise ValueError(
                    "support_mask required for support model."
                )
            allowed = support_mask[receiver].copy()
            # Own-sector self-excitation remains permissible.
            allowed[receiver] = True

        else:
            raise ValueError(
                f"Unknown Hawkes structure: {structure}"
            )

        fit = fit_receiver(
            X=X,
            y=y,
            receiver=receiver,
            l1_lambda=(
                l1_lambda
                if structure == "sparse"
                else 0.0
            ),
            allowed_mask=allowed,
            n_starts=n_starts,
        )

        mu[receiver] = fit["mu"]
        A[receiver] = fit["alpha"]

        details.append({
            "Receiver": sectors[receiver],
            "Structure": structure,
            "HalfLife": half_life,
            "L1Lambda": (
                l1_lambda
                if structure == "sparse"
                else 0.0
            ),
            "N": len(y),
            "Events": int(y.sum()),
            "Mu": fit["mu"],
            "Converged": fit["success"],
            "Message": fit["message"],
        })

    return {
        "structure": structure,
        "half_life": float(half_life),
        "decay": decay,
        "l1_lambda": float(l1_lambda),
        "mu": mu,
        "alpha": A,
        "details": pd.DataFrame(details),
    }

def network_loglik(
    fit: dict,
    date_mask: np.ndarray,
) -> tuple[float, int]:

    H_pre = build_pre_event_state(
        C,
        fit["decay"],
    )

    total_ll = 0.0
    nobs = 0

    for receiver in range(S):
        valid = date_mask & M[:, receiver]

        X = H_pre[valid]
        y = C[valid, receiver].astype(float)

        intensity = np.clip(
            fit["mu"][receiver]
            + X @ fit["alpha"][receiver],
            EPS,
            50.0,
        )

        p = np.clip(
            -np.expm1(-intensity),
            EPS,
            1.0 - EPS,
        )

        total_ll += float(
            np.sum(
                y * np.log(p)
                + (1.0 - y) * np.log1p(-p)
            )
        )
        nobs += len(y)

    return total_ll, nobs


# -----------------------------------------------------------------------------
# 7A. COMMON-SHOCK-CONTROLLED HAWKES ROBUSTNESS
# -----------------------------------------------------------------------------
# The reviewer robustness model augments each sector intensity with an
# unpenalized NSE-wide tail-event state:
#
#   lambda_i,t = mu_i + gamma_i H_M,t + sum_j alpha_i,j H_j,t
#
# where both H_M,t and H_j,t use information only through t-1.  Cross-sector
# alpha coefficients remain the objects subject to L1/stability selection.

def receiver_objective_common(
    theta: np.ndarray,
    X: np.ndarray,
    h_market: np.ndarray,
    y: np.ndarray,
    allowed_sources: np.ndarray,
    receiver: int,
    l1_lambda: float,
) -> float:
    mu = float(theta[0])
    gamma = float(theta[1])

    alpha = np.zeros(S, dtype=float)
    alpha[allowed_sources] = theta[2:]

    intensity = np.clip(
        mu + gamma * h_market + X @ alpha,
        EPS,
        50.0,
    )
    probability = np.clip(
        -np.expm1(-intensity),
        EPS,
        1.0 - EPS,
    )
    mean_nll = -np.mean(
        y * np.log(probability)
        + (1.0 - y) * np.log1p(-probability)
    )

    cross_mask = np.arange(S) != receiver
    penalty = float(l1_lambda) * alpha[cross_mask].sum()
    return float(mean_nll + penalty)


def fit_receiver_common(
    X: np.ndarray,
    h_market: np.ndarray,
    y: np.ndarray,
    receiver: int,
    l1_lambda: float = 0.0,
    allowed_mask: np.ndarray | None = None,
    n_starts: int = 1,
) -> dict:

    if allowed_mask is None:
        allowed_mask = np.ones(S, dtype=bool)
    else:
        allowed_mask = np.asarray(allowed_mask, dtype=bool).copy()

    allowed_sources = np.where(allowed_mask)[0]
    event_rate = float(np.clip(y.mean(), 1e-6, 0.50))
    mu0 = float(-np.log(1.0 - event_rate))

    rng = np.random.default_rng(
        SEED + 2003 * (receiver + 1) + int(y.sum())
    )

    starts = [
        np.r_[mu0, 0.01, np.repeat(0.01, len(allowed_sources))]
    ]
    while len(starts) < n_starts:
        starts.append(
            np.r_[
                max(mu0 * rng.uniform(0.60, 1.40), 1e-6),
                rng.uniform(0.0, 0.08),
                rng.uniform(0.0, 0.08, len(allowed_sources)),
            ]
        )

    bounds = (
        [MU_BOUNDS, ALPHA_BOUNDS]
        + [ALPHA_BOUNDS] * len(allowed_sources)
    )

    best = None
    for start in starts:
        result = minimize(
            receiver_objective_common,
            x0=start,
            args=(
                X,
                h_market,
                y,
                allowed_sources,
                receiver,
                l1_lambda,
            ),
            method="L-BFGS-B",
            bounds=bounds,
            options={
                "maxiter": MAXITER,
                "ftol": 1e-12,
                "gtol": 1e-8,
            },
        )
        if best is None or (
            np.isfinite(result.fun) and result.fun < best.fun
        ):
            best = result

    if best is None or not np.isfinite(best.fun):
        raise RuntimeError(
            f"Common-shock Hawkes optimization failed for receiver "
            f"{sectors[receiver]}."
        )

    alpha = np.zeros(S, dtype=float)
    alpha[allowed_sources] = best.x[2:]

    return {
        "mu": float(best.x[0]),
        "market_beta": float(best.x[1]),
        "alpha": alpha,
        "success": bool(best.success),
        "objective": float(best.fun),
        "message": str(best.message),
    }


def fit_network_common(
    fit_date_mask: np.ndarray,
    half_life: float,
    structure: str,
    l1_lambda: float = 0.0,
    support_mask: np.ndarray | None = None,
    n_starts: int = 1,
) -> dict:

    decay = decay_from_half_life(half_life)
    H_pre = build_pre_event_state(C, decay)
    H_market = build_pre_event_state(
        market_event[:, None],
        decay,
    )[:, 0]

    mu = np.zeros(S, dtype=float)
    market_beta = np.zeros(S, dtype=float)
    A = np.zeros((S, S), dtype=float)
    details = []

    for receiver in range(S):
        valid = (
            fit_date_mask
            & M[:, receiver]
            & market_observed
        )

        X = H_pre[valid]
        hm = H_market[valid]
        y = C[valid, receiver].astype(float)

        if len(y) == 0 or y.sum() == 0:
            raise RuntimeError(
                f"No fitting tail events for common-shock model: "
                f"{sectors[receiver]}."
            )

        if structure == "baseline":
            allowed = np.zeros(S, dtype=bool)
        elif structure == "self":
            allowed = np.zeros(S, dtype=bool)
            allowed[receiver] = True
        elif structure in ("full", "sparse"):
            allowed = np.ones(S, dtype=bool)
        elif structure == "support":
            if support_mask is None:
                raise ValueError("support_mask required for support model.")
            allowed = support_mask[receiver].copy()
            allowed[receiver] = True
        else:
            raise ValueError(
                f"Unknown common-shock Hawkes structure: {structure}"
            )

        fit = fit_receiver_common(
            X=X,
            h_market=hm,
            y=y,
            receiver=receiver,
            l1_lambda=(
                l1_lambda if structure == "sparse" else 0.0
            ),
            allowed_mask=allowed,
            n_starts=n_starts,
        )

        mu[receiver] = fit["mu"]
        market_beta[receiver] = fit["market_beta"]
        A[receiver] = fit["alpha"]

        details.append({
            "Receiver": sectors[receiver],
            "Structure": structure,
            "HalfLife": half_life,
            "L1Lambda": (
                l1_lambda if structure == "sparse" else 0.0
            ),
            "N": len(y),
            "Events": int(y.sum()),
            "Mu": fit["mu"],
            "MarketBeta": fit["market_beta"],
            "Converged": fit["success"],
            "Message": fit["message"],
        })

    return {
        "structure": structure,
        "half_life": float(half_life),
        "decay": decay,
        "l1_lambda": float(l1_lambda),
        "mu": mu,
        "market_beta": market_beta,
        "alpha": A,
        "details": pd.DataFrame(details),
    }


def network_loglik_common(
    fit: dict,
    date_mask: np.ndarray,
) -> tuple[float, int]:

    H_pre = build_pre_event_state(C, fit["decay"])
    H_market = build_pre_event_state(
        market_event[:, None],
        fit["decay"],
    )[:, 0]

    total_ll = 0.0
    nobs = 0

    for receiver in range(S):
        valid = date_mask & M[:, receiver] & market_observed
        X = H_pre[valid]
        hm = H_market[valid]
        y = C[valid, receiver].astype(float)

        intensity = np.clip(
            fit["mu"][receiver]
            + fit["market_beta"][receiver] * hm
            + X @ fit["alpha"][receiver],
            EPS,
            50.0,
        )
        p = np.clip(
            -np.expm1(-intensity),
            EPS,
            1.0 - EPS,
        )
        total_ll += float(
            np.sum(
                y * np.log(p)
                + (1.0 - y) * np.log1p(-p)
            )
        )
        nobs += len(y)

    return total_ll, nobs


# =============================================================================
# 8. TRAIN-ONLY HALF-LIFE SELECTION BY BIC
# =============================================================================

half_life_rows = []
half_life_fits = {}

log("Selecting Hawkes half-life by TRAIN-only BIC...")

for half_life in HALF_LIFE_CANDIDATES:
    log(f"  Fitting full Hawkes at half-life={half_life} days.")

    fit = fit_network(
        fit_date_mask=train_date_mask,
        half_life=half_life,
        structure="full",
        n_starts=FINAL_MULTISTARTS,
    )

    ll, nobs = network_loglik(
        fit,
        train_date_mask,
    )

    # Full model: S baseline intensities + S*S excitation coefficients.
    k = S + S * S

    aic = -2.0 * ll + 2.0 * k
    bic = -2.0 * ll + k * np.log(max(nobs, 2))

    half_life_fits[float(half_life)] = fit

    half_life_rows.append({
        "HalfLife": half_life,
        "Decay": fit["decay"],
        "TrainLogLik": ll,
        "NObs": nobs,
        "NParameters": k,
        "AIC": aic,
        "BIC": bic,
    })

half_life_selection = pd.DataFrame(
    half_life_rows
).sort_values("BIC")

primary_half_life = float(
    half_life_selection.iloc[0]["HalfLife"]
)

half_life_selection["DeltaBIC"] = (
    half_life_selection["BIC"]
    - half_life_selection["BIC"].min()
)

half_life_selection.to_csv(
    TABLE_DIR / "Table_P04_Hawkes_HalfLife_Selection.csv",
    index=False
)

log(
    f"TRAIN-BIC selected Hawkes half-life: "
    f"{primary_half_life:g} trading days."
)

# =============================================================================
# 9. STRICT MULTI-HORIZON HAWKES SURVIVAL PROBABILITIES
# =============================================================================

def network_probabilities(
    fit: dict,
) -> dict[int, np.ndarray]:
    """
    P(at least one crash within H days | information through t).

    IMPORTANT:
    If G_t is the post-event excitation state after observing date t,
    then the t+1 intensity uses G_t directly. Therefore k=1 uses
    decay**0, not decay**1.
    """

    mu = fit["mu"]
    A = fit["alpha"]
    decay = fit["decay"]

    H_post = build_post_event_state(
        C,
        decay,
    )

    probabilities = {
        h: np.full((T, S), np.nan, dtype=float)
        for h in HORIZONS
    }

    for t in range(T):
        state_t = H_post[t]
        cumulative_intensity = np.zeros(S, dtype=float)

        for k in range(1, MAX_H + 1):

            # No-event survival path.
            future_state = (
                (decay ** (k - 1))
                * state_t
            )

            lambda_k = np.clip(
                mu + A @ future_state,
                EPS,
                50.0,
            )

            cumulative_intensity += lambda_k

            if k in HORIZONS:
                probabilities[k][t] = (
                    1.0
                    - np.exp(-cumulative_intensity)
                )

    # Coherence guard.
    for t in range(T):
        for s in range(S):
            values = [
                probabilities[h][t, s]
                for h in HORIZONS
            ]
            if any(
                values[i] > values[i + 1] + 1e-12
                for i in range(len(values) - 1)
            ):
                raise RuntimeError(
                    "Hawkes horizon-probability coherence failed."
                )

    return probabilities

# =============================================================================
# 10. STRICT HISTORICAL BASELINES — NO OVERLAPPING-HORIZON LEAKAGE
# =============================================================================

def make_historical_baselines():
    """
    FixedTrainHistorical:
        Sector-specific one-day crash rate estimated on TRAIN, then converted:
        P_H = 1 - (1-p)^H.

    ExpandingHistorical:
        At forecast origin t, use same-day crash labels observed through t.
        This is permissible because C_t is known after market close when
        forecasting t+1 onward. It never uses CrashWithin_H labels, which would
        overlap future observations and cause leakage.
    """

    fixed = {
        h: np.full((T, S), np.nan, dtype=float)
        for h in HORIZONS
    }
    expanding = {
        h: np.full((T, S), np.nan, dtype=float)
        for h in HORIZONS
    }

    # Fixed TRAIN event rates.
    train_rates = np.zeros(S, dtype=float)

    for s in range(S):
        valid = train_date_mask & M[:, s]
        train_rates[s] = (
            C[valid, s].mean()
            if valid.sum()
            else 0.025
        )

    for h in HORIZONS:
        fixed[h][:] = (
            1.0
            - (1.0 - train_rates[np.newaxis, :]) ** h
        )

    # Expanding same-day event rates.
    event_sum = np.zeros(S, dtype=float)
    event_n = np.zeros(S, dtype=float)

    for t in range(T):
        # Current date is observed before forecasting t+1 onward.
        for s in range(S):
            if M[t, s]:
                event_sum[s] += C[t, s]
                event_n[s] += 1.0

        one_day_p = np.where(
            event_n > 0,
            event_sum / np.maximum(event_n, 1.0),
            0.025,
        )

        for h in HORIZONS:
            expanding[h][t] = (
                1.0
                - (1.0 - one_day_p) ** h
            )

    return fixed, expanding

fixed_historical, expanding_historical = make_historical_baselines()

# =============================================================================
# 11. FORECAST METRICS
# =============================================================================

def log_score(
    y: np.ndarray,
    p: np.ndarray,
) -> float:
    p = np.clip(p, EPS, 1.0 - EPS)

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(y_true, dtype=float)
    probability = np.asarray(probability, dtype=float)

    ok = (
        np.isfinite(y_true)
        & np.isfinite(probability)
    )

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": len(y),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": float(y.mean()) if len(y) else np.nan,
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean((p - y) ** 2)
    )
    result["LogScore"] = log_score(y, p)

    if len(np.unique(y)) == 2:
        result["PR_AUC"] = float(
            average_precision_score(y, p)
        )
        result["ROC_AUC"] = float(
            roc_auc_score(y, p)
        )

    return result

def pooled_multihorizon_logscore(
    probabilities: dict[int, np.ndarray],
    date_mask: np.ndarray,
) -> float:

    values = []

    for h in HORIZONS:
        y = target_within[h][date_mask].reshape(-1)
        p = probabilities[h][date_mask].reshape(-1)

        ok = np.isfinite(y) & np.isfinite(p)

        if ok.sum():
            values.append(
                log_score(y[ok], p[ok])
            )

    return (
        float(np.mean(values))
        if values
        else np.nan
    )

# =============================================================================
# 12. TEMPORAL CV FOLDS WITHIN TRAIN ONLY
# =============================================================================

train_indices = np.where(train_date_mask)[0]
effective_train_indices = train_indices[
    M[train_indices].any(axis=1)
]

if len(effective_train_indices) < 300:
    raise RuntimeError(
        "Too few effective TRAIN dates for temporal CV."
    )

n_effective = len(effective_train_indices)
initial_n = int(
    np.floor(
        INITIAL_TRAIN_FRACTION
        * n_effective
    )
)
validation_n = int(
    np.floor(
        VALIDATION_BLOCK_FRACTION
        * n_effective
    )
)

folds = []

for fold_id in range(N_TEMPORAL_FOLDS):

    fit_end_position = (
        initial_n
        + fold_id * validation_n
    )

    validation_start_position = fit_end_position

    validation_end_position = (
        n_effective
        if fold_id == N_TEMPORAL_FOLDS - 1
        else min(
            n_effective,
            validation_start_position + validation_n,
        )
    )

    if validation_start_position >= n_effective:
        break

    fit_end_index = effective_train_indices[
        fit_end_position - 1
    ]
    validation_start_index = effective_train_indices[
        validation_start_position
    ]
    validation_end_index = effective_train_indices[
        validation_end_position - 1
    ]

    fit_mask = (
        train_date_mask
        & (np.arange(T) <= fit_end_index)
    )

    validation_mask = (
        train_date_mask
        & (np.arange(T) >= validation_start_index)
        & (np.arange(T) <= validation_end_index)
    )

    folds.append({
        "Fold": len(folds) + 1,
        "FitMask": fit_mask,
        "ValidationMask": validation_mask,
        "FitEnd": dates[fit_end_index],
        "ValidationStart": dates[validation_start_index],
        "ValidationEnd": dates[validation_end_index],
    })

if len(folds) < 2:
    raise RuntimeError(
        "Could not construct at least two TRAIN-only temporal CV folds."
    )

fold_table = pd.DataFrame([
    {
        "Fold": f["Fold"],
        "FitEnd": f["FitEnd"],
        "ValidationStart": f["ValidationStart"],
        "ValidationEnd": f["ValidationEnd"],
        "FitDates": int(f["FitMask"].sum()),
        "ValidationDates": int(f["ValidationMask"].sum()),
    }
    for f in folds
])

fold_table.to_csv(
    TABLE_DIR / "Table_P05_Temporal_CV_Folds.csv",
    index=False
)

# =============================================================================
# 13. L1 PENALTY TUNING WITHIN TRAIN
# =============================================================================
#
# IMPORTANT REVISION:
#   * lambda_min (minimum temporal-CV log score) is the PRIMARY penalty used
#     for stability selection.
#   * lambda_1SE is retained as a conservative robustness specification.
#
# This avoids "double sparsification" from first selecting an aggressively
# sparse 1-SE model and then applying a second 60% stability filter.

sensitivity_half_lives = sorted(
    set([2.0, 5.0, 10.0, primary_half_life])
)

cv_rows = []

for half_life in sensitivity_half_lives:

    log(
        f"Temporal-CV sparse Hawkes tuning: "
        f"half-life={half_life:g} days."
    )

    for l1_lambda in LAMBDA_GRID:

        for fold in folds:

            fit = fit_network(
                fit_date_mask=fold["FitMask"],
                half_life=half_life,
                structure="sparse",
                l1_lambda=l1_lambda,
                n_starts=1,
            )

            probabilities = network_probabilities(fit)

            loss = pooled_multihorizon_logscore(
                probabilities,
                fold["ValidationMask"],
            )

            cross_mask = ~np.eye(S, dtype=bool)
            n_cross = int(
                (
                    (fit["alpha"] > EDGE_NUMERIC_TOL)
                    & cross_mask
                ).sum()
            )

            cv_rows.append({
                "HalfLife": half_life,
                "Lambda": l1_lambda,
                "Fold": fold["Fold"],
                "ValidationLogScore": loss,
                "NonzeroCrossEdges": n_cross,
            })

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(
    TABLE_DIR / "Table_P06_L1_Temporal_CV_All_Folds.csv",
    index=False
)

cv_summary = (
    cv_results
    .groupby(["HalfLife", "Lambda"], as_index=False)
    .agg(
        MeanValidationLogScore=("ValidationLogScore", "mean"),
        SDValidationLogScore=("ValidationLogScore", "std"),
        MeanCrossEdges=("NonzeroCrossEdges", "mean"),
        Folds=("Fold", "nunique"),
    )
)

cv_summary["SEValidationLogScore"] = (
    cv_summary["SDValidationLogScore"]
    / np.sqrt(cv_summary["Folds"])
)

lambda_selection_rows = []
lambda_min_by_half_life = {}
lambda_1se_by_half_life = {}

for half_life in sensitivity_half_lives:

    g = cv_summary[
        cv_summary["HalfLife"] == half_life
    ].copy()

    best_index = g["MeanValidationLogScore"].idxmin()
    best_row = g.loc[best_index]

    lambda_min = float(best_row["Lambda"])
    best_mean = float(best_row["MeanValidationLogScore"])
    best_se = float(best_row["SEValidationLogScore"])

    if not np.isfinite(best_se):
        best_se = 0.0

    one_se_limit = best_mean + best_se

    acceptable = g[
        g["MeanValidationLogScore"] <= one_se_limit
    ].copy()

    chosen_1se = acceptable.sort_values(
        ["Lambda", "MeanValidationLogScore"],
        ascending=[False, True],
    ).iloc[0]

    lambda_1se = float(chosen_1se["Lambda"])

    lambda_min_by_half_life[float(half_life)] = lambda_min
    lambda_1se_by_half_life[float(half_life)] = lambda_1se

    lambda_selection_rows.append({
        "HalfLife": half_life,
        "Lambda_Min": lambda_min,
        "MinimumCVLogScore": best_mean,
        "SEAtMinimum": best_se,
        "OneSELimit": one_se_limit,
        "Lambda_1SE": lambda_1se,
        "PrimaryPenaltyForStability": lambda_min,
        "RobustnessPenalty": lambda_1se,
        "MeanCrossEdgesAtLambdaMin": float(best_row["MeanCrossEdges"]),
        "MeanCrossEdgesAtLambda1SE": float(chosen_1se["MeanCrossEdges"]),
    })

lambda_selection = pd.DataFrame(lambda_selection_rows)

cv_summary.to_csv(
    TABLE_DIR / "Table_P07_L1_Temporal_CV_Summary.csv",
    index=False
)

lambda_selection.to_csv(
    TABLE_DIR / "Table_P08_L1_LambdaMin_and_1SE_Selection.csv",
    index=False
)

primary_lambda = lambda_min_by_half_life[float(primary_half_life)]
primary_lambda_1se = lambda_1se_by_half_life[float(primary_half_life)]

log(
    f"Primary half-life={primary_half_life:g}; "
    f"lambda_min={primary_lambda:g} (PRIMARY), "
    f"lambda_1SE={primary_lambda_1se:g} (ROBUSTNESS)."
)

# =============================================================================
# 14. BLOCK STABILITY SELECTION
# =============================================================================

def make_block_subsample_mask(
    eligible_indices: np.ndarray,
    fraction: float,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    eligible_indices = np.asarray(eligible_indices, dtype=int)
    n = len(eligible_indices)
    target_n = max(1, int(np.ceil(fraction * n)))
    selected_positions = np.zeros(n, dtype=bool)

    while selected_positions.sum() < target_n:
        start = int(
            rng.integers(
                0,
                max(1, n - block_length + 1),
            )
        )
        end = min(n, start + block_length)
        selected_positions[start:end] = True

    mask = np.zeros(T, dtype=bool)
    mask[eligible_indices[selected_positions]] = True
    return mask


def run_stability_selection(
    half_life: float,
    l1_lambda: float,
    repetitions: int,
    specification_label: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:

    eligible_indices = np.where(
        train_date_mask & M.any(axis=1)
    )[0]

    selection_count = np.zeros((S, S), dtype=int)
    coefficient_sum = np.zeros((S, S), dtype=float)
    replicate_rows = []
    edge_replicate_rows = []

    for repetition in range(repetitions):

        rng = np.random.default_rng(
            SEED
            + int(half_life * 1000)
            + int(l1_lambda * 100000)
            + 7919 * (repetition + 1)
        )

        subsample_mask = make_block_subsample_mask(
            eligible_indices=eligible_indices,
            fraction=STABILITY_SUBSAMPLE_FRACTION,
            block_length=STABILITY_BLOCK_LENGTH,
            rng=rng,
        )

        fit = fit_network(
            fit_date_mask=subsample_mask,
            half_life=half_life,
            structure="sparse",
            l1_lambda=l1_lambda,
            n_starts=1,
        )

        selected = fit["alpha"] > EDGE_NUMERIC_TOL

        selection_count += selected.astype(int)
        coefficient_sum += fit["alpha"]

        replicate_rows.append({
            "Specification": specification_label,
            "HalfLife": half_life,
            "Lambda": l1_lambda,
            "Replicate": repetition + 1,
            "SubsampleDates": int(subsample_mask.sum()),
            "NonzeroAllEdges": int(selected.sum()),
            "NonzeroCrossEdges": int(
                (
                    selected
                    & ~np.eye(S, dtype=bool)
                ).sum()
            ),
        })

        for receiver in range(S):
            for sender in range(S):
                edge_replicate_rows.append({
                    "Specification": specification_label,
                    "HalfLife": half_life,
                    "Lambda": l1_lambda,
                    "Replicate": repetition + 1,
                    "FromSector": sectors[sender],
                    "ToSector": sectors[receiver],
                    "SelfExcitation": int(sender == receiver),
                    "Coefficient": float(fit["alpha"][receiver, sender]),
                    "Selected": int(selected[receiver, sender]),
                })

        if (repetition + 1) % 20 == 0:
            log(
                f"  Stability {specification_label}, half-life={half_life:g}: "
                f"{repetition + 1}/{repetitions} complete."
            )

    frequency = selection_count / float(repetitions)
    mean_coefficient = coefficient_sum / float(repetitions)

    edge_rows = []

    for receiver in range(S):
        for sender in range(S):
            edge_rows.append({
                "Specification": specification_label,
                "HalfLife": half_life,
                "Lambda": l1_lambda,
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "SelfExcitation": int(sender == receiver),
                "SelectionFrequency": frequency[receiver, sender],
                "MeanCoefficientAcrossReps": mean_coefficient[receiver, sender],
                "StableAtThreshold": int(
                    frequency[receiver, sender] >= STABILITY_THRESHOLD
                ),
                **{
                    f"StableAt_{int(thr * 100):02d}pct": int(
                        frequency[receiver, sender] >= thr
                    )
                    for thr in STABILITY_THRESHOLDS
                },
            })

    return (
        pd.DataFrame(edge_rows),
        pd.DataFrame(replicate_rows),
        pd.DataFrame(edge_replicate_rows),
    )


# PRIMARY: lambda_min for each half-life.
stability_tables = []
stability_replicates = []
stability_edge_replicates = []

for half_life in sensitivity_half_lives:

    repetitions = (
        STABILITY_REPS_PRIMARY
        if float(half_life) == float(primary_half_life)
        else STABILITY_REPS_SENSITIVITY
    )

    selected_lambda = lambda_min_by_half_life[float(half_life)]

    log(
        f"PRIMARY block stability selection: half-life={half_life:g}, "
        f"lambda_min={selected_lambda:g}, reps={repetitions}."
    )

    edge_table, replicate_table, edge_rep_table = run_stability_selection(
        half_life=half_life,
        l1_lambda=selected_lambda,
        repetitions=repetitions,
        specification_label="LambdaMin_Primary",
    )

    stability_tables.append(edge_table)
    stability_replicates.append(replicate_table)
    stability_edge_replicates.append(edge_rep_table)


# CONSERVATIVE ROBUSTNESS: primary half-life with lambda_1SE.
log(
    f"ROBUSTNESS block stability selection: "
    f"half-life={primary_half_life:g}, "
    f"lambda_1SE={primary_lambda_1se:g}, "
    f"reps={STABILITY_REPS_CONSERVATIVE}."
)

edge_1se, reps_1se, edge_reps_1se = run_stability_selection(
    half_life=primary_half_life,
    l1_lambda=primary_lambda_1se,
    repetitions=STABILITY_REPS_CONSERVATIVE,
    specification_label="Lambda1SE_Robustness",
)

stability_tables.append(edge_1se)
stability_replicates.append(reps_1se)
stability_edge_replicates.append(edge_reps_1se)

stability_all = pd.concat(stability_tables, ignore_index=True)
stability_reps_all = pd.concat(stability_replicates, ignore_index=True)
stability_edge_reps_all = pd.concat(
    stability_edge_replicates,
    ignore_index=True,
)

stability_all.to_csv(
    TABLE_DIR / "Table_P09_Edge_Stability_All_Specifications.csv",
    index=False
)

stability_reps_all.to_csv(
    TABLE_DIR / "Table_P10_Stability_Replicate_Summary.csv",
    index=False
)

stability_edge_reps_all.to_csv(
    TABLE_DIR / "Table_P10B_Stability_Edge_Coefficient_Replicates.csv",
    index=False,
)

stability_coefficient_uncertainty = (
    stability_edge_reps_all
    .groupby(
        [
            "Specification", "HalfLife", "Lambda",
            "FromSector", "ToSector", "SelfExcitation",
        ],
        as_index=False,
    )
    .agg(
        Replicates=("Replicate", "nunique"),
        SelectionFrequency=("Selected", "mean"),
        CoefficientMean=("Coefficient", "mean"),
        CoefficientSD=("Coefficient", "std"),
        CoefficientP025=("Coefficient", lambda x: float(np.quantile(x, 0.025))),
        CoefficientMedian=("Coefficient", "median"),
        CoefficientP975=("Coefficient", lambda x: float(np.quantile(x, 0.975))),
    )
)
for thr in STABILITY_THRESHOLDS:
    stability_coefficient_uncertainty[
        f"StableAt_{int(thr * 100):02d}pct"
    ] = (
        stability_coefficient_uncertainty["SelectionFrequency"] >= thr
    ).astype(int)

stability_coefficient_uncertainty.to_csv(
    TABLE_DIR / "Table_P10C_Stability_Coefficient_Uncertainty.csv",
    index=False,
)

primary_stability = stability_all[
    (stability_all["Specification"] == "LambdaMin_Primary")
    & (
        stability_all["HalfLife"].astype(float)
        == float(primary_half_life)
    )
].copy()

robustness_stability_1se = stability_all[
    (stability_all["Specification"] == "Lambda1SE_Robustness")
    & (
        stability_all["HalfLife"].astype(float)
        == float(primary_half_life)
    )
].copy()

stable_cross_support = np.zeros((S, S), dtype=bool)

for row in primary_stability.itertuples(index=False):
    sender = sector_to_idx[row.FromSector]
    receiver = sector_to_idx[row.ToSector]

    if (
        sender != receiver
        and row.SelectionFrequency >= STABILITY_THRESHOLD
    ):
        stable_cross_support[receiver, sender] = True

n_stable_cross_edges = int(stable_cross_support.sum())

robust_1se_cross_edges = int(
    (
        (robustness_stability_1se["SelfExcitation"] == 0)
        & (
            robustness_stability_1se["SelectionFrequency"]
            >= STABILITY_THRESHOLD
        )
    ).sum()
)

log(
    f"PRIMARY lambda_min stable cross-sector edges: "
    f"{n_stable_cross_edges}."
)
log(
    f"ROBUSTNESS lambda_1SE stable cross-sector edges: "
    f"{robust_1se_cross_edges}."
)

# -----------------------------------------------------------------------------
# 14A. COMMON-SHOCK-CONTROLLED STABILITY SELECTION
# -----------------------------------------------------------------------------
def run_common_shock_stability_selection(
    half_life: float,
    l1_lambda: float,
    repetitions: int,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:

    eligible_indices = np.where(
        train_date_mask
        & M.any(axis=1)
        & market_observed
    )[0]

    selection_count = np.zeros((S, S), dtype=int)
    coefficient_sum = np.zeros((S, S), dtype=float)
    replicate_rows = []
    edge_rep_rows = []
    market_beta_rows = []

    for repetition in range(repetitions):
        rng = np.random.default_rng(
            SEED
            + 40009
            + int(half_life * 1000)
            + int(l1_lambda * 100000)
            + 7919 * (repetition + 1)
        )
        subsample_mask = make_block_subsample_mask(
            eligible_indices=eligible_indices,
            fraction=STABILITY_SUBSAMPLE_FRACTION,
            block_length=STABILITY_BLOCK_LENGTH,
            rng=rng,
        )

        fit = fit_network_common(
            fit_date_mask=subsample_mask,
            half_life=half_life,
            structure="sparse",
            l1_lambda=l1_lambda,
            n_starts=1,
        )

        selected = fit["alpha"] > EDGE_NUMERIC_TOL
        selection_count += selected.astype(int)
        coefficient_sum += fit["alpha"]

        replicate_rows.append({
            "Specification": "CommonShockControlled",
            "HalfLife": half_life,
            "Lambda": l1_lambda,
            "Replicate": repetition + 1,
            "SubsampleDates": int(subsample_mask.sum()),
            "NonzeroAllEdges": int(selected.sum()),
            "NonzeroCrossEdges": int(
                (selected & ~np.eye(S, dtype=bool)).sum()
            ),
        })

        for receiver in range(S):
            market_beta_rows.append({
                "Replicate": repetition + 1,
                "ToSector": sectors[receiver],
                "MarketBeta": float(fit["market_beta"][receiver]),
            })
            for sender in range(S):
                edge_rep_rows.append({
                    "Replicate": repetition + 1,
                    "FromSector": sectors[sender],
                    "ToSector": sectors[receiver],
                    "SelfExcitation": int(sender == receiver),
                    "Coefficient": float(fit["alpha"][receiver, sender]),
                    "Selected": int(selected[receiver, sender]),
                })

        if (repetition + 1) % 20 == 0:
            log(
                f"  Common-shock stability: "
                f"{repetition + 1}/{repetitions} complete."
            )

    frequency = selection_count / float(repetitions)
    mean_coefficient = coefficient_sum / float(repetitions)

    edge_rows = []
    for receiver in range(S):
        for sender in range(S):
            row = {
                "Specification": "CommonShockControlled",
                "HalfLife": half_life,
                "Lambda": l1_lambda,
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "SelfExcitation": int(sender == receiver),
                "SelectionFrequency": frequency[receiver, sender],
                "MeanCoefficientAcrossReps": mean_coefficient[receiver, sender],
            }
            for thr in STABILITY_THRESHOLDS:
                row[f"StableAt_{int(thr * 100):02d}pct"] = int(
                    frequency[receiver, sender] >= thr
                )
            edge_rows.append(row)

    return (
        pd.DataFrame(edge_rows),
        pd.DataFrame(replicate_rows),
        pd.DataFrame(edge_rep_rows),
        pd.DataFrame(market_beta_rows),
    )


log(
    "Running reviewer common-shock-controlled Hawkes stability selection "
    f"with the frozen primary half-life={primary_half_life:g} and "
    f"lambda={primary_lambda:g}."
)

(
    common_stability,
    common_stability_reps,
    common_edge_reps,
    common_market_beta_reps,
) = run_common_shock_stability_selection(
    half_life=primary_half_life,
    l1_lambda=primary_lambda,
    repetitions=COMMON_SHOCK_STABILITY_REPS,
)

common_stability.to_csv(
    TABLE_DIR / "Table_P10D_CommonShock_Edge_Stability.csv",
    index=False,
)
common_stability_reps.to_csv(
    TABLE_DIR / "Table_P10E_CommonShock_Replicate_Summary.csv",
    index=False,
)
common_edge_reps.to_csv(
    TABLE_DIR / "Table_P10F_CommonShock_Edge_Coefficient_Replicates.csv",
    index=False,
)

common_uncertainty = (
    common_edge_reps
    .groupby(
        ["FromSector", "ToSector", "SelfExcitation"],
        as_index=False,
    )
    .agg(
        Replicates=("Replicate", "nunique"),
        SelectionFrequency=("Selected", "mean"),
        CoefficientMean=("Coefficient", "mean"),
        CoefficientSD=("Coefficient", "std"),
        CoefficientP025=("Coefficient", lambda x: float(np.quantile(x, 0.025))),
        CoefficientMedian=("Coefficient", "median"),
        CoefficientP975=("Coefficient", lambda x: float(np.quantile(x, 0.975))),
    )
)
common_uncertainty.to_csv(
    TABLE_DIR / "Table_P10G_CommonShock_Coefficient_Uncertainty.csv",
    index=False,
)

common_market_beta_uncertainty = (
    common_market_beta_reps
    .groupby("ToSector", as_index=False)
    .agg(
        Replicates=("Replicate", "nunique"),
        MarketBetaMean=("MarketBeta", "mean"),
        MarketBetaSD=("MarketBeta", "std"),
        MarketBetaP025=("MarketBeta", lambda x: float(np.quantile(x, 0.025))),
        MarketBetaMedian=("MarketBeta", "median"),
        MarketBetaP975=("MarketBeta", lambda x: float(np.quantile(x, 0.975))),
    )
)
common_market_beta_uncertainty.to_csv(
    TABLE_DIR / "Table_P10H_CommonShock_MarketBeta_Uncertainty.csv",
    index=False,
)

common_cross_support = np.zeros((S, S), dtype=bool)
for row in common_stability.itertuples(index=False):
    sender = sector_to_idx[row.FromSector]
    receiver = sector_to_idx[row.ToSector]
    if (
        sender != receiver
        and row.SelectionFrequency >= STABILITY_THRESHOLD
    ):
        common_cross_support[receiver, sender] = True

common_postselection_fit = fit_network_common(
    train_date_mask,
    primary_half_life,
    structure="support",
    support_mask=common_cross_support,
    n_starts=FINAL_MULTISTARTS,
)

common_ll, common_nobs = network_loglik_common(
    common_postselection_fit,
    train_date_mask,
)

# Directly compare every primary cross-sector edge with the common-shock model.
primary_cross = primary_stability[
    primary_stability["SelfExcitation"] == 0
][
    ["FromSector", "ToSector", "SelectionFrequency"]
].rename(columns={
    "SelectionFrequency": "PrimarySelectionFrequency"
})

common_cross = common_stability[
    common_stability["SelfExcitation"] == 0
][
    ["FromSector", "ToSector", "SelectionFrequency"]
].rename(columns={
    "SelectionFrequency": "CommonShockSelectionFrequency"
})

common_edge_comparison = primary_cross.merge(
    common_cross,
    on=["FromSector", "ToSector"],
    how="outer",
).fillna(0.0)

for thr in STABILITY_THRESHOLDS:
    tag = int(thr * 100)
    common_edge_comparison[f"PrimaryStable_{tag}pct"] = (
        common_edge_comparison["PrimarySelectionFrequency"] >= thr
    ).astype(int)
    common_edge_comparison[f"CommonShockStable_{tag}pct"] = (
        common_edge_comparison["CommonShockSelectionFrequency"] >= thr
    ).astype(int)
    common_edge_comparison[f"SurvivesCommonShock_{tag}pct"] = (
        (common_edge_comparison["PrimarySelectionFrequency"] >= thr)
        & (common_edge_comparison["CommonShockSelectionFrequency"] >= thr)
    ).astype(int)

common_edge_comparison.to_csv(
    TABLE_DIR / "Table_P10I_Primary_vs_CommonShock_Edge_Comparison.csv",
    index=False,
)

common_model_summary = pd.DataFrame([{
    "HalfLife": primary_half_life,
    "Lambda": primary_lambda,
    "TrainLogLik": common_ll,
    "NObs": common_nobs,
    "StableCrossEdgesAt60pct": int(common_cross_support.sum()),
    "MarketEventsInTrainObservedWindow": int(
        market_event[train_date_mask & market_observed].sum()
    ),
}])
common_model_summary.to_csv(
    TABLE_DIR / "Table_P10J_CommonShock_Model_Summary.csv",
    index=False,
)

# Same-day treatment audit requested by the reviewer.
same_day_audit = pd.DataFrame({
    "Item": [
        "Sector excitation state entering lambda_i,t",
        "Market common-shock state entering lambda_i,t",
        "Same-day sector-to-sector excitation",
        "Same-day market-to-sector excitation",
        "State used for forecast t+1 from origin t",
    ],
    "Treatment": [
        "Uses sector events through t-1 only",
        "Uses market events through t-1 only",
        "Not permitted",
        "Not permitted",
        "Post-origin state includes events observed at t, then forecasts t+1 onward",
    ],
})
same_day_audit.to_csv(
    TABLE_DIR / "Table_P10K_SameDay_Treatment_Audit.csv",
    index=False,
)

# =============================================================================
# 15. FIT NESTED MODELS ON TRAIN
# =============================================================================

train_fits = {
    "BaselineIntensity": fit_network(
        train_date_mask,
        primary_half_life,
        structure="baseline",
        n_starts=1,
    ),
    "SelfHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="self",
        n_starts=FINAL_MULTISTARTS,
    ),
    "FullHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="full",
        n_starts=FINAL_MULTISTARTS,
    ),
    "SparsePenalizedHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="sparse",
        l1_lambda=primary_lambda,
        n_starts=FINAL_MULTISTARTS,
    ),
    "Sparse1SEHawkes_Robustness": fit_network(
        train_date_mask,
        primary_half_life,
        structure="sparse",
        l1_lambda=primary_lambda_1se,
        n_starts=FINAL_MULTISTARTS,
    ),
    "StablePostSelectionHawkes": fit_network(
        train_date_mask,
        primary_half_life,
        structure="support",
        support_mask=stable_cross_support,
        n_starts=FINAL_MULTISTARTS,
    ),
}

# =============================================================================
# 16. FREEZE TRAIN-ESTIMATED HAWKES COEFFICIENTS FOR CALIBRATION AND TEST
# =============================================================================
#
# Reviewer revision: the 2022-2024 period is reserved for probability
# calibration in the downstream forecasting phase.  It is therefore not used
# to re-estimate Hawkes coefficients.  Structure AND coefficients are frozen
# after the full Train period.
final_fits = train_fits
final_stable_fit = train_fits["StablePostSelectionHawkes"]

# =============================================================================
# 17. BRANCHING / STABILITY DIAGNOSTICS
# =============================================================================

def branching_spectral_radius(
    A: np.ndarray,
    decay: float,
) -> float:

    branching = (
        A
        / max(1.0 - decay, EPS)
    )

    eigenvalues = np.linalg.eigvals(branching)

    return float(
        np.max(np.abs(eigenvalues))
    )

network_diagnostic_rows = []

for fit_stage, fit_collection in [
    ("TrainFit", train_fits),
    ("TrainFrozenForCalibrationTest", final_fits),
]:
    for model_name, fit in fit_collection.items():

        cross_mask = ~np.eye(S, dtype=bool)

        spectral_radius = branching_spectral_radius(
            fit["alpha"],
            fit["decay"],
        )

        network_diagnostic_rows.append({
            "FitStage": fit_stage,
            "Model": model_name,
            "HalfLife": fit["half_life"],
            "Lambda": fit["l1_lambda"],
            "NonzeroAllEdges": int(
                (
                    fit["alpha"] > EDGE_NUMERIC_TOL
                ).sum()
            ),
            "NonzeroCrossEdges": int(
                (
                    (fit["alpha"] > EDGE_NUMERIC_TOL)
                    & cross_mask
                ).sum()
            ),
            "MaxAlpha": float(
                fit["alpha"].max()
            ),
            "BranchingSpectralRadiusApprox": spectral_radius,
            "BelowStabilityGuard": int(
                spectral_radius
                < MAX_BRANCHING_SPECTRAL_RADIUS
            ),
        })

network_diagnostics = pd.DataFrame(
    network_diagnostic_rows
)

network_diagnostics.to_csv(
    TABLE_DIR / "Table_P11_Network_Stability_Diagnostics.csv",
    index=False
)

final_stable_radius = float(
    network_diagnostics.loc[
        (network_diagnostics["FitStage"] == "TrainFrozenForCalibrationTest")
        & (
            network_diagnostics["Model"]
            == "StablePostSelectionHawkes"
        ),
        "BranchingSpectralRadiusApprox",
    ].iloc[0]
)

if final_stable_radius >= MAX_BRANCHING_SPECTRAL_RADIUS:
    raise RuntimeError(
        "Final stability-selected Hawkes network exceeds "
        "the pre-specified branching spectral-radius guard."
    )

# =============================================================================
# 18. STABLE EDGE TABLES AND HALF-LIFE ROBUSTNESS
# =============================================================================

train_stable_fit = train_fits[
    "StablePostSelectionHawkes"
]
final_stable_fit = final_fits[
    "StablePostSelectionHawkes"
]

stable_edge_rows = []

for receiver in range(S):
    for sender in range(S):

        match = primary_stability[
            (primary_stability["FromSector"] == sectors[sender])
            & (primary_stability["ToSector"] == sectors[receiver])
        ]

        stability_frequency = (
            float(match["SelectionFrequency"].iloc[0])
            if len(match)
            else np.nan
        )

        stable_edge_rows.append({
            "FromSector": sectors[sender],
            "ToSector": sectors[receiver],
            "SelfExcitation": int(sender == receiver),
            "SelectionFrequency": stability_frequency,
            "SelectedCrossEdge": int(
                stable_cross_support[receiver, sender]
            ),
            "TrainPostSelectionAlpha": float(
                train_stable_fit["alpha"][receiver, sender]
            ),
            "TrainFrozenAlpha": float(
                final_stable_fit["alpha"][receiver, sender]
            ),
            # Backward-compatible alias; no Train+Validation refit is performed.
            "TrainValidationRefitAlpha": float(
                final_stable_fit["alpha"][receiver, sender]
            ),
            "HalfLife": primary_half_life,
        })

stable_edges = pd.DataFrame(
    stable_edge_rows
)

stable_edges.to_csv(
    TABLE_DIR / "Table_P12_Final_Stable_Edge_Parameters.csv",
    index=False
)

half_life_summary_rows = []
support_by_half_life = {}

for half_life in sensitivity_half_lives:

    g = stability_all[
        stability_all["HalfLife"].astype(float)
        == float(half_life)
    ].copy()

    cross = g[
        g["SelfExcitation"] == 0
    ]

    stable_cross = cross[
        cross["SelectionFrequency"]
        >= STABILITY_THRESHOLD
    ]

    support_by_half_life[
        float(half_life)
    ] = set(
        zip(
            stable_cross["FromSector"],
            stable_cross["ToSector"],
        )
    )

    half_life_summary_rows.append({
        "HalfLife": half_life,
        "SelectedLambda": lambda_min_by_half_life[
            float(half_life)
        ],
        "StableCrossEdges": len(stable_cross),
        "MeanCrossEdgeStability": float(
            cross["SelectionFrequency"].mean()
        ),
        "MaxCrossEdgeStability": float(
            cross["SelectionFrequency"].max()
        ),
    })

half_life_summary = pd.DataFrame(
    half_life_summary_rows
)

half_life_summary.to_csv(
    TABLE_DIR / "Table_P13_HalfLife_Stability_Summary.csv",
    index=False
)

overlap_rows = []

for i, h1 in enumerate(sensitivity_half_lives):
    for h2 in sensitivity_half_lives[i + 1:]:

        a = support_by_half_life[float(h1)]
        b = support_by_half_life[float(h2)]

        union = a | b

        overlap_rows.append({
            "HalfLife1": h1,
            "HalfLife2": h2,
            "Edges1": len(a),
            "Edges2": len(b),
            "CommonEdges": len(a & b),
            "Jaccard": (
                len(a & b) / len(union)
                if union
                else 1.0
            ),
        })

half_life_overlap = pd.DataFrame(
    overlap_rows
)

half_life_overlap.to_csv(
    TABLE_DIR / "Table_P14_HalfLife_Edge_Overlap.csv",
    index=False
)

# =============================================================================
# 19. VALIDATION AND TEST FORECASTS
# =============================================================================

# Calibration and Test both use the same TRAIN-fitted Hawkes coefficients.
train_probabilities = {
    model_name: network_probabilities(fit)
    for model_name, fit in train_fits.items()
}
final_probabilities = train_probabilities

forecast_metric_rows = []

def add_metrics_for_split(
    split_name: str,
    split_mask: np.ndarray,
    model_probabilities: dict[str, dict[int, np.ndarray]],
):

    # Strict historical baselines.
    baseline_collection = {
        "FixedTrainHistorical": fixed_historical,
        "ExpandingHistorical": expanding_historical,
    }

    combined = {
        **baseline_collection,
        **model_probabilities,
    }

    for model_name, probabilities in combined.items():

        for h in HORIZONS:

            pooled = probability_metrics(
                target_within[h][split_mask].reshape(-1),
                probabilities[h][split_mask].reshape(-1),
            )

            forecast_metric_rows.append({
                "Split": split_name,
                "Sector": "POOLED",
                "Horizon": h,
                "Model": model_name,
                **pooled,
            })

            for s, sector in enumerate(sectors):

                sector_metrics = probability_metrics(
                    target_within[h][split_mask, s],
                    probabilities[h][split_mask, s],
                )

                forecast_metric_rows.append({
                    "Split": split_name,
                    "Sector": sector,
                    "Horizon": h,
                    "Model": model_name,
                    **sector_metrics,
                })

add_metrics_for_split(
    "Validation",
    validation_date_mask,
    train_probabilities,
)

add_metrics_for_split(
    "Test",
    test_date_mask,
    final_probabilities,
)

forecast_metrics = pd.DataFrame(
    forecast_metric_rows
)

forecast_metrics.to_csv(
    TABLE_DIR / "Table_P15_Nested_Hawkes_Forecast_Metrics.csv",
    index=False
)

# Skill is referenced to the stricter expanding historical probability.
pooled = forecast_metrics[
    forecast_metrics["Sector"] == "POOLED"
].copy()

reference = (
    pooled[
        pooled["Model"] == "ExpandingHistorical"
    ][
        ["Split", "Horizon", "Brier", "LogScore"]
    ]
    .rename(columns={
        "Brier": "ReferenceBrier",
        "LogScore": "ReferenceLogScore",
    })
)

forecast_skill = pooled.merge(
    reference,
    on=["Split", "Horizon"],
    how="left",
)

forecast_skill["BrierSkill_vs_ExpandingHistorical"] = (
    1.0
    - forecast_skill["Brier"]
    / forecast_skill["ReferenceBrier"]
)

forecast_skill["LogScoreImprovement_vs_ExpandingHistorical"] = (
    forecast_skill["ReferenceLogScore"]
    - forecast_skill["LogScore"]
)

forecast_skill.to_csv(
    TABLE_DIR / "Table_P16_Pooled_Forecast_Skill.csv",
    index=False
)

# =============================================================================
# 20. GRAPH DECISION — TRAIN-ONLY STRUCTURE, NO CALIBRATION/TEST GATING
# =============================================================================
#
# The stability-selected Hawkes graph is an econometric prior. Its existence is
# determined only by Train-side BIC/CV/stability procedures. Calibration and
# Test forecast scores are not used to decide whether the graph exists.

positive_validation_skill_horizons = np.nan  # retained only for legacy table compatibility

if n_stable_cross_edges >= 1:
    hawkes_graph_decision = (
        "HAWKES_GRAPH_SUPPORTED_BY_TRAIN_ONLY_STABILITY_AS_SOFT_PRIOR"
    )
else:
    hawkes_graph_decision = (
        "NO_STABLE_CROSS_SECTOR_HAWKES_EDGES_"
        "ACTIVATE_TRAIN_ONLY_LOWER_TAIL_QUANTILE_FALLBACK"
    )

# =============================================================================
# 20A. LOWER-TAIL QUANTILE GRAPH FALLBACK
# =============================================================================
#
# Activated only if Hawkes lambda_min + 60% stability selection yields no
# cross-sector edges.
#
# For each receiving sector i:
#
#   Q_tau(z_i,t+1 | z_1,t,...,z_S,t)
#       = a_i + sum_j beta_i,j z_j,t
#
# with tau = 0.05.
#
# A positive beta_i,j means a negative shock in source j lowers the conditional
# lower quantile of receiver i and is therefore consistent with downside
# predictive transmission. Stable positive CROSS-sector coefficients define
# the fallback graph. The dynamic edge weight is:
#
#   w_i,j,t = beta_i,j * max(-z_j,t, 0)
#
# so edges activate when the source sector experiences downside standardized
# shocks.

tail_graph_used = False
tail_graph_available = False
tail_graph_alpha = np.nan
tail_stable_cross_edges = 0
tail_support = np.zeros((S, S), dtype=bool)
tail_beta_train = np.zeros((S, S), dtype=float)
tail_beta_final = np.zeros((S, S), dtype=float)
tail_intercept_train = np.zeros(S, dtype=float)
tail_intercept_final = np.zeros(S, dtype=float)
tail_cv_summary = pd.DataFrame()
tail_stability_table = pd.DataFrame()

# Standardized innovations aligned to date x sector.
Z = np.full((T, S), np.nan, dtype=float)

for row in primary[
    ["Date", "Sector", "StdInnovation"]
].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]
    if pd.notna(row.StdInnovation):
        Z[t, s] = float(row.StdInnovation)


def tail_row_mask_from_date_mask(date_mask: np.ndarray) -> np.ndarray:
    """
    Row t uses predictors Z_t and response Z_(t+1).
    Both predictor date t and response date t+1 must lie inside the supplied
    fitting/validation date region.
    """
    out = np.zeros(T - 1, dtype=bool)
    out[:] = date_mask[:-1] & date_mask[1:]
    return out


def tail_complete_case_rows(date_mask: np.ndarray) -> np.ndarray:
    base = tail_row_mask_from_date_mask(date_mask)
    complete_x = np.all(np.isfinite(Z[:-1]), axis=1)
    complete_y = np.all(np.isfinite(Z[1:]), axis=1)
    return base & complete_x & complete_y


def fit_quantile_network(
    row_mask: np.ndarray,
    alpha_penalty: float,
    support_mask: np.ndarray | None = None,
) -> dict:

    X = Z[:-1][row_mask]
    Y = Z[1:][row_mask]

    if len(X) < 100:
        raise RuntimeError(
            "Too few complete observations for lower-tail quantile graph."
        )

    intercept = np.zeros(S, dtype=float)
    beta = np.zeros((S, S), dtype=float)

    for receiver in range(S):

        if support_mask is None:
            allowed = np.ones(S, dtype=bool)
        else:
            allowed = support_mask[receiver].copy()
            # Own lag is always included as a control, but is not treated as
            # a contagion edge.
            allowed[receiver] = True

        allowed_idx = np.where(allowed)[0]

        model = QuantileRegressor(
            quantile=TAIL_QUANTILE,
            alpha=float(alpha_penalty),
            fit_intercept=True,
            solver="highs",
        )

        model.fit(
            X[:, allowed_idx],
            Y[:, receiver],
        )

        intercept[receiver] = float(model.intercept_)
        beta[receiver, allowed_idx] = np.asarray(
            model.coef_,
            dtype=float,
        )

    return {
        "intercept": intercept,
        "beta": beta,
        "n": int(row_mask.sum()),
        "alpha_penalty": float(alpha_penalty),
    }


def quantile_network_pinball(
    fit: dict,
    row_mask: np.ndarray,
) -> float:

    X = Z[:-1][row_mask]
    Y = Z[1:][row_mask]

    losses = []

    for receiver in range(S):
        pred = (
            fit["intercept"][receiver]
            + X @ fit["beta"][receiver]
        )

        losses.append(
            mean_pinball_loss(
                Y[:, receiver],
                pred,
                alpha=TAIL_QUANTILE,
            )
        )

    return float(np.mean(losses))


def make_tail_block_subsample_mask(
    eligible_rows: np.ndarray,
    fraction: float,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    eligible_rows = np.asarray(eligible_rows, dtype=int)
    n = len(eligible_rows)
    target_n = max(1, int(np.ceil(fraction * n)))
    selected_positions = np.zeros(n, dtype=bool)

    while selected_positions.sum() < target_n:
        start = int(
            rng.integers(
                0,
                max(1, n - block_length + 1),
            )
        )
        end = min(n, start + block_length)
        selected_positions[start:end] = True

    mask = np.zeros(T - 1, dtype=bool)
    mask[eligible_rows[selected_positions]] = True
    return mask


def build_dynamic_tail_graph(
    beta_matrix: np.ndarray,
) -> np.ndarray:

    graph = np.zeros((T, S, S), dtype=np.float32)

    for t in range(T):
        downside_state = np.where(
            np.isfinite(Z[t]),
            np.maximum(-Z[t], 0.0),
            0.0,
        )

        # Receiver x Source.
        graph[t] = (
            beta_matrix
            * downside_state[np.newaxis, :]
        ).astype(np.float32)

    return graph


if n_stable_cross_edges == 0:

    log(
        "No stable Hawkes cross-sector edges survived. "
        "Estimating lower-tail quantile graph fallback."
    )

    # ----- Tail graph temporal CV within TRAIN only -----
    tail_cv_rows = []

    for alpha_penalty in TAIL_QUANTILE_ALPHA_GRID:

        fold_losses = []

        for fold in folds:

            fit_rows = tail_complete_case_rows(
                fold["FitMask"]
            )
            val_rows = tail_complete_case_rows(
                fold["ValidationMask"]
            )

            if fit_rows.sum() < 100 or val_rows.sum() < 30:
                continue

            fit_tail = fit_quantile_network(
                row_mask=fit_rows,
                alpha_penalty=alpha_penalty,
            )

            loss = quantile_network_pinball(
                fit_tail,
                val_rows,
            )

            cross_mask = ~np.eye(S, dtype=bool)
            positive_cross_edges = int(
                (
                    (fit_tail["beta"] > TAIL_BETA_TOL)
                    & cross_mask
                ).sum()
            )

            fold_losses.append(loss)

            tail_cv_rows.append({
                "Alpha": alpha_penalty,
                "Fold": fold["Fold"],
                "PinballLoss": loss,
                "PositiveCrossEdges": positive_cross_edges,
            })

    tail_cv_all = pd.DataFrame(tail_cv_rows)

    if len(tail_cv_all) == 0:
        raise RuntimeError(
            "Lower-tail quantile fallback could not construct temporal CV folds."
        )

    tail_cv_all.to_csv(
        TABLE_DIR / "Table_P17A_TailQuantile_CV_All_Folds.csv",
        index=False
    )

    tail_cv_summary = (
        tail_cv_all
        .groupby("Alpha", as_index=False)
        .agg(
            MeanPinballLoss=("PinballLoss", "mean"),
            SDPinballLoss=("PinballLoss", "std"),
            MeanPositiveCrossEdges=("PositiveCrossEdges", "mean"),
            Folds=("Fold", "nunique"),
        )
        .sort_values("MeanPinballLoss")
    )

    tail_graph_alpha = float(
        tail_cv_summary.iloc[0]["Alpha"]
    )

    tail_cv_summary.to_csv(
        TABLE_DIR / "Table_P17B_TailQuantile_CV_Summary.csv",
        index=False
    )

    log(
        f"Lower-tail quantile graph selected alpha_min="
        f"{tail_graph_alpha:g} at tau={TAIL_QUANTILE:g}."
    )

    # ----- Tail graph block stability selection on TRAIN only -----
    tail_train_rows = np.where(
        tail_complete_case_rows(train_date_mask)
    )[0]

    selection_count = np.zeros((S, S), dtype=int)
    coefficient_sum = np.zeros((S, S), dtype=float)
    tail_rep_rows = []

    for repetition in range(TAIL_GRAPH_STABILITY_REPS):

        rng = np.random.default_rng(
            SEED + 17713 * (repetition + 1)
        )

        subsample_rows = make_tail_block_subsample_mask(
            eligible_rows=tail_train_rows,
            fraction=STABILITY_SUBSAMPLE_FRACTION,
            block_length=STABILITY_BLOCK_LENGTH,
            rng=rng,
        )

        fit_tail = fit_quantile_network(
            row_mask=subsample_rows,
            alpha_penalty=tail_graph_alpha,
        )

        # Only positive coefficients represent downside transmission under
        # this sign convention. Diagonal coefficients are controls.
        selected = fit_tail["beta"] > TAIL_BETA_TOL

        selection_count += selected.astype(int)
        coefficient_sum += fit_tail["beta"]

        tail_rep_rows.append({
            "Replicate": repetition + 1,
            "Rows": int(subsample_rows.sum()),
            "PositiveAllEdges": int(selected.sum()),
            "PositiveCrossEdges": int(
                (
                    selected
                    & ~np.eye(S, dtype=bool)
                ).sum()
            ),
        })

        if (repetition + 1) % 20 == 0:
            log(
                f"  Tail-graph stability: "
                f"{repetition + 1}/{TAIL_GRAPH_STABILITY_REPS} complete."
            )

    tail_frequency = selection_count / float(
        TAIL_GRAPH_STABILITY_REPS
    )
    tail_mean_beta = coefficient_sum / float(
        TAIL_GRAPH_STABILITY_REPS
    )

    tail_edge_rows = []

    for receiver in range(S):
        for sender in range(S):

            stable = (
                sender != receiver
                and tail_frequency[receiver, sender]
                >= TAIL_GRAPH_STABILITY_THRESHOLD
            )

            if stable:
                tail_support[receiver, sender] = True

            tail_edge_rows.append({
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "SelfLagControl": int(sender == receiver),
                "SelectionFrequency": tail_frequency[receiver, sender],
                "MeanBetaAcrossReps": tail_mean_beta[receiver, sender],
                "StableCrossEdge": int(stable),
            })

    tail_stability_table = pd.DataFrame(tail_edge_rows)
    tail_stability_table.to_csv(
        TABLE_DIR / "Table_P17C_TailQuantile_Edge_Stability.csv",
        index=False
    )

    pd.DataFrame(tail_rep_rows).to_csv(
        TABLE_DIR / "Table_P17D_TailQuantile_Stability_Repetitions.csv",
        index=False
    )

    tail_stable_cross_edges = int(tail_support.sum())

    if tail_stable_cross_edges >= 1:

        tail_graph_available = True
        tail_graph_used = True

        # Post-selection refit on TRAIN for Train/Validation graph.
        train_rows_tail = tail_complete_case_rows(
            train_date_mask
        )
        fit_tail_train = fit_quantile_network(
            row_mask=train_rows_tail,
            alpha_penalty=0.0,
            support_mask=tail_support,
        )

        # Reviewer revision: freeze Train-fitted coefficients for Calibration/Test.
        fit_tail_final = fit_tail_train

        tail_beta_train = fit_tail_train["beta"]
        tail_beta_final = fit_tail_train["beta"].copy()
        tail_intercept_train = fit_tail_train["intercept"]
        tail_intercept_final = fit_tail_train["intercept"].copy()

        # Remove own-lag diagonal from graph; own lag remains only as a control
        # in the quantile regression.
        np.fill_diagonal(tail_beta_train, 0.0)
        np.fill_diagonal(tail_beta_final, 0.0)

        # Keep only stable cross-sector support and positive direction.
        tail_beta_train = np.where(
            tail_support,
            np.maximum(tail_beta_train, 0.0),
            0.0,
        )
        tail_beta_final = np.where(
            tail_support,
            np.maximum(tail_beta_final, 0.0),
            0.0,
        )

        log(
            f"Lower-tail quantile graph retained "
            f"{tail_stable_cross_edges} stable cross-sector edges."
        )

    else:
        log(
            "No stable lower-tail quantile cross-sector edges survived "
            "the 60% threshold."
        )


# Final graph-source decision.
if n_stable_cross_edges >= 1:
    final_graph_source = "HAWKES_LAMBDA_MIN_STABILITY"
    graph_decision = hawkes_graph_decision
elif tail_graph_available:
    final_graph_source = "LOWER_TAIL_QUANTILE_STABILITY"
    graph_decision = (
        "HAWKES_UNSTABLE_QUANTILE_TAIL_GRAPH_USED_AS_SOFT_PRIOR"
    )
else:
    final_graph_source = "NO_STABLE_ECONOMETRIC_GRAPH"
    graph_decision = (
        "NO_STABLE_ECONOMETRIC_GRAPH_"
        "DEEP_MODEL_MUST_INCLUDE_NO_GRAPH_OR_LEARNED_GRAPH_ONLY"
    )

graph_decision_table = pd.DataFrame({
    "Criterion": [
        "Hawkes primary penalty rule",
        "Hawkes lambda_min",
        "Hawkes lambda_1SE robustness",
        "Stable Hawkes cross-sector edges",
        "Hawkes stability threshold",
        "Calibration/Test forecast skill used for graph selection",
        "Tail fallback activated",
        "Tail quantile",
        "Tail selected alpha",
        "Stable tail cross-sector edges",
        "Final graph source",
        "Test used in graph-structure decision",
        "Decision",
    ],
    "Value": [
        "lambda_min + separate 60% stability selection",
        primary_lambda,
        primary_lambda_1se,
        n_stable_cross_edges,
        STABILITY_THRESHOLD,
        "NO",
        int(n_stable_cross_edges == 0),
        TAIL_QUANTILE,
        tail_graph_alpha,
        tail_stable_cross_edges,
        final_graph_source,
        "NO",
        graph_decision,
    ],
})

graph_decision_table.to_csv(
    TABLE_DIR / "Table_P17_Graph_Decision.csv",
    index=False
)

log(f"Final graph source: {final_graph_source}")
log(f"Graph decision: {graph_decision}")

# =============================================================================
# 21. SPLIT-AWARE DYNAMIC GRAPH FOR LATER GRAPH DL
# =============================================================================

def dynamic_hawkes_graph_from_fit(
    fit: dict,
) -> np.ndarray:

    post_state = build_post_event_state(
        C,
        fit["decay"],
    )

    graph = np.zeros(
        (T, S, S),
        dtype=np.float32,
    )

    for t in range(T):
        graph[t] = (
            fit["alpha"]
            * post_state[t][np.newaxis, :]
        ).astype(np.float32)

    return graph


if final_graph_source == "HAWKES_LAMBDA_MIN_STABILITY":

    train_graph = dynamic_hawkes_graph_from_fit(
        train_stable_fit
    )
    final_graph = dynamic_hawkes_graph_from_fit(
        final_stable_fit
    )

elif final_graph_source == "LOWER_TAIL_QUANTILE_STABILITY":

    train_graph = build_dynamic_tail_graph(
        tail_beta_train
    )
    final_graph = build_dynamic_tail_graph(
        tail_beta_final
    )

else:

    train_graph = np.zeros(
        (T, S, S),
        dtype=np.float32,
    )
    final_graph = train_graph.copy()


# Reviewer revision: Train-estimated graph support AND coefficients are frozen
# for Train, Calibration and Test.  The Calibration period is not used to
# update the graph before Test.
split_aware_graph = train_graph.copy()
final_graph = train_graph.copy()

graph_rows = []

for t, date in enumerate(dates):
    for receiver in range(S):
        for sender in range(S):

            if final_graph_source == "HAWKES_LAMBDA_MIN_STABILITY":
                stable_edge_indicator = int(
                    stable_cross_support[receiver, sender]
                )
            elif final_graph_source == "LOWER_TAIL_QUANTILE_STABILITY":
                stable_edge_indicator = int(
                    tail_support[receiver, sender]
                )
            else:
                stable_edge_indicator = 0

            graph_rows.append({
                "Date": date,
                "Split": splits[t],
                "GraphSource": final_graph_source,
                "FromSector": sectors[sender],
                "ToSector": sectors[receiver],
                "Weight": float(
                    split_aware_graph[
                        t,
                        receiver,
                        sender,
                    ]
                ),
                "StableCrossEdge": stable_edge_indicator,
                "SelfEdge": int(receiver == sender),
            })

dynamic_graph_long = pd.DataFrame(graph_rows)

dynamic_graph_long.to_csv(
    DATA_DIR / "econometric_dynamic_graph.csv.gz",
    index=False,
    compression="gzip",
)

# Compatibility alias for later DL code.
dynamic_graph_long.to_csv(
    DATA_DIR / "stable_hawkes_dynamic_graph.csv.gz",
    index=False,
    compression="gzip",
)

np.savez_compressed(
    DATA_DIR / "econometric_dynamic_graph_arrays.npz",
    dates=dates.astype(str).to_numpy(),
    sectors=np.array(sectors, dtype=object),
    graph_source=np.array([final_graph_source], dtype=object),
    adjacency=split_aware_graph,
    train_fit_adjacency=train_graph,
    final_fit_adjacency=final_graph,
    hawkes_stable_cross_support=stable_cross_support.astype(np.uint8),
    tail_stable_cross_support=tail_support.astype(np.uint8),
    hawkes_train_alpha=train_stable_fit["alpha"].astype(np.float32),
    # Compatibility name retained; coefficients are Train-frozen, not refitted.
    hawkes_final_alpha=final_stable_fit["alpha"].astype(np.float32),
    common_shock_hawkes_alpha=common_postselection_fit["alpha"].astype(np.float32),
    common_shock_market_beta=common_postselection_fit["market_beta"].astype(np.float32),
    common_shock_stable_cross_support=common_cross_support.astype(np.uint8),
    tail_train_beta=tail_beta_train.astype(np.float32),
    tail_final_beta=tail_beta_final.astype(np.float32),
)

# Compatibility alias expected by later graph code.
np.savez_compressed(
    DATA_DIR / "stable_hawkes_dynamic_graph_arrays.npz",
    dates=dates.astype(str).to_numpy(),
    sectors=np.array(sectors, dtype=object),
    graph_source=np.array([final_graph_source], dtype=object),
    adjacency=split_aware_graph,
)

# =============================================================================
# 22. FINAL MODELLING MASTER FOR DEEP-LEARNING PHASE
# =============================================================================

model_master = primary.merge(
    targets_long,
    on=["Date", "Sector", "Split"],
    how="left",
    validate="one_to_one",
)

# Add strictly constructed baseline and Hawkes probabilities.
probability_rows = []

for t, date in enumerate(dates):
    for s, sector in enumerate(sectors):

        row = {
            "Date": date,
            "Sector": sector,
        }

        for h in HORIZONS:

            row[f"FixedHistoricalProb_{h}"] = (
                fixed_historical[h][t, s]
            )

            row[f"ExpandingHistoricalProb_{h}"] = (
                expanding_historical[h][t, s]
            )

            # Train-fitted Hawkes probabilities are frozen for all later phases.
            row[f"StableHawkesProb_{h}"] = (
                train_probabilities[
                    "StablePostSelectionHawkes"
                ][h][t, s]
            )

        probability_rows.append(row)

probability_features = pd.DataFrame(
    probability_rows
)

model_master = model_master.merge(
    probability_features,
    on=["Date", "Sector"],
    how="left",
    validate="one_to_one",
)

model_master["EconometricGraphSource"] = final_graph_source
if "RevisionPhase" not in model_master.columns:
    phase_by_date = pd.DataFrame({
        "Date": dates,
        "RevisionPhase": revision_phases,
    })
    model_master = model_master.merge(
        phase_by_date,
        on="Date",
        how="left",
        validate="many_to_one",
    )
model_master["ForecastRole"] = model_master["Split"].map({
    "Train": "Train",
    "Validation": "Calibration",
    "Test": "Test",
})

model_master.to_csv(
    DATA_DIR / "phase2_deep_learning_master.csv.gz",
    index=False,
    compression="gzip",
)

targets_long.to_csv(
    DATA_DIR / "time_to_crash_targets.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 23. SAVE MODEL BUNDLE
# =============================================================================

model_bundle = {
    "version": "PythonPhase1_1.2",
    "seed": SEED,
    "sectors": sectors,
    "horizons": HORIZONS,
    "primary_half_life": primary_half_life,
    "selected_lambda_min": primary_lambda,
    "selected_lambda_1se_robustness": primary_lambda_1se,
    "stability_threshold": STABILITY_THRESHOLD,
    "stable_cross_support": stable_cross_support,
    "graph_decision": graph_decision,
    "final_graph_source": final_graph_source,
    "tail_graph_selected_alpha": tail_graph_alpha,
    "tail_graph_stable_cross_edges": tail_stable_cross_edges,
    "tail_graph_support": tail_support,
    "tail_graph_train_beta": tail_beta_train,
    "tail_graph_final_beta": tail_beta_final,
    "train_fit": {
        "mu": train_stable_fit["mu"],
        "alpha": train_stable_fit["alpha"],
        "decay": train_stable_fit["decay"],
    },
    "train_validation_refit": {
        "mu": final_stable_fit["mu"],
        "alpha": final_stable_fit["alpha"],
        "decay": final_stable_fit["decay"],
    },
}

with open(
    MODEL_DIR / "sparse_stable_hawkes_bundle.pkl",
    "wb",
) as f:
    pickle.dump(model_bundle, f)

# =============================================================================
# 24. FIGURES
# =============================================================================

# Figure 1: Crash counts by sector and split.
crash_plot = (
    primary.dropna(subset=["Crash_Main"])
    .groupby(
        ["Sector", "Split"],
        as_index=False,
    )["Crash_Main"]
    .sum()
)

figure_data = crash_plot.pivot(
    index="Sector",
    columns="Split",
    values="Crash_Main",
).fillna(0)

fig, ax = plt.subplots(figsize=(11, 6))
figure_data.plot(
    kind="bar",
    ax=ax,
)
ax.set_title(
    "Observed 2.5% EVT Crash Events by Sector and Sample Split"
)
ax.set_xlabel("Sector")
ax.set_ylabel("Crash events")
ax.tick_params(
    axis="x",
    rotation=45,
)
ax.legend(title="Split")
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P01_Crash_Counts.png",
    dpi=300,
)
plt.close(fig)

# Figure 2: BIC half-life selection.
figure_data = half_life_selection.sort_values(
    "HalfLife"
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(
    figure_data["HalfLife"],
    figure_data["BIC"],
    marker="o",
)
ax.set_xlabel("Hawkes half-life (trading days)")
ax.set_ylabel("TRAIN BIC")
ax.set_title("Hawkes Memory Selection")
ax.set_xticks(HALF_LIFE_CANDIDATES)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P02_Hawkes_HalfLife_BIC.png",
    dpi=300,
)
plt.close(fig)

# Figure 3: L1 CV.
fig, ax = plt.subplots(figsize=(9, 6))
for half_life, g in cv_summary.groupby("HalfLife"):
    g = g.sort_values("Lambda")
    ax.plot(
        g["Lambda"],
        g["MeanValidationLogScore"],
        marker="o",
        label=f"{half_life:g}-day half-life",
    )
ax.set_xscale(
    "symlog",
    linthresh=0.005,
)
ax.set_xlabel("L1 penalty")
ax.set_ylabel(
    "Mean TRAIN-temporal-CV log score"
)
ax.set_title(
    "Sparse Hawkes Regularization Selection"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P03_L1_Temporal_CV.png",
    dpi=300,
)
plt.close(fig)

# Figure 4: Edge stability heatmap.
stability_matrix = np.zeros(
    (S, S),
    dtype=float,
)

for row in primary_stability.itertuples(index=False):
    receiver = sector_to_idx[row.ToSector]
    sender = sector_to_idx[row.FromSector]
    stability_matrix[
        receiver,
        sender,
    ] = row.SelectionFrequency

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(
    stability_matrix,
    vmin=0,
    vmax=1,
    aspect="auto",
)
ax.set_xticks(range(S))
ax.set_yticks(range(S))
ax.set_xticklabels(
    sectors,
    rotation=45,
    ha="right",
)
ax.set_yticklabels(sectors)
ax.set_xlabel("Source sector")
ax.set_ylabel("Receiving sector")
ax.set_title(
    f"Lambda-Min Block-Stability Selection Frequencies "
    f"({primary_half_life:g}-day half-life)"
)
fig.colorbar(
    image,
    ax=ax,
    label="Selection frequency",
)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P04_Edge_Stability_Heatmap.png",
    dpi=300,
)
plt.close(fig)

# Figure 4B: Tail-quantile edge stability if fallback was activated.
if n_stable_cross_edges == 0 and len(tail_stability_table):

    tail_stability_matrix = np.zeros((S, S), dtype=float)

    for row in tail_stability_table.itertuples(index=False):
        receiver = sector_to_idx[row.ToSector]
        sender = sector_to_idx[row.FromSector]
        tail_stability_matrix[receiver, sender] = row.SelectionFrequency

    fig, ax = plt.subplots(figsize=(9, 8))
    image = ax.imshow(
        tail_stability_matrix,
        vmin=0,
        vmax=1,
        aspect="auto",
    )
    ax.set_xticks(range(S))
    ax.set_yticks(range(S))
    ax.set_xticklabels(
        sectors,
        rotation=45,
        ha="right",
    )
    ax.set_yticklabels(sectors)
    ax.set_xlabel("Source sector")
    ax.set_ylabel("Receiving sector")
    ax.set_title(
        f"Lower-Tail Quantile Edge Stability "
        f"(tau={TAIL_QUANTILE:g})"
    )
    fig.colorbar(
        image,
        ax=ax,
        label="Selection frequency",
    )
    fig.tight_layout()
    fig.savefig(
        FIG_DIR / "Figure_P04B_Tail_Quantile_Edge_Stability.png",
        dpi=300,
    )
    plt.close(fig)

# Figure 5: Final TEST Brier.
test_pooled = forecast_metrics[
    (forecast_metrics["Split"] == "Test")
    & (forecast_metrics["Sector"] == "POOLED")
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for model_name, g in test_pooled.groupby("Model"):
    g = g.sort_values("Horizon")
    ax.plot(
        g["Horizon"],
        g["Brier"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Brier score (lower is better)"
)
ax.set_xticks(HORIZONS)
ax.set_title(
    "Final Test Probability Forecast Performance"
)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_P05_Test_Brier.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 25. EXCEL WORKBOOK WITH ENGINE FALLBACK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase1_Sparse_Stable_Hawkes.xlsx"
)

if importlib.util.find_spec("xlsxwriter") is not None:
    excel_engine = "xlsxwriter"
elif importlib.util.find_spec("openpyxl") is not None:
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "R Handoff Integrity": r_audit,
    "Sector Audit": sector_audit,
    "Target Summary": target_summary,
    "HalfLife Selection": half_life_selection,
    "Temporal CV Folds": fold_table,
    "L1 CV Summary": cv_summary,
    "L1 Selection": lambda_selection,
    "Edge Stability": stability_all,
    "Stable Edges": stable_edges,
    "HalfLife Summary": half_life_summary,
    "HalfLife Overlap": half_life_overlap,
    "Network Diagnostics": network_diagnostics,
    "Forecast Metrics": forecast_metrics,
    "Forecast Skill": forecast_skill,
    "Graph Decision": graph_decision_table,
    "Hawkes 1SE Robustness": robustness_stability_1se,
    "Tail Quantile CV": tail_cv_summary,
    "Tail Edge Stability": tail_stability_table,
}

if excel_engine is not None:
    try:
        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for sheet_name, dataframe in excel_tables.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name[:31],
                    index=False,
                )

            if excel_engine == "xlsxwriter":
                workbook = writer.book
                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes(1, 0)
                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )
                    worksheet.set_column(
                        0,
                        25,
                        16,
                    )

            else:
                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved using "
            f"{excel_engine}: {excel_path}"
        )

    except Exception as exc:
        log(
            f"WARNING: Excel export failed: {repr(exc)}"
        )
else:
    log(
        "WARNING: neither xlsxwriter nor openpyxl "
        "is available. CSV outputs remain complete."
    )

# =============================================================================
# 26. METADATA FOR DEEP-LEARNING PHASE
# =============================================================================

python_metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "1",
    "ScriptVersion": "1.3",
    "GeneratedAt": datetime.now().isoformat(),
    "RandomSeed": SEED,
    "RScriptVersionRecorded": metadata.get("ScriptVersion"),
    "PrimarySectors": sectors,
    "ForecastHorizons": HORIZONS,
    "SelectedHawkesHalfLife": primary_half_life,
    "SelectedL1Lambda_Min_Primary": primary_lambda,
    "SelectedL1Lambda_1SE_Robustness": primary_lambda_1se,
    "StabilityThreshold": STABILITY_THRESHOLD,
    "StabilityThresholdSensitivity": STABILITY_THRESHOLDS,
    "StableHawkesCrossEdges": n_stable_cross_edges,
    "CommonShockStableCrossEdgesAt60pct": int(common_cross_support.sum()),
    "StableTailQuantileCrossEdges": tail_stable_cross_edges,
    "FinalGraphSource": final_graph_source,
    "GraphDecision": graph_decision,
    "CalibrationUsedForStructureSelection": False,
    "TestUsedForStructureSelection": False,
    "HawkesCoefficientPolicy": "Train-fitted coefficients frozen for Calibration and Test",
    "TestCoefficientRefitSample": "None",
}

with open(
    DATA_DIR / "python_phase1_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        python_metadata,
        f,
        indent=2,
        default=str,
    )

# =============================================================================
# 27. FINAL INTEGRITY CHECKS
# =============================================================================

integrity_checks = [
    (
        "Corrected R EVT parameter variation passed",
        shape_sd >= 1e-6 and scale_sd >= 1e-6,
    ),
    (
        "R Crash_Main equals 2.5% EVT label",
        main_mismatch == 0,
    ),
    (
        "R EVT thresholds correctly ordered",
        threshold_order_ok,
    ),
    (
        "No Test dates used in temporal CV",
        not any(
            np.any(
                fold["FitMask"]
                & test_date_mask
            )
            or np.any(
                fold["ValidationMask"]
                & test_date_mask
            )
            for fold in folds
        ),
    ),
    (
        "Train-frozen stable Hawkes parameters finite",
        bool(
            np.all(
                np.isfinite(
                    final_stable_fit["mu"]
                )
            )
            and np.all(
                np.isfinite(
                    final_stable_fit["alpha"]
                )
            )
        ),
    ),
    (
        "Train-frozen stable Hawkes parameters non-negative",
        bool(
            np.all(
                final_stable_fit["mu"] > 0
            )
            and np.all(
                final_stable_fit["alpha"] >= -1e-12
            )
        ),
    ),
    (
        "Train-frozen stable network below branching guard",
        final_stable_radius
        < MAX_BRANCHING_SPECTRAL_RADIUS,
    ),
    (
        "Common-shock-controlled fit finite",
        bool(
            np.all(np.isfinite(common_postselection_fit["mu"]))
            and np.all(np.isfinite(common_postselection_fit["market_beta"]))
            and np.all(np.isfinite(common_postselection_fit["alpha"]))
        ),
    ),
    (
        "Calibration outcomes not used to refit Hawkes coefficients",
        final_fits is train_fits,
    ),
    (
        "Same-day market and sector excitation excluded from lambda_t",
        True,
    ),
    (
        "Final graph source resolved without Calibration/Test selection",
        final_graph_source in {
            "HAWKES_LAMBDA_MIN_STABILITY",
            "LOWER_TAIL_QUANTILE_STABILITY",
            "NO_STABLE_ECONOMETRIC_GRAPH",
        },
    ),
    (
        "Validation targets contain both classes",
        all(
            len(
                np.unique(
                    target_within[h][
                        validation_date_mask
                    ][
                        np.isfinite(
                            target_within[h][
                                validation_date_mask
                            ]
                        )
                    ]
                )
            ) == 2
            for h in HORIZONS
        ),
    ),
    (
        "Test targets contain both classes",
        all(
            len(
                np.unique(
                    target_within[h][
                        test_date_mask
                    ][
                        np.isfinite(
                            target_within[h][
                                test_date_mask
                            ]
                        )
                    ]
                )
            ) == 2
            for h in HORIZONS
        ),
    ),
]

integrity_table = pd.DataFrame(
    integrity_checks,
    columns=["Check", "Passed"],
)

integrity_table.to_csv(
    TABLE_DIR / "Table_P18_Final_Integrity_Checks.csv",
    index=False
)

if not integrity_table["Passed"].all():
    failed = integrity_table.loc[
        ~integrity_table["Passed"],
        "Check",
    ].tolist()

    raise RuntimeError(
        f"Final Python Phase 1 integrity checks failed: {failed}"
    )

# =============================================================================
# 28. ZIP OUTPUTS
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

zip_output = (
    ZIP_DIR
    / "Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as zf:

    for path in OUTPUT_ROOT.rglob("*"):

        if (
            path.is_file()
            and path != zip_output
        ):
            zf.write(
                path,
                arcname=path.relative_to(
                    OUTPUT_ROOT
                ),
            )

log(
    f"All Python Phase 1 outputs zipped to: "
    f"{zip_output}"
)

# =============================================================================
# 29. CONSOLE SUMMARY
# =============================================================================

print("\n" + "=" * 96)
print("PYTHON PHASE 1.3 REVIEWER REVISION COMPLETED SUCCESSFULLY")
print("=" * 96)

print(
    f"Input: final R handoff "
    f"({INPUT_ZIP_PATH.name})"
)

print(
    f"Primary sectors: {S}"
)

for sector in sectors:
    print(f"  - {sector}")

print(
    f"\nSelected Hawkes half-life: "
    f"{primary_half_life:g} trading days"
)

print(
    f"Selected L1 lambda_min (primary): "
    f"{primary_lambda:g}"
)

print(
    f"Stable Hawkes cross-sector edges: "
    f"{n_stable_cross_edges}"
)

print(
    f"Final graph source: "
    f"{final_graph_source}"
)

print(
    f"Graph decision: "
    f"{graph_decision}"
)

print("\nStable cross-sector edges:")

stable_cross_final = stable_edges[
    stable_edges["SelectedCrossEdge"] == 1
].sort_values(
    [
        "SelectionFrequency",
        "TrainValidationRefitAlpha",
    ],
    ascending=False,
)

if len(stable_cross_final):
    for row in stable_cross_final.itertuples(index=False):
        print(
            f"  {row.FromSector} -> {row.ToSector}: "
            f"stability={row.SelectionFrequency:.3f}, "
            f"alpha_frozen={row.TrainFrozenAlpha:.6f}"
        )
else:
    print("  None survived the stability threshold.")

if final_graph_source == "LOWER_TAIL_QUANTILE_STABILITY":
    print("\nStable lower-tail quantile cross-sector edges:")
    tail_selected_console = tail_stability_table[
        tail_stability_table["StableCrossEdge"] == 1
    ].sort_values(
        ["SelectionFrequency", "MeanBetaAcrossReps"],
        ascending=False,
    )
    for row in tail_selected_console.itertuples(index=False):
        print(
            f"  {row.FromSector} -> {row.ToSector}: "
            f"stability={row.SelectionFrequency:.3f}, "
            f"mean_beta={row.MeanBetaAcrossReps:.6f}"
        )

print("\nDeep-learning handoff files:")
print(
    f"  {DATA_DIR / 'phase2_deep_learning_master.csv.gz'}"
)
print(
    f"  {DATA_DIR / 'econometric_dynamic_graph.csv.gz'}"
)
print(
    f"  {DATA_DIR / 'econometric_dynamic_graph_arrays.npz'}"
)
print(
    f"  {MODEL_DIR / 'sparse_stable_hawkes_bundle.pkl'}"
)

print("\nZIP to send back for review:")
print(f"  {zip_output}")

print("=" * 96)

# =============================================================================
# 30. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip ..."
    )

    files.download(
        str(zip_output)
    )

except ImportError:
    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: {zip_output}"
    )

# =============================================================================
# END
# =============================================================================


[2026-09-12 09:47:08] ================================================================================================
[2026-09-12 09:47:08] PYTHON PHASE 1 START — DIRECT R HANDOFF
[2026-09-12 09:47:08] Python: 3.13.15
[2026-09-12 09:47:08] Platform: Linux-6.6.122+-x86_64-with-glibc2.39
[2026-09-12 09:47:08] Seed: 20260901

Please upload the FINAL R handoff file:
    04_Python_Handoff.zip



Saving 04_Python_Handoff.zip to 04_Python_Handoff.zip
[2026-09-12 09:53:11] Validated uploaded R handoff: /content/04_Python_Handoff.zip
[2026-09-12 09:53:11] Accepted R handoff: /content/04_Python_Handoff.zip
[2026-09-12 09:53:11] Primary sectors (6): ['Banking', 'Commercial and services', 'Energy and Petroleum', 'Insurance', 'Investment', 'Manufacturing and Allied']
[2026-09-12 09:53:11] R master: 27,368 rows, 2,488 dates, 11 total sectors.
[2026-09-12 09:53:12] Event panel: 2488 dates x 6 sectors. Train tail events=141; Calibration tail events=73; Test tail events=66; market tail events observed=41.
[2026-09-12 09:53:12] Selecting Hawkes half-life by TRAIN-only BIC...
[2026-09-12 09:53:12]   Fitting full Hawkes at half-life=1 days.
[2026-09-12 09:53:14]   Fitting full Hawkes at half-life=2 days.
[2026-09-12 09:53:15]   Fitting full Hawkes at half-life=5 days.
[2026-09-12 09:53:16]   Fitting full Hawkes at half-life=10 days.
[2026-09-12 09:53:17]   Fitting full Hawkes at half-life=22

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
# =============================================================================
# PYTHON PHASE 2.2 — FIVE-SEED ROBUSTNESS / GRAPH PLACEBO REPLICATION
#
# Forecasting Sectoral Crash Risk and Contagion:
# Integrating Dynamic Volatility, Extreme-Value Modelling and Graph Deep Learning
#
# INPUT
# -----
# Python_Phase1_ReviewerRevision_v1_3_All_Outputs.zip
#
# PURPOSE
# -------
# This script consumes the frozen Phase-1 econometric/contagion output and
# estimates the forecasting models used in the main paper:
#
#   1. Historical probability baselines.
#   2. Stability-selected Hawkes probability baseline.
#   3. XGBoost benchmark (four horizon classifiers; monotone rearrangement).
#   4. LSTM discrete-time survival model.
#   5. Temporal Transformer discrete-time survival benchmark.
#   6. Graph Survival Transformer — no graph prior.
#   7. Graph Survival Transformer — random static graph placebo.
#   8. Graph Survival Transformer — static stability-selected Hawkes graph.
#   9. PROPOSED: Graph Survival Transformer with the dynamic Hawkes graph
#      supplied as a SOFT attention prior.
#
# KEY DESIGN PRINCIPLES
# ---------------------
# * Only the uploaded NSE/R-derived data are used. No external predictors.
# * Train / Validation / Test are chronological; no random data split.
# * Forecast-origin features contain information available through day t only.
# * Time-to-crash outcomes are RECONSTRUCTED HERE with split-aware censoring.
#   Therefore Validation targets never use Test outcomes and Train targets
#   never use Validation outcomes.
# * 1/5/10/22-day neural probabilities are derived from one 22-day daily
#   hazard path:
#
#       P(T <= H) = 1 - product_{k=1}^H [1 - h_k].
#
#   Hence neural probabilities are coherent by construction.
# * XGBoost is a conventional benchmark and uses separate horizon classifiers;
#   its four probabilities are monotonically rearranged after calibration.
# * Deep models use the SAME economic predictors. Graph models differ only in
#   their graph prior, permitting clean ablation tests.
# * The Hawkes graph is a SOFT prior: it biases graph attention but does not
#   hard-mask other sector interactions.
# * Train is internally split into TrainFit and TrainTune for model/epoch selection.
# * After tuning, every model is reinitialized and refitted on the full Train sample.
# * The 2022-2024 Validation-labelled period is used ONLY for probability calibration.
# * Test is evaluated once after tuning, refitting, and calibration are locked.
# * Neural rare-event weighting uses sqrt imbalance capped at 3; all neural
#   models receive post-hoc hazard temperature calibration only on the dedicated
#   2022-2024 calibration segment (legacy R label: Validation).
# * Main probability metrics: Brier Score, Log Score, PR-AUC, ROC-AUC and
#   calibration intercept/slope. Accuracy is deliberately not a headline metric.
# * Paired moving-block bootstrap inference is reported for the proposed model
#   versus every benchmark on the untouched Test sample.
#
# VERSION: 2.3 — REVIEWER REVISION
# DATE: 2026-09-12
# =============================================================================

from __future__ import annotations

import os
import sys
import gc
import json
import math
import time
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)

# =============================================================================
# 0. REPRODUCIBILITY / CONFIGURATION
# =============================================================================

SEED = 20260901

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

INPUT_ZIP = os.getenv(
    "NSE_PHASE1_ZIP",
    "/content/Python_Phase1_ReviewerRevision_v1_3_All_Outputs.zip"
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PHASE2_OUTPUT_DIR",
    "/content/Sectoral_Tail_Event_Python_Phase2_2_ReviewerRevision"
))

HORIZONS = [1, 5, 10, 22]
MAX_HORIZON = 22
LOOKBACK = int(os.getenv("NSE_LOOKBACK", "60"))

# Reviewer-revision chronology. The R/Phase-1 handoff carries these exact
# RevisionPhase labels. The legacy Split label "Validation" is retained only
# for backward-compatible file structure; analytically it is Calibration only.
TRAIN_FIT_END = pd.Timestamp("2021-01-29")
TRAIN_TUNE_START = pd.Timestamp("2021-02-01")
TRAIN_TUNE_END = pd.Timestamp("2022-07-29")
CALIBRATION_START = pd.Timestamp("2022-08-01")
CALIBRATION_END = pd.Timestamp("2024-07-31")
TEST_START = pd.Timestamp("2024-08-01")

# Deep-learning hyperparameters. These are intentionally compact because the
# dataset contains six sectors and ~2,500 trading dates.
BATCH_SIZE = int(os.getenv("NSE_BATCH_SIZE", "128"))
GRAPH_BATCH_SIZE = int(os.getenv("NSE_GRAPH_BATCH_SIZE", "32"))
MAX_EPOCHS = int(os.getenv("NSE_MAX_EPOCHS", "60"))
PATIENCE = int(os.getenv("NSE_PATIENCE", "8"))
LEARNING_RATE = float(os.getenv("NSE_LEARNING_RATE", "0.001"))
WEIGHT_DECAY = float(os.getenv("NSE_WEIGHT_DECAY", "0.0001"))
D_MODEL = int(os.getenv("NSE_D_MODEL", "48"))
N_HEADS = int(os.getenv("NSE_N_HEADS", "4"))
N_TRANSFORMER_LAYERS = int(os.getenv("NSE_TRANSFORMER_LAYERS", "2"))
DROPOUT = float(os.getenv("NSE_DROPOUT", "0.10"))
SECTOR_EMBED_DIM = int(os.getenv("NSE_SECTOR_EMBED_DIM", "8"))

# Positive hazard weighting.
#
# CORRECTION 2.1:
# The Phase-2 neural cap of 10 produced a hazard weight near 6.7 and required
# very strong post-hoc temperature flattening. Neural survival models now use a
# conservative cap of 3. XGBoost retains the previous cap of 10 so the strong
# tabular benchmark is otherwise unchanged.
NEURAL_MAX_POS_WEIGHT = float(
    os.getenv("NSE_NEURAL_MAX_POS_WEIGHT", "3.0")
)
XGB_MAX_POS_WEIGHT = float(
    os.getenv("NSE_XGB_MAX_POS_WEIGHT", "10.0")
)

# Magnitude-preserving Hawkes graph transformation.
#
# A positive Train-only q99 reference is mapped to 1:
#
#     scaled(A) = log(1 + A / q99_train) / log(2)
#
# Values are globally clipped only at 3 to limit extreme numerical leverage.
# Crucially, there is NO row normalization, so 0.001 remains much weaker than
# 0.10 after transformation.
GRAPH_SCALE_QUANTILE = float(
    os.getenv("NSE_GRAPH_SCALE_QUANTILE", "0.99")
)
GRAPH_SCALE_CLIP = float(
    os.getenv("NSE_GRAPH_SCALE_CLIP", "3.0")
)

# XGBoost.
XGB_MAX_ROUNDS = int(os.getenv("NSE_XGB_MAX_ROUNDS", "1500"))
XGB_EARLY_STOP = int(os.getenv("NSE_XGB_EARLY_STOP", "75"))

# Moving-block bootstrap.
BOOTSTRAP_REPS = int(os.getenv("NSE_BOOTSTRAP_REPS", "500"))
BOOTSTRAP_BLOCK = int(os.getenv("NSE_BOOTSTRAP_BLOCK", "22"))

EPS = 1e-8

PROPOSED_MODEL = "DynamicHawkesGraphTransformer"

# =============================================================================
# 1. OUTPUT DIRECTORIES / LOGGING
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_Phase1_Extracted"

for directory in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR, MODEL_DIR,
    DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase2_2_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("=" * 100)
log("PYTHON PHASE 2.3 START — NESTED TUNING / FULL-TRAIN REFIT / FIVE-SEED ROBUSTNESS")
log(f"Python={sys.version.split()[0]}; platform={platform.platform()}; seed={SEED}")

# =============================================================================
# 2. PACKAGE CHECKS
# =============================================================================

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is not None:
        return
    pip_name = pip_name or import_name
    log(f"Installing missing package: {pip_name}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pip_name]
    )

ensure_package("torch")
ensure_package("xgboost")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import xgboost as xgb

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log(f"PyTorch={torch.__version__}; device={DEVICE}")

# =============================================================================
# 3. ALWAYS ASK USER TO UPLOAD PHASE-1 ZIP IN COLAB
# =============================================================================

REQUIRED_PHASE1_MEMBERS = {
    "phase2_deep_learning_master.csv.gz",
    "econometric_dynamic_graph_arrays.npz",
    "python_phase1_metadata.json",
    "Table_P12_Final_Stable_Edge_Parameters.csv",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as archive:
            return {
                Path(name).name
                for name in archive.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_phase1_zip(path: Path) -> bool:
    return (
        path.exists()
        and path.suffix.lower() == ".zip"
        and REQUIRED_PHASE1_MEMBERS.issubset(zip_basenames(path))
    )

def resolve_phase1_zip(configured: str) -> Path:
    """
    Colab behaviour: ALWAYS open a file picker. This avoids accidentally using
    a stale R handoff or previous Python ZIP already present in /content.
    """
    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the completed Phase-1 ZIP:\n"
                "    Python_Phase1_ReviewerRevision_v1_3_All_Outputs.zip\n"
            )
            uploaded = files.upload()

            candidates = [
                Path("/content") / name
                for name in uploaded
                if name.lower().endswith(".zip")
            ]

            valid = [p for p in candidates if is_valid_phase1_zip(p)]

            if len(valid) == 1:
                log(f"Validated uploaded Phase-1 ZIP: {valid[0]}")
                return valid[0]

            if len(valid) > 1:
                print(
                    "\nMore than one valid Phase-1 ZIP was uploaded. "
                    "Please upload exactly one ZIP.\n"
                )
                continue

            for p in candidates:
                missing = sorted(
                    REQUIRED_PHASE1_MEMBERS - zip_basenames(p)
                )
                log(
                    f"Rejected '{p.name}'. Missing Phase-1 members: {missing}"
                )

            print(
                "\nThe selected ZIP is not the required completed Phase-1 ZIP. "
                "Please choose:\n"
                "    Python_Phase1_ReviewerRevision_v1_3_All_Outputs.zip\n"
            )

    except ImportError:
        configured_path = Path(configured)

        if is_valid_phase1_zip(configured_path):
            return configured_path

        # Local / notebook fallback: locate a validated ZIP by contents.
        for folder in [Path.cwd(), Path("/mnt/data")]:
            if folder.exists():
                for candidate in folder.glob("*.zip"):
                    if is_valid_phase1_zip(candidate):
                        return candidate

        raise FileNotFoundError(
            "Could not locate a valid Phase-1 ZIP. "
            "Set NSE_PHASE1_ZIP to the correct path."
        )

INPUT_ZIP_PATH = resolve_phase1_zip(INPUT_ZIP)
log(f"Accepted Phase-1 ZIP: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' in Phase-1 ZIP; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("phase2_deep_learning_master.csv.gz")
GRAPH_FILE = find_one("econometric_dynamic_graph_arrays.npz")
META_FILE = find_one("python_phase1_metadata.json")
EDGE_FILE = find_one("Table_P12_Final_Stable_Edge_Parameters.csv")

# =============================================================================
# 4. LOAD / AUDIT PHASE-1 OUTPUT
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

with open(META_FILE, "r", encoding="utf-8") as handle:
    phase1_meta = json.load(handle)

graph_npz = np.load(GRAPH_FILE, allow_pickle=True)
stable_edges = pd.read_csv(EDGE_FILE)

# Require the reviewer-revised Phase-1 chronology/graph policy. This prevents
# accidental use of the older Train+Validation Hawkes coefficient refit.
phase1_version = str(phase1_meta.get("ScriptVersion", ""))
phase1_hawkes_policy = str(phase1_meta.get("HawkesCoefficientPolicy", ""))
if phase1_version != "1.3":
    raise RuntimeError(
        "Phase 2.3 requires Python Phase 1 reviewer revision v1.3. "
        f"Received ScriptVersion={phase1_version!r}."
    )
if "frozen for Calibration and Test" not in phase1_hawkes_policy:
    raise RuntimeError(
        "Phase-1 Hawkes coefficients are not documented as Train-fitted and "
        "frozen for Calibration/Test. Use the reviewer-revised Phase-1 ZIP."
    )

sectors = graph_npz["sectors"].astype(str).tolist()
S = len(sectors)
sector_to_idx = {sector: i for i, sector in enumerate(sectors)}

dates = pd.DatetimeIndex(graph_npz["dates"].astype(str))
T = len(dates)
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

graph_source = str(graph_npz["graph_source"].reshape(-1)[0])
dynamic_graph_all = graph_npz["adjacency"].astype(np.float32)
dynamic_graph_train_fit = graph_npz["train_fit_adjacency"].astype(np.float32)
dynamic_graph_final_fit = graph_npz["final_fit_adjacency"].astype(np.float32)
hawkes_train_alpha = graph_npz["hawkes_train_alpha"].astype(np.float32)
hawkes_final_alpha = graph_npz["hawkes_final_alpha"].astype(np.float32)

if dynamic_graph_all.shape != (T, S, S):
    raise RuntimeError(
        f"Unexpected graph shape: {dynamic_graph_all.shape}; "
        f"expected {(T, S, S)}."
    )

if set(master["Sector"].unique()) != set(sectors):
    raise RuntimeError("Master sectors do not match graph-array sectors.")

if master["Date"].nunique() != T:
    raise RuntimeError("Master dates do not match graph-array dates.")

if graph_source != "HAWKES_LAMBDA_MIN_STABILITY":
    log(
        f"WARNING: graph source is '{graph_source}', not the expected Hawkes "
        "lambda-min stability graph. The code will still proceed."
    )

# Ensure every Date-Sector combination is unique and complete.
if master.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found in DL master.")

counts = master.groupby("Date")["Sector"].nunique()
if not (counts == S).all():
    raise RuntimeError("The DL master is not a complete six-sector date panel.")

# Date-level chronology.
legacy_split_by_date = (
    master[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .reindex(dates)
)
splits = legacy_split_by_date.astype(str).to_numpy()

if set(np.unique(splits)) != {"Train", "Validation", "Test"}:
    raise RuntimeError(f"Unexpected legacy split labels: {np.unique(splits)}")

if "RevisionPhase" not in master.columns:
    raise RuntimeError(
        "Reviewer-revised Phase 2.3 requires RevisionPhase from the R/Phase-1 handoff."
    )

revision_phase_by_date = (
    master[["Date", "RevisionPhase"]]
    .drop_duplicates()
    .set_index("Date")["RevisionPhase"]
    .reindex(dates)
)
revision_phases = revision_phase_by_date.astype(str).to_numpy()
expected_revision_phases = {"TrainFit", "TrainTune", "Calibration", "Test"}
if set(np.unique(revision_phases)) != expected_revision_phases:
    raise RuntimeError(
        f"Unexpected RevisionPhase labels: {np.unique(revision_phases)}"
    )

train_fit_date_mask = revision_phases == "TrainFit"
train_tune_date_mask = revision_phases == "TrainTune"
full_train_date_mask = train_fit_date_mask | train_tune_date_mask
calibration_date_mask = revision_phases == "Calibration"
test_date_mask = revision_phases == "Test"

# Backward-compatible aliases used by later reporting code. Importantly,
# validation_date_mask now means the dedicated calibration period only.
train_date_mask = full_train_date_mask
validation_date_mask = calibration_date_mask

if not np.array_equal(full_train_date_mask, splits == "Train"):
    raise RuntimeError("RevisionPhase TrainFit+TrainTune does not equal legacy Train.")
if not np.array_equal(calibration_date_mask, splits == "Validation"):
    raise RuntimeError("RevisionPhase Calibration does not equal legacy Validation.")
if not np.array_equal(test_date_mask, splits == "Test"):
    raise RuntimeError("RevisionPhase Test does not equal legacy Test.")

# Hard date-boundary audit.
def _date_bounds(mask):
    d = dates[mask]
    return (pd.Timestamp(d.min()), pd.Timestamp(d.max()))

if _date_bounds(train_fit_date_mask)[1] != TRAIN_FIT_END:
    raise RuntimeError("TrainFit end date differs from frozen reviewer design.")
if _date_bounds(train_tune_date_mask) != (TRAIN_TUNE_START, TRAIN_TUNE_END):
    raise RuntimeError("TrainTune dates differ from frozen reviewer design.")
if _date_bounds(calibration_date_mask) != (CALIBRATION_START, CALIBRATION_END):
    raise RuntimeError("Calibration dates differ from frozen reviewer design.")
if _date_bounds(test_date_mask)[0] != TEST_START:
    raise RuntimeError("Test start date differs from frozen reviewer design.")

log(
    f"Loaded {len(master):,} sector-days; {T:,} dates; {S} sectors; "
    f"graph source={graph_source}."
)
log(
    "Reviewer chronology: "
    f"TrainFit={train_fit_date_mask.sum()} dates, "
    f"TrainTune={train_tune_date_mask.sum()}, "
    f"Calibration={calibration_date_mask.sum()}, "
    f"Test={test_date_mask.sum()}."
)

# =============================================================================
# 5. RECONSTRUCT CURRENT CRASH PANEL
# =============================================================================

C = np.zeros((T, S), dtype=np.float32)
M = np.zeros((T, S), dtype=bool)

for row in master[["Date", "Sector", "Crash_Main"]].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[row.Sector]

    if pd.notna(row.Crash_Main):
        C[t, s] = float(row.Crash_Main)
        M[t, s] = True

if C[train_date_mask].sum() <= 0:
    raise RuntimeError("No Train crash events found.")

# =============================================================================
# 6. DUAL SPLIT-AWARE SURVIVAL TARGETS — NESTED TEMPORAL DESIGN
# =============================================================================
# Two target systems are required:
#   (A) INNER targets stop at TrainFit/TrainTune boundaries and are used only
#       for epoch/round selection.
#   (B) FINAL targets stop at Train/Calibration/Test boundaries and are used for
#       full-Train refitting, dedicated calibration, and locked Test evaluation.
# This prevents the inner tuning stage from seeing future TrainTune outcomes
# while still allowing the final full-Train fit to use all pre-calibration data.

def build_split_aware_targets(boundary_labels: np.ndarray):
    hazard_target = np.zeros((T, S, MAX_HORIZON), dtype=np.float32)
    hazard_mask = np.zeros((T, S, MAX_HORIZON), dtype=np.float32)
    event_time = np.full((T, S), np.nan, dtype=np.float32)
    censor_time = np.full((T, S), np.nan, dtype=np.float32)
    event_observed = np.full((T, S), np.nan, dtype=np.float32)
    horizon = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }

    for t in range(T):
        label_t = boundary_labels[t]
        for s in range(S):
            if not M[t, s]:
                continue

            observed_until = 0
            event_k = None

            for k in range(1, MAX_HORIZON + 1):
                future_t = t + k
                if future_t >= T:
                    break
                if boundary_labels[future_t] != label_t:
                    break
                if not M[future_t, s]:
                    break

                observed_until = k
                if C[future_t, s] == 1:
                    event_k = k
                    break

            if event_k is not None:
                event_observed[t, s] = 1.0
                event_time[t, s] = float(event_k)
                censor_time[t, s] = float(event_k)
                hazard_mask[t, s, :event_k] = 1.0
                hazard_target[t, s, event_k - 1] = 1.0
            else:
                event_observed[t, s] = 0.0
                censor_time[t, s] = float(observed_until)
                if observed_until > 0:
                    hazard_mask[t, s, :observed_until] = 1.0

            for h in HORIZONS:
                if event_k is not None and event_k <= h:
                    horizon[h][t, s] = 1.0
                elif observed_until >= h:
                    horizon[h][t, s] = 0.0

    return hazard_target, hazard_mask, event_time, censor_time, event_observed, horizon

# Inner tuning boundaries.
(
    inner_daily_hazard_target,
    inner_daily_hazard_mask,
    inner_event_time,
    inner_censor_time,
    inner_event_observed,
    inner_horizon_target,
) = build_split_aware_targets(revision_phases)

# Final Train / Calibration / Test boundaries.
final_boundary_labels = np.where(
    full_train_date_mask,
    "Train",
    np.where(calibration_date_mask, "Calibration", "Test"),
)
(
    daily_hazard_target,
    daily_hazard_mask,
    split_event_time,
    split_censor_time,
    split_event_observed,
    horizon_target,
) = build_split_aware_targets(final_boundary_labels)

# Save both target audits.
target_audit_rows = []
for design_name, phase_specs, htarget in [
    (
        "InnerTuning",
        [
            ("TrainFit", train_fit_date_mask),
            ("TrainTune", train_tune_date_mask),
        ],
        inner_horizon_target,
    ),
    (
        "FinalCalibrationTest",
        [
            ("Train", full_train_date_mask),
            ("Calibration", calibration_date_mask),
            ("Test", test_date_mask),
        ],
        horizon_target,
    ),
]:
    for phase_name, phase_mask in phase_specs:
        for h in HORIZONS:
            y = htarget[h][phase_mask].reshape(-1)
            ok = np.isfinite(y)
            target_audit_rows.append({
                "TargetDesign": design_name,
                "Split": phase_name,
                "Sector": "POOLED",
                "Horizon": h,
                "ValidTargets": int(ok.sum()),
                "Events": int(np.nansum(y)),
                "EventRate": float(np.nanmean(y)) if ok.any() else np.nan,
            })
            for s, sector in enumerate(sectors):
                ys = htarget[h][phase_mask, s]
                oks = np.isfinite(ys)
                target_audit_rows.append({
                    "TargetDesign": design_name,
                    "Split": phase_name,
                    "Sector": sector,
                    "Horizon": h,
                    "ValidTargets": int(oks.sum()),
                    "Events": int(np.nansum(ys)),
                    "EventRate": float(np.nanmean(ys)) if oks.any() else np.nan,
                })

target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(
    TABLE_DIR / "Table_D01_Split_Aware_Target_Audit.csv",
    index=False,
)
log("Nested split-aware survival targets reconstructed successfully.")

# =============================================================================
# 7. FEATURE GOVERNANCE — EXPLICITLY REMOVE FUTURE / TARGET VARIABLES
# =============================================================================

# Predictors intentionally excluded because they are outcomes, future-event
# summaries, benchmark predictions, IDs/text, or direct target thresholds.
EXCLUDE_COLUMNS = {
    "Date",
    "Sector",
    "Split",
    "PrimarySector",
    "ExclusionReason",
    "EconometricGraphSource",

    # Survival / future labels.
    "EventObservedWithin22",
    "EventTime",
    "CensorTime",
    "CrashWithin_1",
    "CrashWithin_5",
    "CrashWithin_10",
    "CrashWithin_22",

    # Baseline/model predictions.
    "FixedHistoricalProb_1",
    "FixedHistoricalProb_5",
    "FixedHistoricalProb_10",
    "FixedHistoricalProb_22",
    "ExpandingHistoricalProb_1",
    "ExpandingHistoricalProb_5",
    "ExpandingHistoricalProb_10",
    "ExpandingHistoricalProb_22",
    "StableHawkesProb_1",
    "StableHawkesProb_5",
    "StableHawkesProb_10",
    "StableHawkesProb_22",

    # Redundant current-label copy.
    "CurrentCrash",

    # Direct crash thresholds / alternative crash outcomes are excluded to keep
    # the feature set economically interpretable and avoid near-target proxies.
    "CrashThreshold_001",
    "CrashThreshold_0025",
    "CrashThreshold_005",
    "Crash_001",
    "Crash_0025",
    "Crash_005",

    # Optimizer diagnostics are not economic predictors.
    "EVT_fit_method",
    "EVT_fit_convergence",
    "EVT_loglik",
    "EVT_nll_improvement",
    "EVT_at_boundary",
    "EVT_refit_id",
}

# Crash_Main at forecast origin t IS permitted: a crash observed today is valid
# information when forecasting t+1 onward and is central to excitation dynamics.
candidate_features = []

for column in master.columns:
    if column in EXCLUDE_COLUMNS:
        continue

    if pd.api.types.is_numeric_dtype(master[column]):
        candidate_features.append(column)

# Explicitly retain current crash state and observation indicator.
for mandatory in ["Crash_Main", "CurrentCrashObserved"]:
    if mandatory in master.columns and mandatory not in candidate_features:
        candidate_features.append(mandatory)

# Remove accidental future-like names defensively.
forbidden_name_fragments = [
    "CrashWithin_",
    "EventTime",
    "CensorTime",
    "Prob_",
]

feature_columns = [
    c for c in candidate_features
    if not any(fragment in c for fragment in forbidden_name_fragments)
]

if "Crash_Main" not in feature_columns:
    raise RuntimeError("Crash_Main should be available as current-state input.")

log(f"Selected {len(feature_columns)} leakage-screened numeric predictors.")

pd.DataFrame({
    "Feature": feature_columns
}).to_csv(
    TABLE_DIR / "Table_D02_Model_Features.csv",
    index=False
)

# =============================================================================
# 8. PANELIZE, EVENT-SAFE IMPUTE, TRAIN-ONLY STANDARDIZE
# =============================================================================

# Reindex each sector to the graph date order.
feature_panel_raw = np.full(
    (T, S, len(feature_columns)),
    np.nan,
    dtype=np.float64,
)

for s, sector in enumerate(sectors):
    sector_df = (
        master[master["Sector"] == sector]
        .set_index("Date")
        .reindex(dates)
    )

    feature_panel_raw[:, s, :] = (
        sector_df[feature_columns]
        .astype(float)
        .to_numpy()
    )

# -------------------------------------------------------------------------
# CORRECTION 2.1A — Crash_Main is an EVENT indicator and must never be
# forward-filled. Missing crash states are represented as Crash_Main = 0
# together with CurrentCrashObserved = 0. Thus the model can distinguish
# "observed non-crash" from "crash state unavailable" without propagating a
# previous crash across later ineligible sector-days.
# -------------------------------------------------------------------------

feature_panel_preimpute = feature_panel_raw.copy()

crash_feature_index = feature_columns.index("Crash_Main")
feature_panel_preimpute[:, :, crash_feature_index] = C.astype(np.float64)

if "CurrentCrashObserved" in feature_columns:
    crash_observed_feature_index = feature_columns.index(
        "CurrentCrashObserved"
    )
    feature_panel_preimpute[
        :, :, crash_observed_feature_index
    ] = M.astype(np.float64)
else:
    crash_observed_feature_index = None

# Past-only forward fill for CONTINUOUS / STATE predictors.
# Crash_Main and CurrentCrashObserved are already complete after the explicit
# event-safe assignment above, so they cannot be propagated by ffill.
feature_panel_ffill = feature_panel_preimpute.copy()

for s in range(S):
    frame = pd.DataFrame(
        feature_panel_ffill[:, s, :],
        columns=feature_columns,
    )
    feature_panel_ffill[:, s, :] = frame.ffill().to_numpy()

# Inner-Train medians determine which features exist and which missingness
# indicators are admitted. This prevents the TrainTune segment from influencing
# feature-set selection. The same feature set is then retained for the final
# full-Train refit.
trainfit_values = feature_panel_ffill[train_fit_date_mask]
trainfit_median_initial = np.nanmedian(trainfit_values, axis=(0, 1))
valid_feature_mask = np.isfinite(trainfit_median_initial)

if not valid_feature_mask.all():
    dropped = [
        feature_columns[i]
        for i in np.where(~valid_feature_mask)[0]
    ]
    log(f"Dropping all-missing TrainFit features: {dropped}")
    feature_columns = [
        feature_columns[i]
        for i in np.where(valid_feature_mask)[0]
    ]
    feature_panel_ffill = feature_panel_ffill[:, :, valid_feature_mask]

raw_after_feature_filter = feature_panel_raw[:, :, valid_feature_mask]
raw_trainfit = raw_after_feature_filter[train_fit_date_mask]
missing_rate = np.mean(~np.isfinite(raw_trainfit), axis=(0, 1))
missing_indicator_candidate = missing_rate >= 0.02

for special_name in ["Crash_Main", "CurrentCrashObserved"]:
    if special_name in feature_columns:
        missing_indicator_candidate[feature_columns.index(special_name)] = False

missing_indicator_indices = np.where(missing_indicator_candidate)[0]
missing_indicators = (
    ~np.isfinite(raw_after_feature_filter[:, :, missing_indicator_indices])
).astype(np.float64)
missing_indicator_names = [
    f"MISS__{feature_columns[i]}"
    for i in missing_indicator_indices
]

def build_preprocessed_panel(reference_mask: np.ndarray, label: str):
    reference_values = feature_panel_ffill[reference_mask]
    medians = np.nanmedian(reference_values, axis=(0, 1))
    if not np.all(np.isfinite(medians)):
        bad = [
            feature_columns[i]
            for i in np.where(~np.isfinite(medians))[0]
        ]
        raise RuntimeError(
            f"{label} preprocessing has non-finite medians after feature freeze: {bad}"
        )

    numeric = feature_panel_ffill.copy()
    for j in range(numeric.shape[2]):
        bad = ~np.isfinite(numeric[:, :, j])
        numeric[:, :, j][bad] = medians[j]

    if len(missing_indicator_indices):
        numeric = np.concatenate([numeric, missing_indicators], axis=2)

    ref_block = numeric[reference_mask]
    means = ref_block.mean(axis=(0, 1))
    stds = ref_block.std(axis=(0, 1))
    stds[stds < 1e-8] = 1.0

    panel = ((numeric - means[None, None, :]) / stds[None, None, :]).astype(np.float32)
    return panel, medians, means, stds

model_feature_names = feature_columns + missing_indicator_names

# Tuning preprocessing uses TrainFit only.
(
    X_panel_inner,
    trainfit_median,
    trainfit_mean,
    trainfit_std,
) = build_preprocessed_panel(train_fit_date_mask, "TrainFit")

# Final preprocessing is recomputed on all Train, with the feature set frozen.
(
    X_panel,
    train_median,
    train_mean,
    train_std,
) = build_preprocessed_panel(full_train_date_mask, "FullTrain")

F_DIM = X_panel.shape[2]
if X_panel_inner.shape[2] != F_DIM:
    raise RuntimeError("Inner and final preprocessing produced different feature dimensions.")

# Audit the special event imputation before scaling.
crash_after_special = feature_panel_preimpute[
    :, :, crash_feature_index
]
original_crash_missing = ~np.isfinite(
    feature_panel_raw[
        :, :, crash_feature_index
    ]
)

crash_missing_set_to_zero = bool(
    np.all(
        crash_after_special[
            original_crash_missing
        ] == 0
    )
)

if not crash_missing_set_to_zero:
    raise RuntimeError(
        "Crash_Main event-safe imputation failed."
    )

preprocess_bundle = {
    "features": model_feature_names,
    "base_features": feature_columns,
    "missing_indicator_features": missing_indicator_names,
    "trainfit_median": trainfit_median,
    "trainfit_mean": trainfit_mean,
    "trainfit_std": trainfit_std,
    "full_train_median": train_median,
    "full_train_mean": train_mean,
    "full_train_std": train_std,
    "lookback": LOOKBACK,
    "sectors": sectors,
    "CrashMainImputation": (
        "Missing Crash_Main -> 0; "
        "CurrentCrashObserved -> 0; "
        "Crash_Main is never forward-filled"
    ),
}

with open(
    MODEL_DIR
    / "feature_preprocessing_bundle.pkl",
    "wb",
) as handle:
    pickle.dump(
        preprocess_bundle,
        handle,
    )

pd.DataFrame({
    "Feature": model_feature_names,
    "TrainFitMeanBeforeScaling": trainfit_mean,
    "TrainFitStdBeforeScaling": trainfit_std,
    "FullTrainMeanBeforeScaling": train_mean,
    "FullTrainStdBeforeScaling": train_std,
}).to_csv(
    TABLE_DIR
    / "Table_D03_Feature_Scaling.csv",
    index=False,
)

event_imputation_audit = pd.DataFrame({
    "Item": [
        "Original missing Crash_Main cells",
        "Missing Crash_Main cells set to zero",
        "Crash_Main forward-filled",
        "CurrentCrashObserved used",
    ],
    "Value": [
        int(original_crash_missing.sum()),
        int(
            (
                crash_after_special[
                    original_crash_missing
                ] == 0
            ).sum()
        ),
        "NO",
        "YES",
    ],
})

event_imputation_audit.to_csv(
    TABLE_DIR
    / "Table_D03A_Crash_Event_Imputation_Audit.csv",
    index=False,
)

log(
    f"Final neural feature dimension={F_DIM}; "
    f"missingness indicators="
    f"{len(missing_indicator_names)}."
)
log(
    "Crash_Main event-safe imputation applied: "
    "missing events set to zero and never forward-filled."
)

# =============================================================================
# 9. MAGNITUDE-PRESERVING GRAPH SCALING / ABLATION PRIORS
# =============================================================================
#
# CORRECTION 2.1B
# ----------------
# Phase 2 row-normalized each receiver row at each date. For a sparse Hawkes
# network, that operation can turn the only active incoming edge into weight 1
# irrespective of whether its raw excitation contribution is tiny or large.
#
# Here we preserve ABSOLUTE dynamic excitation magnitude. A single Train-only
# global scale is estimated from positive dynamic Hawkes weights:
#
#     q = Q_0.99(A_train | A_train > 0)
#
#     scaled(A) = log(1 + A/q) / log(2)
#
# Therefore q maps to 1, weaker states remain weak, stronger states remain
# stronger, and the exact same transformation is used for Train, Validation,
# Test, static Hawkes and random-placebo priors. No Test information enters q.

def clean_nonnegative_graph(
    A: np.ndarray,
) -> np.ndarray:
    A = np.asarray(
        A,
        dtype=np.float32,
    )

    return np.where(
        np.isfinite(A)
        & (A > 0),
        A,
        0.0,
    ).astype(np.float32)

raw_dynamic_train = clean_nonnegative_graph(
    dynamic_graph_train_fit[
        train_fit_date_mask
    ]
)

positive_train_dynamic = raw_dynamic_train[
    raw_dynamic_train > 0
]

if len(positive_train_dynamic) == 0:
    raise RuntimeError(
        "No positive Train Hawkes graph weights available "
        "for magnitude-preserving scaling."
    )

GRAPH_TRAIN_Q = float(
    np.quantile(
        positive_train_dynamic,
        GRAPH_SCALE_QUANTILE,
    )
)

if not np.isfinite(GRAPH_TRAIN_Q) or GRAPH_TRAIN_Q <= 0:
    raise RuntimeError(
        "Invalid Train-only Hawkes graph scale reference."
    )

def scale_graph_magnitude(
    A: np.ndarray,
    reference: float = GRAPH_TRAIN_Q,
) -> np.ndarray:
    """
    Train-reference global log scaling. There is NO row normalization.
    """
    A = clean_nonnegative_graph(A)

    scaled = (
        np.log1p(
            A / max(reference, EPS)
        )
        / np.log(2.0)
    )

    scaled = np.clip(
        scaled,
        0.0,
        GRAPH_SCALE_CLIP,
    )

    return scaled.astype(np.float32)

# Phase-1 v1.3 freezes Hawkes support AND coefficients after Train. There is no
# Train+Calibration coefficient refit. Enforce that policy before graph models.
if not np.allclose(hawkes_train_alpha, hawkes_final_alpha, atol=1e-12, rtol=0):
    raise RuntimeError(
        "Phase-1 train/final Hawkes coefficients differ. Reviewer revision requires "
        "Train-fitted coefficients frozen for Calibration and Test."
    )

dynamic_graph_scaled = scale_graph_magnitude(dynamic_graph_all)

# Static Hawkes graph is the same Train-fitted coefficient matrix at every date.
static_train = scale_graph_magnitude(hawkes_train_alpha)
static_graph_by_date = np.repeat(static_train[None, :, :], T, axis=0)

# No econometric graph prior.
zero_graph_by_date = np.zeros(
    (T, S, S),
    dtype=np.float32,
)

# Random static placebo:
# preserve the number of directed CROSS-sector edges in the stable Hawkes
# support and use the mean raw positive Hawkes alpha as the placebo edge
# magnitude before applying the SAME Train-derived graph transformation.
rng_graph = np.random.default_rng(
    SEED
)

cross_support = (
    (hawkes_train_alpha > 0)
    & (~np.eye(S, dtype=bool))
)

n_cross_edges = int(
    cross_support.sum()
)

all_cross_positions = [
    (i, j)
    for i in range(S)
    for j in range(S)
    if i != j
]

chosen_positions = rng_graph.choice(
    len(all_cross_positions),
    size=max(1, n_cross_edges),
    replace=False,
)

random_static_raw = np.zeros(
    (S, S),
    dtype=np.float32,
)

positive_cross_values = (
    hawkes_train_alpha[
        cross_support
    ]
)

random_raw_weight = (
    float(
        np.mean(
            positive_cross_values
        )
    )
    if len(positive_cross_values)
    else GRAPH_TRAIN_Q
)

for idx in np.atleast_1d(
    chosen_positions
):
    receiver, source = (
        all_cross_positions[
            int(idx)
        ]
    )
    random_static_raw[
        receiver,
        source,
    ] = random_raw_weight

random_static = scale_graph_magnitude(
    random_static_raw
)

random_graph_by_date = np.repeat(
    random_static[
        None, :, :
    ],
    T,
    axis=0,
)

# -------------------------------------------------------------------------
# Explicit node-level contagion-state summaries.
#
# Matrix convention is A[receiver, source].
#
# IncomingRisk_i   = sum_{j != i} A[i,j]
# OutgoingRisk_i   = sum_{r != i} A[r,i]
# MaxIncomingRisk_i= max_{j != i} A[i,j]
#
# These are derived INSIDE the graph model from whichever prior that ablation
# receives. NoGraph therefore receives exact zeros; random/static receive their
# corresponding controls; dynamic receives time-varying Hawkes state.
# -------------------------------------------------------------------------

GRAPH_STATE_FEATURE_NAMES = [
    "HawkesIncomingRisk",
    "HawkesOutgoingRisk",
    "HawkesMaxIncomingRisk",
]

def graph_state_numpy(
    graph_array: np.ndarray,
) -> np.ndarray:
    graph_array = np.asarray(
        graph_array,
        dtype=np.float32,
    )

    cross = graph_array.copy()

    diag = np.arange(S)
    cross[..., diag, diag] = 0.0

    incoming = cross.sum(
        axis=-1
    )
    outgoing = cross.sum(
        axis=-2
    )
    max_incoming = cross.max(
        axis=-1
    )

    return np.stack(
        [
            incoming,
            outgoing,
            max_incoming,
        ],
        axis=-1,
    ).astype(np.float32)

dynamic_graph_state = graph_state_numpy(
    dynamic_graph_scaled
)
static_graph_state = graph_state_numpy(
    static_graph_by_date
)
random_graph_state = graph_state_numpy(
    random_graph_by_date
)
zero_graph_state = graph_state_numpy(
    zero_graph_by_date
)

# Graph scaling audit.
graph_scaling_audit = pd.DataFrame({
    "Item": [
        "Scaling method",
        "Train-only graph quantile",
        "Train-only q reference",
        "Global clip",
        "Row normalization used",
        "Positive Train raw minimum",
        "Positive Train raw median",
        "Positive Train raw q90",
        "Positive Train raw q99",
        "Positive Train raw maximum",
        "Scaled Train positive median",
        "Scaled Train positive q90",
        "Scaled Train positive q99",
        "Scaled Train positive maximum",
    ],
    "Value": [
        "log1p(A/q_train)/log(2)",
        GRAPH_SCALE_QUANTILE,
        GRAPH_TRAIN_Q,
        GRAPH_SCALE_CLIP,
        "NO",
        float(np.min(positive_train_dynamic)),
        float(np.median(positive_train_dynamic)),
        float(np.quantile(positive_train_dynamic, 0.90)),
        float(np.quantile(positive_train_dynamic, 0.99)),
        float(np.max(positive_train_dynamic)),
        float(np.median(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ]
        )),
        float(np.quantile(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ],
            0.90,
        )),
        float(np.quantile(
            dynamic_graph_scaled[
                train_date_mask
            ][
                dynamic_graph_scaled[
                    train_date_mask
                ] > 0
            ],
            0.99,
        )),
        float(np.max(
            dynamic_graph_scaled[
                train_date_mask
            ]
        )),
    ],
})

graph_scaling_audit.to_csv(
    TABLE_DIR
    / "Table_D04A_Graph_Magnitude_Scaling_Audit.csv",
    index=False,
)

# Per-edge magnitude preservation diagnostics for stable cross-sector edges.
edge_scaling_rows = []

for receiver in range(S):
    for source in range(S):

        if receiver == source:
            continue

        raw_series = (
            dynamic_graph_train_fit[
                train_date_mask,
                receiver,
                source,
            ].astype(float)
        )

        scaled_series = (
            dynamic_graph_scaled[
                train_date_mask,
                receiver,
                source,
            ].astype(float)
        )

        if np.any(raw_series > 0):

            if (
                np.std(raw_series) > 0
                and np.std(scaled_series) > 0
            ):
                correlation = float(
                    np.corrcoef(
                        raw_series,
                        scaled_series,
                    )[0, 1]
                )
            else:
                correlation = np.nan

            edge_scaling_rows.append({
                "FromSector": sectors[source],
                "ToSector": sectors[receiver],
                "RawMinimum": float(np.min(raw_series)),
                "RawMedian": float(np.median(raw_series)),
                "RawMaximum": float(np.max(raw_series)),
                "ScaledMinimum": float(np.min(scaled_series)),
                "ScaledMedian": float(np.median(scaled_series)),
                "ScaledMaximum": float(np.max(scaled_series)),
                "RawScaledPearsonCorrelation": correlation,
                "UniqueScaledValues": int(
                    len(
                        np.unique(
                            np.round(
                                scaled_series,
                                8,
                            )
                        )
                    )
                ),
            })

edge_scaling_diagnostics = pd.DataFrame(
    edge_scaling_rows
)

edge_scaling_diagnostics.to_csv(
    TABLE_DIR
    / "Table_D04B_Edge_Magnitude_Preservation.csv",
    index=False,
)

graph_state_summary = pd.DataFrame({
    "Feature": GRAPH_STATE_FEATURE_NAMES,
    "TrainMean_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].mean()
        )
        for k in range(3)
    ],
    "TrainSD_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].std()
        )
        for k in range(3)
    ],
    "TrainMax_Dynamic": [
        float(
            dynamic_graph_state[
                train_date_mask,
                :,
                k,
            ].max()
        )
        for k in range(3)
    ],
})

graph_state_summary.to_csv(
    TABLE_DIR
    / "Table_D04C_Graph_State_Features.csv",
    index=False,
)

graph_ablation_summary = pd.DataFrame({
    "GraphVariant": [
        "NoGraph",
        "RandomStaticGraph",
        "StaticHawkesGraph",
        "DynamicHawkesGraph",
    ],
    "Description": [
        (
            "Zero econometric prior; graph attention remains data-learned; "
            "graph-state summaries are zero."
        ),
        (
            "Random directed prior with Hawkes cross-edge density; "
            "same Train-derived magnitude scale."
        ),
        (
            "Time-invariant stability-selected Hawkes coefficient prior; "
            "same Train-derived magnitude scale."
        ),
        (
            "Time-varying stability-selected Hawkes excitation prior; "
            "absolute excitation magnitude preserved."
        ),
    ],
    "CrossEdgesTrainPrior": [
        0,
        int(
            (
                random_static
                * (~np.eye(S, dtype=bool))
            > 0
            ).sum()
        ),
        int(
            (
                static_train
                * (~np.eye(S, dtype=bool))
            > 0
            ).sum()
        ),
        int(
            (
                np.any(
                    dynamic_graph_scaled[
                        train_date_mask
                    ] > 0,
                    axis=0,
                )
                & (~np.eye(S, dtype=bool))
            ).sum()
        ),
    ],
    "RowNormalized": [
        "NO",
        "NO",
        "NO",
        "NO",
    ],
    "ExplicitNodeGraphState": [
        "ZERO",
        "YES",
        "YES",
        "YES",
    ],
})

graph_ablation_summary.to_csv(
    TABLE_DIR
    / "Table_D04_Graph_Ablation_Design.csv",
    index=False,
)

log(
    "Magnitude-preserving graph scaling applied. "
    f"TrainFit q{GRAPH_SCALE_QUANTILE:.2f}="
    f"{GRAPH_TRAIN_Q:.8f}; row normalization disabled."
)

# =============================================================================
# 10. SAMPLE INDEX CONSTRUCTION
# =============================================================================

# Sector-specific samples for XGBoost/LSTM/Transformer.
sector_sample_rows = []

for t in range(LOOKBACK - 1, T):
    split_name = splits[t]

    for s, sector in enumerate(sectors):

        if not M[t, s]:
            continue

        # Require at least one observed future risk day.
        if daily_hazard_mask[t, s].sum() <= 0:
            continue

        sector_sample_rows.append({
            "t": t,
            "s": s,
            "Date": dates[t],
            "Sector": sector,
            "Split": split_name,
            "RevisionPhase": revision_phases[t],
            "InnerAtRiskDays": float(inner_daily_hazard_mask[t, s].sum()),
        })

sector_samples = pd.DataFrame(sector_sample_rows)

# Graph samples are date-level and carry all sectors.
graph_sample_rows = []

for t in range(LOOKBACK - 1, T):
    if daily_hazard_mask[t].sum() <= 0:
        continue

    graph_sample_rows.append({
        "t": t,
        "Date": dates[t],
        "Split": splits[t],
        "RevisionPhase": revision_phases[t],
        "InnerAtRiskCells": float(inner_daily_hazard_mask[t].sum()),
    })

graph_samples = pd.DataFrame(graph_sample_rows)

sector_samples.to_csv(
    DATA_DIR / "sector_sample_index.csv.gz",
    index=False,
    compression="gzip",
)

graph_samples.to_csv(
    DATA_DIR / "graph_sample_index.csv.gz",
    index=False,
    compression="gzip",
)

log(
    f"Sector samples={len(sector_samples):,}; "
    f"graph-date samples={len(graph_samples):,}."
)

# =============================================================================
# 11. DATASET CLASSES
# =============================================================================

class SectorSequenceDataset(Dataset):
    def __init__(
        self,
        sample_frame: pd.DataFrame,
        x_panel: np.ndarray,
        hazard_target: np.ndarray,
        hazard_mask: np.ndarray,
    ):
        self.rows = sample_frame.reset_index(drop=True)
        self.x_panel = x_panel
        self.hazard_target = hazard_target
        self.hazard_mask = hazard_mask

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        t = int(row["t"])
        s = int(row["s"])
        x = self.x_panel[t - LOOKBACK + 1:t + 1, s, :]
        y = self.hazard_target[t, s, :]
        mask = self.hazard_mask[t, s, :]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(s, dtype=torch.long),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )

class GraphSequenceDataset(Dataset):
    def __init__(
        self,
        sample_frame: pd.DataFrame,
        graph_by_date: np.ndarray,
        x_panel: np.ndarray,
        hazard_target: np.ndarray,
        hazard_mask: np.ndarray,
    ):
        self.rows = sample_frame.reset_index(drop=True)
        self.graph_by_date = graph_by_date
        self.x_panel = x_panel
        self.hazard_target = hazard_target
        self.hazard_mask = hazard_mask

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        t = int(row["t"])
        x = self.x_panel[t - LOOKBACK + 1:t + 1, :, :]
        g = self.graph_by_date[t - LOOKBACK + 1:t + 1]
        y = self.hazard_target[t, :, :]
        mask = self.hazard_mask[t, :, :]
        return (
            torch.tensor(x, dtype=torch.float32),
            torch.tensor(g, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(mask, dtype=torch.float32),
            torch.tensor(t, dtype=torch.long),
        )

# =============================================================================
# 12. SURVIVAL LOSS / PROBABILITY UTILITIES
# =============================================================================

def compute_hazard_pos_weight(
    target: np.ndarray,
    mask: np.ndarray,
    date_mask: np.ndarray,
) -> float:
    valid = (mask > 0) & date_mask[:, None, None]
    y = target[valid]
    positives = float(y.sum())
    negatives = float(len(y) - positives)
    if positives <= 0:
        return 1.0
    raw = math.sqrt(max(negatives / positives, 1.0))
    return float(np.clip(raw, 1.0, NEURAL_MAX_POS_WEIGHT))

INNER_HAZARD_POS_WEIGHT = compute_hazard_pos_weight(
    inner_daily_hazard_target,
    inner_daily_hazard_mask,
    train_fit_date_mask,
)
FULL_HAZARD_POS_WEIGHT = compute_hazard_pos_weight(
    daily_hazard_target,
    daily_hazard_mask,
    full_train_date_mask,
)
# Backward-compatible metadata alias; final models use the full-Train weight.
HAZARD_POS_WEIGHT = FULL_HAZARD_POS_WEIGHT
log(
    f"Neural hazard weights: TrainFit tuning={INNER_HAZARD_POS_WEIGHT:.4f}; "
    f"FullTrain refit={FULL_HAZARD_POS_WEIGHT:.4f}"
)

def masked_survival_bce(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
    pos_weight: float,
) -> torch.Tensor:
    base = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    weight = torch.where(
        targets > 0.5,
        torch.tensor(
            pos_weight,
            dtype=base.dtype,
            device=base.device,
        ),
        torch.tensor(
            1.0,
            dtype=base.dtype,
            device=base.device,
        ),
    )

    weighted = base * weight * mask
    denom = torch.clamp(mask.sum(), min=1.0)

    return weighted.sum() / denom

def masked_unweighted_survival_nll(
    logits: torch.Tensor,
    targets: torch.Tensor,
    mask: torch.Tensor,
) -> torch.Tensor:
    base = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )

    return (base * mask).sum() / torch.clamp(mask.sum(), min=1.0)

def hazard_logits_to_horizon_probs(
    logits: np.ndarray,
    temperature: float = 1.0,
    bias: float = 0.0,
) -> dict[int, np.ndarray]:
    calibrated_logits = logits / max(temperature, 1e-4) + bias
    hazard = 1.0 / (1.0 + np.exp(-np.clip(calibrated_logits, -30, 30)))

    survival = np.cumprod(1.0 - hazard, axis=-1)
    cumulative_event = 1.0 - survival

    return {
        h: cumulative_event[..., h - 1]
        for h in HORIZONS
    }

# =============================================================================
# 13. NEURAL MODEL DEFINITIONS
# =============================================================================

class LSTMSurvival(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        hidden_dim: int = D_MODEL,
    ):
        super().__init__()

        self.sector_embedding = nn.Embedding(
            n_sectors,
            SECTOR_EMBED_DIM,
        )

        self.lstm = nn.LSTM(
            input_size=feature_dim + SECTOR_EMBED_DIM,
            hidden_size=hidden_dim,
            num_layers=2,
            dropout=DROPOUT,
            batch_first=True,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, MAX_HORIZON),
        )

    def forward(self, x, sector_idx):
        embedding = self.sector_embedding(sector_idx)
        embedding_seq = embedding[:, None, :].expand(
            -1,
            x.size(1),
            -1,
        )

        z = torch.cat([x, embedding_seq], dim=-1)
        output, _ = self.lstm(z)

        return self.head(output[:, -1, :])


class TemporalTransformerSurvival(nn.Module):
    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            feature_dim,
            d_model,
        )

        self.sector_embedding = nn.Embedding(
            n_sectors,
            d_model,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(1, LOOKBACK, d_model)
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=N_TRANSFORMER_LAYERS,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, MAX_HORIZON),
        )

    def forward(self, x, sector_idx):
        z = self.input_projection(x)

        sector_emb = self.sector_embedding(
            sector_idx
        )[:, None, :]

        z = (
            z
            + sector_emb
            + self.position_embedding[:, :x.size(1), :]
        )

        z = self.encoder(z)

        return self.head(z[:, -1, :])


class SoftGraphAttention(nn.Module):
    """
    Multi-head node attention with an additive econometric graph bias.

    The prior is SOFT: non-edge pairs remain available to learned attention.
    A learnable positive eta controls how strongly the econometric graph
    influences attention scores.
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float,
    ):
        super().__init__()

        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads.")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

        # softplus(raw_eta) ensures eta >= 0.
        self.raw_eta = nn.Parameter(
            torch.tensor(0.0)
        )

    def forward(self, h, graph_prior):
        # h: B x L x S x D
        # graph_prior: B x L x S(receiver) x S(source)

        B, L, S_, D = h.shape

        q = self.q_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        k = self.k_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        v = self.v_proj(h).view(
            B, L, S_, self.n_heads, self.head_dim
        ).permute(0, 1, 3, 2, 4)

        # receiver i queries source j.
        scores = torch.einsum(
            "blhid,blhjd->blhij",
            q,
            k,
        ) / math.sqrt(self.head_dim)

        eta = F.softplus(self.raw_eta)

        graph_bias = graph_prior[:, :, None, :, :]
        scores = scores + eta * graph_bias

        attention = torch.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        message = torch.einsum(
            "blhij,blhjd->blhid",
            attention,
            v,
        )

        message = message.permute(
            0, 1, 3, 2, 4
        ).contiguous().view(B, L, S_, D)

        return self.norm(
            h + self.out_proj(message)
        )


class GraphSurvivalTransformer(nn.Module):
    """
    Temporal graph survival Transformer with TWO econometric graph channels:

      1. a soft additive edge prior in cross-sector attention;
      2. explicit node-level contagion-state summaries:
           incoming risk,
           outgoing risk,
           maximum incoming risk.

    The same architecture is used for NoGraph / Random / Static / Dynamic
    ablations. Only graph_prior changes. For NoGraph all three state summaries
    are identically zero.
    """

    def __init__(
        self,
        feature_dim: int,
        n_sectors: int,
        d_model: int = D_MODEL,
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            feature_dim,
            d_model,
        )

        self.node_embedding = nn.Embedding(
            n_sectors,
            d_model,
        )

        self.position_embedding = nn.Parameter(
            torch.zeros(
                1,
                LOOKBACK,
                1,
                d_model,
            )
        )

        self.graph_attention = SoftGraphAttention(
            d_model=d_model,
            n_heads=N_HEADS,
            dropout=DROPOUT,
        )

        # Bias=False is deliberate: a zero graph prior must inject exactly zero
        # graph-state signal in the NoGraph ablation.
        self.graph_state_projection = nn.Linear(
            3,
            d_model,
            bias=False,
        )

        # Non-negative learnable gate for explicit graph-state summaries.
        self.raw_graph_state_eta = nn.Parameter(
            torch.tensor(0.0)
        )

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=N_HEADS,
            dim_feedforward=4 * d_model,
            dropout=DROPOUT,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.temporal_encoder = nn.TransformerEncoder(
            temporal_layer,
            num_layers=N_TRANSFORMER_LAYERS,
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(
                d_model,
                MAX_HORIZON,
            ),
        )

    @staticmethod
    def graph_state_features(
        graph_prior: torch.Tensor,
    ) -> torch.Tensor:
        """
        graph_prior:
            B x L x S(receiver) x S(source)

        Returns:
            B x L x S x 3
            [incoming, outgoing, max incoming], excluding diagonal edges.
        """

        S_ = graph_prior.size(-1)

        eye = torch.eye(
            S_,
            dtype=graph_prior.dtype,
            device=graph_prior.device,
        )

        cross = (
            graph_prior
            * (
                1.0
                - eye[
                    None,
                    None,
                    :, :,
                ]
            )
        )

        incoming = cross.sum(
            dim=-1
        )

        outgoing = cross.sum(
            dim=-2
        )

        max_incoming = cross.max(
            dim=-1
        ).values

        return torch.stack(
            [
                incoming,
                outgoing,
                max_incoming,
            ],
            dim=-1,
        )

    def forward(
        self,
        x,
        graph_prior,
    ):
        # x: B x L x S x F
        # graph_prior: B x L x S(receiver) x S(source)

        B, L, S_, _ = x.shape

        z = self.input_projection(x)

        node_ids = torch.arange(
            S_,
            device=x.device,
        )

        node_emb = self.node_embedding(
            node_ids
        )[
            None,
            None,
            :,
            :,
        ]

        z = (
            z
            + node_emb
            + self.position_embedding[
                :, :L, :, :
            ]
        )

        # Explicit node-level econometric contagion state.
        graph_state = self.graph_state_features(
            graph_prior
        )

        graph_state_embedding = (
            self.graph_state_projection(
                graph_state
            )
        )

        graph_state_eta = F.softplus(
            self.raw_graph_state_eta
        )

        z = (
            z
            + graph_state_eta
            * graph_state_embedding
        )

        # Cross-sector attention with magnitude-preserving soft graph bias.
        z = self.graph_attention(
            z,
            graph_prior,
        )

        # Temporal Transformer independently for each sector after graph mixing.
        z = z.permute(
            0,
            2,
            1,
            3,
        ).contiguous().view(
            B * S_,
            L,
            -1,
        )

        z = self.temporal_encoder(
            z
        )

        last = z[:, -1, :]

        logits = self.head(
            last
        ).view(
            B,
            S_,
            MAX_HORIZON,
        )

        return logits


# =============================================================================
# 14. NESTED TRAIN-TUNE + FULL-TRAIN REFIT HELPERS
# =============================================================================

@dataclass
class TuneResult:
    best_epoch: int
    best_tune_nll: float
    history: pd.DataFrame

@dataclass
class RefitResult:
    state: dict
    history: pd.DataFrame

def make_sector_loader(
    frame: pd.DataFrame,
    shuffle: bool,
    *,
    x_panel: np.ndarray = None,
    hazard_target: np.ndarray = None,
    hazard_mask: np.ndarray = None,
) -> DataLoader:
    return DataLoader(
        SectorSequenceDataset(
            frame,
            X_panel if x_panel is None else x_panel,
            daily_hazard_target if hazard_target is None else hazard_target,
            daily_hazard_mask if hazard_mask is None else hazard_mask,
        ),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

def make_graph_loader(
    frame: pd.DataFrame,
    graph_by_date: np.ndarray,
    shuffle: bool,
    *,
    x_panel: np.ndarray = None,
    hazard_target: np.ndarray = None,
    hazard_mask: np.ndarray = None,
) -> DataLoader:
    return DataLoader(
        GraphSequenceDataset(
            frame,
            graph_by_date,
            X_panel if x_panel is None else x_panel,
            daily_hazard_target if hazard_target is None else hazard_target,
            daily_hazard_mask if hazard_mask is None else hazard_mask,
        ),
        batch_size=GRAPH_BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
    )

def _train_epoch_sector(model, loader, optimizer, pos_weight):
    model.train()
    loss_num = 0.0
    mask_num = 0.0
    for x, sector_idx, y, mask, _ in loader:
        x = x.to(DEVICE); sector_idx = sector_idx.to(DEVICE)
        y = y.to(DEVICE); mask = mask.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x, sector_idx)
        loss = masked_survival_bce(logits, y, mask, pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        nmask = float(mask.sum().item())
        loss_num += float(loss.item()) * nmask
        mask_num += nmask
    return loss_num / max(mask_num, 1.0)

def _eval_sector_nll(model, loader):
    model.eval()
    loss_num = 0.0
    mask_num = 0.0
    with torch.no_grad():
        for x, sector_idx, y, mask, _ in loader:
            x = x.to(DEVICE); sector_idx = sector_idx.to(DEVICE)
            y = y.to(DEVICE); mask = mask.to(DEVICE)
            logits = model(x, sector_idx)
            loss = masked_unweighted_survival_nll(logits, y, mask)
            nmask = float(mask.sum().item())
            loss_num += float(loss.item()) * nmask
            mask_num += nmask
    return loss_num / max(mask_num, 1.0)

def _train_epoch_graph(model, loader, optimizer, pos_weight):
    model.train()
    loss_num = 0.0
    mask_num = 0.0
    for x, graph_prior, y, mask, _ in loader:
        x = x.to(DEVICE); graph_prior = graph_prior.to(DEVICE)
        y = y.to(DEVICE); mask = mask.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits = model(x, graph_prior)
        loss = masked_survival_bce(logits, y, mask, pos_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        nmask = float(mask.sum().item())
        loss_num += float(loss.item()) * nmask
        mask_num += nmask
    return loss_num / max(mask_num, 1.0)

def _eval_graph_nll(model, loader):
    model.eval()
    loss_num = 0.0
    mask_num = 0.0
    with torch.no_grad():
        for x, graph_prior, y, mask, _ in loader:
            x = x.to(DEVICE); graph_prior = graph_prior.to(DEVICE)
            y = y.to(DEVICE); mask = mask.to(DEVICE)
            logits = model(x, graph_prior)
            loss = masked_unweighted_survival_nll(logits, y, mask)
            nmask = float(mask.sum().item())
            loss_num += float(loss.item()) * nmask
            mask_num += nmask
    return loss_num / max(mask_num, 1.0)

def tune_sector_neural_model(model: nn.Module, model_name: str) -> TuneResult:
    train_frame = sector_samples[
        (sector_samples["RevisionPhase"] == "TrainFit")
        & (sector_samples["InnerAtRiskDays"] > 0)
    ]
    tune_frame = sector_samples[
        (sector_samples["RevisionPhase"] == "TrainTune")
        & (sector_samples["InnerAtRiskDays"] > 0)
    ]
    train_loader = make_sector_loader(
        train_frame, True, x_panel=X_panel_inner,
        hazard_target=inner_daily_hazard_target,
        hazard_mask=inner_daily_hazard_mask,
    )
    tune_loader = make_sector_loader(
        tune_frame, False, x_panel=X_panel_inner,
        hazard_target=inner_daily_hazard_target,
        hazard_mask=inner_daily_hazard_mask,
    )
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_epoch = 0; best_tune = np.inf; patience_counter = 0; rows = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = _train_epoch_sector(
            model, train_loader, optimizer, INNER_HAZARD_POS_WEIGHT
        )
        tune_nll = _eval_sector_nll(model, tune_loader)
        rows.append({
            "Model": model_name, "Stage": "InnerTuning", "Epoch": epoch,
            "TrainFitWeightedLoss": train_loss, "TrainTuneNLL": tune_nll,
        })
        log(
            f"{model_name}: tune epoch={epoch:02d}, "
            f"TrainFit={train_loss:.5f}, TrainTuneNLL={tune_nll:.5f}"
        )
        if tune_nll < best_tune - 1e-5:
            best_tune = tune_nll; best_epoch = epoch; patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break
    if best_epoch <= 0:
        raise RuntimeError(f"No selected epoch for {model_name}.")
    return TuneResult(best_epoch, best_tune, pd.DataFrame(rows))

def refit_sector_neural_model(
    model: nn.Module, model_name: str, epochs: int
) -> RefitResult:
    frame = sector_samples[sector_samples["Split"] == "Train"]
    loader = make_sector_loader(frame, True)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rows = []
    for epoch in range(1, int(epochs) + 1):
        loss = _train_epoch_sector(
            model, loader, optimizer, FULL_HAZARD_POS_WEIGHT
        )
        rows.append({
            "Model": model_name, "Stage": "FullTrainRefit", "Epoch": epoch,
            "FullTrainWeightedLoss": loss,
        })
        log(f"{model_name}: refit epoch={epoch:02d}/{epochs}, FullTrain={loss:.5f}")
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return RefitResult(state, pd.DataFrame(rows))

def tune_graph_neural_model(
    model: nn.Module, model_name: str, graph_by_date: np.ndarray
) -> TuneResult:
    train_frame = graph_samples[
        (graph_samples["RevisionPhase"] == "TrainFit")
        & (graph_samples["InnerAtRiskCells"] > 0)
    ]
    tune_frame = graph_samples[
        (graph_samples["RevisionPhase"] == "TrainTune")
        & (graph_samples["InnerAtRiskCells"] > 0)
    ]
    train_loader = make_graph_loader(
        train_frame, graph_by_date, True, x_panel=X_panel_inner,
        hazard_target=inner_daily_hazard_target,
        hazard_mask=inner_daily_hazard_mask,
    )
    tune_loader = make_graph_loader(
        tune_frame, graph_by_date, False, x_panel=X_panel_inner,
        hazard_target=inner_daily_hazard_target,
        hazard_mask=inner_daily_hazard_mask,
    )
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_epoch = 0; best_tune = np.inf; patience_counter = 0; rows = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = _train_epoch_graph(
            model, train_loader, optimizer, INNER_HAZARD_POS_WEIGHT
        )
        tune_nll = _eval_graph_nll(model, tune_loader)
        rows.append({
            "Model": model_name, "Stage": "InnerTuning", "Epoch": epoch,
            "TrainFitWeightedLoss": train_loss, "TrainTuneNLL": tune_nll,
        })
        log(
            f"{model_name}: tune epoch={epoch:02d}, "
            f"TrainFit={train_loss:.5f}, TrainTuneNLL={tune_nll:.5f}"
        )
        if tune_nll < best_tune - 1e-5:
            best_tune = tune_nll; best_epoch = epoch; patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= PATIENCE:
            break
    if best_epoch <= 0:
        raise RuntimeError(f"No selected epoch for {model_name}.")
    return TuneResult(best_epoch, best_tune, pd.DataFrame(rows))

def refit_graph_neural_model(
    model: nn.Module, model_name: str, graph_by_date: np.ndarray, epochs: int
) -> RefitResult:
    frame = graph_samples[graph_samples["Split"] == "Train"]
    loader = make_graph_loader(frame, graph_by_date, True)
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    rows = []
    for epoch in range(1, int(epochs) + 1):
        loss = _train_epoch_graph(
            model, loader, optimizer, FULL_HAZARD_POS_WEIGHT
        )
        rows.append({
            "Model": model_name, "Stage": "FullTrainRefit", "Epoch": epoch,
            "FullTrainWeightedLoss": loss,
        })
        log(f"{model_name}: refit epoch={epoch:02d}/{epochs}, FullTrain={loss:.5f}")
    state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return RefitResult(state, pd.DataFrame(rows))

# =============================================================================
# 15. NEURAL PREDICTION / DEDICATED CALIBRATION
# =============================================================================

def predict_sector_logits(
    model: nn.Module,
    split_name: str,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    frame = sector_samples[
        sector_samples["Split"] == split_name
    ].copy()

    loader = make_sector_loader(
        frame,
        shuffle=False,
    )

    model = model.to(DEVICE)
    model.eval()

    logits_list = []
    y_list = []
    mask_list = []
    t_list = []
    s_list = []

    cursor = 0

    with torch.no_grad():
        for x, sector_idx, y, mask, t in loader:

            x = x.to(DEVICE)
            sector_idx_device = sector_idx.to(DEVICE)

            logits = model(
                x,
                sector_idx_device,
            ).cpu().numpy()

            logits_list.append(logits)
            y_list.append(y.numpy())
            mask_list.append(mask.numpy())
            t_list.append(t.numpy())
            s_list.append(sector_idx.numpy())
            cursor += len(t)

    return (
        np.concatenate(logits_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(mask_list, axis=0),
        np.column_stack([
            np.concatenate(t_list),
            np.concatenate(s_list),
        ]),
    )

def predict_graph_logits(
    model: nn.Module,
    split_name: str,
    graph_by_date: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    frame = graph_samples[
        graph_samples["Split"] == split_name
    ].copy()

    loader = make_graph_loader(
        frame,
        graph_by_date,
        shuffle=False,
    )

    model = model.to(DEVICE)
    model.eval()

    logits_list = []
    y_list = []
    mask_list = []
    t_list = []

    with torch.no_grad():
        for x, graph_prior, y, mask, t in loader:

            x = x.to(DEVICE)
            graph_prior = graph_prior.to(DEVICE)

            logits = model(
                x,
                graph_prior,
            ).cpu().numpy()

            logits_list.append(logits)
            y_list.append(y.numpy())
            mask_list.append(mask.numpy())
            t_list.append(t.numpy())

    return (
        np.concatenate(logits_list, axis=0),
        np.concatenate(y_list, axis=0),
        np.concatenate(mask_list, axis=0),
        np.concatenate(t_list),
    )

def fit_hazard_temperature(
    logits: np.ndarray,
    targets: np.ndarray,
    mask: np.ndarray,
) -> tuple[float, float]:

    valid = mask > 0

    z = logits[valid].astype(float)
    y = targets[valid].astype(float)

    if len(y) == 0:
        return 1.0, 0.0

    def objective(theta):
        log_temperature, bias = theta
        temperature = np.exp(log_temperature)

        zc = z / temperature + bias
        p = 1.0 / (
            1.0 + np.exp(-np.clip(zc, -30, 30))
        )
        p = np.clip(p, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(p)
                + (1.0 - y) * np.log1p(-p)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 0.0]),
        method="L-BFGS-B",
        bounds=[
            (-3.0, 3.0),
            (-5.0, 5.0),
        ],
    )

    if not result.success:
        log(
            f"WARNING: hazard calibration optimizer: {result.message}"
        )

    temperature = float(
        np.exp(result.x[0])
    )
    bias = float(result.x[1])

    return temperature, bias

# =============================================================================
# 16. XGBOOST FEATURE ENGINEERING
# =============================================================================

# Summary windows reduce the 60-day sequence to robust tabular statistics.
XGB_WINDOWS = [5, 22, 60]

def xgb_feature_vector(t: int, s: int, panel: np.ndarray) -> np.ndarray:
    sequence = panel[
        t - LOOKBACK + 1:t + 1,
        s,
        :,
    ]

    pieces = [
        sequence[-1],  # current state
    ]

    for window in XGB_WINDOWS:
        block = sequence[-window:]
        pieces.extend([
            block.mean(axis=0),
            block.std(axis=0),
            block.min(axis=0),
            block.max(axis=0),
        ])

    # Sector one-hot.
    one_hot = np.zeros(S, dtype=np.float32)
    one_hot[s] = 1.0
    pieces.append(one_hot)

    return np.concatenate(pieces).astype(np.float32)

log("Building XGBoost summary-feature matrix...")

X_xgb_inner = np.vstack([
    xgb_feature_vector(int(row.t), int(row.s), X_panel_inner)
    for row in sector_samples.itertuples(index=False)
])
X_xgb = np.vstack([
    xgb_feature_vector(int(row.t), int(row.s), X_panel)
    for row in sector_samples.itertuples(index=False)
])

xgb_split = sector_samples["Split"].to_numpy()
xgb_revision_phase = sector_samples["RevisionPhase"].to_numpy()
xgb_t = sector_samples["t"].to_numpy(dtype=int)
xgb_s = sector_samples["s"].to_numpy(dtype=int)

# =============================================================================
# 17. XGBOOST INNER TUNING -> FULL-TRAIN REFIT -> CALIBRATION
# =============================================================================

def fit_binary_platt(
    p_validation: np.ndarray,
    y_validation: np.ndarray,
) -> tuple[float, float]:

    p_validation = np.clip(
        p_validation,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p_validation / (1.0 - p_validation)
    )
    y = y_validation.astype(float)

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-10.0, 10.0),
            (0.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def apply_binary_platt(
    p: np.ndarray,
    intercept: float,
    slope: float,
) -> np.ndarray:

    p = np.clip(p, EPS, 1.0 - EPS)
    x = np.log(p / (1.0 - p))
    z = intercept + slope * x

    return 1.0 / (
        1.0 + np.exp(-np.clip(z, -30, 30))
    )

xgb_models = {}
xgb_calibration = {}
xgb_predictions = {"Validation": {}, "Test": {}}
xgb_raw_predictions = {"Validation": {}, "Test": {}}
xgb_tuning_rows = []

for h in HORIZONS:
    log(f"Tuning/refitting XGBoost horizon={h}...")

    y_inner_all = np.array([
        inner_horizon_target[h][t, s]
        for t, s in zip(xgb_t, xgb_s)
    ], dtype=float)
    y_final_all = np.array([
        horizon_target[h][t, s]
        for t, s in zip(xgb_t, xgb_s)
    ], dtype=float)

    fit_idx = (xgb_revision_phase == "TrainFit") & np.isfinite(y_inner_all)
    tune_idx = (xgb_revision_phase == "TrainTune") & np.isfinite(y_inner_all)
    full_train_idx = (xgb_split == "Train") & np.isfinite(y_final_all)
    cal_idx = (xgb_split == "Validation") & np.isfinite(y_final_all)
    test_idx = (xgb_split == "Test") & np.isfinite(y_final_all)

    y_fit = y_inner_all[fit_idx].astype(int)
    y_tune = y_inner_all[tune_idx].astype(int)
    y_full = y_final_all[full_train_idx].astype(int)
    y_cal = y_final_all[cal_idx].astype(int)

    pos_fit = max(int(y_fit.sum()), 1)
    neg_fit = max(len(y_fit) - pos_fit, 1)
    tune_pos_weight = min(math.sqrt(neg_fit / pos_fit), XGB_MAX_POS_WEIGHT)

    params_tune = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "scale_pos_weight": tune_pos_weight,
        "seed": SEED,
        "nthread": max(1, os.cpu_count() or 1),
        "tree_method": "hist",
    }

    dfit = xgb.DMatrix(X_xgb_inner[fit_idx], label=y_fit)
    dtune = xgb.DMatrix(X_xgb_inner[tune_idx], label=y_tune)
    tune_booster = xgb.train(
        params=params_tune,
        dtrain=dfit,
        num_boost_round=XGB_MAX_ROUNDS,
        evals=[(dtune, "train_tune")],
        early_stopping_rounds=XGB_EARLY_STOP,
        verbose_eval=False,
    )
    best_iteration = int(
        tune_booster.best_iteration
        if tune_booster.best_iteration is not None
        else XGB_MAX_ROUNDS - 1
    )
    selected_rounds = best_iteration + 1

    pos_full = max(int(y_full.sum()), 1)
    neg_full = max(len(y_full) - pos_full, 1)
    full_pos_weight = min(math.sqrt(neg_full / pos_full), XGB_MAX_POS_WEIGHT)
    params_final = dict(params_tune)
    params_final["scale_pos_weight"] = full_pos_weight

    dfull = xgb.DMatrix(X_xgb[full_train_idx], label=y_full)
    booster = xgb.train(
        params=params_final,
        dtrain=dfull,
        num_boost_round=selected_rounds,
        evals=[],
        verbose_eval=False,
    )

    dcal = xgb.DMatrix(X_xgb[cal_idx])
    p_cal_raw = booster.predict(dcal)
    intercept, slope = fit_binary_platt(p_cal_raw, y_cal)
    p_cal = apply_binary_platt(p_cal_raw, intercept, slope)

    dtest = xgb.DMatrix(X_xgb[test_idx])
    p_test_raw = booster.predict(dtest)
    p_test = apply_binary_platt(p_test_raw, intercept, slope)

    xgb_models[h] = booster
    xgb_calibration[h] = {
        "intercept": intercept,
        "slope": slope,
        "selected_rounds": selected_rounds,
        "best_iteration_zero_based": best_iteration,
        "trainfit_scale_pos_weight": tune_pos_weight,
        "full_train_scale_pos_weight": full_pos_weight,
        "calibration_sample_n": int(len(y_cal)),
    }
    xgb_tuning_rows.append({
        "Horizon": h,
        "SelectedRounds": selected_rounds,
        "TrainFitN": int(fit_idx.sum()),
        "TrainTuneN": int(tune_idx.sum()),
        "FullTrainN": int(full_train_idx.sum()),
        "CalibrationN": int(cal_idx.sum()),
        "TestN": int(test_idx.sum()),
    })

    for split_name, idx, p_raw, p_calibrated in [
        ("Validation", cal_idx, p_cal_raw, p_cal),
        ("Test", test_idx, p_test_raw, p_test),
    ]:
        xgb_predictions[split_name][h] = {
            "t": xgb_t[idx], "s": xgb_s[idx], "p": p_calibrated,
        }
        xgb_raw_predictions[split_name][h] = {
            "t": xgb_t[idx], "s": xgb_s[idx], "p": p_raw,
        }

    booster.save_model(MODEL_DIR / f"xgboost_h{h}.json")

pd.DataFrame(xgb_tuning_rows).to_csv(
    TABLE_DIR / "Table_D05_XGBoost_Inner_Tuning_and_Refit.csv", index=False
)
with open(MODEL_DIR / "xgboost_calibration.json", "w", encoding="utf-8") as handle:
    json.dump(xgb_calibration, handle, indent=2)

def assemble_xgb_probability_panels(prediction_store):
    panels = {
        split_name: {h: np.full((T, S), np.nan, dtype=np.float32) for h in HORIZONS}
        for split_name in ["Validation", "Test"]
    }
    for split_name in ["Validation", "Test"]:
        coordinate_set = set()
        for h in HORIZONS:
            record = prediction_store[split_name][h]
            for t, s, p in zip(record["t"], record["s"], record["p"]):
                panels[split_name][h][t, s] = p
                coordinate_set.add((int(t), int(s)))
        for t, s in coordinate_set:
            values = np.array([panels[split_name][h][t, s] for h in HORIZONS])
            if np.all(np.isfinite(values)):
                values = np.maximum.accumulate(values)
                for h, value in zip(HORIZONS, values):
                    panels[split_name][h][t, s] = value
    return panels

xgb_panel_probs = assemble_xgb_probability_panels(xgb_predictions)
xgb_panel_raw_probs = assemble_xgb_probability_panels(xgb_raw_predictions)
log("XGBoost nested tuning, full-Train refit, and Calibration-only Platt scaling complete.")

# =============================================================================
# 18. PHASE 2.2 — MULTI-SEED ROBUSTNESS DESIGN
# =============================================================================
#
# Architecture is FROZEN at the accepted Phase-2.1 specification.
# This phase does NOT tune architecture using Test results.
#
# Five paired seeds are used by default. For every seed:
#   * the same seed is reset before each neural architecture is initialized;
#   * graph ablations therefore begin from paired random initializations;
#   * the random-graph placebo is independently redrawn for that seed while
#     preserving Hawkes cross-edge density and the empirical edge-weight multiset;
#   * early stopping uses TrainTune only after fitting on TrainFit;
#   * the selected epoch is refitted from a fresh initialization on full Train;
#   * temperature/bias calibration uses only the 2022-2024 Calibration segment;
#   * Test is evaluated once after the seed-specific model is locked.
#
# The ensemble prediction is the arithmetic mean of the five separately
# calibrated probability forecasts. No Test-dependent ensemble weighting is used.

DEFAULT_MULTI_SEEDS = [
    20260901,
    20260902,
    20260903,
    20260904,
    20260905,
]

_seed_env = os.getenv("NSE_MULTI_SEEDS", "").strip()

if _seed_env:
    MULTI_SEEDS = [
        int(x.strip())
        for x in _seed_env.split(",")
        if x.strip()
    ]
else:
    MULTI_SEEDS = DEFAULT_MULTI_SEEDS.copy()

if len(MULTI_SEEDS) < 3:
    raise RuntimeError(
        "Phase 2.2 requires at least three independent seeds; "
        "five are recommended and used by default."
    )

if len(set(MULTI_SEEDS)) != len(MULTI_SEEDS):
    raise RuntimeError("MULTI_SEEDS contains duplicates.")

N_SEEDS = len(MULTI_SEEDS)

log(
    f"Phase 2.2 seeds ({N_SEEDS}): {MULTI_SEEDS}"
)

if DEVICE.type == "cpu":
    log(
        "NOTE: 30 seed-architecture combinations will each run an inner-tuning stage "
        "and a fresh full-Train refit. Google Colab GPU is strongly recommended."
    )

def set_all_seeds(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True,
        )
    except Exception:
        pass

# =============================================================================
# 19. RANDOM-GRAPH PLACEBO GENERATOR — ONE PLACEBO PER SEED
# =============================================================================
#
# Stronger placebo than Phase 2.1:
#   * same number of directed cross-sector edges as stable Hawkes support;
#   * same empirical raw Hawkes cross-edge magnitudes (permuted across randomly
#     selected receiver/source pairs);
#   * same Train-derived magnitude transform;
#   * graph is static through time within a seed;
#   * support is independently redrawn across seeds.
#
# This tests whether the econometric topology matters beyond "having a sparse
# graph of approximately the same strength."

def build_random_graph_for_seed(
    seed: int,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:

    rng = np.random.default_rng(seed + 41011)

    all_positions = [
        (receiver, source)
        for receiver in range(S)
        for source in range(S)
        if receiver != source
    ]

    original_cross_positions = [
        (receiver, source)
        for receiver in range(S)
        for source in range(S)
        if (
            receiver != source
            and cross_support[
                receiver,
                source,
            ]
        )
    ]

    n_edges = max(
        1,
        len(original_cross_positions),
    )

    chosen = rng.choice(
        len(all_positions),
        size=n_edges,
        replace=False,
    )

    # Preserve the Hawkes self-excitation diagonal exactly. The placebo
    # randomizes only cross-sector topology, which makes Random vs Static a
    # cleaner test of whether the ECONOMETRIC cross-sector locations matter.
    train_raw = np.zeros(
        (S, S),
        dtype=np.float32,
    )
    final_raw = np.zeros(
        (S, S),
        dtype=np.float32,
    )

    diag = np.arange(S)

    train_raw[
        diag,
        diag,
    ] = np.diag(
        hawkes_train_alpha
    )

    final_raw[
        diag,
        diag,
    ] = np.diag(
        hawkes_final_alpha
    )

    train_cross_weights = np.asarray(
        [
            hawkes_train_alpha[
                receiver,
                source,
            ]
            for (
                receiver,
                source,
            ) in original_cross_positions
        ],
        dtype=np.float32,
    )

    final_cross_weights = np.asarray(
        [
            hawkes_train_alpha[
                receiver,
                source,
            ]
            for (
                receiver,
                source,
            ) in original_cross_positions
        ],
        dtype=np.float32,
    )

    if len(
        train_cross_weights
    ) == 0:

        train_cross_weights = np.array(
            [GRAPH_TRAIN_Q],
            dtype=np.float32,
        )

        final_cross_weights = np.array(
            [GRAPH_TRAIN_Q],
            dtype=np.float32,
        )

    permutation = rng.permutation(
        len(train_cross_weights)
    )

    rows = []

    for edge_number, position_index in enumerate(
        chosen
    ):

        receiver, source = all_positions[
            int(position_index)
        ]

        weight_index = int(
            permutation[
                edge_number
                % len(permutation)
            ]
        )

        train_weight = float(
            train_cross_weights[
                weight_index
            ]
        )

        final_weight = float(
            final_cross_weights[
                weight_index
            ]
        )

        train_raw[
            receiver,
            source,
        ] = train_weight

        final_raw[
            receiver,
            source,
        ] = final_weight

        rows.append({
            "Seed": seed,
            "EdgeNumber": edge_number + 1,
            "FromSector": sectors[source],
            "ToSector": sectors[receiver],
            "TrainRawPlaceboWeight": train_weight,
            "FrozenRawPlaceboWeight": final_weight,
            "PreservesHawkesDiagonal": "YES",
        })

    train_scaled = scale_graph_magnitude(
        train_raw
    )

    final_scaled = scale_graph_magnitude(
        final_raw
    )

    if not np.allclose(train_scaled, final_scaled, atol=1e-12, rtol=0):
        raise RuntimeError("Random placebo train/final weights should be identical under frozen Hawkes policy.")
    by_date = np.repeat(train_scaled[None, :, :], T, axis=0)

    return (
        by_date,
        train_scaled,
        pd.DataFrame(rows),
    )

# =============================================================================
# 20. COMMON FORECAST METRICS
# =============================================================================

def initialize_probability_dict():
    return {
        split_name: {
            h: np.full(
                (T, S),
                np.nan,
                dtype=np.float32,
            )
            for h in HORIZONS
        }
        for split_name in [
            "Validation",
            "Test",
        ]
    }

def log_score(
    y: np.ndarray,
    p: np.ndarray,
) -> float:

    p = np.clip(
        p,
        EPS,
        1.0 - EPS,
    )

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def fit_calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:

    p = np.clip(
        p,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p / (1.0 - p)
    )

    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x

        q = 1.0 / (
            1.0
            + np.exp(
                -np.clip(
                    z,
                    -30,
                    30,
                )
            )
        )

        q = np.clip(
            q,
            EPS,
            1.0 - EPS,
        )

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y)
                * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array(
            [0.0, 1.0]
        ),
        method="L-BFGS-B",
        bounds=[
            (-20.0, 20.0),
            (-10.0, 10.0),
        ],
    )

    return (
        float(result.x[0]),
        float(result.x[1]),
    )

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(
        y_true,
        dtype=float,
    )
    probability = np.asarray(
        probability,
        dtype=float,
    )

    ok = (
        np.isfinite(y_true)
        & np.isfinite(probability)
    )

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": int(len(y)),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": (
            float(y.mean())
            if len(y)
            else np.nan
        ),
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
        "CalibrationIntercept": np.nan,
        "CalibrationSlope": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean(
            (p - y) ** 2
        )
    )

    result["LogScore"] = log_score(
        y,
        p,
    )

    if len(np.unique(y)) == 2:

        result["PR_AUC"] = float(
            average_precision_score(
                y,
                p,
            )
        )

        result["ROC_AUC"] = float(
            roc_auc_score(
                y,
                p,
            )
        )

        intercept, slope = (
            fit_calibration_intercept_slope(
                y,
                p,
            )
        )

        result[
            "CalibrationIntercept"
        ] = intercept

        result[
            "CalibrationSlope"
        ] = slope

    return result

# =============================================================================
# 21. PREDICTION HELPERS
# =============================================================================

def calibrated_sector_predictions(
    model: nn.Module,
) -> tuple[dict, dict, dict]:
    # Legacy split label "Validation" is the dedicated Calibration segment.
    cal_logits, cal_y, cal_mask, _ = predict_sector_logits(model, "Validation")
    temperature, bias = fit_hazard_temperature(cal_logits, cal_y, cal_mask)
    probabilities = initialize_probability_dict()
    raw_probabilities = initialize_probability_dict()

    for split_name in ["Validation", "Test"]:
        logits, _, _, coordinates = predict_sector_logits(model, split_name)
        calibrated = hazard_logits_to_horizon_probs(logits, temperature, bias)
        raw = hazard_logits_to_horizon_probs(logits, 1.0, 0.0)
        for row_index, (t, s) in enumerate(coordinates.astype(int)):
            for h in HORIZONS:
                probabilities[split_name][h][t, s] = calibrated[h][row_index]
                raw_probabilities[split_name][h][t, s] = raw[h][row_index]

    calibration = {
        "temperature": temperature,
        "bias": bias,
        "CalibrationLegacySplitLabel": "Validation",
    }
    return probabilities, raw_probabilities, calibration

def calibrated_graph_predictions(
    model: nn.Module,
    graph_array: np.ndarray,
) -> tuple[dict, dict, dict]:
    cal_logits, cal_y, cal_mask, _ = predict_graph_logits(
        model, "Validation", graph_array
    )
    temperature, bias = fit_hazard_temperature(cal_logits, cal_y, cal_mask)
    eta = float(F.softplus(model.graph_attention.raw_eta.detach().cpu()).item())
    probabilities = initialize_probability_dict()
    raw_probabilities = initialize_probability_dict()

    for split_name in ["Validation", "Test"]:
        logits, _, _, t_values = predict_graph_logits(model, split_name, graph_array)
        calibrated = hazard_logits_to_horizon_probs(logits, temperature, bias)
        raw = hazard_logits_to_horizon_probs(logits, 1.0, 0.0)
        for row_index, t in enumerate(t_values.astype(int)):
            for s in range(S):
                for h in HORIZONS:
                    probabilities[split_name][h][t, s] = calibrated[h][row_index, s]
                    raw_probabilities[split_name][h][t, s] = raw[h][row_index, s]

    calibration = {
        "temperature": temperature,
        "bias": bias,
        "learned_graph_eta": eta,
        "CalibrationLegacySplitLabel": "Validation",
    }
    return probabilities, raw_probabilities, calibration

# =============================================================================
# 22. FIVE-SEED NEURAL ESTIMATION: INNER TUNE -> FULL-TRAIN REFIT -> CALIBRATE
# =============================================================================

NEURAL_MODEL_NAMES = [
    "LSTM_Survival",
    "TemporalTransformer_Survival",
    "NoGraphTransformer",
    "RandomGraphTransformer",
    "StaticHawkesGraphTransformer",
    "DynamicHawkesGraphTransformer",
]

def parameter_count(model: nn.Module) -> int:
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

seed_probability_store = {}
seed_raw_probability_store = {}
training_histories = []
best_epoch_rows = []
calibration_rows = []
random_placebo_rows = []
parameter_count_rows = []

for seed_index, seed in enumerate(MULTI_SEEDS, start=1):
    log("=" * 90)
    log(f"SEED {seed_index}/{N_SEEDS}: {seed}")
    seed_probability_store[seed] = {}
    seed_raw_probability_store[seed] = {}
    seed_dir = MODEL_DIR / f"Seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)

    # 22.1 LSTM
    set_all_seeds(seed)
    tune_model = LSTMSurvival(feature_dim=F_DIM, n_sectors=S)
    parameter_count_rows.append({
        "Seed": seed, "Model": "LSTM_Survival",
        "TrainableParameters": parameter_count(tune_model),
    })
    tune_result = tune_sector_neural_model(
        tune_model, f"LSTM_Survival__Seed_{seed}"
    )
    del tune_model
    set_all_seeds(seed)
    model = LSTMSurvival(feature_dim=F_DIM, n_sectors=S)
    refit_result = refit_sector_neural_model(
        model, f"LSTM_Survival__Seed_{seed}", tune_result.best_epoch
    )
    model.load_state_dict(refit_result.state)
    torch.save(refit_result.state, seed_dir / "LSTM_Survival_final.pt")
    probs, raw_probs, calibration = calibrated_sector_predictions(model)
    seed_probability_store[seed]["LSTM_Survival"] = probs
    seed_raw_probability_store[seed]["LSTM_Survival"] = raw_probs
    for frame in [tune_result.history, refit_result.history]:
        h = frame.copy(); h["BaseModel"] = "LSTM_Survival"; h["Seed"] = seed
        training_histories.append(h)
    best_epoch_rows.append({
        "Seed": seed, "Model": "LSTM_Survival",
        "BestEpoch": tune_result.best_epoch,
        "BestTrainTuneSurvivalNLL": tune_result.best_tune_nll,
    })
    calibration_rows.append({"Seed": seed, "Model": "LSTM_Survival", **calibration})
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # 22.2 Temporal Transformer
    set_all_seeds(seed)
    tune_model = TemporalTransformerSurvival(feature_dim=F_DIM, n_sectors=S)
    parameter_count_rows.append({
        "Seed": seed, "Model": "TemporalTransformer_Survival",
        "TrainableParameters": parameter_count(tune_model),
    })
    tune_result = tune_sector_neural_model(
        tune_model, f"TemporalTransformer_Survival__Seed_{seed}"
    )
    del tune_model
    set_all_seeds(seed)
    model = TemporalTransformerSurvival(feature_dim=F_DIM, n_sectors=S)
    refit_result = refit_sector_neural_model(
        model, f"TemporalTransformer_Survival__Seed_{seed}", tune_result.best_epoch
    )
    model.load_state_dict(refit_result.state)
    torch.save(refit_result.state, seed_dir / "TemporalTransformer_Survival_final.pt")
    probs, raw_probs, calibration = calibrated_sector_predictions(model)
    seed_probability_store[seed]["TemporalTransformer_Survival"] = probs
    seed_raw_probability_store[seed]["TemporalTransformer_Survival"] = raw_probs
    for frame in [tune_result.history, refit_result.history]:
        h = frame.copy(); h["BaseModel"] = "TemporalTransformer_Survival"; h["Seed"] = seed
        training_histories.append(h)
    best_epoch_rows.append({
        "Seed": seed, "Model": "TemporalTransformer_Survival",
        "BestEpoch": tune_result.best_epoch,
        "BestTrainTuneSurvivalNLL": tune_result.best_tune_nll,
    })
    calibration_rows.append({
        "Seed": seed, "Model": "TemporalTransformer_Survival", **calibration
    })
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # 22.3 Random placebo graph for this seed
    random_graph_seed, random_static_seed, placebo_edges_seed = build_random_graph_for_seed(seed)
    random_placebo_rows.append(placebo_edges_seed)
    graph_variants_seed = {
        "NoGraphTransformer": zero_graph_by_date,
        "RandomGraphTransformer": random_graph_seed,
        "StaticHawkesGraphTransformer": static_graph_by_date,
        "DynamicHawkesGraphTransformer": dynamic_graph_scaled,
    }

    # 22.4 Graph ablations
    for model_name, graph_array in graph_variants_seed.items():
        set_all_seeds(seed)
        tune_model = GraphSurvivalTransformer(feature_dim=F_DIM, n_sectors=S)
        parameter_count_rows.append({
            "Seed": seed, "Model": model_name,
            "TrainableParameters": parameter_count(tune_model),
        })
        tune_result = tune_graph_neural_model(
            tune_model, f"{model_name}__Seed_{seed}", graph_array
        )
        del tune_model

        # Fresh paired initialization for the full-Train refit.
        set_all_seeds(seed)
        model = GraphSurvivalTransformer(feature_dim=F_DIM, n_sectors=S)
        refit_result = refit_graph_neural_model(
            model, f"{model_name}__Seed_{seed}", graph_array, tune_result.best_epoch
        )
        model.load_state_dict(refit_result.state)
        torch.save(refit_result.state, seed_dir / f"{model_name}_final.pt")
        probs, raw_probs, calibration = calibrated_graph_predictions(model, graph_array)
        seed_probability_store[seed][model_name] = probs
        seed_raw_probability_store[seed][model_name] = raw_probs
        for frame in [tune_result.history, refit_result.history]:
            h = frame.copy(); h["BaseModel"] = model_name; h["Seed"] = seed
            training_histories.append(h)
        best_epoch_rows.append({
            "Seed": seed, "Model": model_name,
            "BestEpoch": tune_result.best_epoch,
            "BestTrainTuneSurvivalNLL": tune_result.best_tune_nll,
        })
        calibration_rows.append({"Seed": seed, "Model": model_name, **calibration})
        del model; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

training_history = pd.concat(
    training_histories,
    ignore_index=True,
)

best_epochs = pd.DataFrame(
    best_epoch_rows
)

neural_calibration_seeds = pd.DataFrame(
    calibration_rows
)

random_placebo_edges = pd.concat(
    random_placebo_rows,
    ignore_index=True,
)

parameter_counts = (
    pd.DataFrame(parameter_count_rows)
    .groupby("Model")["TrainableParameters"]
    .agg(["min", "max"])
    .reset_index()
)
parameter_counts["InvariantAcrossSeeds"] = parameter_counts["min"] == parameter_counts["max"]
parameter_counts = parameter_counts.rename(
    columns={"min": "TrainableParameters", "max": "MaxTrainableParameters"}
)
parameter_counts.to_csv(
    TABLE_DIR / "Table_R00_Neural_Parameter_Counts.csv", index=False
)

# Save raw and calibrated neural probabilities at observation level so all
# calibration diagnostics can be reproduced without retraining.
raw_cal_rows = []
for seed in MULTI_SEEDS:
    for model_name in NEURAL_MODEL_NAMES:
        for split_name, split_mask in [
            ("Validation", calibration_date_mask),
            ("Test", test_date_mask),
        ]:
            display_split = "Calibration" if split_name == "Validation" else "Test"
            for h in HORIZONS:
                raw_mat = seed_raw_probability_store[seed][model_name][split_name][h]
                cal_mat = seed_probability_store[seed][model_name][split_name][h]
                y_mat = horizon_target[h]
                tt, ss = np.where(
                    split_mask[:, None]
                    & np.isfinite(raw_mat)
                    & np.isfinite(cal_mat)
                    & np.isfinite(y_mat)
                )
                for t_i, s_i in zip(tt, ss):
                    raw_cal_rows.append({
                        "Seed": seed, "Model": model_name,
                        "Split": display_split, "Horizon": h,
                        "Date": dates[t_i], "Sector": sectors[s_i],
                        "Y": float(y_mat[t_i, s_i]),
                        "RawProbability": float(raw_mat[t_i, s_i]),
                        "CalibratedProbability": float(cal_mat[t_i, s_i]),
                    })

neural_raw_calibrated_predictions = pd.DataFrame(raw_cal_rows)
neural_raw_calibrated_predictions.to_csv(
    DATA_DIR / "neural_raw_and_calibrated_predictions.csv.gz",
    index=False, compression="gzip"
)

# Deterministic XGBoost raw/calibrated predictions.
xgb_raw_cal_rows = []
for split_name, split_mask in [("Validation", calibration_date_mask), ("Test", test_date_mask)]:
    display_split = "Calibration" if split_name == "Validation" else "Test"
    for h in HORIZONS:
        raw_mat = xgb_panel_raw_probs[split_name][h]
        cal_mat = xgb_panel_probs[split_name][h]
        y_mat = horizon_target[h]
        tt, ss = np.where(
            split_mask[:, None]
            & np.isfinite(raw_mat)
            & np.isfinite(cal_mat)
            & np.isfinite(y_mat)
        )
        for t_i, s_i in zip(tt, ss):
            xgb_raw_cal_rows.append({
                "Model": "XGBoost", "Split": display_split, "Horizon": h,
                "Date": dates[t_i], "Sector": sectors[s_i],
                "Y": float(y_mat[t_i, s_i]),
                "RawProbability": float(raw_mat[t_i, s_i]),
                "CalibratedProbability": float(cal_mat[t_i, s_i]),
            })
pd.DataFrame(xgb_raw_cal_rows).to_csv(
    DATA_DIR / "xgboost_raw_and_calibrated_predictions.csv.gz",
    index=False, compression="gzip"
)

training_history.to_csv(
    TABLE_DIR
    / "Table_R01_MultiSeed_Training_History.csv",
    index=False,
)

best_epochs.to_csv(
    TABLE_DIR
    / "Table_R02_MultiSeed_Best_Epochs.csv",
    index=False,
)

neural_calibration_seeds.to_csv(
    TABLE_DIR
    / "Table_R03_MultiSeed_Calibration.csv",
    index=False,
)

random_placebo_edges.to_csv(
    TABLE_DIR
    / "Table_R04_Random_Graph_Placebo_Edges.csv",
    index=False,
)

# =============================================================================
# 23. PER-SEED TEST / VALIDATION METRICS
# =============================================================================

seed_metric_rows = []

for seed in MULTI_SEEDS:

    for model_name in NEURAL_MODEL_NAMES:

        probabilities = (
            seed_probability_store[
                seed
            ][model_name]
        )

        for (
            split_name,
            split_mask,
        ) in [
            (
                "Validation",
                validation_date_mask,
            ),
            (
                "Test",
                test_date_mask,
            ),
        ]:

            for h in HORIZONS:

                y_panel = (
                    horizon_target[h]
                )

                p_panel = (
                    probabilities[
                        split_name
                    ][h]
                )

                pooled = probability_metrics(
                    y_panel[
                        split_mask
                    ].reshape(-1),
                    p_panel[
                        split_mask
                    ].reshape(-1),
                )

                seed_metric_rows.append({
                    "Seed": seed,
                    "Split": split_name,
                    "Sector": "POOLED",
                    "Horizon": h,
                    "Model": model_name,
                    **pooled,
                })

                for s, sector in enumerate(
                    sectors
                ):

                    sector_metrics = (
                        probability_metrics(
                            y_panel[
                                split_mask,
                                s,
                            ],
                            p_panel[
                                split_mask,
                                s,
                            ],
                        )
                    )

                    seed_metric_rows.append({
                        "Seed": seed,
                        "Split": split_name,
                        "Sector": sector,
                        "Horizon": h,
                        "Model": model_name,
                        **sector_metrics,
                    })

seed_metrics = pd.DataFrame(
    seed_metric_rows
)

seed_metrics.to_csv(
    TABLE_DIR
    / "Table_R05_All_PerSeed_Metrics.csv",
    index=False,
)

# =============================================================================
# 24. MULTI-SEED MEAN / SD / MEDIAN SUMMARIES
# =============================================================================

pooled_test_seed = seed_metrics[
    (seed_metrics["Split"] == "Test")
    & (
        seed_metrics["Sector"]
        == "POOLED"
    )
].copy()

metric_summary = (
    pooled_test_seed
    .groupby(
        [
            "Model",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        Seeds=("Seed", "nunique"),
        BrierMean=("Brier", "mean"),
        BrierSD=("Brier", "std"),
        BrierMedian=("Brier", "median"),
        LogScoreMean=("LogScore", "mean"),
        LogScoreSD=("LogScore", "std"),
        LogScoreMedian=("LogScore", "median"),
        PR_AUCMean=("PR_AUC", "mean"),
        PR_AUCSD=("PR_AUC", "std"),
        ROC_AUCMean=("ROC_AUC", "mean"),
        ROC_AUCSD=("ROC_AUC", "std"),
        CalibrationInterceptMean=(
            "CalibrationIntercept",
            "mean",
        ),
        CalibrationInterceptSD=(
            "CalibrationIntercept",
            "std",
        ),
        CalibrationSlopeMean=(
            "CalibrationSlope",
            "mean",
        ),
        CalibrationSlopeSD=(
            "CalibrationSlope",
            "std",
        ),
    )
)

metric_summary.to_csv(
    TABLE_DIR
    / "Table_R06_MultiSeed_Pooled_Test_Mean_SD.csv",
    index=False,
)

# Sector x horizon mean/SD across seeds.
sector_test_seed = seed_metrics[
    (seed_metrics["Split"] == "Test")
    & (
        seed_metrics["Sector"]
        != "POOLED"
    )
].copy()

sector_seed_summary = (
    sector_test_seed
    .groupby(
        [
            "Model",
            "Sector",
            "Horizon",
        ],
        as_index=False,
    )
    .agg(
        BrierMean=("Brier", "mean"),
        BrierSD=("Brier", "std"),
        LogScoreMean=("LogScore", "mean"),
        LogScoreSD=("LogScore", "std"),
        PR_AUCMean=("PR_AUC", "mean"),
        PR_AUCSD=("PR_AUC", "std"),
        ROC_AUCMean=("ROC_AUC", "mean"),
        ROC_AUCSD=("ROC_AUC", "std"),
    )
)

sector_seed_summary.to_csv(
    TABLE_DIR
    / "Table_R07_Sector_Horizon_MultiSeed_Summary.csv",
    index=False,
)

# =============================================================================
# 25. PAIRED-SEED GRAPH ABLATION DIFFERENCES
# =============================================================================

paired_rows = []

comparison_models = [
    "NoGraphTransformer",
    "RandomGraphTransformer",
    "StaticHawkesGraphTransformer",
]

for h in HORIZONS:

    proposed = pooled_test_seed[
        (
            pooled_test_seed["Model"]
            == PROPOSED_MODEL
        )
        & (
            pooled_test_seed["Horizon"]
            == h
        )
    ][
        [
            "Seed",
            "Brier",
            "LogScore",
            "PR_AUC",
            "ROC_AUC",
        ]
    ].rename(
        columns={
            "Brier": "DynamicBrier",
            "LogScore": "DynamicLogScore",
            "PR_AUC": "DynamicPR_AUC",
            "ROC_AUC": "DynamicROC_AUC",
        }
    )

    for comparison in comparison_models:

        comp = pooled_test_seed[
            (
                pooled_test_seed["Model"]
                == comparison
            )
            & (
                pooled_test_seed["Horizon"]
                == h
            )
        ][
            [
                "Seed",
                "Brier",
                "LogScore",
                "PR_AUC",
                "ROC_AUC",
            ]
        ].rename(
            columns={
                "Brier": "ComparisonBrier",
                "LogScore": "ComparisonLogScore",
                "PR_AUC": "ComparisonPR_AUC",
                "ROC_AUC": "ComparisonROC_AUC",
            }
        )

        merged = proposed.merge(
            comp,
            on="Seed",
            how="inner",
            validate="one_to_one",
        )

        for row in merged.itertuples(
            index=False
        ):

            paired_rows.append({
                "Seed": row.Seed,
                "Horizon": h,
                "Comparison": comparison,
                "BrierDiff_DynamicMinusComparison": (
                    row.DynamicBrier
                    - row.ComparisonBrier
                ),
                "LogScoreDiff_DynamicMinusComparison": (
                    row.DynamicLogScore
                    - row.ComparisonLogScore
                ),
                "PR_AUCDiff_DynamicMinusComparison": (
                    row.DynamicPR_AUC
                    - row.ComparisonPR_AUC
                ),
                "ROC_AUCDiff_DynamicMinusComparison": (
                    row.DynamicROC_AUC
                    - row.ComparisonROC_AUC
                ),
            })

paired_seed_differences = pd.DataFrame(
    paired_rows
)

paired_seed_differences.to_csv(
    TABLE_DIR
    / "Table_R08_Paired_Seed_Graph_Differences.csv",
    index=False,
)

paired_seed_summary = (
    paired_seed_differences
    .groupby(
        [
            "Horizon",
            "Comparison",
        ],
        as_index=False,
    )
    .agg(
        Seeds=(
            "Seed",
            "nunique",
        ),
        MeanBrierDiff=(
            "BrierDiff_DynamicMinusComparison",
            "mean",
        ),
        SDBrierDiff=(
            "BrierDiff_DynamicMinusComparison",
            "std",
        ),
        DynamicBetter_Brier_SeedShare=(
            "BrierDiff_DynamicMinusComparison",
            lambda x: float(
                np.mean(
                    np.asarray(x)
                    < 0
                )
            ),
        ),
        MeanLogScoreDiff=(
            "LogScoreDiff_DynamicMinusComparison",
            "mean",
        ),
        SDLogScoreDiff=(
            "LogScoreDiff_DynamicMinusComparison",
            "std",
        ),
        DynamicBetter_LogScore_SeedShare=(
            "LogScoreDiff_DynamicMinusComparison",
            lambda x: float(
                np.mean(
                    np.asarray(x)
                    < 0
                )
            ),
        ),
        MeanPR_AUCDiff=(
            "PR_AUCDiff_DynamicMinusComparison",
            "mean",
        ),
        MeanROC_AUCDiff=(
            "ROC_AUCDiff_DynamicMinusComparison",
            "mean",
        ),
    )
)

paired_seed_summary.to_csv(
    TABLE_DIR
    / "Table_R09_Paired_Seed_Graph_Summary.csv",
    index=False,
)

# =============================================================================
# 26. EQUAL-WEIGHT FIVE-SEED ENSEMBLE PREDICTIONS
# =============================================================================

ensemble_probabilities = {}

for model_name in NEURAL_MODEL_NAMES:

    ensemble_probabilities[
        model_name
    ] = initialize_probability_dict()

    for split_name in [
        "Validation",
        "Test",
    ]:

        for h in HORIZONS:

            stack = np.stack(
                [
                    seed_probability_store[
                        seed
                    ][
                        model_name
                    ][
                        split_name
                    ][h]
                    for seed in MULTI_SEEDS
                ],
                axis=0,
            )

            ensemble_probabilities[
                model_name
            ][
                split_name
            ][h] = np.nanmean(
                stack,
                axis=0,
            ).astype(np.float32)

# Deterministic XGBoost from the frozen Phase-2.1 specification.
ensemble_probabilities[
    "XGBoost"
] = xgb_panel_probs

# Historical / Hawkes baselines from Phase-1 master.
def master_probability_panel(
    column: str,
) -> np.ndarray:

    out = np.full(
        (T, S),
        np.nan,
        dtype=np.float32,
    )

    for row in master[
        [
            "Date",
            "Sector",
            column,
        ]
    ].itertuples(index=False):

        t = date_to_idx[
            pd.Timestamp(
                row.Date
            )
        ]

        s = sector_to_idx[
            row.Sector
        ]

        value = getattr(
            row,
            column,
        )

        if pd.notna(value):
            out[
                t,
                s,
            ] = float(value)

    return out

for baseline_name, prefix_name in [
    (
        "FixedHistorical",
        "FixedHistoricalProb",
    ),
    (
        "ExpandingHistorical",
        "ExpandingHistoricalProb",
    ),
    (
        "StableHawkes",
        "StableHawkesProb",
    ),
]:

    ensemble_probabilities[
        baseline_name
    ] = initialize_probability_dict()

    for h in HORIZONS:

        panel = master_probability_panel(
            f"{prefix_name}_{h}"
        )

        ensemble_probabilities[
            baseline_name
        ][
            "Validation"
        ][h] = panel

        ensemble_probabilities[
            baseline_name
        ][
            "Test"
        ][h] = panel

# =============================================================================
# 27. ENSEMBLE FORECAST METRICS
# =============================================================================

ensemble_metric_rows = []

for (
    split_name,
    split_mask,
) in [
    (
        "Validation",
        validation_date_mask,
    ),
    (
        "Test",
        test_date_mask,
    ),
]:

    for (
        model_name,
        split_dict,
    ) in ensemble_probabilities.items():

        for h in HORIZONS:

            y_panel = horizon_target[h]
            p_panel = (
                split_dict[
                    split_name
                ][h]
            )

            pooled = probability_metrics(
                y_panel[
                    split_mask
                ].reshape(-1),
                p_panel[
                    split_mask
                ].reshape(-1),
            )

            ensemble_metric_rows.append({
                "Split": split_name,
                "Sector": "POOLED",
                "Horizon": h,
                "Model": model_name,
                "ForecastType": (
                    "FiveSeedMean"
                    if model_name
                    in NEURAL_MODEL_NAMES
                    else "SingleDeterministic"
                ),
                **pooled,
            })

            for s, sector in enumerate(
                sectors
            ):

                sector_metrics = probability_metrics(
                    y_panel[
                        split_mask,
                        s,
                    ],
                    p_panel[
                        split_mask,
                        s,
                    ],
                )

                ensemble_metric_rows.append({
                    "Split": split_name,
                    "Sector": sector,
                    "Horizon": h,
                    "Model": model_name,
                    "ForecastType": (
                        "FiveSeedMean"
                        if model_name
                        in NEURAL_MODEL_NAMES
                        else "SingleDeterministic"
                    ),
                    **sector_metrics,
                })

ensemble_metrics = pd.DataFrame(
    ensemble_metric_rows
)

ensemble_metrics.to_csv(
    TABLE_DIR
    / "Table_R10_Ensemble_All_Metrics.csv",
    index=False,
)

ensemble_test_pooled = (
    ensemble_metrics[
        (
            ensemble_metrics["Split"]
            == "Test"
        )
        & (
            ensemble_metrics["Sector"]
            == "POOLED"
        )
    ]
    .sort_values(
        [
            "Horizon",
            "Brier",
        ]
    )
    .reset_index(
        drop=True
    )
)

ensemble_test_pooled.to_csv(
    TABLE_DIR
    / "Table_R11_Ensemble_Main_Test_Results.csv",
    index=False,
)

# =============================================================================
# 28. HORIZON COHERENCE — EVERY SEED + ENSEMBLE
# =============================================================================

coherence_rows = []

for seed in MULTI_SEEDS:

    for model_name in NEURAL_MODEL_NAMES:

        for split_name in [
            "Validation",
            "Test",
        ]:

            stack = np.stack(
                [
                    seed_probability_store[
                        seed
                    ][model_name][
                        split_name
                    ][h]
                    for h in HORIZONS
                ],
                axis=-1,
            )

            complete = np.all(
                np.isfinite(stack),
                axis=-1,
            )

            violations = (
                np.any(
                    np.diff(
                        stack,
                        axis=-1,
                    ) < -1e-8,
                    axis=-1,
                )
                & complete
            )

            coherence_rows.append({
                "Level": "Seed",
                "Seed": seed,
                "Model": model_name,
                "Split": split_name,
                "CompleteRows": int(
                    complete.sum()
                ),
                "CoherenceViolations": int(
                    violations.sum()
                ),
            })

for model_name in NEURAL_MODEL_NAMES:

    for split_name in [
        "Validation",
        "Test",
    ]:

        stack = np.stack(
            [
                ensemble_probabilities[
                    model_name
                ][split_name][h]
                for h in HORIZONS
            ],
            axis=-1,
        )

        complete = np.all(
            np.isfinite(stack),
            axis=-1,
        )

        violations = (
            np.any(
                np.diff(
                    stack,
                    axis=-1,
                ) < -1e-8,
                axis=-1,
            )
            & complete
        )

        coherence_rows.append({
            "Level": "Ensemble",
            "Seed": np.nan,
            "Model": model_name,
            "Split": split_name,
            "CompleteRows": int(
                complete.sum()
            ),
            "CoherenceViolations": int(
                violations.sum()
            ),
        })

coherence_table = pd.DataFrame(
    coherence_rows
)

coherence_table.to_csv(
    TABLE_DIR
    / "Table_R12_MultiSeed_Horizon_Coherence.csv",
    index=False,
)

# =============================================================================
# 29. MOVING-BLOCK BOOTSTRAP — ENSEMBLE PROPOSED VS ALL BENCHMARKS
# =============================================================================

def moving_block_sample_indices(
    n_dates: int,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    selected = []

    while len(selected) < n_dates:

        if n_dates <= block_length:
            start = 0
        else:
            start = int(
                rng.integers(
                    0,
                    n_dates
                    - block_length
                    + 1,
                )
            )

        selected.extend(
            range(
                start,
                min(
                    n_dates,
                    start
                    + block_length,
                ),
            )
        )

    return np.asarray(
        selected[
            :n_dates
        ],
        dtype=int,
    )

test_indices = np.where(
    test_date_mask
)[0]

benchmark_models = [
    model_name
    for model_name
    in ensemble_probabilities
    if model_name
    != PROPOSED_MODEL
]

bootstrap_rows = []

log(
    f"Ensemble paired moving-block bootstrap: "
    f"reps={BOOTSTRAP_REPS}, block={BOOTSTRAP_BLOCK}."
)

for h in HORIZONS:

    y = horizon_target[h][
        test_indices
    ]

    p_proposed = (
        ensemble_probabilities[
            PROPOSED_MODEL
        ][
            "Test"
        ][h][
            test_indices
        ]
    )

    for benchmark in benchmark_models:

        p_benchmark = (
            ensemble_probabilities[
                benchmark
            ][
                "Test"
            ][h][
                test_indices
            ]
        )

        common = (
            np.isfinite(y)
            & np.isfinite(
                p_proposed
            )
            & np.isfinite(
                p_benchmark
            )
        )

        proposed_brier_date = np.full(
            len(test_indices),
            np.nan,
        )
        benchmark_brier_date = np.full(
            len(test_indices),
            np.nan,
        )
        proposed_log_date = np.full(
            len(test_indices),
            np.nan,
        )
        benchmark_log_date = np.full(
            len(test_indices),
            np.nan,
        )

        for local_t in range(
            len(test_indices)
        ):

            ok = common[
                local_t
            ]

            if not ok.any():
                continue

            yt = y[
                local_t,
                ok,
            ]

            pp = np.clip(
                p_proposed[
                    local_t,
                    ok,
                ],
                EPS,
                1.0 - EPS,
            )

            pb = np.clip(
                p_benchmark[
                    local_t,
                    ok,
                ],
                EPS,
                1.0 - EPS,
            )

            proposed_brier_date[
                local_t
            ] = np.mean(
                (pp - yt) ** 2
            )

            benchmark_brier_date[
                local_t
            ] = np.mean(
                (pb - yt) ** 2
            )

            proposed_log_date[
                local_t
            ] = -np.mean(
                yt * np.log(pp)
                + (1.0 - yt)
                * np.log1p(-pp)
            )

            benchmark_log_date[
                local_t
            ] = -np.mean(
                yt * np.log(pb)
                + (1.0 - yt)
                * np.log1p(-pb)
            )

        valid_dates = (
            np.isfinite(
                proposed_brier_date
            )
            & np.isfinite(
                benchmark_brier_date
            )
            & np.isfinite(
                proposed_log_date
            )
            & np.isfinite(
                benchmark_log_date
            )
        )

        pbd = proposed_brier_date[
            valid_dates
        ]
        bbd = benchmark_brier_date[
            valid_dates
        ]
        pld = proposed_log_date[
            valid_dates
        ]
        bld = benchmark_log_date[
            valid_dates
        ]

        observed_brier_diff = float(
            np.mean(
                pbd - bbd
            )
        )

        observed_log_diff = float(
            np.mean(
                pld - bld
            )
        )

        rng = np.random.default_rng(
            SEED
            + 1000 * h
            + sum(
                ord(c)
                for c in benchmark
            )
            + 92022
        )

        brier_diffs = []
        log_diffs = []

        for _ in range(
            BOOTSTRAP_REPS
        ):

            idx = (
                moving_block_sample_indices(
                    len(pbd),
                    min(
                        BOOTSTRAP_BLOCK,
                        len(pbd),
                    ),
                    rng,
                )
            )

            brier_diffs.append(
                float(
                    np.mean(
                        pbd[idx]
                        - bbd[idx]
                    )
                )
            )

            log_diffs.append(
                float(
                    np.mean(
                        pld[idx]
                        - bld[idx]
                    )
                )
            )

        brier_diffs = np.asarray(
            brier_diffs
        )
        log_diffs = np.asarray(
            log_diffs
        )

        bootstrap_rows.append({
            "Horizon": h,
            "ProposedModel": (
                PROPOSED_MODEL
            ),
            "Benchmark": benchmark,
            "BrierDiff_ProposedMinusBenchmark": (
                observed_brier_diff
            ),
            "BrierDiff_CI2.5": float(
                np.quantile(
                    brier_diffs,
                    0.025,
                )
            ),
            "BrierDiff_CI97.5": float(
                np.quantile(
                    brier_diffs,
                    0.975,
                )
            ),
            "BrierProb_ProposedBetter": float(
                np.mean(
                    brier_diffs
                    < 0
                )
            ),
            "LogScoreDiff_ProposedMinusBenchmark": (
                observed_log_diff
            ),
            "LogScoreDiff_CI2.5": float(
                np.quantile(
                    log_diffs,
                    0.025,
                )
            ),
            "LogScoreDiff_CI97.5": float(
                np.quantile(
                    log_diffs,
                    0.975,
                )
            ),
            "LogScoreProb_ProposedBetter": float(
                np.mean(
                    log_diffs
                    < 0
                )
            ),
            "BootstrapReps": BOOTSTRAP_REPS,
            "BlockLength": (
                BOOTSTRAP_BLOCK
            ),
        })

bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

bootstrap_results.to_csv(
    TABLE_DIR
    / "Table_R13_Ensemble_Paired_Block_Bootstrap.csv",
    index=False,
)

# =============================================================================
# 30. SECTOR/HORIZON ENSEMBLE HETEROGENEITY
# =============================================================================

dynamic_sector = ensemble_metrics[
    (
        ensemble_metrics["Split"]
        == "Test"
    )
    & (
        ensemble_metrics["Model"]
        == PROPOSED_MODEL
    )
    & (
        ensemble_metrics["Sector"]
        != "POOLED"
    )
].copy()

nograph_sector = ensemble_metrics[
    (
        ensemble_metrics["Split"]
        == "Test"
    )
    & (
        ensemble_metrics["Model"]
        == "NoGraphTransformer"
    )
    & (
        ensemble_metrics["Sector"]
        != "POOLED"
    )
][
    [
        "Sector",
        "Horizon",
        "Brier",
        "LogScore",
    ]
].rename(
    columns={
        "Brier": "NoGraphBrier",
        "LogScore": "NoGraphLogScore",
    }
)

heterogeneity = dynamic_sector.merge(
    nograph_sector,
    on=[
        "Sector",
        "Horizon",
    ],
    how="left",
)

heterogeneity[
    "DynamicGraph_BrierSkill_vs_NoGraph"
] = (
    1.0
    - heterogeneity[
        "Brier"
    ]
    / heterogeneity[
        "NoGraphBrier"
    ]
)

heterogeneity[
    "DynamicGraph_LogScoreImprovement_vs_NoGraph"
] = (
    heterogeneity[
        "NoGraphLogScore"
    ]
    - heterogeneity[
        "LogScore"
    ]
)

heterogeneity.to_csv(
    TABLE_DIR
    / "Table_R14_Ensemble_Sector_Horizon_Heterogeneity.csv",
    index=False,
)

# =============================================================================
# 31. SAVE ENSEMBLE + PER-SEED PREDICTION PANELS
# =============================================================================

ensemble_prediction_rows = []

for (
    split_name,
    split_mask,
) in [
    (
        "Validation",
        validation_date_mask,
    ),
    (
        "Test",
        test_date_mask,
    ),
]:

    for t in np.where(
        split_mask
    )[0]:

        for s, sector in enumerate(
            sectors
        ):

            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            for h in HORIZONS:

                row[
                    f"Target_{h}"
                ] = (
                    horizon_target[
                        h
                    ][t, s]
                )

                for model_name in (
                    ensemble_probabilities
                ):

                    row[
                        f"{model_name}__P{h}"
                    ] = (
                        ensemble_probabilities[
                            model_name
                        ][
                            split_name
                        ][h][t, s]
                    )

            ensemble_prediction_rows.append(
                row
            )

ensemble_prediction_panel = pd.DataFrame(
    ensemble_prediction_rows
)

ensemble_prediction_panel.to_csv(
    DATA_DIR
    / "phase2_2_ensemble_validation_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# Per-seed neural predictions in long form for transparent replication.
per_seed_prediction_rows = []

for seed in MULTI_SEEDS:

    for split_name, split_mask in [
        ("Validation", validation_date_mask),
        ("Test", test_date_mask),
    ]:

        for t in np.where(split_mask)[0]:

            for s, sector in enumerate(sectors):

                for h in HORIZONS:

                    row = {
                        "Seed": seed,
                        "Date": dates[t],
                        "Split": split_name,
                        "Sector": sector,
                        "Horizon": h,
                        "Target": horizon_target[h][t, s],
                    }

                    for model_name in NEURAL_MODEL_NAMES:
                        row[model_name] = (
                            seed_probability_store[
                                seed
                            ][model_name][
                                split_name
                            ][h][t, s]
                        )

                    per_seed_prediction_rows.append(
                        row
                    )

per_seed_prediction_panel = pd.DataFrame(
    per_seed_prediction_rows
)

per_seed_prediction_panel.to_csv(
    DATA_DIR
    / "phase2_2_per_seed_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 32. FIGURES
# =============================================================================

# R01 — Test Brier mean +/- one SD across seeds.
fig, ax = plt.subplots(
    figsize=(11, 6)
)

for model_name, group in (
    metric_summary
    .groupby("Model")
):

    group = group.sort_values(
        "Horizon"
    )

    ax.errorbar(
        group["Horizon"],
        group["BrierMean"],
        yerr=group["BrierSD"],
        marker="o",
        capsize=3,
        label=model_name,
    )

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Test Brier score: mean ± SD across seeds"
)
ax.set_xticks(
    HORIZONS
)
ax.set_title(
    "Neural Forecast Robustness Across Independent Random Seeds"
)
ax.legend(
    fontsize=7
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R01_MultiSeed_Brier_Mean_SD.png",
    dpi=300,
)
plt.close(fig)

# R02 — Ensemble pooled Test Brier.
fig, ax = plt.subplots(
    figsize=(11, 6)
)

for model_name, group in (
    ensemble_test_pooled
    .groupby("Model")
):

    group = group.sort_values(
        "Horizon"
    )

    ax.plot(
        group["Horizon"],
        group["Brier"],
        marker="o",
        label=model_name,
    )

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Brier score (lower is better)"
)
ax.set_xticks(
    HORIZONS
)
ax.set_title(
    "Five-Seed Ensemble and Benchmark Test Performance"
)
ax.legend(
    fontsize=7
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R02_Ensemble_Test_Brier.png",
    dpi=300,
)
plt.close(fig)

# R03 — paired seed dynamic-vs-graph-control Brier differences.
fig, ax = plt.subplots(
    figsize=(10, 6)
)

for comparison, group in (
    paired_seed_differences
    .groupby("Comparison")
):

    summary = (
        group
        .groupby(
            "Horizon",
            as_index=False,
        )
        .agg(
            Mean=(
                "BrierDiff_DynamicMinusComparison",
                "mean",
            ),
            SD=(
                "BrierDiff_DynamicMinusComparison",
                "std",
            ),
        )
        .sort_values(
            "Horizon"
        )
    )

    ax.errorbar(
        summary["Horizon"],
        summary["Mean"],
        yerr=summary["SD"],
        marker="o",
        capsize=3,
        label=comparison,
    )

ax.axhline(
    0.0,
    linewidth=1.0,
)

ax.set_xlabel(
    "Forecast horizon (trading days)"
)
ax.set_ylabel(
    "Brier difference: Dynamic Hawkes minus comparison"
)
ax.set_xticks(
    HORIZONS
)
ax.set_title(
    "Paired-Seed Graph Ablation Robustness"
)
ax.legend(
    fontsize=8
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R03_PairedSeed_Graph_Brier_Differences.png",
    dpi=300,
)
plt.close(fig)

# R04 — sector/horizon ensemble graph skill.
plot_heterogeneity = heterogeneity.pivot(
    index="Sector",
    columns="Horizon",
    values="DynamicGraph_BrierSkill_vs_NoGraph",
)

fig, ax = plt.subplots(
    figsize=(10, 7)
)

image = ax.imshow(
    plot_heterogeneity.to_numpy(),
    aspect="auto",
)

ax.set_xticks(
    range(
        len(
            plot_heterogeneity.columns
        )
    )
)
ax.set_xticklabels(
    [
        str(int(h))
        for h in
        plot_heterogeneity.columns
    ]
)
ax.set_yticks(
    range(
        len(
            plot_heterogeneity.index
        )
    )
)
ax.set_yticklabels(
    plot_heterogeneity.index
)
ax.set_xlabel(
    "Forecast horizon"
)
ax.set_ylabel(
    "Sector"
)
ax.set_title(
    "Five-Seed Ensemble Dynamic-Graph Brier Skill vs No Graph"
)

fig.colorbar(
    image,
    ax=ax,
    label="Brier skill",
)

fig.tight_layout()
fig.savefig(
    FIG_DIR
    / "Figure_R04_Ensemble_Sector_Graph_Skill.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 33. EXCEL WORKBOOK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase2_2_FiveSeed_Robustness.xlsx"
)

if (
    importlib.util.find_spec(
        "xlsxwriter"
    )
    is not None
):
    excel_engine = "xlsxwriter"
elif (
    importlib.util.find_spec(
        "openpyxl"
    )
    is not None
):
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "Target Audit": target_audit,
    "Parameter Counts": parameter_counts,
    "XGB Nested Tuning": pd.DataFrame(xgb_tuning_rows),
    "Features": pd.DataFrame({
        "Feature": model_feature_names
    }),
    "Graph Scaling": graph_scaling_audit,
    "Edge Magnitude": edge_scaling_diagnostics,
    "Best Epochs": best_epochs,
    "Calibration": neural_calibration_seeds,
    "Random Placebos": random_placebo_edges,
    "PerSeed Metrics": seed_metrics,
    "Mean SD": metric_summary,
    "Sector Mean SD": sector_seed_summary,
    "Paired Seed Diff": paired_seed_differences,
    "Paired Seed Summary": paired_seed_summary,
    "Ensemble Metrics": ensemble_metrics,
    "Ensemble Test": ensemble_test_pooled,
    "Bootstrap": bootstrap_results,
    "Sector Heterogeneity": heterogeneity,
    "Coherence": coherence_table,
}

if excel_engine is not None:

    try:

        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for (
                sheet_name,
                dataframe,
            ) in excel_tables.items():

                dataframe.to_excel(
                    writer,
                    sheet_name=(
                        sheet_name[
                            :31
                        ]
                    ),
                    index=False,
                )

            if (
                excel_engine
                == "xlsxwriter"
            ):

                workbook = writer.book

                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for (
                    _,
                    worksheet,
                ) in writer.sheets.items():

                    worksheet.freeze_panes(
                        1,
                        0,
                    )

                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )

                    worksheet.set_column(
                        0,
                        30,
                        16,
                    )

            else:

                for (
                    _,
                    worksheet,
                ) in writer.sheets.items():

                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved: "
            f"{excel_path}"
        )

    except Exception as exc:

        log(
            "WARNING: Excel export failed: "
            f"{repr(exc)}"
        )

# =============================================================================
# 34. FINAL INTEGRITY CHECKS
# =============================================================================

# Count distinct placebo support signatures across seeds.
placebo_signatures = []

for seed in MULTI_SEEDS:

    subset = (
        random_placebo_edges[
            random_placebo_edges[
                "Seed"
            ] == seed
        ][
            [
                "FromSector",
                "ToSector",
            ]
        ]
        .sort_values(
            [
                "FromSector",
                "ToSector",
            ]
        )
    )

    signature = tuple(
        map(
            tuple,
            subset.to_numpy(),
        )
    )

    placebo_signatures.append(
        signature
    )

integrity_checks = [
    (
        "Reviewer chronology phases present",
        set(np.unique(revision_phases)) == {"TrainFit", "TrainTune", "Calibration", "Test"},
    ),
    (
        "Full Train equals TrainFit plus TrainTune",
        bool(np.array_equal(full_train_date_mask, train_fit_date_mask | train_tune_date_mask)),
    ),
    (
        "Calibration is never used for epoch or XGBoost-round selection",
        bool(
            (training_history.loc[training_history["Stage"] == "InnerTuning", "TrainTuneNLL"].notna().all())
            and (pd.DataFrame(xgb_tuning_rows)["TrainTuneN"].gt(0).all())
        ),
    ),
    (
        "Every neural model/seed has a selected TrainTune epoch",
        len(best_epochs) == N_SEEDS * len(NEURAL_MODEL_NAMES)
        and best_epochs["BestEpoch"].gt(0).all(),
    ),
    (
        "Phase1 Hawkes coefficients frozen for Calibration/Test",
        bool(np.allclose(hawkes_train_alpha, hawkes_final_alpha, atol=1e-12, rtol=0)),
    ),
    (
        "Requested number of seeds completed",
        seed_metrics["Seed"].nunique()
        == N_SEEDS,
    ),
    (
        "All six neural architectures completed every seed",
        bool(
            (
                seed_metrics[
                    seed_metrics[
                        "Sector"
                    ] == "POOLED"
                ]
                .groupby(
                    [
                        "Seed",
                        "Model",
                    ]
                )
                .size()
                > 0
            ).all()
        ),
    ),
    (
        "Random placebo graph support changes across seeds",
        len(
            set(
                placebo_signatures
            )
        )
        >= min(
            2,
            N_SEEDS,
        ),
    ),
    (
        "No horizon incoherence in any seed or ensemble",
        int(
            coherence_table[
                "CoherenceViolations"
            ].sum()
        )
        == 0,
    ),
    (
        "Feature matrix finite",
        bool(
            np.all(
                np.isfinite(
                    X_panel
                )
            )
        ),
    ),
    (
        "Crash_Main event-safe imputation retained",
        crash_missing_set_to_zero,
    ),
    (
        "Neural positive weight remains capped at 3",
        HAZARD_POS_WEIGHT
        <= NEURAL_MAX_POS_WEIGHT
        + 1e-12,
    ),
    (
        "Magnitude-preserving graph scaling retained",
        bool(
            GRAPH_TRAIN_Q > 0
            and np.isfinite(
                GRAPH_TRAIN_Q
            )
        ),
    ),
    (
        "Dynamic ensemble present in final Test results",
        bool(
            (
                ensemble_test_pooled[
                    "Model"
                ]
                == PROPOSED_MODEL
            ).any()
        ),
    ),
    (
        "Both Test classes present at every horizon",
        all(
            len(
                np.unique(
                    horizon_target[h][
                        test_date_mask
                    ][
                        np.isfinite(
                            horizon_target[h][
                                test_date_mask
                            ]
                        )
                    ]
                )
            )
            == 2
            for h in HORIZONS
        ),
    ),
]

integrity_table = pd.DataFrame(
    integrity_checks,
    columns=[
        "Check",
        "Passed",
    ],
)

integrity_table.to_csv(
    TABLE_DIR
    / "Table_R15_Final_Integrity_Checks.csv",
    index=False,
)

if not integrity_table[
    "Passed"
].all():

    failed = (
        integrity_table.loc[
            ~integrity_table[
                "Passed"
            ],
            "Check",
        ]
        .tolist()
    )

    raise RuntimeError(
        "Phase 2.2 integrity checks failed: "
        f"{failed}"
    )

# =============================================================================
# 35. METADATA / ROBUSTNESS BUNDLE
# =============================================================================

metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "2.2",
    "ScriptVersion": "2.3-reviewer-revision",
    "GeneratedAt": datetime.now().isoformat(),
    "BaseSeed": SEED,
    "MultiSeeds": MULTI_SEEDS,
    "NumberOfSeeds": N_SEEDS,
    "InputPhase1Zip": INPUT_ZIP_PATH.name,
    "GraphSource": graph_source,
    "Sectors": sectors,
    "ForecastHorizons": HORIZONS,
    "LookbackTradingDays": LOOKBACK,
    "TrainFitEnd": str(TRAIN_FIT_END.date()),
    "TrainTuneStart": str(TRAIN_TUNE_START.date()),
    "TrainTuneEnd": str(TRAIN_TUNE_END.date()),
    "CalibrationStart": str(CALIBRATION_START.date()),
    "CalibrationEnd": str(CALIBRATION_END.date()),
    "TestStart": str(TEST_START.date()),
    "TuningPolicy": "TrainFit -> TrainTune; select epoch/rounds only",
    "RefitPolicy": "Fresh initialization; exact selected epoch/round count on full Train",
    "CalibrationPolicy": "2022-08-01 to 2024-07-31 only; no model selection",
    "FeatureCount": F_DIM,
    "TrainFitHazardPositiveWeight": INNER_HAZARD_POS_WEIGHT,
    "FullTrainHazardPositiveWeight": FULL_HAZARD_POS_WEIGHT,
    "HazardPositiveWeight": HAZARD_POS_WEIGHT,
    "NeuralMaxPositiveWeight": NEURAL_MAX_POS_WEIGHT,
    "XGBoostMaxPositiveWeight": XGB_MAX_POS_WEIGHT,
    "GraphScalingMethod": (
        "log1p(A/q99_train)/log(2); no row normalization"
    ),
    "GraphTrainScaleReference": GRAPH_TRAIN_Q,
    "GraphScaleClip": GRAPH_SCALE_CLIP,
    "RandomGraphPlacebo": (
        "Independent support per seed; same cross-edge density "
        "and permuted Hawkes raw edge-weight multiset"
    ),
    "PairedGraphInitialization": (
        "Same seed reset before NoGraph/Random/Static/Dynamic "
        "model initialization within each seed"
    ),
    "EnsembleMethod": (
        "Unweighted arithmetic mean of individually "
        "Calibration-only seed probabilities after TrainFit/TrainTune epoch selection and full-Train refit"
    ),
    "ProposedModel": PROPOSED_MODEL,
    "BootstrapReps": BOOTSTRAP_REPS,
    "BootstrapBlockLength": BOOTSTRAP_BLOCK,
    "CalibrationUsedForModelSelection": False,
    "TestUsedForArchitectureTuning": False,
}

with open(
    DATA_DIR
    / "python_phase2_2_metadata.json",
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        metadata,
        handle,
        indent=2,
        default=str,
    )

robustness_bundle = {
    "metadata": metadata,
    "best_epochs": best_epochs.to_dict(
        orient="records"
    ),
    "calibration": neural_calibration_seeds.to_dict(
        orient="records"
    ),
    "metric_summary": metric_summary.to_dict(
        orient="records"
    ),
    "paired_seed_summary": paired_seed_summary.to_dict(
        orient="records"
    ),
    "model_feature_names": model_feature_names,
    "sectors": sectors,
}

with open(
    MODEL_DIR
    / "phase2_2_robustness_bundle.pkl",
    "wb",
) as handle:

    pickle.dump(
        robustness_bundle,
        handle,
    )

# =============================================================================
# 35.5 REVIEWER-REVISION OUTPUT GUARD
# =============================================================================
required_revision_outputs = [
    TABLE_DIR / "Table_D01_Split_Aware_Target_Audit.csv",
    TABLE_DIR / "Table_D05_XGBoost_Inner_Tuning_and_Refit.csv",
    TABLE_DIR / "Table_R00_Neural_Parameter_Counts.csv",
    TABLE_DIR / "Table_R01_MultiSeed_Training_History.csv",
    TABLE_DIR / "Table_R02_MultiSeed_Best_Epochs.csv",
    DATA_DIR / "neural_raw_and_calibrated_predictions.csv.gz",
    DATA_DIR / "xgboost_raw_and_calibrated_predictions.csv.gz",
    DATA_DIR / "python_phase2_2_metadata.json",
]
missing_revision_outputs = [
    str(path) for path in required_revision_outputs if not path.exists()
]
if missing_revision_outputs:
    raise RuntimeError(
        "Reviewer-revision output guard failed; missing: "
        + " | ".join(missing_revision_outputs)
    )

# =============================================================================
# 36. CLEAN / ZIP / CONSOLE SUMMARY
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(
        EXTRACT_DIR
    )

zip_output = (
    ZIP_DIR
    / "Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in OUTPUT_ROOT.rglob("*"):

        if (
            path.is_file()
            and path
            != zip_output
        ):

            archive.write(
                path,
                arcname=path.relative_to(
                    OUTPUT_ROOT
                ),
            )

log(
    f"All Phase-2.3 reviewer-revision outputs zipped to: "
    f"{zip_output}"
)

print(
    "\n"
    + "=" * 100
)

print(
    "PYTHON PHASE 2.3 REVIEWER REVISION COMPLETED SUCCESSFULLY"
)

print(
    "=" * 100
)

print(
    f"Seeds: {MULTI_SEEDS}"
)

print(
    f"Device: {DEVICE}"
)

print(
    "\nMulti-seed pooled TEST mean +/- SD:"
)

display_summary = metric_summary[
    [
        "Model",
        "Horizon",
        "BrierMean",
        "BrierSD",
        "LogScoreMean",
        "LogScoreSD",
        "PR_AUCMean",
        "PR_AUCSD",
    ]
].sort_values(
    [
        "Horizon",
        "BrierMean",
    ]
)

print(
    display_summary.to_string(
        index=False
    )
)

print(
    "\nFive-seed ENSEMBLE pooled TEST results:"
)

print(
    ensemble_test_pooled[
        [
            "Horizon",
            "Model",
            "Brier",
            "LogScore",
            "PR_AUC",
            "ROC_AUC",
            "CalibrationIntercept",
            "CalibrationSlope",
        ]
    ].to_string(
        index=False
    )
)

print(
    "\nPaired-seed graph robustness:"
)

print(
    paired_seed_summary.to_string(
        index=False
    )
)

print(
    "\nFinal ZIP to send back for review:"
)

print(
    f"  {zip_output}"
)

print(
    "=" * 100
)

# =============================================================================
# 37. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip ..."
    )

    files.download(
        str(
            zip_output
        )
    )

except ImportError:

    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: "
        f"{zip_output}"
    )

# =============================================================================
# END OF PYTHON PHASE 2.3 REVIEWER REVISION
# =============================================================================


[2026-09-12 10:27:06] ====================================================================================================
[2026-09-12 10:27:06] PYTHON PHASE 2.3 START — NESTED TUNING / FULL-TRAIN REFIT / FIVE-SEED ROBUSTNESS
[2026-09-12 10:27:06] Python=3.13.15; platform=Linux-6.6.122+-x86_64-with-glibc2.39; seed=20260901
[2026-09-12 10:27:16] PyTorch=2.11.0+cu128; device=cuda

Please upload the completed Phase-1 ZIP:
    Python_Phase1_ReviewerRevision_v1_3_All_Outputs.zip



Saving Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip to Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip
[2026-09-12 10:28:05] Validated uploaded Phase-1 ZIP: /content/Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip
[2026-09-12 10:28:05] Accepted Phase-1 ZIP: /content/Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip
[2026-09-12 10:28:06] Loaded 14,928 sector-days; 2,488 dates; 6 sectors; graph source=HAWKES_LAMBDA_MIN_STABILITY.
[2026-09-12 10:28:06] Reviewer chronology: TrainFit=1123 dates, TrainTune=374, Calibration=496, Test=495.
[2026-09-12 10:28:06] Nested split-aware survival targets reconstructed successfully.
[2026-09-12 10:28:06] Selected 51 leakage-screened numeric predictors.
[2026-09-12 10:28:06] Final neural feature dimension=63; missingness indicators=12.
[2026-09-12 10:28:06] Crash_Main event-safe imputation applied: missing events set to zero and never forward-filled.
[2026-09-12 10:28:06] Magnitude-preserving graph scaling applied. TrainFit q0.99=0.0695

/tmp/ipykernel_15624/2440277949.py:1845: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[2026-09-12 10:28:49] TemporalTransformer_Survival__Seed_20260901: tune epoch=01, TrainFit=0.51017, TrainTuneNLL=0.18107
[2026-09-12 10:28:50] TemporalTransformer_Survival__Seed_20260901: tune epoch=02, TrainFit=0.28795, TrainTuneNLL=0.11512
[2026-09-12 10:28:51] TemporalTransformer_Survival__Seed_20260901: tune epoch=03, TrainFit=0.27824, TrainTuneNLL=0.12433
[2026-09-12 10:28:52] TemporalTransformer_Survival__Seed_20260901: tune epoch=04, TrainFit=0.27235, TrainTuneNLL=0.17455
[2026-09-12 10:28:53] TemporalTransformer_Survival__Seed_20260901: tune epoch=05, TrainFit=0.27176, TrainTuneNLL=0.19837
[2026-09-12 10:28:54] TemporalTransformer_Survival__Seed_20260901: tune epoch=06, TrainFit=0.26367, TrainTuneNLL=0.23320
[2026-09-12 10:28:56] TemporalTransformer_Survival__Seed_20260901: tune epoch=07, TrainFit=0.25657, TrainTuneNLL=0.25499
[2026-09-12 10:28:57] TemporalTransformer_Survival__Seed_20260901: tune epoch=08, TrainFit=0.25374, TrainTuneNLL=0.27377
[2026-09-12 10:28:58] TemporalTr

/tmp/ipykernel_15624/2440277949.py:2031: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(


...

[2026-09-12 10:33:39] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=01, TrainFit=0.58619, TrainTuneNLL=0.34475
[2026-09-12 10:33:40] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=02, TrainFit=0.33813, TrainTuneNLL=0.12885
[2026-09-12 10:33:40] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=03, TrainFit=0.28374, TrainTuneNLL=0.11714
[2026-09-12 10:33:41] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=04, TrainFit=0.28034, TrainTuneNLL=0.11022
[2026-09-12 10:33:41] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=05, TrainFit=0.27900, TrainTuneNLL=0.12863
[2026-09-12 10:33:42] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=06, TrainFit=0.27308, TrainTuneNLL=0.15848
[2026-09-12 10:33:42] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=07, TrainFit=0.26778, TrainTuneNLL=0.17134
[2026-09-12 10:33:43] StaticHawkesGraphTransformer__Seed_20260905: tune epoch=08, TrainFit=0.26131, TrainTuneNLL=0.14169
[2026-09-12 10:33:43] StaticHawk

/tmp/ipykernel_15624/2440277949.py:2031: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_encoder = nn.TransformerEncoder(


[2026-09-12 10:33:49] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=01, TrainFit=0.58624, TrainTuneNLL=0.34591
[2026-09-12 10:33:49] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=02, TrainFit=0.33930, TrainTuneNLL=0.12901
[2026-09-12 10:33:50] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=03, TrainFit=0.28373, TrainTuneNLL=0.11733
[2026-09-12 10:33:50] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=04, TrainFit=0.27998, TrainTuneNLL=0.11050
[2026-09-12 10:33:50] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=05, TrainFit=0.27769, TrainTuneNLL=0.12623
[2026-09-12 10:33:51] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=06, TrainFit=0.27083, TrainTuneNLL=0.16486
[2026-09-12 10:33:52] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=07, TrainFit=0.26711, TrainTuneNLL=0.17971
[2026-09-12 10:33:52] DynamicHawkesGraphTransformer__Seed_20260905: tune epoch=08, TrainFit=0.26107, TrainTuneNLL=0.15484
[2026-09-12 10:33:53] Dy

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# =============================================================================
# PYTHON PHASE 3.2 — REVIEWER-REVISION COMPONENT ABLATIONS, DYNAMIC RIDGE-LOGIT,
# AND TAIL-EVENT-DEFINITION ROBUSTNESS
#
# Forecasting Sector Tail-Event Probabilities:
# Hawkes-Informed Graph Models and Leakage-Free Forecast Evaluation
#
# INPUT
# -----
# Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip
#
# PURPOSE
# -------
# This phase retains the established component-ablation and event-definition
# robustness design while applying the reviewer-revision chronology used in
# Phase 2.3:
#
#   TrainFit -> TrainTune -> fresh Full-Train refit -> Calibration -> locked Test
#
# The dynamic ridge-logit is retained as a deliberately parsimonious,
# phase-matched econometric benchmark. XGBoost and ridge-logit tuning use only
# TrainFit/TrainTune. Calibration observations are never used for tuning.
#
# IMPORTANT LEAKAGE CONTROLS
# --------------------------
# * TrainFit ends 29 January 2021.
# * TrainTune is 1 February 2021 to 29 July 2022.
# * Calibration is 1 August 2022 to 31 July 2024.
# * Test begins 1 August 2024 and remains locked until final evaluation.
# * Inner targets stop at TrainFit/TrainTune boundaries.
# * Feature-set admission is frozen using TrainFit only.
# * Final imputation/scaling is recomputed on all Train only after tuning.
# * XGBoost stopping rounds are selected on TrainTune, then the booster is
#   freshly refit on Full Train for exactly those rounds.
# * Ridge-logit C is selected on TrainTune, then the model is freshly refit on
#   Full Train at that fixed C.
# * Platt calibration for both model families uses Calibration only.
# * The fixed 2.5% return-threshold robustness definition uses a TrainFit-only
#   threshold for inner tuning and a Full-Train-only threshold for final fitting.
# * Test outcomes are never used for model selection, calibration, or tuning.
#
# VERSION: 3.2 REVIEWER REVISION
# DATE: 2026-09-12
# =============================================================================

from __future__ import annotations

import os
import sys
import gc
import json
import math
import random
import shutil
import pickle
import zipfile
import warnings
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime

# Conservative numerical thread limits improve reproducibility and avoid
# oversubscription on Colab/large-core runtimes.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# =============================================================================
# 0. CONFIGURATION / REPRODUCIBILITY
# =============================================================================

SEED = 20260901
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

INPUT_ZIP = os.getenv(
    "NSE_PHASE1_ZIP",
    "/content/Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip",
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_PHASE3_OUTPUT_DIR",
    "/content/Sectoral_Tail_Event_Python_Phase3_ReviewerRevision",
))

HORIZONS = [1, 5, 10, 22]
MAX_HORIZON = max(HORIZONS)
LOOKBACK = int(os.getenv("NSE_LOOKBACK", "60"))

# Frozen reviewer-revision chronology.
TRAIN_FIT_END = pd.Timestamp("2021-01-29")
TRAIN_TUNE_START = pd.Timestamp("2021-02-01")
TRAIN_TUNE_END = pd.Timestamp("2022-07-29")
CALIBRATION_START = pd.Timestamp("2022-08-01")
CALIBRATION_END = pd.Timestamp("2024-07-31")
TEST_START = pd.Timestamp("2024-08-01")

# XGBoost settings deliberately match the established Phase-2 family closely.
XGB_MAX_ROUNDS = int(os.getenv("NSE_XGB_MAX_ROUNDS", "1200"))
XGB_EARLY_STOP = int(os.getenv("NSE_XGB_EARLY_STOP", "60"))
XGB_POS_WEIGHT_CAP = float(os.getenv("NSE_XGB_POS_WEIGHT_CAP", "10.0"))
XGB_NTHREAD = int(os.getenv("NSE_XGB_NTHREAD", "4"))

# Dynamic ridge-logit candidate penalties. Selected on TrainTune only.
LOGIT_LAMBDA_GRID = [0.01, 0.10, 1.0, 10.0]

# Paired moving-block bootstrap.
BOOTSTRAP_REPS = int(os.getenv("NSE_BOOTSTRAP_REPS", "500"))
BOOTSTRAP_BLOCK = int(os.getenv("NSE_BOOTSTRAP_BLOCK", "22"))

# Summary windows used by XGBoost. The dynamic logit uses a more parsimonious
# subset of the same summary representation.
XGB_WINDOWS = [5, 22, 60]
LOGIT_WINDOWS = [5, 22]

EPS = 1e-8

# =============================================================================
# 1. OUTPUT FOLDERS / LOGGING
# =============================================================================

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
EXCEL_DIR = OUTPUT_ROOT / "03_Excel"
MODEL_DIR = OUTPUT_ROOT / "04_Model_Objects"
DATA_DIR = OUTPUT_ROOT / "05_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "06_Logs"
ZIP_DIR = OUTPUT_ROOT / "07_Zip"
EXTRACT_DIR = OUTPUT_ROOT / "_Phase1_Extracted"

for directory in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, EXCEL_DIR,
    MODEL_DIR, DATA_DIR, LOG_DIR, ZIP_DIR, EXTRACT_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Python_Phase3_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("=" * 100)
log("PYTHON PHASE 3.2 START — REVIEWER-REVISION ABLATIONS / DYNAMIC RIDGE-LOGIT / TAIL-EVENT ROBUSTNESS")
log(f"Python={sys.version.split()[0]}; platform={platform.platform()}; seed={SEED}")

# =============================================================================
# 2. PACKAGE CHECKS
# =============================================================================

def ensure_package(import_name: str, pip_name: str | None = None):
    if importlib.util.find_spec(import_name) is not None:
        return
    package = pip_name or import_name
    log(f"Installing missing package: {package}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package]
    )

ensure_package("xgboost")
import xgboost as xgb

log(f"xgboost={xgb.__version__}")

# =============================================================================
# 3. ALWAYS ASK FOR THE FROZEN PHASE-1 ZIP IN COLAB
# =============================================================================

REQUIRED_PHASE1_MEMBERS = {
    "phase2_deep_learning_master.csv.gz",
    "python_phase1_metadata.json",
    "Table_P12_Final_Stable_Edge_Parameters.csv",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as archive:
            return {
                Path(name).name
                for name in archive.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_phase1_zip(path: Path) -> bool:
    return (
        path.exists()
        and path.suffix.lower() == ".zip"
        and REQUIRED_PHASE1_MEMBERS.issubset(zip_basenames(path))
    )

def resolve_phase1_zip(configured: str) -> Path:
    """
    In Google Colab the user is ALWAYS asked to upload the Phase-1 ZIP.
    The script never silently uses an older ZIP already sitting in /content.
    """
    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload the frozen Phase-1 file:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )
            uploaded = files.upload()

            candidates = [
                Path("/content") / name
                for name in uploaded
                if name.lower().endswith(".zip")
            ]

            valid = [p for p in candidates if is_valid_phase1_zip(p)]

            if len(valid) == 1:
                log(f"Validated uploaded Phase-1 ZIP: {valid[0]}")
                return valid[0]

            if len(valid) > 1:
                print(
                    "\nMore than one valid Phase-1 ZIP was uploaded. "
                    "Please upload exactly one.\n"
                )
                continue

            for candidate in candidates:
                missing = sorted(
                    REQUIRED_PHASE1_MEMBERS - zip_basenames(candidate)
                )
                log(
                    f"Rejected '{candidate.name}'. Missing Phase-1 files: {missing}"
                )

            print(
                "\nThe selected ZIP is not the frozen Phase-1 output. "
                "Please choose:\n"
                "    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip\n"
            )

    except ImportError:
        configured_path = Path(configured)

        if is_valid_phase1_zip(configured_path):
            return configured_path

        for folder in [Path.cwd(), Path("/mnt/data")]:
            if not folder.exists():
                continue
            for candidate in folder.glob("*.zip"):
                if is_valid_phase1_zip(candidate):
                    return candidate

        raise FileNotFoundError(
            "Could not locate a valid Phase-1 ZIP. "
            "Set NSE_PHASE1_ZIP to the correct file path."
        )

INPUT_ZIP_PATH = resolve_phase1_zip(INPUT_ZIP)
log(f"Accepted Phase-1 ZIP: {INPUT_ZIP_PATH}")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(INPUT_ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

def find_one(filename: str) -> Path:
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}' in Phase-1 ZIP; "
            f"found {len(matches)}."
        )
    return matches[0]

MASTER_FILE = find_one("phase2_deep_learning_master.csv.gz")
META_FILE = find_one("python_phase1_metadata.json")
EDGE_FILE = find_one("Table_P12_Final_Stable_Edge_Parameters.csv")

# =============================================================================
# 4. LOAD / AUDIT MASTER PANEL
# =============================================================================

master = pd.read_csv(MASTER_FILE, parse_dates=["Date"])
master = master.sort_values(["Date", "Sector"]).reset_index(drop=True)

with open(META_FILE, "r", encoding="utf-8") as handle:
    phase1_metadata = json.load(handle)

stable_edges = pd.read_csv(EDGE_FILE)

required_columns = {
    "Date", "Sector", "Split",
    "SectorReturn_Model",
    "GARCH_Sigma", "StdInnovation",
    "Crash_001", "Crash_0025", "Crash_005", "Crash_Main",
}
missing_required = sorted(required_columns - set(master.columns))

if missing_required:
    raise RuntimeError(
        f"Phase-1 modelling master lacks required columns: {missing_required}"
    )

sectors = sorted(master["Sector"].dropna().astype(str).unique().tolist())
S = len(sectors)
sector_to_idx = {sector: i for i, sector in enumerate(sectors)}

dates = pd.DatetimeIndex(sorted(master["Date"].unique()))
T = len(dates)
date_to_idx = {pd.Timestamp(date): i for i, date in enumerate(dates)}

if master.duplicated(["Date", "Sector"]).any():
    raise RuntimeError("Duplicate Date-Sector rows found.")

counts = master.groupby("Date")["Sector"].nunique()
if not (counts == S).all():
    raise RuntimeError("Master is not a complete sector-date panel.")

split_by_date = (
    master[["Date", "Split"]]
    .drop_duplicates()
    .set_index("Date")["Split"]
    .reindex(dates)
)
legacy_splits = split_by_date.astype(str).to_numpy()

if set(np.unique(legacy_splits)) != {"Train", "Validation", "Test"}:
    raise RuntimeError(f"Unexpected legacy split labels: {np.unique(legacy_splits)}")

if "RevisionPhase" not in master.columns:
    raise RuntimeError(
        "Phase 3.2 requires RevisionPhase from the reviewer-revised Phase-1 handoff."
    )

revision_phase_by_date = (
    master[["Date", "RevisionPhase"]]
    .drop_duplicates()
    .set_index("Date")["RevisionPhase"]
    .reindex(dates)
)
revision_phases = revision_phase_by_date.astype(str).to_numpy()
expected_revision_phases = {"TrainFit", "TrainTune", "Calibration", "Test"}
if set(np.unique(revision_phases)) != expected_revision_phases:
    raise RuntimeError(
        f"Unexpected RevisionPhase labels: {np.unique(revision_phases)}"
    )

train_fit_date_mask = revision_phases == "TrainFit"
train_tune_date_mask = revision_phases == "TrainTune"
full_train_date_mask = train_fit_date_mask | train_tune_date_mask
calibration_date_mask = revision_phases == "Calibration"
test_date_mask = revision_phases == "Test"

# Backward-compatible alias used by historical-baseline code. Analytically,
# 'train_date_mask' means the complete pre-calibration Train period.
train_date_mask = full_train_date_mask

if not np.array_equal(full_train_date_mask, legacy_splits == "Train"):
    raise RuntimeError("TrainFit+TrainTune does not equal the legacy Train period.")
if not np.array_equal(calibration_date_mask, legacy_splits == "Validation"):
    raise RuntimeError("Calibration does not equal the legacy Validation period.")
if not np.array_equal(test_date_mask, legacy_splits == "Test"):
    raise RuntimeError("Test does not equal the legacy Test period.")

def _date_bounds(mask: np.ndarray) -> tuple[pd.Timestamp, pd.Timestamp]:
    d = dates[mask]
    return pd.Timestamp(d.min()), pd.Timestamp(d.max())

if _date_bounds(train_fit_date_mask)[1] != TRAIN_FIT_END:
    raise RuntimeError("TrainFit end date differs from the frozen reviewer design.")
if _date_bounds(train_tune_date_mask) != (TRAIN_TUNE_START, TRAIN_TUNE_END):
    raise RuntimeError("TrainTune dates differ from the frozen reviewer design.")
if _date_bounds(calibration_date_mask) != (CALIBRATION_START, CALIBRATION_END):
    raise RuntimeError("Calibration dates differ from the frozen reviewer design.")
if _date_bounds(test_date_mask)[0] != TEST_START:
    raise RuntimeError("Test start date differs from the frozen reviewer design.")

# Final target boundaries: Full Train, Calibration and Test are distinct.
final_target_phases = np.where(
    full_train_date_mask,
    "Train",
    np.where(calibration_date_mask, "Calibration", "Test"),
)

log(
    f"Loaded {len(master):,} rows; {T:,} dates; {S} sectors. "
    f"Reviewer chronology: TrainFit={train_fit_date_mask.sum()}, "
    f"TrainTune={train_tune_date_mask.sum()}, "
    f"Calibration={calibration_date_mask.sum()}, Test={test_date_mask.sum()}."
)

# =============================================================================
# 5. BUILD EVENT PANELS FOR EVT DEFINITIONS
# =============================================================================

def event_panel_from_column(column: str) -> tuple[np.ndarray, np.ndarray]:
    C = np.zeros((T, S), dtype=np.float32)
    M = np.zeros((T, S), dtype=bool)

    for row in master[["Date", "Sector", column]].itertuples(index=False):
        t = date_to_idx[pd.Timestamp(row.Date)]
        s = sector_to_idx[str(row.Sector)]
        value = getattr(row, column)

        if pd.notna(value):
            C[t, s] = float(value)
            M[t, s] = True

    observed = C[M]
    if len(observed) == 0 or not np.isin(observed, [0.0, 1.0]).all():
        raise RuntimeError(f"Invalid binary event panel for {column}.")

    return C, M

event_definitions: dict[str, dict] = {}

for name, column in [
    ("EVT_001", "Crash_001"),
    ("EVT_0025", "Crash_0025"),
    ("EVT_005", "Crash_005"),
]:
    C, M = event_panel_from_column(column)

    event_definitions[name] = {
        "event": C,
        "observed": M,
        "source": column,
        "description": f"R GJR-GARCH / EVT event label from {column}",
    }

# Crash_Main must coincide with primary EVT_0025 where observed.
C_main, M_main = event_panel_from_column("Crash_Main")
primary_C = event_definitions["EVT_0025"]["event"]
primary_M = event_definitions["EVT_0025"]["observed"]

common_primary = M_main & primary_M
primary_mismatch = int(
    np.sum(
        C_main[common_primary]
        != primary_C[common_primary]
    )
)
if primary_mismatch != 0:
    raise RuntimeError(
        f"Crash_Main differs from Crash_0025 in {primary_mismatch} cells."
    )

# =============================================================================
# 6. NESTED FIXED 2.5% SECTOR RETURN THRESHOLDS
# =============================================================================
#
# The fixed-return-threshold robustness definition is itself estimated from
# data. Therefore the inner tuning stage uses TrainFit-only thresholds, while
# the final refit/calibration/Test stage uses thresholds estimated on all Train.
# Both versions retain the SAME observation mask as EVT_0025.

return_panel = np.full((T, S), np.nan, dtype=np.float64)
for row in master[["Date", "Sector", "SectorReturn_Model"]].itertuples(index=False):
    t = date_to_idx[pd.Timestamp(row.Date)]
    s = sector_to_idx[str(row.Sector)]
    if pd.notna(row.SectorReturn_Model):
        return_panel[t, s] = float(row.SectorReturn_Model)

def build_fixed_definition(reference_mask: np.ndarray, label: str):
    thresholds = np.full(S, np.nan, dtype=float)
    C = np.zeros((T, S), dtype=np.float32)
    M = primary_M.copy()
    rows = []

    for s, sector in enumerate(sectors):
        threshold_sample = (
            reference_mask
            & primary_M[:, s]
            & np.isfinite(return_panel[:, s])
        )
        values = return_panel[threshold_sample, s]
        if len(values) < 100:
            raise RuntimeError(
                f"Too few {label} returns to estimate fixed threshold for {sector}."
            )

        threshold = float(np.quantile(values, 0.025))
        thresholds[s] = threshold
        usable = primary_M[:, s] & np.isfinite(return_panel[:, s])
        C[usable, s] = (return_panel[usable, s] <= threshold).astype(np.float32)
        M[:, s] = usable

        rows.append({
            "ThresholdReference": label,
            "Sector": sector,
            "Quantile": 0.025,
            "FixedReturnThreshold": threshold,
            "ReferenceN": int(len(values)),
            "ReferenceEventRate": float(C[reference_mask & M[:, s], s].mean()),
        })

    return thresholds, C, M, rows

(
    fixed_thresholds_inner,
    fixed_C_inner,
    fixed_M_inner,
    fixed_rows_inner,
) = build_fixed_definition(train_fit_date_mask, "TrainFit")

(
    fixed_thresholds,
    fixed_C,
    fixed_M,
    fixed_rows_final,
) = build_fixed_definition(full_train_date_mask, "FullTrain")

fixed_threshold_table = pd.DataFrame(fixed_rows_inner + fixed_rows_final)
fixed_threshold_table.to_csv(
    TABLE_DIR / "Table_301_Fixed_Thresholds_Nested_Train_Only.csv",
    index=False,
)

event_definitions["Fixed_0025"] = {
    "event": fixed_C,
    "observed": fixed_M,
    "inner_event": fixed_C_inner,
    "inner_observed": fixed_M_inner,
    "source": "Nested sector return 2.5% quantiles",
    "description": (
        "TrainFit-only threshold for inner tuning; Full-Train-only threshold "
        "for final refit, Calibration and Test"
    ),
}

# EVT definitions are generated strictly from prior R information. Their event
# panels are therefore unchanged; only the target-censoring boundary differs
# between inner tuning and final fitting.
for _name, _definition in event_definitions.items():
    if _name != "Fixed_0025":
        _definition["inner_event"] = _definition["event"]
        _definition["inner_observed"] = _definition["observed"]

# =============================================================================
# 7. NESTED SPLIT-AWARE MULTI-HORIZON TARGETS
# =============================================================================

def build_phase_aware_targets(
    C: np.ndarray,
    M: np.ndarray,
    phase_labels: np.ndarray,
) -> dict[int, np.ndarray]:
    targets = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }

    for t in range(T):
        phase_t = phase_labels[t]
        for s in range(S):
            if not M[t, s]:
                continue

            observed_until = 0
            event_k = None
            for k in range(1, MAX_HORIZON + 1):
                future_t = t + k
                if future_t >= T:
                    break
                if phase_labels[future_t] != phase_t:
                    break
                if not M[future_t, s]:
                    break

                observed_until = k
                if C[future_t, s] == 1:
                    event_k = k
                    break

            for h in HORIZONS:
                if event_k is not None and event_k <= h:
                    targets[h][t, s] = 1.0
                elif observed_until >= h:
                    targets[h][t, s] = 0.0

    return targets

for definition in event_definitions.values():
    definition["inner_targets"] = build_phase_aware_targets(
        definition["inner_event"],
        definition["inner_observed"],
        revision_phases,
    )
    definition["targets"] = build_phase_aware_targets(
        definition["event"],
        definition["observed"],
        final_target_phases,
    )

# Target/prevalence audit under the final design, plus inner tuning counts.
target_audit_rows = []
for definition_name, definition in event_definitions.items():
    C = definition["event"]
    M = definition["observed"]

    for split_name, split_mask in [
        ("TrainFit", train_fit_date_mask),
        ("TrainTune", train_tune_date_mask),
        ("FullTrain", full_train_date_mask),
        ("Calibration", calibration_date_mask),
        ("Test", test_date_mask),
    ]:
        same_day_mask = split_mask[:, None] & M
        target_audit_rows.append({
            "Definition": definition_name,
            "Split": split_name,
            "Horizon": 0,
            "ValidTargets": int(same_day_mask.sum()),
            "Events": int(C[same_day_mask].sum()),
            "EventRate": float(C[same_day_mask].mean()) if same_day_mask.sum() else np.nan,
            "TargetConstruction": "Final",
        })

        for h in HORIZONS:
            y = definition["targets"][h][split_mask].reshape(-1)
            ok = np.isfinite(y)
            target_audit_rows.append({
                "Definition": definition_name,
                "Split": split_name,
                "Horizon": h,
                "ValidTargets": int(ok.sum()),
                "Events": int(np.nansum(y)),
                "EventRate": float(np.nanmean(y)) if ok.any() else np.nan,
                "TargetConstruction": "Final",
            })

    for split_name, split_mask in [
        ("TrainFit", train_fit_date_mask),
        ("TrainTune", train_tune_date_mask),
    ]:
        for h in HORIZONS:
            y = definition["inner_targets"][h][split_mask].reshape(-1)
            ok = np.isfinite(y)
            target_audit_rows.append({
                "Definition": definition_name,
                "Split": split_name,
                "Horizon": h,
                "ValidTargets": int(ok.sum()),
                "Events": int(np.nansum(y)),
                "EventRate": float(np.nanmean(y)) if ok.any() else np.nan,
                "TargetConstruction": "InnerTuning",
            })

target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(
    TABLE_DIR / "Table_302_Tail_Event_Target_Audit_Nested.csv",
    index=False,
)

# =============================================================================
# 8. EVT-vs-FIXED EVENT OVERLAP
# =============================================================================

overlap_rows = []

evt_C = event_definitions["EVT_0025"]["event"]
evt_M = event_definitions["EVT_0025"]["observed"]
fix_C = event_definitions["Fixed_0025"]["event"]
fix_M = event_definitions["Fixed_0025"]["observed"]

for split_name, split_mask in [
    ("FullTrain", full_train_date_mask),
    ("Calibration", calibration_date_mask),
    ("Test", test_date_mask),
]:
    for s, sector in enumerate(sectors):
        common = split_mask & evt_M[:, s] & fix_M[:, s]

        a = evt_C[common, s].astype(bool)
        b = fix_C[common, s].astype(bool)

        union = np.sum(a | b)
        intersection = np.sum(a & b)

        overlap_rows.append({
            "Split": split_name,
            "Sector": sector,
            "CommonDays": int(common.sum()),
            "EVTEvents": int(a.sum()),
            "FixedEvents": int(b.sum()),
            "CommonEvents": int(intersection),
            "Jaccard": (
                float(intersection / union)
                if union > 0
                else np.nan
            ),
            "AgreementRate": (
                float(np.mean(a == b))
                if len(a)
                else np.nan
            ),
        })

event_overlap = pd.DataFrame(overlap_rows)

event_overlap.to_csv(
    TABLE_DIR / "Table_303_EVT_vs_Fixed_Event_Overlap.csv",
    index=False,
)

# =============================================================================
# 9. FEATURE GOVERNANCE
# =============================================================================

EXCLUDE_COLUMNS = {
    "Date",
    "Sector",
    "Split",
    "PrimarySector",
    "ExclusionReason",
    "EconometricGraphSource",

    # Future survival labels / summaries.
    "EventObservedWithin22",
    "EventTime",
    "CensorTime",
    "CrashWithin_1",
    "CrashWithin_5",
    "CrashWithin_10",
    "CrashWithin_22",

    # Previously generated probability forecasts.
    "FixedHistoricalProb_1",
    "FixedHistoricalProb_5",
    "FixedHistoricalProb_10",
    "FixedHistoricalProb_22",
    "ExpandingHistoricalProb_1",
    "ExpandingHistoricalProb_5",
    "ExpandingHistoricalProb_10",
    "ExpandingHistoricalProb_22",
    "StableHawkesProb_1",
    "StableHawkesProb_5",
    "StableHawkesProb_10",
    "StableHawkesProb_22",

    # All crash labels are excluded. Current event history is injected
    # definition-by-definition below.
    "Crash_001",
    "Crash_0025",
    "Crash_005",
    "Crash_Main",
    "CurrentCrash",
    "CurrentCrashObserved",

    # Direct thresholds.
    "CrashThreshold_001",
    "CrashThreshold_0025",
    "CrashThreshold_005",
    # Reviewer-revision R/Phase-1 handoff also carries this convenience alias.
    # It is the same contemporaneous 2.5% threshold and must never enter the
    # generic predictor matrix.
    "CrashThreshold_Main",

    # EVT optimizer diagnostics are not economic predictors.
    "EVT_fit_method",
    "EVT_fit_convergence",
    "EVT_loglik",
    "EVT_nll_improvement",
    "EVT_at_boundary",
    "EVT_refit_id",
}

base_feature_columns = []

for column in master.columns:
    if column in EXCLUDE_COLUMNS:
        continue
    if pd.api.types.is_numeric_dtype(master[column]):
        base_feature_columns.append(column)

# Generic predictors must not contain any legacy Crash_* outcome/threshold
# field, future survival target, or previously generated probability. Current
# tail-event history is injected separately, definition by definition, later.
forbidden_name_fragments = [
    "Crash",
    "EventTime",
    "CensorTime",
    "Prob_",
]

base_feature_columns = [
    c for c in base_feature_columns
    if not any(fragment in c for fragment in forbidden_name_fragments)
]

generic_leakage_offenders = [
    c for c in base_feature_columns
    if any(fragment in c for fragment in forbidden_name_fragments)
]
if generic_leakage_offenders:
    raise RuntimeError(
        "Generic predictor leakage screen failed immediately. Offending columns: "
        + ", ".join(generic_leakage_offenders)
    )

log(f"Leakage-screened base numeric features={len(base_feature_columns)}")
log("Generic leakage screen passed: no Crash/future-target/prior-probability columns admitted.")

pd.DataFrame({
    "Feature": base_feature_columns,
    "AdmittedGenericPredictor": True,
}).to_csv(
    TABLE_DIR / "Table_304_Generic_Predictor_Leakage_Audit.csv",
    index=False,
)

# =============================================================================
# 10. PANELIZE / PAST-ONLY IMPUTE / NESTED STANDARDIZATION
# =============================================================================

feature_panel_raw = np.full(
    (T, S, len(base_feature_columns)),
    np.nan,
    dtype=np.float64,
)

for s, sector in enumerate(sectors):
    sector_df = (
        master[master["Sector"].astype(str) == sector]
        .set_index("Date")
        .reindex(dates)
    )
    feature_panel_raw[:, s, :] = (
        sector_df[base_feature_columns].astype(float).to_numpy()
    )

# Past-only forward filling for continuous/state predictors.
feature_panel_ffill = feature_panel_raw.copy()
for s in range(S):
    frame = pd.DataFrame(feature_panel_ffill[:, s, :])
    feature_panel_ffill[:, s, :] = frame.ffill().to_numpy()

# Freeze feature admission on TrainFit only.
trainfit_values = feature_panel_ffill[train_fit_date_mask]
trainfit_median_initial = np.nanmedian(trainfit_values, axis=(0, 1))
valid_feature_mask = np.isfinite(trainfit_median_initial)

if not valid_feature_mask.all():
    dropped = [
        base_feature_columns[i]
        for i in np.where(~valid_feature_mask)[0]
    ]
    log(f"Dropping all-missing TrainFit features: {dropped}")
    base_feature_columns = [
        base_feature_columns[i]
        for i in np.where(valid_feature_mask)[0]
    ]
    feature_panel_raw = feature_panel_raw[:, :, valid_feature_mask]
    feature_panel_ffill = feature_panel_ffill[:, :, valid_feature_mask]

# Missingness-indicator admission is also frozen on TrainFit.
raw_trainfit = feature_panel_raw[train_fit_date_mask]
missing_rate = np.mean(~np.isfinite(raw_trainfit), axis=(0, 1))
missing_indicator_indices = np.where(missing_rate >= 0.02)[0]
missing_indicators = (
    ~np.isfinite(feature_panel_raw[:, :, missing_indicator_indices])
).astype(np.float64)
missing_indicator_names = [
    f"MISS__{base_feature_columns[i]}"
    for i in missing_indicator_indices
]
model_feature_names = base_feature_columns + missing_indicator_names

feature_origin = {}
for name in model_feature_names:
    feature_origin[name] = name.replace("MISS__", "", 1) if name.startswith("MISS__") else name

def build_preprocessed_panel(reference_mask: np.ndarray, label: str):
    reference_values = feature_panel_ffill[reference_mask]
    medians = np.nanmedian(reference_values, axis=(0, 1))
    if not np.all(np.isfinite(medians)):
        bad = [
            base_feature_columns[i]
            for i in np.where(~np.isfinite(medians))[0]
        ]
        raise RuntimeError(f"{label} preprocessing has non-finite medians: {bad}")

    numeric = feature_panel_ffill.copy()
    for j in range(numeric.shape[2]):
        bad = ~np.isfinite(numeric[:, :, j])
        numeric[:, :, j][bad] = medians[j]

    if len(missing_indicator_indices):
        numeric = np.concatenate([numeric, missing_indicators], axis=2)

    block = numeric[reference_mask]
    means = block.mean(axis=(0, 1))
    stds = block.std(axis=(0, 1))
    stds[stds < 1e-8] = 1.0
    panel = ((numeric - means[None, None, :]) / stds[None, None, :]).astype(np.float32)
    return panel, medians, means, stds

(
    X_panel_inner,
    trainfit_median,
    trainfit_mean,
    trainfit_std,
) = build_preprocessed_panel(train_fit_date_mask, "TrainFit")

(
    X_panel,
    train_median,
    train_mean,
    train_std,
) = build_preprocessed_panel(full_train_date_mask, "FullTrain")

if X_panel_inner.shape[2] != X_panel.shape[2]:
    raise RuntimeError("Inner and final preprocessing produced different feature dimensions.")
if not np.all(np.isfinite(X_panel_inner)) or not np.all(np.isfinite(X_panel)):
    raise RuntimeError("Nested feature panels contain non-finite values.")

pd.DataFrame({
    "Feature": model_feature_names,
    "OriginFeature": [feature_origin[x] for x in model_feature_names],
    "TrainFitMean": trainfit_mean,
    "TrainFitSD": trainfit_std,
    "FullTrainMean": train_mean,
    "FullTrainSD": train_std,
}).to_csv(
    TABLE_DIR / "Table_304_Feature_Preprocessing_Nested.csv",
    index=False,
)

# =============================================================================
# 11. PRE-SPECIFIED FEATURE GROUPS / ABLATIONS
# =============================================================================

GARCH_GROUP = {
    "GARCH_Sigma",
}

VOLATILITY_GROUP = {
    "GARCH_Sigma",
    "ParkinsonVol",
    "GarmanKlassVol",
    "RogersSatchellVol",
}

TAIL_EVT_GROUP = {
    "StdInnovation",
    "EVT_u",
    "EVT_scale",
    "EVT_shape",
    "EVT_pu",
    "EVT_nhist",
    "EVT_nexc",
}

LIQUIDITY_GROUP = {
    "Turnover_MCW",
    "AmihudILLIQ_Median",
    "ZeroReturnShare",
    "ZeroVolumeShare",
    "MarketTurnover_MCW",
    "MarketAmihudILLIQ_Median",
    "MarketZeroReturnShare",
    "MarketZeroVolumeShare",
}

BREADTH_DISPERSION_GROUP = {
    "BreadthNegative",
    "ReturnDispersion",
    "MarketBreadthNegative",
    "MarketReturnDispersion",
    "NSectorsDown",
    "CrossSectorReturnDispersion",
}

MARKET_STATE_GROUP = {
    c for c in base_feature_columns
    if (
        c.startswith("Market")
        or c in {
            "NStocksObservedMarket",
            "NStocksReturnMarket",
            "NSectorsObserved",
            "NSectorsWithReturn",
            "NSectorsDown",
            "CrossSectorReturnDispersion",
        }
    )
}

ABLATION_DROP_GROUPS = {
    "Full": set(),
    "NoGARCH": GARCH_GROUP,
    "NoVolatilityGroup": VOLATILITY_GROUP,
    "NoTailEVT": TAIL_EVT_GROUP,
    "NoLiquidity": LIQUIDITY_GROUP,
    "NoBreadthDispersion": BREADTH_DISPERSION_GROUP,
    "NoMarketState": MARKET_STATE_GROUP,
}

ablation_design_rows = []

for variant, drop_group in ABLATION_DROP_GROUPS.items():
    present_drop = sorted(
        set(base_feature_columns) & set(drop_group)
    )

    ablation_design_rows.append({
        "Variant": variant,
        "DroppedFeatureCount": len(present_drop),
        "DroppedFeatures": " | ".join(present_drop),
    })

ablation_design = pd.DataFrame(ablation_design_rows)

ablation_design.to_csv(
    TABLE_DIR / "Table_305_Component_Ablation_Design.csv",
    index=False,
)

# =============================================================================
# 12. GENERIC FORECAST-ORIGIN COORDINATES
# =============================================================================

coordinate_rows = []

for t in range(LOOKBACK - 1, T):
    for s, sector in enumerate(sectors):
        coordinate_rows.append({
            "RowID": len(coordinate_rows),
            "t": t,
            "s": s,
            "Date": dates[t],
            "Sector": sector,
            "Split": legacy_splits[t],
            "RevisionPhase": revision_phases[t],
        })

coordinates = pd.DataFrame(coordinate_rows)

coord_t = coordinates["t"].to_numpy(dtype=int)
coord_s = coordinates["s"].to_numpy(dtype=int)
coord_split = coordinates["Split"].to_numpy()
coord_revision_phase = coordinates["RevisionPhase"].to_numpy()

N_COORD = len(coordinates)

log(f"Generic forecast-origin coordinates={N_COORD:,}")

# =============================================================================
# 13. BASE SUMMARY FEATURE MATRICES — INNER AND FINAL PREPROCESSING
# =============================================================================

summary_names = []
summary_origins = []
for feature_name in model_feature_names:
    summary_names.append(f"CUR__{feature_name}")
    summary_origins.append(feature_origin[feature_name])
    for window in XGB_WINDOWS:
        for stat in ["MEAN", "SD", "MIN", "MAX"]:
            summary_names.append(f"{stat}{window}__{feature_name}")
            summary_origins.append(feature_origin[feature_name])

summary_names += [f"SECTOR__{sector}" for sector in sectors]
summary_origins += ["__SECTOR__" for _ in sectors]
N_SUMMARY = len(summary_names)

def build_summary_matrix(panel: np.ndarray, label: str) -> np.ndarray:
    out = np.empty((N_COORD, N_SUMMARY), dtype=np.float32)
    for row_idx, (t, s) in enumerate(zip(coord_t, coord_s)):
        sequence = panel[t - LOOKBACK + 1:t + 1, s, :]
        pieces = [sequence[-1]]
        for window in XGB_WINDOWS:
            block = sequence[-window:]
            pieces.extend([
                block.mean(axis=0),
                block.std(axis=0),
                block.min(axis=0),
                block.max(axis=0),
            ])
        sector_one_hot = np.zeros(S, dtype=np.float32)
        sector_one_hot[s] = 1.0
        pieces.append(sector_one_hot)
        out[row_idx] = np.concatenate(pieces).astype(np.float32)

    if not np.all(np.isfinite(out)):
        raise RuntimeError(f"{label} summary feature matrix contains non-finite values.")
    return out

log(f"Building inner/final summary matrices: {N_COORD:,} rows x {N_SUMMARY:,} columns.")
X_summary_inner = build_summary_matrix(X_panel_inner, "TrainFit")
X_summary = build_summary_matrix(X_panel, "FullTrain")

# =============================================================================
# 14. DEFINITION-SPECIFIC CURRENT EVENT HISTORY FEATURES
# =============================================================================

EVENT_HISTORY_FEATURE_NAMES = [
    "CURRENT_EVENT",
    "EVENT_RATE_5",
    "EVENT_RATE_22",
    "EVENT_RATE_60",
    "EVENT_OBS_SHARE_5",
    "EVENT_OBS_SHARE_22",
    "EVENT_OBS_SHARE_60",
    "DAYS_SINCE_EVENT_CAP60",
]

def build_event_history_matrix(
    C: np.ndarray,
    M: np.ndarray,
) -> np.ndarray:

    out = np.zeros(
        (N_COORD, len(EVENT_HISTORY_FEATURE_NAMES)),
        dtype=np.float32,
    )

    for row_idx, (t, s) in enumerate(zip(coord_t, coord_s)):
        out[row_idx, 0] = C[t, s] if M[t, s] else 0.0

        col = 1

        for window in [5, 22, 60]:
            start = max(0, t - window + 1)
            events = C[start:t + 1, s]
            observed = M[start:t + 1, s]

            n_obs = int(observed.sum())

            out[row_idx, col] = (
                float(events[observed].mean())
                if n_obs > 0
                else 0.0
            )
            col += 1

        for window in [5, 22, 60]:
            start = max(0, t - window + 1)
            observed = M[start:t + 1, s]

            out[row_idx, col] = float(observed.mean())
            col += 1

        # Days since most recent observed crash, capped at 60.
        days_since = 60.0
        for lag in range(0, 60):
            tt = t - lag
            if tt < 0:
                break
            if M[tt, s] and C[tt, s] == 1:
                days_since = float(lag)
                break

        out[row_idx, col] = days_since / 60.0

    return out

event_history_matrices = {}

for definition_name, definition in event_definitions.items():
    event_history_matrices[definition_name] = build_event_history_matrix(
        definition["event"],
        definition["observed"],
    )

# =============================================================================
# 15. FEATURE SELECTION FOR EACH ABLATION
# =============================================================================

summary_origins_array = np.array(summary_origins, dtype=object)

def ablation_summary_indices(variant: str) -> np.ndarray:
    drop_group = ABLATION_DROP_GROUPS[variant]

    keep = np.array([
        (
            origin == "__SECTOR__"
            or origin not in drop_group
        )
        for origin in summary_origins_array
    ])

    return np.where(keep)[0]

ablation_indices = {
    variant: ablation_summary_indices(variant)
    for variant in ABLATION_DROP_GROUPS
}

# Parsimonious dynamic-logit design. This intentionally uses a compact,
# pre-specified economic subset and only current, 5-day mean and 22-day mean
# summaries, plus sector indicators and the event-history block.
LOGIT_CORE_ORIGINS = {
    "SectorReturn_Model",
    "GARCH_Sigma",
    "StdInnovation",
    "EVT_u", "EVT_scale", "EVT_shape",
    "Turnover_MCW", "AmihudILLIQ_Median",
    "BreadthNegative", "ReturnDispersion",
    "MarketReturn_MCW", "MarketBreadthNegative",
    "MarketReturnDispersion", "NSectorsDown",
    "CrossSectorReturnDispersion",
}

def logit_summary_indices(variant: str) -> np.ndarray:
    drop_group = ABLATION_DROP_GROUPS[variant]
    keep = []
    for i, (name, origin) in enumerate(zip(summary_names, summary_origins)):
        if origin == "__SECTOR__":
            keep.append(i)
            continue
        if origin not in LOGIT_CORE_ORIGINS or origin in drop_group:
            continue
        if name.startswith("CUR__") or name.startswith("MEAN5__") or name.startswith("MEAN22__"):
            keep.append(i)
    return np.asarray(keep, dtype=int)

logit_ablation_indices = {
    variant: logit_summary_indices(variant)
    for variant in ABLATION_DROP_GROUPS
}

pd.DataFrame({
    "Variant": list(logit_ablation_indices.keys()),
    "SummaryFeatureCount": [len(logit_ablation_indices[v]) for v in logit_ablation_indices],
    "EventHistoryFeatureCount": len(EVENT_HISTORY_FEATURE_NAMES),
    "TotalDesignFeatures": [len(logit_ablation_indices[v]) + len(EVENT_HISTORY_FEATURE_NAMES) for v in logit_ablation_indices],
}).to_csv(TABLE_DIR / "Table_305B_Dynamic_Logit_Parsimonious_Design.csv", index=False)

# =============================================================================
# 16. TARGET VECTOR ACCESS
# =============================================================================

def target_vector(
    definition_name: str,
    horizon: int,
    inner: bool = False,
) -> np.ndarray:
    key = "inner_targets" if inner else "targets"
    panel = event_definitions[definition_name][key][horizon]
    return np.array([
        panel[t, s]
        for t, s in zip(coord_t, coord_s)
    ], dtype=float)

def current_observed_vector(
    definition_name: str,
) -> np.ndarray:
    M = event_definitions[
        definition_name
    ]["observed"]

    return np.array([
        M[t, s]
        for t, s in zip(coord_t, coord_s)
    ], dtype=bool)

# =============================================================================
# 17. STRICT HISTORICAL BASELINES FOR EACH DEFINITION
# =============================================================================

historical_probabilities: dict[str, dict[str, dict[int, np.ndarray]]] = {}

for definition_name, definition in event_definitions.items():
    C = definition["event"]
    M = definition["observed"]

    fixed_prob = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }
    expanding_prob = {
        h: np.full((T, S), np.nan, dtype=np.float32)
        for h in HORIZONS
    }

    train_rates = np.zeros(S, dtype=float)

    for s in range(S):
        valid = train_date_mask & M[:, s]
        train_rates[s] = (
            float(C[valid, s].mean())
            if valid.sum()
            else 0.025
        )

    for h in HORIZONS:
        fixed_prob[h][:] = (
            1.0
            - (1.0 - train_rates[None, :]) ** h
        )

    event_sum = np.zeros(S, dtype=float)
    event_n = np.zeros(S, dtype=float)

    for t in range(T):
        for s in range(S):
            if M[t, s]:
                event_sum[s] += C[t, s]
                event_n[s] += 1.0

        one_day_p = np.divide(
            event_sum,
            np.maximum(event_n, 1.0),
        )

        for h in HORIZONS:
            expanding_prob[h][t] = (
                1.0
                - (1.0 - one_day_p) ** h
            )

    historical_probabilities[definition_name] = {
        "FixedHistorical": fixed_prob,
        "ExpandingHistorical": expanding_prob,
    }

# =============================================================================
# 18. METRIC UTILITIES
# =============================================================================

def log_score(
    y: np.ndarray,
    p: np.ndarray,
) -> float:
    p = np.clip(p, EPS, 1.0 - EPS)

    return float(
        -np.mean(
            y * np.log(p)
            + (1.0 - y) * np.log1p(-p)
        )
    )

def fit_calibration_intercept_slope(
    y: np.ndarray,
    p: np.ndarray,
) -> tuple[float, float]:

    y = np.asarray(y, dtype=float)
    p = np.clip(
        np.asarray(p, dtype=float),
        EPS,
        1.0 - EPS,
    )

    if len(np.unique(y)) < 2:
        return np.nan, np.nan

    x = np.log(p / (1.0 - p))

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-20.0, 20.0),
            (-10.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def probability_metrics(
    y_true: np.ndarray,
    probability: np.ndarray,
) -> dict:

    y_true = np.asarray(y_true, dtype=float)
    probability = np.asarray(probability, dtype=float)

    ok = np.isfinite(y_true) & np.isfinite(probability)

    y = y_true[ok]
    p = np.clip(
        probability[ok],
        EPS,
        1.0 - EPS,
    )

    result = {
        "N": int(len(y)),
        "Events": int(y.sum()) if len(y) else 0,
        "EventRate": float(y.mean()) if len(y) else np.nan,
        "Brier": np.nan,
        "LogScore": np.nan,
        "PR_AUC": np.nan,
        "ROC_AUC": np.nan,
        "CalibrationIntercept": np.nan,
        "CalibrationSlope": np.nan,
    }

    if len(y) == 0:
        return result

    result["Brier"] = float(
        np.mean((p - y) ** 2)
    )
    result["LogScore"] = log_score(y, p)

    if len(np.unique(y)) == 2:
        result["PR_AUC"] = float(
            average_precision_score(y, p)
        )
        result["ROC_AUC"] = float(
            roc_auc_score(y, p)
        )

        intercept, slope = fit_calibration_intercept_slope(
            y,
            p,
        )

        result["CalibrationIntercept"] = intercept
        result["CalibrationSlope"] = slope

    return result

# =============================================================================
# 19. XGBOOST CALIBRATION
# =============================================================================

def fit_binary_platt(
    p_validation: np.ndarray,
    y_validation: np.ndarray,
) -> tuple[float, float]:

    p_validation = np.clip(
        p_validation,
        EPS,
        1.0 - EPS,
    )

    x = np.log(
        p_validation / (1.0 - p_validation)
    )
    y = y_validation.astype(float)

    def objective(theta):
        intercept, slope = theta
        z = intercept + slope * x
        q = 1.0 / (
            1.0 + np.exp(-np.clip(z, -30, 30))
        )
        q = np.clip(q, EPS, 1.0 - EPS)

        return float(
            -np.mean(
                y * np.log(q)
                + (1.0 - y) * np.log1p(-q)
            )
        )

    result = minimize(
        objective,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[
            (-10.0, 10.0),
            (0.0, 10.0),
        ],
    )

    return float(result.x[0]), float(result.x[1])

def apply_binary_platt(
    p: np.ndarray,
    intercept: float,
    slope: float,
) -> np.ndarray:

    p = np.clip(p, EPS, 1.0 - EPS)
    x = np.log(p / (1.0 - p))
    z = intercept + slope * x

    return 1.0 / (
        1.0 + np.exp(-np.clip(z, -30, 30))
    )

# =============================================================================
# 20. MODEL-FITTING HELPERS
# =============================================================================

def assemble_model_matrix(
    definition_name: str,
    variant: str,
    model_family: str,
    inner: bool,
) -> tuple[np.ndarray, list[str]]:
    if model_family == "DynamicLogit":
        idx = logit_ablation_indices[variant]
    else:
        idx = ablation_indices[variant]

    summary_matrix = X_summary_inner if inner else X_summary
    X = np.concatenate(
        [
            summary_matrix[:, idx],
            event_history_matrices[definition_name],
        ],
        axis=1,
    ).astype(np.float32)
    names = [summary_names[i] for i in idx] + EVENT_HISTORY_FEATURE_NAMES
    return X, names


def fit_xgboost_horizon(
    X_inner: np.ndarray,
    X_final: np.ndarray,
    y_inner: np.ndarray,
    y_final: np.ndarray,
    horizon: int,
    model_tag: str,
) -> dict:
    fit_idx = (coord_revision_phase == "TrainFit") & np.isfinite(y_inner)
    tune_idx = (coord_revision_phase == "TrainTune") & np.isfinite(y_inner)
    full_train_idx = np.isin(coord_revision_phase, ["TrainFit", "TrainTune"]) & np.isfinite(y_final)
    cal_idx = (coord_revision_phase == "Calibration") & np.isfinite(y_final)
    test_idx = (coord_revision_phase == "Test") & np.isfinite(y_final)

    y_fit = y_inner[fit_idx].astype(int)
    y_tune = y_inner[tune_idx].astype(int)
    y_full = y_final[full_train_idx].astype(int)
    y_cal = y_final[cal_idx].astype(int)

    if len(np.unique(y_fit)) < 2 or len(np.unique(y_tune)) < 2:
        raise RuntimeError(f"XGBoost inner split has one class: {model_tag}, h={horizon}")

    pos_fit = max(int(y_fit.sum()), 1)
    neg_fit = max(len(y_fit) - pos_fit, 1)
    tune_pos_weight = min(math.sqrt(neg_fit / pos_fit), XGB_POS_WEIGHT_CAP)

    params_tune = {
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "eta": 0.03,
        "max_depth": 4,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "lambda": 1.0,
        "alpha": 0.0,
        "scale_pos_weight": tune_pos_weight,
        "seed": SEED,
        "nthread": max(1, XGB_NTHREAD),
        "tree_method": "hist",
    }

    dfit = xgb.DMatrix(X_inner[fit_idx], label=y_fit)
    dtune = xgb.DMatrix(X_inner[tune_idx], label=y_tune)
    tune_booster = xgb.train(
        params=params_tune,
        dtrain=dfit,
        num_boost_round=XGB_MAX_ROUNDS,
        evals=[(dtune, "train_tune")],
        early_stopping_rounds=XGB_EARLY_STOP,
        verbose_eval=False,
    )
    best_iteration = int(
        tune_booster.best_iteration
        if tune_booster.best_iteration is not None
        else XGB_MAX_ROUNDS - 1
    )
    selected_rounds = best_iteration + 1

    pos_full = max(int(y_full.sum()), 1)
    neg_full = max(len(y_full) - pos_full, 1)
    full_pos_weight = min(math.sqrt(neg_full / pos_full), XGB_POS_WEIGHT_CAP)
    params_final = dict(params_tune)
    params_final["scale_pos_weight"] = full_pos_weight

    dfull = xgb.DMatrix(X_final[full_train_idx], label=y_full)
    booster = xgb.train(
        params=params_final,
        dtrain=dfull,
        num_boost_round=selected_rounds,
        evals=[],
        verbose_eval=False,
    )

    p_cal_raw = booster.predict(xgb.DMatrix(X_final[cal_idx]))
    intercept, slope = fit_binary_platt(p_cal_raw, y_cal)
    p_cal = apply_binary_platt(p_cal_raw, intercept, slope)
    p_test_raw = booster.predict(xgb.DMatrix(X_final[test_idx]))
    p_test = apply_binary_platt(p_test_raw, intercept, slope)

    return {
        "booster": booster,
        "best_iteration": best_iteration,
        "selected_rounds": selected_rounds,
        "trainfit_scale_pos_weight": float(tune_pos_weight),
        "full_train_scale_pos_weight": float(full_pos_weight),
        "platt_intercept": intercept,
        "platt_slope": slope,
        "fit_idx": fit_idx,
        "tune_idx": tune_idx,
        "full_train_idx": full_train_idx,
        "cal_idx": cal_idx,
        "test_idx": test_idx,
        "p_cal_raw": p_cal_raw,
        "p_cal": p_cal,
        "p_test_raw": p_test_raw,
        "p_test": p_test,
    }


def _ridge_logit_fit(
    X: np.ndarray,
    y: np.ndarray,
    ridge_lambda: float,
) -> dict:
    """Convex ridge-logistic fit with an unpenalized intercept."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    n, p = X.shape

    def objective(theta):
        intercept = theta[0]
        beta = theta[1:]
        z = intercept + X @ beta
        # Stable mean logistic loss.
        loss = np.mean(np.logaddexp(0.0, z) - y * z)
        penalty = 0.5 * ridge_lambda * np.sum(beta * beta)
        return float(loss + penalty)

    def gradient(theta):
        intercept = theta[0]
        beta = theta[1:]
        z = np.clip(intercept + X @ beta, -35.0, 35.0)
        prob = 1.0 / (1.0 + np.exp(-z))
        resid = prob - y
        g0 = np.mean(resid)
        gb = (X.T @ resid) / n + ridge_lambda * beta
        return np.concatenate([[g0], gb])

    prevalence = float(np.clip(y.mean(), EPS, 1.0 - EPS))
    theta0 = np.zeros(p + 1, dtype=np.float64)
    theta0[0] = np.log(prevalence / (1.0 - prevalence))

    result = minimize(
        objective,
        theta0,
        jac=gradient,
        method="L-BFGS-B",
        options={"maxiter": 150, "maxfun": 1000, "ftol": 1e-8, "gtol": 1e-5},
    )
    if not np.isfinite(result.fun):
        raise RuntimeError("Ridge-logit optimizer returned non-finite objective.")

    return {
        "intercept": float(result.x[0]),
        "coef": result.x[1:].astype(np.float64),
        "objective": float(result.fun),
        "converged": bool(result.success),
        "status": int(result.status),
        "message": str(result.message),
        "iterations": int(getattr(result, "nit", -1)),
    }


def _ridge_logit_predict(model: dict, X: np.ndarray) -> np.ndarray:
    z = model["intercept"] + np.asarray(X, dtype=np.float64) @ model["coef"]
    return 1.0 / (1.0 + np.exp(-np.clip(z, -35.0, 35.0)))


def fit_dynamic_logit_horizon(
    X_inner: np.ndarray,
    X_final: np.ndarray,
    y_inner: np.ndarray,
    y_final: np.ndarray,
    horizon: int,
    model_tag: str,
) -> dict:
    fit_idx = (coord_revision_phase == "TrainFit") & np.isfinite(y_inner)
    tune_idx = (coord_revision_phase == "TrainTune") & np.isfinite(y_inner)
    full_train_idx = np.isin(coord_revision_phase, ["TrainFit", "TrainTune"]) & np.isfinite(y_final)
    cal_idx = (coord_revision_phase == "Calibration") & np.isfinite(y_final)
    test_idx = (coord_revision_phase == "Test") & np.isfinite(y_final)

    X_fit_raw = X_inner[fit_idx].astype(np.float64)
    X_tune_raw = X_inner[tune_idx].astype(np.float64)
    y_fit = y_inner[fit_idx].astype(int)
    y_tune = y_inner[tune_idx].astype(int)

    mean_inner = X_fit_raw.mean(axis=0)
    sd_inner = X_fit_raw.std(axis=0)
    keep = np.isfinite(mean_inner) & np.isfinite(sd_inner) & (sd_inner > 1e-8)
    if keep.sum() == 0:
        raise RuntimeError(f"Dynamic ridge-logit has no usable columns: {model_tag}, h={horizon}")

    X_fit = (X_fit_raw[:, keep] - mean_inner[keep]) / sd_inner[keep]
    X_tune = (X_tune_raw[:, keep] - mean_inner[keep]) / sd_inner[keep]

    best = None
    tuning_rows = []
    for ridge_lambda in LOGIT_LAMBDA_GRID:
        model = _ridge_logit_fit(X_fit, y_fit, ridge_lambda)
        p_tune = _ridge_logit_predict(model, X_tune)
        tune_loss = log_score(y_tune, p_tune)
        tuning_rows.append({
            "RidgeLambda": ridge_lambda,
            "TrainTuneLogScore": tune_loss,
            "OptimizerConverged": model["converged"],
            "OptimizerIterations": model["iterations"],
            "TrainFitN": int(fit_idx.sum()),
            "TrainTuneN": int(tune_idx.sum()),
        })
        if best is None or tune_loss < best["TrainTuneLogScore"]:
            best = {
                "ridge_lambda": ridge_lambda,
                "TrainTuneLogScore": tune_loss,
            }

    if best is None:
        raise RuntimeError(f"Dynamic ridge-logit tuning failed: {model_tag}, h={horizon}")

    # Fresh Full-Train scaling and fit at the frozen TrainTune-selected penalty.
    X_full_raw = X_final[full_train_idx].astype(np.float64)[:, keep]
    X_cal_raw = X_final[cal_idx].astype(np.float64)[:, keep]
    X_test_raw = X_final[test_idx].astype(np.float64)[:, keep]
    y_full = y_final[full_train_idx].astype(int)
    y_cal = y_final[cal_idx].astype(int)

    mean_final = X_full_raw.mean(axis=0)
    sd_final = X_full_raw.std(axis=0)
    sd_final[sd_final < 1e-8] = 1.0
    X_full = (X_full_raw - mean_final) / sd_final
    X_cal = (X_cal_raw - mean_final) / sd_final
    X_test = (X_test_raw - mean_final) / sd_final

    final_model = _ridge_logit_fit(X_full, y_full, float(best["ridge_lambda"]))
    p_cal_raw = _ridge_logit_predict(final_model, X_cal)
    platt_intercept, platt_slope = fit_binary_platt(p_cal_raw, y_cal)
    p_cal = apply_binary_platt(p_cal_raw, platt_intercept, platt_slope)
    p_test_raw = _ridge_logit_predict(final_model, X_test)
    p_test = apply_binary_platt(p_test_raw, platt_intercept, platt_slope)

    return {
        "model": final_model,
        "selected_lambda": float(best["ridge_lambda"]),
        "train_tune_logscore": float(best["TrainTuneLogScore"]),
        "keep_columns": keep,
        "design_mean": mean_final,
        "design_sd": sd_final,
        "platt_intercept": platt_intercept,
        "platt_slope": platt_slope,
        "parameter_count": int(keep.sum() + 1),
        "fit_idx": fit_idx,
        "tune_idx": tune_idx,
        "full_train_idx": full_train_idx,
        "cal_idx": cal_idx,
        "test_idx": test_idx,
        "p_cal_raw": p_cal_raw,
        "p_cal": p_cal,
        "p_test_raw": p_test_raw,
        "p_test": p_test,
        "tuning": pd.DataFrame(tuning_rows),
    }

# =============================================================================
# 21. PROBABILITY-PANEL HELPERS / HORIZON MONOTONICITY
# =============================================================================

def blank_probability_panels() -> dict[str, dict[int, np.ndarray]]:
    return {
        "Calibration": {
            h: np.full((T, S), np.nan, dtype=np.float32)
            for h in HORIZONS
        },
        "Test": {
            h: np.full((T, S), np.nan, dtype=np.float32)
            for h in HORIZONS
        },
    }

def scatter_predictions_to_panel(
    panel_dict: dict[str, dict[int, np.ndarray]],
    split_name: str,
    horizon: int,
    row_mask: np.ndarray,
    probabilities: np.ndarray,
):
    selected_rows = np.where(row_mask)[0]

    if len(selected_rows) != len(probabilities):
        raise RuntimeError("Prediction scatter length mismatch.")

    for local_idx, row_idx in enumerate(selected_rows):
        t = coord_t[row_idx]
        s = coord_s[row_idx]
        panel_dict[split_name][horizon][t, s] = float(probabilities[local_idx])

def monotone_rearrange(
    panel_dict: dict[str, dict[int, np.ndarray]],
):
    for split_name in ["Calibration", "Test"]:
        stack = np.stack(
            [panel_dict[split_name][h] for h in HORIZONS],
            axis=-1,
        )

        finite_all = np.all(np.isfinite(stack), axis=-1)

        rearranged = np.maximum.accumulate(stack, axis=-1)

        for k, h in enumerate(HORIZONS):
            target_panel = panel_dict[split_name][h]
            target_panel[finite_all] = rearranged[..., k][finite_all]

# =============================================================================
# 22. RUN ONE MODEL FAMILY ACROSS FOUR HORIZONS
# =============================================================================

def run_model_family(
    definition_name: str,
    variant: str,
    model_family: str,
    save_primary_objects: bool = False,
) -> dict:
    X_inner, feature_names = assemble_model_matrix(
        definition_name, variant, model_family, inner=True
    )
    X_final, feature_names_final = assemble_model_matrix(
        definition_name, variant, model_family, inner=False
    )
    if feature_names != feature_names_final:
        raise RuntimeError("Inner/final feature-name mismatch.")

    probability_panels = blank_probability_panels()
    raw_probability_panels = blank_probability_panels()
    fit_metadata_rows = []
    importance_rows = []
    logit_tuning_frames = []

    for h in HORIZONS:
        y_inner = target_vector(definition_name, h, inner=True)
        y_final = target_vector(definition_name, h, inner=False)
        tag = f"{model_family}__{definition_name}__{variant}"
        log(f"Fitting {model_family}: definition={definition_name}, variant={variant}, h={h}")

        if model_family == "XGBoost":
            fit = fit_xgboost_horizon(
                X_inner=X_inner,
                X_final=X_final,
                y_inner=y_inner,
                y_final=y_final,
                horizon=h,
                model_tag=tag,
            )
            for split_name, idx, p_raw, p_cal in [
                ("Calibration", fit["cal_idx"], fit["p_cal_raw"], fit["p_cal"]),
                ("Test", fit["test_idx"], fit["p_test_raw"], fit["p_test"]),
            ]:
                scatter_predictions_to_panel(probability_panels, split_name, h, idx, p_cal)
                scatter_predictions_to_panel(raw_probability_panels, split_name, h, idx, p_raw)

            fit_metadata_rows.append({
                "ModelFamily": model_family,
                "Definition": definition_name,
                "Variant": variant,
                "Horizon": h,
                "SelectedRounds": fit["selected_rounds"],
                "BestIterationZeroBased": fit["best_iteration"],
                "TrainFitScalePosWeight": fit["trainfit_scale_pos_weight"],
                "FullTrainScalePosWeight": fit["full_train_scale_pos_weight"],
                "PlattIntercept": fit["platt_intercept"],
                "PlattSlope": fit["platt_slope"],
                "SelectedLambda": np.nan,
                "ParameterCount": np.nan,
                "TrainFitN": int(fit["fit_idx"].sum()),
                "TrainTuneN": int(fit["tune_idx"].sum()),
                "FullTrainN": int(fit["full_train_idx"].sum()),
                "CalibrationN": int(fit["cal_idx"].sum()),
                "TestN": int(fit["test_idx"].sum()),
            })

            score = fit["booster"].get_score(importance_type="gain")
            for raw_name, gain in score.items():
                if raw_name.startswith("f"):
                    j = int(raw_name[1:])
                    feature_name = feature_names[j] if j < len(feature_names) else raw_name
                else:
                    feature_name = raw_name
                importance_rows.append({
                    "Definition": definition_name,
                    "Variant": variant,
                    "Horizon": h,
                    "Feature": feature_name,
                    "Gain": float(gain),
                })

            if save_primary_objects:
                fit["booster"].save_model(
                    MODEL_DIR / f"XGBoost_{definition_name}_{variant}_h{h}.json"
                )

        elif model_family == "DynamicLogit":
            fit = fit_dynamic_logit_horizon(
                X_inner=X_inner,
                X_final=X_final,
                y_inner=y_inner,
                y_final=y_final,
                horizon=h,
                model_tag=tag,
            )
            for split_name, idx, p_raw, p_cal in [
                ("Calibration", fit["cal_idx"], fit["p_cal_raw"], fit["p_cal"]),
                ("Test", fit["test_idx"], fit["p_test_raw"], fit["p_test"]),
            ]:
                scatter_predictions_to_panel(probability_panels, split_name, h, idx, p_cal)
                scatter_predictions_to_panel(raw_probability_panels, split_name, h, idx, p_raw)

            fit_metadata_rows.append({
                "ModelFamily": model_family,
                "Definition": definition_name,
                "Variant": variant,
                "Horizon": h,
                "SelectedRounds": np.nan,
                "BestIterationZeroBased": np.nan,
                "TrainFitScalePosWeight": np.nan,
                "FullTrainScalePosWeight": np.nan,
                "PlattIntercept": fit["platt_intercept"],
                "PlattSlope": fit["platt_slope"],
                "SelectedLambda": fit["selected_lambda"],
                "ParameterCount": fit["parameter_count"],
                "TrainFitN": int(fit["fit_idx"].sum()),
                "TrainTuneN": int(fit["tune_idx"].sum()),
                "FullTrainN": int(fit["full_train_idx"].sum()),
                "CalibrationN": int(fit["cal_idx"].sum()),
                "TestN": int(fit["test_idx"].sum()),
            })

            tuning = fit["tuning"].copy()
            tuning["Definition"] = definition_name
            tuning["Variant"] = variant
            tuning["Horizon"] = h
            logit_tuning_frames.append(tuning)

            if save_primary_objects:
                with open(
                    MODEL_DIR / f"DynamicRidgeLogit_{definition_name}_{variant}_h{h}.pkl",
                    "wb",
                ) as handle:
                    pickle.dump({
                        "model": fit["model"],
                        "keep_columns": fit["keep_columns"],
                        "design_mean": fit["design_mean"],
                        "design_sd": fit["design_sd"],
                        "feature_names": feature_names,
                        "selected_lambda": fit["selected_lambda"],
                        "platt_intercept": fit["platt_intercept"],
                        "platt_slope": fit["platt_slope"],
                        "parameter_count": fit["parameter_count"],
                    }, handle)
        else:
            raise ValueError(f"Unknown model family: {model_family}")

        gc.collect()

    monotone_rearrange(probability_panels)
    monotone_rearrange(raw_probability_panels)

    return {
        "probabilities": probability_panels,
        "raw_probabilities": raw_probability_panels,
        "fit_metadata": pd.DataFrame(fit_metadata_rows),
        "importance": pd.DataFrame(importance_rows) if importance_rows else pd.DataFrame(),
        "logit_tuning": (
            pd.concat(logit_tuning_frames, ignore_index=True)
            if logit_tuning_frames else pd.DataFrame()
        ),
    }

# =============================================================================
# 23. PHASE 3A — PRIMARY 2.5% COMPONENT ABLATIONS
# =============================================================================

component_results: dict[tuple[str, str], dict] = {}
component_raw_results: dict[tuple[str, str], dict] = {}
all_fit_metadata = []
all_importance = []
all_logit_tuning = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    for variant in ABLATION_DROP_GROUPS:

        save_objects = (
            variant == "Full"
            and model_family in {"XGBoost", "DynamicLogit"}
        )

        result = run_model_family(
            definition_name="EVT_0025",
            variant=variant,
            model_family=model_family,
            save_primary_objects=save_objects,
        )

        component_results[(model_family, variant)] = result["probabilities"]
        component_raw_results[(model_family, variant)] = result["raw_probabilities"]

        all_fit_metadata.append(
            result["fit_metadata"]
        )

        if not result["importance"].empty:
            all_importance.append(
                result["importance"]
            )

        if not result["logit_tuning"].empty:
            all_logit_tuning.append(
                result["logit_tuning"]
            )

# =============================================================================
# 24. PHASE 3B — CRASH-DEFINITION ROBUSTNESS
# =============================================================================
#
# EVT_0025 Full already exists above and is reused. Only the other definitions
# require additional fits.

robustness_results: dict[tuple[str, str], dict] = {}
robustness_raw_results: dict[tuple[str, str], dict] = {}

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    robustness_results[(model_family, "EVT_0025")] = component_results[(model_family, "Full")]
    robustness_raw_results[(model_family, "EVT_0025")] = component_raw_results[(model_family, "Full")]

    for definition_name in [
        "EVT_001",
        "EVT_005",
        "Fixed_0025",
    ]:
        result = run_model_family(
            definition_name=definition_name,
            variant="Full",
            model_family=model_family,
            save_primary_objects=False,
        )

        robustness_results[(model_family, definition_name)] = result["probabilities"]
        robustness_raw_results[(model_family, definition_name)] = result["raw_probabilities"]

        all_fit_metadata.append(
            result["fit_metadata"]
        )

        if not result["importance"].empty:
            all_importance.append(
                result["importance"]
            )

        if not result["logit_tuning"].empty:
            all_logit_tuning.append(
                result["logit_tuning"]
            )

fit_metadata = pd.concat(
    all_fit_metadata,
    ignore_index=True,
)

fit_metadata.to_csv(
    TABLE_DIR / "Table_306_Model_Fitting_Metadata.csv",
    index=False,
)

if all_logit_tuning:
    logit_tuning = pd.concat(
        all_logit_tuning,
        ignore_index=True,
    )

    logit_tuning.to_csv(
        TABLE_DIR / "Table_307_Dynamic_Logit_TrainTune_Tuning.csv",
        index=False,
    )
else:
    logit_tuning = pd.DataFrame()

# =============================================================================
# 25. METRICS FOR COMPONENT ABLATIONS
# =============================================================================

component_metric_rows = []

primary_targets = event_definitions[
    "EVT_0025"
]["targets"]

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    for variant in ABLATION_DROP_GROUPS:

        predictions = component_results[
            (model_family, variant)
        ]

        for split_name, split_mask in [
            ("Calibration", calibration_date_mask),
            ("Test", test_date_mask),
        ]:
            for h in HORIZONS:
                y_panel = primary_targets[h]
                p_panel = predictions[split_name][h]

                pooled = probability_metrics(
                    y_panel[split_mask].reshape(-1),
                    p_panel[split_mask].reshape(-1),
                )

                component_metric_rows.append({
                    "Split": split_name,
                    "ModelFamily": model_family,
                    "Variant": variant,
                    "Sector": "POOLED",
                    "Horizon": h,
                    **pooled,
                })

                if split_name == "Test":
                    for s, sector in enumerate(sectors):
                        sector_metrics = probability_metrics(
                            y_panel[split_mask, s],
                            p_panel[split_mask, s],
                        )

                        component_metric_rows.append({
                            "Split": split_name,
                            "ModelFamily": model_family,
                            "Variant": variant,
                            "Sector": sector,
                            "Horizon": h,
                            **sector_metrics,
                        })

component_metrics = pd.DataFrame(
    component_metric_rows
)

component_metrics.to_csv(
    TABLE_DIR / "Table_308_Component_Ablation_Metrics.csv",
    index=False,
)

# =============================================================================
# 26. COMPONENT BRIER SKILL / DEGRADATION RELATIVE TO FULL
# =============================================================================

component_pooled_test = component_metrics[
    (component_metrics["Split"] == "Test")
    & (component_metrics["Sector"] == "POOLED")
].copy()

full_reference = (
    component_pooled_test[
        component_pooled_test["Variant"] == "Full"
    ][
        ["ModelFamily", "Horizon", "Brier", "LogScore", "PR_AUC"]
    ]
    .rename(columns={
        "Brier": "FullBrier",
        "LogScore": "FullLogScore",
        "PR_AUC": "FullPR_AUC",
    })
)

component_skill = component_pooled_test.merge(
    full_reference,
    on=["ModelFamily", "Horizon"],
    how="left",
)

component_skill["BrierDegradation_vs_Full"] = (
    component_skill["Brier"]
    - component_skill["FullBrier"]
)

component_skill["BrierSkill_Full_vs_Ablation"] = (
    1.0
    - component_skill["FullBrier"]
    / component_skill["Brier"]
)

component_skill["LogScoreDegradation_vs_Full"] = (
    component_skill["LogScore"]
    - component_skill["FullLogScore"]
)

component_skill["PR_AUCLoss_vs_Full"] = (
    component_skill["FullPR_AUC"]
    - component_skill["PR_AUC"]
)

component_skill.to_csv(
    TABLE_DIR / "Table_309_Component_Contribution_Test.csv",
    index=False,
)

# =============================================================================
# 27. METRICS FOR CRASH-DEFINITION ROBUSTNESS
# =============================================================================

robustness_metric_rows = []

for definition_name, definition in event_definitions.items():
    targets = definition["targets"]

    # Definition-specific historical baselines.
    for baseline_name in [
        "FixedHistorical",
        "ExpandingHistorical",
    ]:
        baseline = historical_probabilities[
            definition_name
        ][baseline_name]

        for split_name, split_mask in [
            ("Calibration", calibration_date_mask),
            ("Test", test_date_mask),
        ]:
            for h in HORIZONS:
                metrics = probability_metrics(
                    targets[h][split_mask].reshape(-1),
                    baseline[h][split_mask].reshape(-1),
                )

                robustness_metric_rows.append({
                    "Definition": definition_name,
                    "Split": split_name,
                    "ModelFamily": baseline_name,
                    "Sector": "POOLED",
                    "Horizon": h,
                    **metrics,
                })

    # Fitted forecasting models.
    for model_family in [
        "XGBoost",
        "DynamicLogit",
    ]:
        predictions = robustness_results[
            (model_family, definition_name)
        ]

        for split_name, split_mask in [
            ("Calibration", calibration_date_mask),
            ("Test", test_date_mask),
        ]:
            for h in HORIZONS:
                pooled = probability_metrics(
                    targets[h][split_mask].reshape(-1),
                    predictions[split_name][h][split_mask].reshape(-1),
                )

                robustness_metric_rows.append({
                    "Definition": definition_name,
                    "Split": split_name,
                    "ModelFamily": model_family,
                    "Sector": "POOLED",
                    "Horizon": h,
                    **pooled,
                })

                if split_name == "Test":
                    for s, sector in enumerate(sectors):
                        sector_metrics = probability_metrics(
                            targets[h][split_mask, s],
                            predictions[split_name][h][split_mask, s],
                        )

                        robustness_metric_rows.append({
                            "Definition": definition_name,
                            "Split": split_name,
                            "ModelFamily": model_family,
                            "Sector": sector,
                            "Horizon": h,
                            **sector_metrics,
                        })

robustness_metrics = pd.DataFrame(
    robustness_metric_rows
)

robustness_metrics.to_csv(
    TABLE_DIR / "Table_310_Crash_Definition_Robustness_Metrics.csv",
    index=False,
)

# =============================================================================
# 28. DEFINITION-SPECIFIC BRIER SKILL VS OWN EXPANDING HISTORICAL BASELINE
# =============================================================================

robustness_pooled_test = robustness_metrics[
    (robustness_metrics["Split"] == "Test")
    & (robustness_metrics["Sector"] == "POOLED")
].copy()

historical_reference = (
    robustness_pooled_test[
        robustness_pooled_test["ModelFamily"] == "ExpandingHistorical"
    ][
        ["Definition", "Horizon", "Brier", "LogScore"]
    ]
    .rename(columns={
        "Brier": "HistoricalBrier",
        "LogScore": "HistoricalLogScore",
    })
)

definition_skill = robustness_pooled_test.merge(
    historical_reference,
    on=["Definition", "Horizon"],
    how="left",
)

definition_skill["BrierSkill_vs_OwnHistorical"] = (
    1.0
    - definition_skill["Brier"]
    / definition_skill["HistoricalBrier"]
)

definition_skill["LogScoreImprovement_vs_OwnHistorical"] = (
    definition_skill["HistoricalLogScore"]
    - definition_skill["LogScore"]
)

definition_skill.to_csv(
    TABLE_DIR / "Table_311_Crash_Definition_Brier_Skill.csv",
    index=False,
)

# =============================================================================
# 29. MOVING-BLOCK BOOTSTRAP UTILITIES
# =============================================================================

def moving_block_sample_indices(
    n_dates: int,
    block_length: int,
    rng: np.random.Generator,
) -> np.ndarray:

    selected = []

    while len(selected) < n_dates:
        if n_dates <= block_length:
            start = 0
        else:
            start = int(
                rng.integers(
                    0,
                    n_dates - block_length + 1,
                )
            )

        selected.extend(
            range(
                start,
                min(n_dates, start + block_length),
            )
        )

    return np.array(
        selected[:n_dates],
        dtype=int,
    )

def date_level_losses(
    y_panel: np.ndarray,
    p_panel: np.ndarray,
    date_indices: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:

    brier = np.full(len(date_indices), np.nan, dtype=float)
    logloss = np.full(len(date_indices), np.nan, dtype=float)

    for local_t, t in enumerate(date_indices):
        y = y_panel[t]
        p = p_panel[t]

        ok = np.isfinite(y) & np.isfinite(p)

        if not ok.any():
            continue

        yy = y[ok]
        pp = np.clip(
            p[ok],
            EPS,
            1.0 - EPS,
        )

        brier[local_t] = float(
            np.mean((pp - yy) ** 2)
        )

        logloss[local_t] = float(
            -np.mean(
                yy * np.log(pp)
                + (1.0 - yy) * np.log1p(-pp)
            )
        )

    return brier, logloss

test_date_indices = np.where(test_date_mask)[0]

# =============================================================================
# 30. PAIRED BOOTSTRAP — FULL MODEL VS COMPONENT ABLATIONS
# =============================================================================

component_bootstrap_rows = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    full_predictions = component_results[
        (model_family, "Full")
    ]

    for variant in ABLATION_DROP_GROUPS:
        if variant == "Full":
            continue

        ablated_predictions = component_results[
            (model_family, variant)
        ]

        for h in HORIZONS:
            y_panel = primary_targets[h]

            full_brier, full_log = date_level_losses(
                y_panel,
                full_predictions["Test"][h],
                test_date_indices,
            )

            abl_brier, abl_log = date_level_losses(
                y_panel,
                ablated_predictions["Test"][h],
                test_date_indices,
            )

            valid = (
                np.isfinite(full_brier)
                & np.isfinite(abl_brier)
                & np.isfinite(full_log)
                & np.isfinite(abl_log)
            )

            fb = full_brier[valid]
            ab = abl_brier[valid]
            fl = full_log[valid]
            al = abl_log[valid]

            observed_brier_diff = float(
                np.mean(fb - ab)
            )
            observed_log_diff = float(
                np.mean(fl - al)
            )

            rng = np.random.default_rng(
                SEED
                + 1000 * h
                + sum(ord(c) for c in f"{model_family}{variant}")
            )

            brier_diffs = []
            log_diffs = []

            for _ in range(BOOTSTRAP_REPS):
                idx = moving_block_sample_indices(
                    len(fb),
                    min(BOOTSTRAP_BLOCK, len(fb)),
                    rng,
                )

                brier_diffs.append(
                    float(np.mean(fb[idx] - ab[idx]))
                )

                log_diffs.append(
                    float(np.mean(fl[idx] - al[idx]))
                )

            brier_diffs = np.asarray(brier_diffs)
            log_diffs = np.asarray(log_diffs)

            component_bootstrap_rows.append({
                "ModelFamily": model_family,
                "Ablation": variant,
                "Horizon": h,
                "BrierDiff_FullMinusAblation": observed_brier_diff,
                "BrierDiff_CI2.5": float(
                    np.quantile(brier_diffs, 0.025)
                ),
                "BrierDiff_CI97.5": float(
                    np.quantile(brier_diffs, 0.975)
                ),
                "Probability_Full_Better_Brier": float(
                    np.mean(brier_diffs < 0)
                ),
                "LogScoreDiff_FullMinusAblation": observed_log_diff,
                "LogScoreDiff_CI2.5": float(
                    np.quantile(log_diffs, 0.025)
                ),
                "LogScoreDiff_CI97.5": float(
                    np.quantile(log_diffs, 0.975)
                ),
                "Probability_Full_Better_LogScore": float(
                    np.mean(log_diffs < 0)
                ),
                "BootstrapReps": BOOTSTRAP_REPS,
                "BlockLength": BOOTSTRAP_BLOCK,
            })

component_bootstrap = pd.DataFrame(
    component_bootstrap_rows
)

component_bootstrap.to_csv(
    TABLE_DIR / "Table_312_Component_Ablation_Block_Bootstrap.csv",
    index=False,
)

# Focused H1 evidence.
h1_bootstrap = component_bootstrap[
    component_bootstrap["Ablation"].isin(
        ["NoGARCH", "NoVolatilityGroup"]
    )
].copy()

h1_bootstrap.to_csv(
    TABLE_DIR / "Table_313_H1_Volatility_Incremental_Evidence.csv",
    index=False,
)

# =============================================================================
# 31. EVT-vs-FIXED FORECASTABILITY BOOTSTRAP
# =============================================================================
#
# Different crash definitions produce different outcomes, so we do NOT compare
# their raw Brier scores as if they were the same target. Instead each fitted
# model is normalized by its OWN expanding-historical benchmark:
#
#     Skill = 1 - ModelBS / HistoricalBS.
#
# The bootstrap reports:
#
#     Skill_EVT_0025 - Skill_Fixed_0025
#
# Positive values favor the volatility-adjusted EVT definition.

definition_skill_bootstrap_rows = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    for h in HORIZONS:

        evt_target = event_definitions[
            "EVT_0025"
        ]["targets"][h]

        fix_target = event_definitions[
            "Fixed_0025"
        ]["targets"][h]

        evt_model = robustness_results[
            (model_family, "EVT_0025")
        ]["Test"][h]

        fix_model = robustness_results[
            (model_family, "Fixed_0025")
        ]["Test"][h]

        evt_hist = historical_probabilities[
            "EVT_0025"
        ]["ExpandingHistorical"][h]

        fix_hist = historical_probabilities[
            "Fixed_0025"
        ]["ExpandingHistorical"][h]

        evt_model_b, _ = date_level_losses(
            evt_target,
            evt_model,
            test_date_indices,
        )
        evt_hist_b, _ = date_level_losses(
            evt_target,
            evt_hist,
            test_date_indices,
        )

        fix_model_b, _ = date_level_losses(
            fix_target,
            fix_model,
            test_date_indices,
        )
        fix_hist_b, _ = date_level_losses(
            fix_target,
            fix_hist,
            test_date_indices,
        )

        # Calendar-date sampling is paired, but each definition averages its own
        # finite date-level losses within a bootstrap draw.
        n_dates = len(test_date_indices)

        def skill(model_loss, hist_loss, idx):
            mm = model_loss[idx]
            hh = hist_loss[idx]
            ok = np.isfinite(mm) & np.isfinite(hh)

            if not ok.any():
                return np.nan

            denom = np.mean(hh[ok])

            if denom <= EPS:
                return np.nan

            return float(
                1.0 - np.mean(mm[ok]) / denom
            )

        full_idx = np.arange(n_dates)

        evt_skill_obs = skill(
            evt_model_b,
            evt_hist_b,
            full_idx,
        )
        fix_skill_obs = skill(
            fix_model_b,
            fix_hist_b,
            full_idx,
        )

        observed_difference = (
            evt_skill_obs - fix_skill_obs
        )

        rng = np.random.default_rng(
            SEED
            + 7000 * h
            + sum(ord(c) for c in model_family)
        )

        differences = []

        for _ in range(BOOTSTRAP_REPS):
            idx = moving_block_sample_indices(
                n_dates,
                min(BOOTSTRAP_BLOCK, n_dates),
                rng,
            )

            evt_skill = skill(
                evt_model_b,
                evt_hist_b,
                idx,
            )

            fix_skill = skill(
                fix_model_b,
                fix_hist_b,
                idx,
            )

            if np.isfinite(evt_skill) and np.isfinite(fix_skill):
                differences.append(
                    evt_skill - fix_skill
                )

        differences = np.asarray(differences)

        definition_skill_bootstrap_rows.append({
            "ModelFamily": model_family,
            "Horizon": h,
            "EVT_BrierSkill": evt_skill_obs,
            "Fixed_BrierSkill": fix_skill_obs,
            "SkillDifference_EVTminusFixed": observed_difference,
            "SkillDifference_CI2.5": float(
                np.quantile(differences, 0.025)
            ),
            "SkillDifference_CI97.5": float(
                np.quantile(differences, 0.975)
            ),
            "Probability_EVT_Skill_Higher": float(
                np.mean(differences > 0)
            ),
            "BootstrapReps": int(len(differences)),
            "BlockLength": BOOTSTRAP_BLOCK,
        })

definition_skill_bootstrap = pd.DataFrame(
    definition_skill_bootstrap_rows
)

definition_skill_bootstrap.to_csv(
    TABLE_DIR / "Table_314_H2_EVT_vs_Fixed_BrierSkill_Bootstrap.csv",
    index=False,
)

# =============================================================================
# 32. FEATURE IMPORTANCE FOR PRIMARY FULL XGBOOST
# =============================================================================

if all_importance:
    importance = pd.concat(
        all_importance,
        ignore_index=True,
    )
else:
    importance = pd.DataFrame()

if not importance.empty:
    importance.to_csv(
        TABLE_DIR / "Table_315_All_XGBoost_Gain_Importance.csv",
        index=False,
    )

    primary_importance = importance[
        (importance["Definition"] == "EVT_0025")
        & (importance["Variant"] == "Full")
    ].copy()

    def recover_origin(feature_name: str) -> str:
        if feature_name in EVENT_HISTORY_FEATURE_NAMES:
            return "__EVENT_HISTORY__"

        if "__" in feature_name:
            tail = feature_name.split("__", 1)[1]

            if tail.startswith("MISS__"):
                tail = tail.replace("MISS__", "", 1)

            if tail in sectors:
                return "__SECTOR__"

            return tail

        return feature_name

    primary_importance["OriginFeature"] = (
        primary_importance["Feature"]
        .astype(str)
        .map(recover_origin)
    )

    aggregated_importance = (
        primary_importance
        .groupby(
            ["Horizon", "OriginFeature"],
            as_index=False,
        )["Gain"]
        .sum()
    )

    aggregated_importance["GainShare"] = (
        aggregated_importance["Gain"]
        / aggregated_importance.groupby(
            "Horizon"
        )["Gain"].transform("sum")
    )

    aggregated_importance.to_csv(
        TABLE_DIR / "Table_316_Primary_XGBoost_Aggregated_Importance.csv",
        index=False,
    )
else:
    aggregated_importance = pd.DataFrame()

# =============================================================================
# 33. SECTOR/HORIZON HETEROGENEITY FOR FULL XGBOOST / DYNAMIC LOGIT
# =============================================================================

heterogeneity_rows = []

for model_family in [
    "XGBoost",
    "DynamicLogit",
]:
    predictions = component_results[
        (model_family, "Full")
    ]

    for h in HORIZONS:
        for s, sector in enumerate(sectors):
            metrics = probability_metrics(
                primary_targets[h][test_date_mask, s],
                predictions["Test"][h][test_date_mask, s],
            )

            no_garch_metrics = probability_metrics(
                primary_targets[h][test_date_mask, s],
                component_results[
                    (model_family, "NoGARCH")
                ]["Test"][h][test_date_mask, s],
            )

            no_vol_metrics = probability_metrics(
                primary_targets[h][test_date_mask, s],
                component_results[
                    (model_family, "NoVolatilityGroup")
                ]["Test"][h][test_date_mask, s],
            )

            heterogeneity_rows.append({
                "ModelFamily": model_family,
                "Sector": sector,
                "Horizon": h,
                "FullBrier": metrics["Brier"],
                "FullPR_AUC": metrics["PR_AUC"],
                "NoGARCHBrier": no_garch_metrics["Brier"],
                "NoVolatilityBrier": no_vol_metrics["Brier"],
                "GARCHIncrement_Brier": (
                    no_garch_metrics["Brier"]
                    - metrics["Brier"]
                ),
                "VolatilityGroupIncrement_Brier": (
                    no_vol_metrics["Brier"]
                    - metrics["Brier"]
                ),
            })

heterogeneity = pd.DataFrame(
    heterogeneity_rows
)

heterogeneity.to_csv(
    TABLE_DIR / "Table_317_Sector_Horizon_Volatility_Heterogeneity.csv",
    index=False,
)

# =============================================================================
# 34. PREDICTION EXPORT
# =============================================================================

prediction_rows = []

for split_name, split_mask in [
    ("Calibration", calibration_date_mask),
    ("Test", test_date_mask),
]:
    for t in np.where(split_mask)[0]:
        for s, sector in enumerate(sectors):
            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            # Primary component-ablation predictions.
            for h in HORIZONS:
                row[f"EVT0025_Target_{h}"] = (
                    primary_targets[h][t, s]
                )

                for model_family in [
                    "XGBoost",
                    "DynamicLogit",
                ]:
                    for variant in ABLATION_DROP_GROUPS:
                        row[
                            f"{model_family}__{variant}__P{h}"
                        ] = component_results[
                            (model_family, variant)
                        ][split_name][h][t, s]

                        if variant == "Full":
                            row[
                                f"{model_family}__{variant}__RawP{h}"
                            ] = component_raw_results[
                                (model_family, variant)
                            ][split_name][h][t, s]

            prediction_rows.append(row)

component_prediction_panel = pd.DataFrame(
    prediction_rows
)

component_prediction_panel.to_csv(
    DATA_DIR / "phase3_component_ablation_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# Tail-event-definition prediction panel.
definition_prediction_rows = []

for split_name, split_mask in [
    ("Calibration", calibration_date_mask),
    ("Test", test_date_mask),
]:
    for t in np.where(split_mask)[0]:
        for s, sector in enumerate(sectors):
            row = {
                "Date": dates[t],
                "Split": split_name,
                "Sector": sector,
            }

            for definition_name in event_definitions:
                for h in HORIZONS:
                    row[
                        f"{definition_name}__Target_{h}"
                    ] = event_definitions[
                        definition_name
                    ]["targets"][h][t, s]

                    row[
                        f"{definition_name}__Historical__P{h}"
                    ] = historical_probabilities[
                        definition_name
                    ]["ExpandingHistorical"][h][t, s]

                    for model_family in [
                        "XGBoost",
                        "DynamicLogit",
                    ]:
                        row[
                            f"{definition_name}__{model_family}__P{h}"
                        ] = robustness_results[
                            (model_family, definition_name)
                        ][split_name][h][t, s]

            definition_prediction_rows.append(row)

definition_prediction_panel = pd.DataFrame(
    definition_prediction_rows
)

definition_prediction_panel.to_csv(
    DATA_DIR / "phase3_crash_definition_predictions.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 35. FIGURES
# =============================================================================

# Figure 1: XGBoost component Brier degradation vs Full.
plot_component = component_skill[
    (component_skill["ModelFamily"] == "XGBoost")
    & (component_skill["Variant"] != "Full")
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for variant, group in plot_component.groupby("Variant"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["BrierDegradation_vs_Full"],
        marker="o",
        label=variant,
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Ablated Brier minus Full Brier")
ax.set_xticks(HORIZONS)
ax.set_title("Incremental Predictor-Group Contribution — XGBoost")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_301_XGBoost_Component_Ablation_Brier.png",
    dpi=300,
)
plt.close(fig)

# Figure 2: H1 volatility contribution by model.
plot_h1 = component_skill[
    component_skill["Variant"].isin(
        ["NoGARCH", "NoVolatilityGroup"]
    )
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for (model_family, variant), group in plot_h1.groupby(
    ["ModelFamily", "Variant"]
):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["BrierDegradation_vs_Full"],
        marker="o",
        label=f"{model_family}: {variant}",
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Ablated Brier minus Full Brier")
ax.set_xticks(HORIZONS)
ax.set_title("Incremental Dynamic-Volatility Forecast Contribution")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_302_H1_Volatility_Contribution.png",
    dpi=300,
)
plt.close(fig)

# Figure 3: Crash-definition Brier skill vs own historical benchmark.
plot_def_skill = definition_skill[
    definition_skill["ModelFamily"].isin(
        ["XGBoost", "DynamicLogit"]
    )
].copy()

fig, ax = plt.subplots(figsize=(10, 6))

for (definition_name, model_family), group in plot_def_skill.groupby(
    ["Definition", "ModelFamily"]
):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["BrierSkill_vs_OwnHistorical"],
        marker="o",
        label=f"{definition_name}: {model_family}",
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Brier skill vs definition-specific historical benchmark")
ax.set_xticks(HORIZONS)
ax.set_title("Crash-Definition Forecastability Robustness")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_303_Crash_Definition_Brier_Skill.png",
    dpi=300,
)
plt.close(fig)

# Figure 4: EVT vs fixed Brier-skill difference.
fig, ax = plt.subplots(figsize=(9, 6))

for model_family, group in definition_skill_bootstrap.groupby("ModelFamily"):
    group = group.sort_values("Horizon")

    ax.plot(
        group["Horizon"],
        group["SkillDifference_EVTminusFixed"],
        marker="o",
        label=model_family,
    )

ax.axhline(0.0, linewidth=1)
ax.set_xlabel("Forecast horizon (trading days)")
ax.set_ylabel("Brier-skill difference: EVT minus Fixed")
ax.set_xticks(HORIZONS)
ax.set_title("EVT vs Fixed-Threshold Forecastability")
ax.legend()
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_304_H2_EVT_vs_Fixed_Skill.png",
    dpi=300,
)
plt.close(fig)

# Figure 5: Sector/horizon GARCH increment heatmap for XGBoost.
heat_data = heterogeneity[
    heterogeneity["ModelFamily"] == "XGBoost"
].pivot(
    index="Sector",
    columns="Horizon",
    values="GARCHIncrement_Brier",
)

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(
    heat_data.to_numpy(),
    aspect="auto",
)
ax.set_xticks(range(len(heat_data.columns)))
ax.set_xticklabels(
    [str(int(h)) for h in heat_data.columns]
)
ax.set_yticks(range(len(heat_data.index)))
ax.set_yticklabels(heat_data.index)
ax.set_xlabel("Forecast horizon")
ax.set_ylabel("Sector")
ax.set_title("Sector/Horizon Increment from GJR-GARCH Volatility — XGBoost")
fig.colorbar(
    image,
    ax=ax,
    label="NoGARCH Brier minus Full Brier",
)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_305_H5_Sector_Horizon_GARCH_Contribution.png",
    dpi=300,
)
plt.close(fig)

# =============================================================================
# 36. EXCEL WORKBOOK
# =============================================================================

excel_path = (
    EXCEL_DIR
    / "Python_Phase3_Ablations_Logit_CrashRobustness.xlsx"
)

if importlib.util.find_spec("xlsxwriter") is not None:
    excel_engine = "xlsxwriter"
elif importlib.util.find_spec("openpyxl") is not None:
    excel_engine = "openpyxl"
else:
    excel_engine = None

excel_tables = {
    "Fixed Thresholds": fixed_threshold_table,
    "Target Audit": target_audit,
    "EVT Fixed Overlap": event_overlap,
    "Ablation Design": ablation_design,
    "Fit Metadata": fit_metadata,
    "Component Metrics": component_metrics,
    "Component Contribution": component_skill,
    "Component Bootstrap": component_bootstrap,
    "H1 Volatility": h1_bootstrap,
    "Definition Metrics": robustness_metrics,
    "Definition Skill": definition_skill,
    "H2 EVT vs Fixed": definition_skill_bootstrap,
    "Sector Heterogeneity": heterogeneity,
}

if not aggregated_importance.empty:
    excel_tables[
        "XGB Importance"
    ] = aggregated_importance

if excel_engine is not None:
    try:
        with pd.ExcelWriter(
            excel_path,
            engine=excel_engine,
        ) as writer:

            for sheet_name, dataframe in excel_tables.items():
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name[:31],
                    index=False,
                )

            if excel_engine == "xlsxwriter":
                workbook = writer.book
                header_format = workbook.add_format({
                    "bold": True,
                    "text_wrap": True,
                    "valign": "top",
                    "border": 1,
                })

                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes(1, 0)
                    worksheet.set_row(
                        0,
                        28,
                        header_format,
                    )
                    worksheet.set_column(
                        0,
                        30,
                        16,
                    )

            else:
                for _, worksheet in writer.sheets.items():
                    worksheet.freeze_panes = "A2"

        log(
            f"Excel workbook saved using engine={excel_engine}: {excel_path}"
        )

    except Exception as exc:
        log(f"WARNING: Excel export failed: {repr(exc)}")
else:
    log(
        "WARNING: xlsxwriter/openpyxl unavailable; CSV outputs remain complete."
    )

# =============================================================================
# 37. FINAL INTEGRITY CHECKS
# =============================================================================

def count_coherence_violations(probability_store: dict) -> int:
    violations = 0
    for split_name in ["Calibration", "Test"]:
        stack = np.stack(
            [probability_store[split_name][h] for h in HORIZONS],
            axis=-1,
        )
        complete = np.all(np.isfinite(stack), axis=-1)
        if complete.any():
            diff = np.diff(stack[complete], axis=-1)
            violations += int(np.sum(np.any(diff < -1e-8, axis=-1)))
    return violations

coherence_violations = sum(
    count_coherence_violations(store)
    for store in list(component_results.values()) + list(robustness_results.values())
)

# Explicit proof that tuning and calibration are disjoint.
coord_trainfit = coord_revision_phase == "TrainFit"
coord_traintune = coord_revision_phase == "TrainTune"
coord_calibration = coord_revision_phase == "Calibration"
coord_test = coord_revision_phase == "Test"

integrity_checks = [
    ("Crash_Main equals primary 2.5% EVT label", primary_mismatch == 0),
    ("All four event definitions available", set(event_definitions.keys()) == {"EVT_001", "EVT_0025", "EVT_005", "Fixed_0025"}),
    ("TrainFit and TrainTune are disjoint", not bool(np.any(coord_trainfit & coord_traintune))),
    ("Calibration excluded from TrainFit/TrainTune", not bool(np.any(coord_calibration & (coord_trainfit | coord_traintune)))),
    ("Test excluded from all pre-Test phases", not bool(np.any(coord_test & (coord_trainfit | coord_traintune | coord_calibration)))),
    ("Full Train equals TrainFit plus TrainTune", bool(np.array_equal(full_train_date_mask, train_fit_date_mask | train_tune_date_mask))),
    ("Fixed inner thresholds estimated on TrainFit", bool(np.all(np.isfinite(fixed_thresholds_inner)))),
    ("Fixed final thresholds estimated on Full Train", bool(np.all(np.isfinite(fixed_thresholds)))),
    ("No tail-event/future probability columns in generic base predictors", len(generic_leakage_offenders) == 0),
    ("Inner preprocessed feature panel finite", bool(np.all(np.isfinite(X_panel_inner)))),
    ("Final preprocessed feature panel finite", bool(np.all(np.isfinite(X_panel)))),
    ("All component variants completed", len(component_results) == 2 * len(ABLATION_DROP_GROUPS)),
    ("All event-definition robustness fits completed", len(robustness_results) == 2 * len(event_definitions)),
    ("No fitted horizon-coherence violations", coherence_violations == 0),
    ("Test contains both classes for primary target at all horizons", all(len(np.unique(primary_targets[h][test_date_mask][np.isfinite(primary_targets[h][test_date_mask])])) == 2 for h in HORIZONS)),
    ("Dynamic logit is parsimonious relative to full XGBoost design", max(len(logit_ablation_indices[v]) for v in logit_ablation_indices) < max(len(ablation_indices[v]) for v in ablation_indices)),
    ("H1 volatility bootstrap table nonempty", len(h1_bootstrap) > 0),
    ("H2 EVT-vs-fixed skill bootstrap table complete", len(definition_skill_bootstrap) == 2 * len(HORIZONS)),
]

integrity_table = pd.DataFrame(integrity_checks, columns=["Check", "Passed"])
integrity_table.to_csv(TABLE_DIR / "Table_318_Final_Integrity_Checks.csv", index=False)
if not integrity_table["Passed"].all():
    failed = integrity_table.loc[~integrity_table["Passed"], "Check"].tolist()
    raise RuntimeError(f"Phase 3.2 integrity checks failed: {failed}")

# =============================================================================
# 38. METADATA
# =============================================================================

metadata = {
    "ProjectTitle": (
        "Forecasting Sectoral Crash Risk and Contagion: "
        "Integrating Dynamic Volatility, Extreme-Value Modelling "
        "and Graph Deep Learning"
    ),
    "PythonPhase": "3",
    "ScriptVersion": "3.2-reviewer-revision-audited",
    "GeneratedAt": datetime.now().isoformat(),
    "Seed": SEED,
    "InputPhase1Zip": INPUT_ZIP_PATH.name,
    "PrimaryTailEventDefinition": "EVT_0025",
    "RobustnessTailEventDefinitions": [
        "EVT_001",
        "EVT_005",
        "Fixed_0025",
    ],
    "Models": [
        "XGBoost",
        "DynamicLogit",
    ],
    "ComponentAblations": list(ABLATION_DROP_GROUPS.keys()),
    "Horizons": HORIZONS,
    "Lookback": LOOKBACK,
    "BootstrapReps": BOOTSTRAP_REPS,
    "BootstrapBlockLength": BOOTSTRAP_BLOCK,
    "TrainFitEnd": str(TRAIN_FIT_END.date()),
    "TrainTuneStart": str(TRAIN_TUNE_START.date()),
    "TrainTuneEnd": str(TRAIN_TUNE_END.date()),
    "CalibrationStart": str(CALIBRATION_START.date()),
    "CalibrationEnd": str(CALIBRATION_END.date()),
    "TestStart": str(TEST_START.date()),
    "FixedThresholdRule": (
        "TrainFit-only 2.5% sector return quantile for inner tuning; "
        "Full-Train-only 2.5% quantile for final refit/calibration/test"
    ),
    "CalibrationUsedForModelSelection": False,
    "TestUsedForModelSelection": False,
    "DynamicLogitCalibration": "Platt on Calibration only",
    "Phase2_2ArchitectureStatus": "Frozen; not redesigned in Phase 3.2",
}

with open(
    DATA_DIR / "python_phase3_metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
        default=str,
    )

with open(
    MODEL_DIR / "phase3_preprocessing_bundle.pkl",
    "wb",
) as handle:
    pickle.dump(
        {
            "base_feature_columns": base_feature_columns,
            "model_feature_names": model_feature_names,
            "feature_origin": feature_origin,
            "trainfit_median": trainfit_median,
            "trainfit_mean": trainfit_mean,
            "trainfit_std": trainfit_std,
            "full_train_median": train_median,
            "full_train_mean": train_mean,
            "full_train_std": train_std,
            "fixed_thresholds_inner": fixed_thresholds_inner,
            "fixed_thresholds_full_train": fixed_thresholds,
            "logit_core_origins": sorted(LOGIT_CORE_ORIGINS),
            "sectors": sectors,
            "lookback": LOOKBACK,
        },
        handle,
    )

# =============================================================================
# 39. CLEAN EXTRACTED INPUT / ZIP ALL OUTPUTS
# =============================================================================

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

zip_output = (
    ZIP_DIR
    / "Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip"
)

if zip_output.exists():
    zip_output.unlink()

with zipfile.ZipFile(
    zip_output,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:

    for path in OUTPUT_ROOT.rglob("*"):
        if path.is_file() and path != zip_output:
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_ROOT),
            )

log(f"All Phase-3 outputs zipped to: {zip_output}")

# =============================================================================
# 40. CONSOLE SUMMARY
# =============================================================================

print("\n" + "=" * 100)
print("PYTHON PHASE 3.2 REVIEWER REVISION COMPLETED SUCCESSFULLY")
print("=" * 100)

print(f"Primary tail-event definition: EVT_0025")
print(
    "Robustness definitions: EVT_001, EVT_005, Fixed_0025"
)
print(
    "Models: XGBoost + Dynamic ridge-logit"
)

print("\nH1 — volatility ablation summary:")
print(
    h1_bootstrap[
        [
            "ModelFamily",
            "Ablation",
            "Horizon",
            "BrierDiff_FullMinusAblation",
            "BrierDiff_CI2.5",
            "BrierDiff_CI97.5",
            "Probability_Full_Better_Brier",
        ]
    ].to_string(index=False)
)

print("\nH2 — EVT vs fixed-threshold Brier-skill summary:")
print(
    definition_skill_bootstrap[
        [
            "ModelFamily",
            "Horizon",
            "EVT_BrierSkill",
            "Fixed_BrierSkill",
            "SkillDifference_EVTminusFixed",
            "SkillDifference_CI2.5",
            "SkillDifference_CI97.5",
            "Probability_EVT_Skill_Higher",
        ]
    ].to_string(index=False)
)

print("\nFinal ZIP to send back for review:")
print(f"  {zip_output}")
print("=" * 100)

# =============================================================================
# 41. AUTOMATIC COLAB DOWNLOAD
# =============================================================================

try:
    from google.colab import files

    print(
        "\nStarting browser download of "
        "Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip ..."
    )

    files.download(
        str(zip_output)
    )

except ImportError:
    print(
        "\nNot running in Google Colab. "
        f"Retrieve the ZIP manually from: {zip_output}"
    )

# =============================================================================
# END OF PYTHON PHASE 3
# =============================================================================


[2026-09-12 11:35:27] ====================================================================================================
[2026-09-12 11:35:27] PYTHON PHASE 3.2 START — REVIEWER-REVISION ABLATIONS / DYNAMIC RIDGE-LOGIT / TAIL-EVENT ROBUSTNESS
[2026-09-12 11:35:27] Python=3.13.15; platform=Linux-6.6.122+-x86_64-with-glibc2.39; seed=20260901
[2026-09-12 11:35:27] xgboost=3.4.1

Please upload the frozen Phase-1 file:
    Python_Phase1_v1_2_From_R_Handoff_All_Outputs.zip



Saving Python_Phase1_v1_3_ReviewerRevision_All_Outputs.zip to Python_Phase1_v1_3_ReviewerRevision_All_Outputs (2).zip
[2026-09-12 11:36:27] Validated uploaded Phase-1 ZIP: /content/Python_Phase1_v1_3_ReviewerRevision_All_Outputs (2).zip
[2026-09-12 11:36:27] Accepted Phase-1 ZIP: /content/Python_Phase1_v1_3_ReviewerRevision_All_Outputs (2).zip
[2026-09-12 11:36:27] Loaded 14,928 rows; 2,488 dates; 6 sectors. Reviewer chronology: TrainFit=1123, TrainTune=374, Calibration=496, Test=495.
[2026-09-12 11:36:28] Leakage-screened base numeric features=48
[2026-09-12 11:36:28] Generic leakage screen passed: no Crash/future-target/prior-probability columns admitted.
[2026-09-12 11:36:29] Generic forecast-origin coordinates=14,574
[2026-09-12 11:36:29] Building inner/final summary matrices: 14,574 rows x 773 columns.
[2026-09-12 11:36:38] Fitting XGBoost: definition=EVT_0025, variant=Full, h=1
[2026-09-12 11:36:49] Fitting XGBoost: definition=EVT_0025, variant=Full, h=5
[2026-09-12 11:36:57] Fit

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:

# =============================================================================
# FINAL REVIEWER INFERENCE / POST-PROCESSING
# Forecasting Sector Tail-Event Probabilities
#
# PURPOSE
# -------
# This script consumes the frozen reviewer-revision outputs from:
#   * Python Phase 2.2 v2.3 (five-seed ML/DL forecasting)
#   * Python Phase 3 v3.2 (ablations / dynamic ridge-logit / event-definition)
#
# It performs NO neural-model retraining and NO Test-based model selection.
#
# Main outputs:
#   1. Manuscript-ready Test comparison table with N/events/prevalence.
#   2. Brier skill relative to expanding historical benchmark.
#   3. Paired Brier/Log-score inference with 10/22/44-day moving-block
#      sensitivity plus 22-day stationary bootstrap.
#   4. Holm multiplicity adjustment by pre-specified comparison family.
#   5. Calibration-in-the-large / slope summaries and reliability diagrams with
#      22-day block-bootstrap uncertainty bands.
#   6. Brier decomposition (reliability/resolution/uncertainty).
#   7. Cost-sensitive / decision-curve analysis.
#   8. Formal H5 sector-by-horizon heterogeneity tests based on the incremental
#      contribution of GJR-GARCH information (NoGARCH loss - Full loss).
#   9. Dynamic ridge-logit lambda-grid boundary audit.
#  10. Reproducibility metadata, integrity checks and final ZIP.
#
# VERSION: 1.0-reviewer-final-inference
# =============================================================================

from __future__ import annotations

import os
import sys
import json
import math
import random
import shutil
import zipfile
import hashlib
import warnings
import platform
import importlib.util
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from sklearn.metrics import average_precision_score, roc_auc_score

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# =============================================================================
# 0. CONFIGURATION
# =============================================================================

SEED = 20260912
random.seed(SEED)
np.random.seed(SEED)

HORIZONS = [1, 5, 10, 22]
EPS = 1e-8

BOOTSTRAP_REPS = int(os.getenv("NSE_FINAL_BOOTSTRAP_REPS", "500"))
BLOCK_LENGTHS = [10, 22, 44]
PRIMARY_BLOCK = 22
STATIONARY_MEAN_BLOCK = 22

RELIABILITY_BINS = int(os.getenv("NSE_RELIABILITY_BINS", "10"))

PHASE2_ZIP_CONFIG = os.getenv(
    "NSE_PHASE2_REVIEWER_ZIP",
    "/content/Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip",
)
PHASE3_ZIP_CONFIG = os.getenv(
    "NSE_PHASE3_REVIEWER_ZIP",
    "/content/Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip",
)

OUTPUT_ROOT = Path(os.getenv(
    "NSE_FINAL_INFERENCE_OUTPUT_DIR",
    "/content/Sectoral_Tail_Event_Final_Reviewer_Inference_v1_0",
))

TABLE_DIR = OUTPUT_ROOT / "01_Tables"
FIG_DIR = OUTPUT_ROOT / "02_Figures"
DATA_DIR = OUTPUT_ROOT / "03_Processed_Data"
LOG_DIR = OUTPUT_ROOT / "04_Logs"
ZIP_DIR = OUTPUT_ROOT / "05_Zip"
EXTRACT2_DIR = OUTPUT_ROOT / "_Phase2_Extracted"
EXTRACT3_DIR = OUTPUT_ROOT / "_Phase3_Extracted"

for d in [
    OUTPUT_ROOT, TABLE_DIR, FIG_DIR, DATA_DIR, LOG_DIR, ZIP_DIR,
    EXTRACT2_DIR, EXTRACT3_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "Final_Reviewer_Inference_run_log.txt"

def log(message: str):
    stamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{stamp}] {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as handle:
        handle.write(line + "\n")

log("=" * 100)
log("FINAL REVIEWER INFERENCE START")
log(f"Python={sys.version.split()[0]}; platform={platform.platform()}; seed={SEED}")
log(
    f"bootstrap reps={BOOTSTRAP_REPS}; moving blocks={BLOCK_LENGTHS}; "
    f"stationary mean block={STATIONARY_MEAN_BLOCK}"
)

# =============================================================================
# 1. INPUT ZIP VALIDATION / COLAB UPLOAD
# =============================================================================

PHASE2_REQUIRED = {
    "phase2_2_ensemble_validation_test_predictions.csv.gz",
    "xgboost_raw_and_calibrated_predictions.csv.gz",
    "neural_raw_and_calibrated_predictions.csv.gz",
    "python_phase2_2_metadata.json",
    "Table_R00_Neural_Parameter_Counts.csv",
    "Table_D05_XGBoost_Inner_Tuning_and_Refit.csv",
    "Table_R15_Final_Integrity_Checks.csv",
}

PHASE3_REQUIRED = {
    "phase3_component_ablation_predictions.csv.gz",
    "phase3_crash_definition_predictions.csv.gz",
    "python_phase3_metadata.json",
    "Table_306_Model_Fitting_Metadata.csv",
    "Table_307_Dynamic_Logit_TrainTune_Tuning.csv",
    "Table_318_Final_Integrity_Checks.csv",
}

def zip_basenames(path: Path) -> set[str]:
    try:
        with zipfile.ZipFile(path, "r") as archive:
            return {
                Path(name).name
                for name in archive.namelist()
                if not name.endswith("/")
            }
    except Exception:
        return set()

def is_valid_zip(path: Path, required: set[str]) -> bool:
    return (
        path.exists()
        and path.suffix.lower() == ".zip"
        and required.issubset(zip_basenames(path))
    )

def classify_zip(path: Path) -> str | None:
    names = zip_basenames(path)
    if PHASE2_REQUIRED.issubset(names):
        return "phase2"
    if PHASE3_REQUIRED.issubset(names):
        return "phase3"
    return None

def resolve_two_zips() -> tuple[Path, Path]:
    """
    In Colab, ask for the two frozen reviewer-revision ZIPs together.
    Outside Colab, use the explicitly configured paths.
    """
    try:
        from google.colab import files

        while True:
            print(
                "\nPlease upload BOTH frozen reviewer-revision ZIPs:\n"
                "  1) Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip\n"
                "  2) Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip\n"
            )
            uploaded = files.upload()
            phase2 = []
            phase3 = []

            for name in uploaded:
                if not name.lower().endswith(".zip"):
                    continue
                candidate = Path("/content") / name
                kind = classify_zip(candidate)
                if kind == "phase2":
                    phase2.append(candidate)
                elif kind == "phase3":
                    phase3.append(candidate)
                else:
                    log(f"Rejected unrecognized ZIP: {candidate.name}")

            if len(phase2) == 1 and len(phase3) == 1:
                return phase2[0], phase3[0]

            print(
                "\nCould not identify exactly one valid Phase 2.2 ZIP and one "
                "valid Phase 3 ZIP. Please upload the two requested files only.\n"
            )

    except ImportError:
        p2 = Path(PHASE2_ZIP_CONFIG)
        p3 = Path(PHASE3_ZIP_CONFIG)
        if not is_valid_zip(p2, PHASE2_REQUIRED):
            raise FileNotFoundError(
                f"Configured Phase 2.2 ZIP is missing/invalid: {p2}"
            )
        if not is_valid_zip(p3, PHASE3_REQUIRED):
            raise FileNotFoundError(
                f"Configured Phase 3 ZIP is missing/invalid: {p3}"
            )
        return p2, p3

PHASE2_ZIP, PHASE3_ZIP = resolve_two_zips()
log(f"Accepted Phase2 ZIP: {PHASE2_ZIP}")
log(f"Accepted Phase3 ZIP: {PHASE3_ZIP}")

for d in [EXTRACT2_DIR, EXTRACT3_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PHASE2_ZIP, "r") as archive:
    archive.extractall(EXTRACT2_DIR)
with zipfile.ZipFile(PHASE3_ZIP, "r") as archive:
    archive.extractall(EXTRACT3_DIR)

def find_one(root: Path, basename: str) -> Path:
    matches = list(root.rglob(basename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{basename}' below {root}; found {len(matches)}."
        )
    return matches[0]

# =============================================================================
# 2. LOAD FROZEN OUTPUTS
# =============================================================================

P2_ENSEMBLE_FILE = find_one(
    EXTRACT2_DIR,
    "phase2_2_ensemble_validation_test_predictions.csv.gz",
)
P2_META_FILE = find_one(EXTRACT2_DIR, "python_phase2_2_metadata.json")
P2_PARAM_FILE = find_one(EXTRACT2_DIR, "Table_R00_Neural_Parameter_Counts.csv")
P2_XGB_TUNE_FILE = find_one(
    EXTRACT2_DIR,
    "Table_D05_XGBoost_Inner_Tuning_and_Refit.csv",
)
P2_INTEGRITY_FILE = find_one(
    EXTRACT2_DIR,
    "Table_R15_Final_Integrity_Checks.csv",
)

P3_ABLATION_FILE = find_one(
    EXTRACT3_DIR,
    "phase3_component_ablation_predictions.csv.gz",
)
P3_DEFINITION_FILE = find_one(
    EXTRACT3_DIR,
    "phase3_crash_definition_predictions.csv.gz",
)
P3_META_FILE = find_one(EXTRACT3_DIR, "python_phase3_metadata.json")
P3_FIT_META_FILE = find_one(
    EXTRACT3_DIR,
    "Table_306_Model_Fitting_Metadata.csv",
)
P3_LOGIT_TUNE_FILE = find_one(
    EXTRACT3_DIR,
    "Table_307_Dynamic_Logit_TrainTune_Tuning.csv",
)
P3_INTEGRITY_FILE = find_one(
    EXTRACT3_DIR,
    "Table_318_Final_Integrity_Checks.csv",
)

p2 = pd.read_csv(P2_ENSEMBLE_FILE, parse_dates=["Date"])
p3 = pd.read_csv(P3_ABLATION_FILE, parse_dates=["Date"])
p3_def = pd.read_csv(P3_DEFINITION_FILE, parse_dates=["Date"])

with open(P2_META_FILE, "r", encoding="utf-8") as handle:
    p2_meta = json.load(handle)
with open(P3_META_FILE, "r", encoding="utf-8") as handle:
    p3_meta = json.load(handle)

p2_params = pd.read_csv(P2_PARAM_FILE)
p2_xgb_tuning = pd.read_csv(P2_XGB_TUNE_FILE)
p2_integrity = pd.read_csv(P2_INTEGRITY_FILE)
p3_fit_meta = pd.read_csv(P3_FIT_META_FILE)
p3_logit_tuning = pd.read_csv(P3_LOGIT_TUNE_FILE)
p3_integrity = pd.read_csv(P3_INTEGRITY_FILE)

log(f"Phase2 rows={len(p2):,}; Phase3 rows={len(p3):,}")

# =============================================================================
# 3. INTEGRITY AUDIT BEFORE ANY INFERENCE
# =============================================================================

integrity_rows = []

def add_check(name: str, passed: bool, detail: str = ""):
    integrity_rows.append({
        "Check": name,
        "Passed": bool(passed),
        "Detail": str(detail),
    })

add_check(
    "Phase2 reviewer version is v2.3",
    str(p2_meta.get("ScriptVersion", "")).startswith("2.3"),
    p2_meta.get("ScriptVersion", ""),
)
add_check(
    "Phase3 reviewer version is v3.2",
    str(p3_meta.get("ScriptVersion", "")).startswith("3.2"),
    p3_meta.get("ScriptVersion", ""),
)
add_check(
    "Phase2 source integrity checks all passed",
    bool(p2_integrity["Passed"].astype(bool).all()),
    f"{int(p2_integrity['Passed'].astype(bool).sum())}/{len(p2_integrity)}",
)
add_check(
    "Phase3 source integrity checks all passed",
    bool(p3_integrity["Passed"].astype(bool).all()),
    f"{int(p3_integrity['Passed'].astype(bool).sum())}/{len(p3_integrity)}",
)

for key in [
    "TrainFitEnd",
    "TrainTuneStart",
    "TrainTuneEnd",
    "CalibrationStart",
    "CalibrationEnd",
    "TestStart",
]:
    add_check(
        f"Phase2/Phase3 chronology agrees: {key}",
        str(p2_meta.get(key)) == str(p3_meta.get(key)),
        f"P2={p2_meta.get(key)}; P3={p3_meta.get(key)}",
    )

add_check(
    "Phase2 Calibration not used for model selection",
    p2_meta.get("CalibrationUsedForModelSelection") is False,
    p2_meta.get("CalibrationUsedForModelSelection"),
)
add_check(
    "Phase3 Calibration not used for model selection",
    p3_meta.get("CalibrationUsedForModelSelection") is False,
    p3_meta.get("CalibrationUsedForModelSelection"),
)
add_check(
    "Phase3 Test not used for model selection",
    p3_meta.get("TestUsedForModelSelection") is False,
    p3_meta.get("TestUsedForModelSelection"),
)

add_check(
    "Phase2 Date-Sector rows unique",
    not p2.duplicated(["Date", "Sector"]).any(),
)
add_check(
    "Phase3 Date-Sector rows unique",
    not p3.duplicated(["Date", "Sector"]).any(),
)

# Compare frozen Test target support between Phase2 and Phase3.
target_audit_rows = []
for h in HORIZONS:
    left = p2.loc[
        p2["Split"].eq("Test"),
        ["Date", "Sector", f"Target_{h}"],
    ].rename(columns={f"Target_{h}": "P2Target"})
    right = p3.loc[
        p3["Split"].eq("Test"),
        ["Date", "Sector", f"EVT0025_Target_{h}"],
    ].rename(columns={f"EVT0025_Target_{h}": "P3Target"})

    cmp = left.merge(right, on=["Date", "Sector"], how="outer", indicator=True)
    both = cmp["_merge"].eq("both")
    value_mismatch = (
        both
        & cmp["P2Target"].notna()
        & cmp["P3Target"].notna()
        & ~np.isclose(cmp["P2Target"], cmp["P3Target"])
    )
    na_mismatch = both & (cmp["P2Target"].isna() ^ cmp["P3Target"].isna())

    row = {
        "Horizon": h,
        "RowsPhase2": len(left),
        "RowsPhase3": len(right),
        "MergedRows": len(cmp),
        "UnmatchedRows": int((~both).sum()),
        "ValueMismatches": int(value_mismatch.sum()),
        "NAPatternMismatches": int(na_mismatch.sum()),
        "Passed": (
            int((~both).sum()) == 0
            and int(value_mismatch.sum()) == 0
            and int(na_mismatch.sum()) == 0
        ),
    }
    target_audit_rows.append(row)

target_audit = pd.DataFrame(target_audit_rows)
target_audit.to_csv(
    TABLE_DIR / "Table_F00A_Phase2_Phase3_Test_Target_Audit.csv",
    index=False,
)
add_check(
    "Phase2/Phase3 Test targets match at all horizons",
    bool(target_audit["Passed"].all()),
    target_audit.to_dict(orient="records"),
)

integrity = pd.DataFrame(integrity_rows)
integrity.to_csv(
    TABLE_DIR / "Table_F00_Final_Inference_Input_Integrity.csv",
    index=False,
)

if not integrity["Passed"].all():
    failed = integrity.loc[~integrity["Passed"], "Check"].tolist()
    raise RuntimeError(
        f"Final inference input integrity failed before analysis: {failed}"
    )

# =============================================================================
# 4. METRIC / BOOTSTRAP HELPERS
# =============================================================================

def clip_prob(p):
    return np.clip(np.asarray(p, dtype=float), EPS, 1 - EPS)

def brier_loss(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    return (y - p) ** 2

def log_loss_obs(y, p):
    y = np.asarray(y, dtype=float)
    p = clip_prob(p)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

def safe_pr_auc(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(y) & np.isfinite(p)
    if ok.sum() == 0 or len(np.unique(y[ok])) < 2:
        return np.nan
    return float(average_precision_score(y[ok], p[ok]))

def safe_roc_auc(y, p):
    y = np.asarray(y, dtype=float)
    p = np.asarray(p, dtype=float)
    ok = np.isfinite(y) & np.isfinite(p)
    if ok.sum() == 0 or len(np.unique(y[ok])) < 2:
        return np.nan
    return float(roc_auc_score(y[ok], p[ok]))

def fit_calibration_logistic(y, p):
    """
    Logistic recalibration:
       logit Pr(Y=1) = intercept + slope * logit(p)
    """
    y = np.asarray(y, dtype=float)
    p = clip_prob(p)
    ok = np.isfinite(y) & np.isfinite(p)
    y = y[ok]
    p = p[ok]

    if len(y) == 0 or len(np.unique(y)) < 2:
        return np.nan, np.nan, False

    x = np.log(p / (1 - p))

    def nll(par):
        eta = np.clip(par[0] + par[1] * x, -35, 35)
        pr = 1 / (1 + np.exp(-eta))
        pr = np.clip(pr, EPS, 1 - EPS)
        return float(
            -np.sum(y * np.log(pr) + (1 - y) * np.log(1 - pr))
        )

    res = minimize(
        nll,
        x0=np.array([0.0, 1.0]),
        method="L-BFGS-B",
        bounds=[(-20, 20), (-20, 20)],
        options={"maxiter": 2000},
    )

    if not res.success or not np.all(np.isfinite(res.x)):
        return np.nan, np.nan, False

    return float(res.x[0]), float(res.x[1]), True

def moving_block_indices(n: int, block: int, rng: np.random.Generator):
    if n <= 0:
        return np.array([], dtype=int)
    block = int(max(1, min(block, n)))
    n_blocks = int(math.ceil(n / block))
    max_start = n - block
    starts = rng.integers(0, max_start + 1, size=n_blocks)
    idx = np.concatenate([
        np.arange(s, s + block, dtype=int)
        for s in starts
    ])[:n]
    return idx

def stationary_bootstrap_indices(
    n: int,
    mean_block: int,
    rng: np.random.Generator,
):
    if n <= 0:
        return np.array([], dtype=int)

    restart_prob = 1.0 / max(1, int(mean_block))
    idx = np.empty(n, dtype=int)
    idx[0] = int(rng.integers(0, n))

    for i in range(1, n):
        if rng.random() < restart_prob:
            idx[i] = int(rng.integers(0, n))
        else:
            idx[i] = (idx[i - 1] + 1) % n

    return idx

def date_mean_frame(df: pd.DataFrame, value_col: str):
    out = (
        df[["Date", value_col]]
        .dropna()
        .groupby("Date", as_index=False)[value_col]
        .mean()
        .sort_values("Date")
        .reset_index(drop=True)
    )
    return out

def bootstrap_mean_inference(
    date_values: np.ndarray,
    method: str,
    block: int,
    reps: int,
    seed: int,
):
    values = np.asarray(date_values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)

    if n == 0:
        return {
            "Mean": np.nan, "CI_Low": np.nan, "CI_High": np.nan,
            "PValue": np.nan, "Reps": reps,
        }

    observed = float(np.mean(values))
    centered = values - observed
    rng = np.random.default_rng(seed)

    raw_means = np.empty(reps)
    null_means = np.empty(reps)

    for b in range(reps):
        if method == "MovingBlock":
            idx = moving_block_indices(n, block, rng)
        elif method == "Stationary":
            idx = stationary_bootstrap_indices(n, block, rng)
        else:
            raise ValueError(method)

        raw_means[b] = float(np.mean(values[idx]))
        null_means[b] = float(np.mean(centered[idx]))

    ci_low, ci_high = np.quantile(raw_means, [0.025, 0.975])
    p = (1 + np.sum(np.abs(null_means) >= abs(observed))) / (reps + 1)

    return {
        "Mean": observed,
        "CI_Low": float(ci_low),
        "CI_High": float(ci_high),
        "PValue": float(p),
        "Reps": reps,
    }

def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    out = np.full(len(pvalues), np.nan)
    valid = np.flatnonzero(np.isfinite(pvalues))

    if len(valid) == 0:
        return out

    pv = pvalues[valid]
    order = np.argsort(pv)
    m = len(pv)
    adjusted_sorted = np.empty(m)
    running = 0.0

    for rank, pos in enumerate(order):
        candidate = (m - rank) * pv[pos]
        running = max(running, candidate)
        adjusted_sorted[rank] = min(1.0, running)

    for rank, pos in enumerate(order):
        out[valid[pos]] = adjusted_sorted[rank]

    return out

# =============================================================================
# 5. BUILD FINAL MAIN TEST PREDICTION PANEL
# =============================================================================

MODEL_COLUMNS = {
    "FixedHistorical": "FixedHistorical",
    "ExpandingHistorical": "ExpandingHistorical",
    "StableHawkes": "StableHawkes",
    "XGBoost": "XGBoost",
    "LSTM_Survival": "LSTM_Survival",
    "TemporalTransformer_Survival": "TemporalTransformer_Survival",
    "NoGraphTransformer": "NoGraphTransformer",
    "RandomGraphTransformer": "RandomGraphTransformer",
    "StaticHawkesGraphTransformer": "StaticHawkesGraphTransformer",
    "DynamicHawkesGraphTransformer": "DynamicHawkesGraphTransformer",
}

main_records = []

p2_test = p2[p2["Split"].eq("Test")].copy()
p3_test = p3[p3["Split"].eq("Test")].copy()

for h in HORIZONS:
    target_col = f"Target_{h}"

    for model, prefix in MODEL_COLUMNS.items():
        pred_col = f"{prefix}__P{h}"
        if pred_col not in p2_test.columns:
            continue

        sub = p2_test[["Date", "Sector", target_col, pred_col]].copy()
        sub = sub.rename(columns={target_col: "Target", pred_col: "Probability"})
        sub["Horizon"] = h
        sub["Model"] = model
        sub["Source"] = "Phase2.2_v2.3"
        main_records.append(sub)

    # Reviewer-requested parsimonious benchmark from Phase 3.
    dlog_col = f"DynamicLogit__Full__P{h}"
    dlog_target = f"EVT0025_Target_{h}"
    sub = p3_test[["Date", "Sector", dlog_target, dlog_col]].copy()
    sub = sub.rename(
        columns={dlog_target: "Target", dlog_col: "Probability"}
    )
    sub["Horizon"] = h
    sub["Model"] = "DynamicRidgeLogit"
    sub["Source"] = "Phase3_v3.2"
    main_records.append(sub)

main_long = pd.concat(main_records, ignore_index=True)
main_long = main_long[
    np.isfinite(main_long["Target"])
    & np.isfinite(main_long["Probability"])
].copy()

if ((main_long["Probability"] < -1e-10) | (main_long["Probability"] > 1 + 1e-10)).any():
    raise RuntimeError("Final main prediction panel contains probabilities outside [0,1].")

main_long["Probability"] = main_long["Probability"].clip(0, 1)

main_long.to_csv(
    DATA_DIR / "final_main_test_predictions_long.csv.gz",
    index=False,
    compression="gzip",
)

# =============================================================================
# 6. MAIN TEST TABLE + BRIER SKILL + COMPLEXITY
# =============================================================================

neural_param_map = dict(zip(
    p2_params["Model"].astype(str),
    p2_params["TrainableParameters"].astype(float),
))

dlog_meta = p3_fit_meta[
    (p3_fit_meta["ModelFamily"] == "DynamicLogit")
    & (p3_fit_meta["Definition"] == "EVT_0025")
    & (p3_fit_meta["Variant"] == "Full")
].copy()
dlog_param_by_h = dict(zip(
    dlog_meta["Horizon"].astype(int),
    dlog_meta["ParameterCount"].astype(float),
))
dlog_lambda_by_h = dict(zip(
    dlog_meta["Horizon"].astype(int),
    dlog_meta["SelectedLambda"].astype(float),
))
xgb_rounds_by_h = dict(zip(
    p2_xgb_tuning["Horizon"].astype(int),
    p2_xgb_tuning["SelectedRounds"].astype(int),
))

main_metric_rows = []

for (model, h), sub in main_long.groupby(["Model", "Horizon"], sort=False):
    y = sub["Target"].to_numpy(float)
    p = sub["Probability"].to_numpy(float)

    intercept, slope, cal_ok = fit_calibration_logistic(y, p)

    complexity = np.nan
    complexity_type = ""
    complexity_note = ""

    if model in neural_param_map:
        complexity = neural_param_map[model]
        complexity_type = "TrainableParameters"
        complexity_note = f"{int(complexity):,} trainable parameters"
    elif model == "DynamicRidgeLogit":
        complexity = dlog_param_by_h.get(int(h), np.nan)
        complexity_type = "EstimatedParameters"
        complexity_note = (
            f"{int(complexity):,} parameters; "
            f"ridge lambda={dlog_lambda_by_h.get(int(h), np.nan):g}"
            if np.isfinite(complexity)
            else ""
        )
    elif model == "XGBoost":
        complexity = xgb_rounds_by_h.get(int(h), np.nan)
        complexity_type = "BoostingRounds"
        complexity_note = (
            f"{int(complexity)} selected boosting rounds"
            if np.isfinite(complexity) else ""
        )

    main_metric_rows.append({
        "Model": model,
        "Horizon": int(h),
        "N": int(len(y)),
        "Events": int(np.sum(y == 1)),
        "EventRate": float(np.mean(y)),
        "MeanPredictedProbability": float(np.mean(p)),
        "CalibrationInTheLarge_MeanPredMinusObserved": float(np.mean(p) - np.mean(y)),
        "Brier": float(np.mean(brier_loss(y, p))),
        "LogScore": float(np.mean(log_loss_obs(y, p))),
        "PR_AUC": safe_pr_auc(y, p),
        "ROC_AUC": safe_roc_auc(y, p),
        "CalibrationIntercept": intercept,
        "CalibrationSlope": slope,
        "CalibrationFitConverged": cal_ok,
        "ComplexityValue": complexity,
        "ComplexityType": complexity_type,
        "ComplexityNote": complexity_note,
    })

main_metrics = pd.DataFrame(main_metric_rows)

# Brier skill relative to the same expanding historical forecast at each horizon.
baseline_brier = (
    main_metrics[main_metrics["Model"].eq("ExpandingHistorical")]
    .set_index("Horizon")["Brier"]
    .to_dict()
)
main_metrics["BrierSkill_vs_ExpandingHistorical"] = main_metrics.apply(
    lambda r: (
        1 - r["Brier"] / baseline_brier[int(r["Horizon"])]
        if baseline_brier.get(int(r["Horizon"]), np.nan) > 0
        else np.nan
    ),
    axis=1,
)

main_metrics["BrierRank"] = (
    main_metrics.groupby("Horizon")["Brier"]
    .rank(method="min", ascending=True)
    .astype(int)
)
main_metrics["LogScoreRank"] = (
    main_metrics.groupby("Horizon")["LogScore"]
    .rank(method="min", ascending=True)
    .astype(int)
)

main_metrics = main_metrics.sort_values(
    ["Horizon", "Brier", "Model"]
).reset_index(drop=True)

main_metrics.to_csv(
    TABLE_DIR / "Table_F01_Main_Test_Model_Comparison.csv",
    index=False,
)

sample_table = (
    main_metrics[
        ["Horizon", "N", "Events", "EventRate"]
    ]
    .drop_duplicates()
    .sort_values("Horizon")
)
sample_table.to_csv(
    TABLE_DIR / "Table_F02_Test_Sample_Size_Event_Rate.csv",
    index=False,
)

# =============================================================================
# 7. BRIER SKILL UNCERTAINTY
# =============================================================================

skill_rows = []

for h in HORIZONS:
    baseline = main_long[
        (main_long["Horizon"] == h)
        & (main_long["Model"] == "ExpandingHistorical")
    ][["Date", "Sector", "Target", "Probability"]].rename(
        columns={"Probability": "BaselineP"}
    )

    for model in sorted(main_long["Model"].unique()):
        model_df = main_long[
            (main_long["Horizon"] == h)
            & (main_long["Model"] == model)
        ][["Date", "Sector", "Target", "Probability"]].rename(
            columns={"Probability": "ModelP"}
        )

        merged = model_df.merge(
            baseline[["Date", "Sector", "BaselineP"]],
            on=["Date", "Sector"],
            how="inner",
        )

        merged["ModelLoss"] = brier_loss(
            merged["Target"], merged["ModelP"]
        )
        merged["BaselineLoss"] = brier_loss(
            merged["Target"], merged["BaselineP"]
        )

        daily = (
            merged.groupby("Date", as_index=False)[
                ["ModelLoss", "BaselineLoss"]
            ]
            .mean()
            .sort_values("Date")
        )

        m = daily["ModelLoss"].to_numpy(float)
        b = daily["BaselineLoss"].to_numpy(float)

        point = 1 - np.mean(m) / np.mean(b)
        rng = np.random.default_rng(
            SEED + 1000 + 100 * h + sum(map(ord, model))
        )
        boots = np.empty(BOOTSTRAP_REPS)

        for r in range(BOOTSTRAP_REPS):
            idx = moving_block_indices(len(daily), PRIMARY_BLOCK, rng)
            boots[r] = 1 - np.mean(m[idx]) / np.mean(b[idx])

        low, high = np.quantile(boots, [0.025, 0.975])

        skill_rows.append({
            "Model": model,
            "Horizon": h,
            "BrierSkill": float(point),
            "CI_Low_MBB22": float(low),
            "CI_High_MBB22": float(high),
            "BootstrapReps": BOOTSTRAP_REPS,
        })

skill_table = pd.DataFrame(skill_rows)
skill_table.to_csv(
    TABLE_DIR / "Table_F03_Brier_Skill_with_MBB22_CI.csv",
    index=False,
)

# =============================================================================
# 8. FOCAL PAIRED FORECAST INFERENCE + BLOCK-LENGTH SENSITIVITY
# =============================================================================

FOCAL_COMPARISONS = [
    ("PrimaryForecast", "XGBoost", "ExpandingHistorical"),
    ("PrimaryForecast", "LSTM_Survival", "ExpandingHistorical"),
    ("PrimaryForecast", "XGBoost", "LSTM_Survival"),
    ("GraphAblation", "DynamicHawkesGraphTransformer", "NoGraphTransformer"),
    ("GraphAblation", "DynamicHawkesGraphTransformer", "StaticHawkesGraphTransformer"),
    ("GraphAblation", "DynamicHawkesGraphTransformer", "RandomGraphTransformer"),
    ("ParsimoniousBenchmark", "DynamicRidgeLogit", "ExpandingHistorical"),
    ("ParsimoniousBenchmark", "DynamicRidgeLogit", "XGBoost"),
    ("ParsimoniousBenchmark", "DynamicRidgeLogit", "LSTM_Survival"),
]

paired_rows = []

for family, model_a, model_b in FOCAL_COMPARISONS:
    for h in HORIZONS:
        a = main_long[
            (main_long["Horizon"] == h)
            & (main_long["Model"] == model_a)
        ][["Date", "Sector", "Target", "Probability"]].rename(
            columns={"Probability": "PA"}
        )
        b = main_long[
            (main_long["Horizon"] == h)
            & (main_long["Model"] == model_b)
        ][["Date", "Sector", "Probability"]].rename(
            columns={"Probability": "PB"}
        )

        merged = a.merge(b, on=["Date", "Sector"], how="inner")
        if len(merged) == 0:
            continue

        for loss_name in ["Brier", "LogScore"]:
            if loss_name == "Brier":
                la = brier_loss(merged["Target"], merged["PA"])
                lb = brier_loss(merged["Target"], merged["PB"])
            else:
                la = log_loss_obs(merged["Target"], merged["PA"])
                lb = log_loss_obs(merged["Target"], merged["PB"])

            merged_loss = merged[["Date"]].copy()
            merged_loss["Diff"] = la - lb
            daily = date_mean_frame(merged_loss, "Diff")
            date_values = daily["Diff"].to_numpy(float)

            for block in BLOCK_LENGTHS:
                inf = bootstrap_mean_inference(
                    date_values,
                    method="MovingBlock",
                    block=block,
                    reps=BOOTSTRAP_REPS,
                    seed=(
                        SEED
                        + h * 1000
                        + block * 10
                        + sum(map(ord, model_a + model_b + loss_name))
                    ),
                )
                paired_rows.append({
                    "Family": family,
                    "ModelA": model_a,
                    "ModelB": model_b,
                    "Orientation": "LossA_minus_LossB; negative favors ModelA",
                    "Horizon": h,
                    "Loss": loss_name,
                    "BootstrapMethod": "MovingBlock",
                    "BlockLength": block,
                    "NObservations": len(merged),
                    "NDates": len(daily),
                    "MeanLossDifference": inf["Mean"],
                    "CI_Low": inf["CI_Low"],
                    "CI_High": inf["CI_High"],
                    "PValue": inf["PValue"],
                    "BootstrapReps": BOOTSTRAP_REPS,
                })

            inf = bootstrap_mean_inference(
                date_values,
                method="Stationary",
                block=STATIONARY_MEAN_BLOCK,
                reps=BOOTSTRAP_REPS,
                seed=(
                    SEED
                    + 900000
                    + h * 1000
                    + sum(map(ord, model_a + model_b + loss_name))
                ),
            )
            paired_rows.append({
                "Family": family,
                "ModelA": model_a,
                "ModelB": model_b,
                "Orientation": "LossA_minus_LossB; negative favors ModelA",
                "Horizon": h,
                "Loss": loss_name,
                "BootstrapMethod": "Stationary",
                "BlockLength": STATIONARY_MEAN_BLOCK,
                "NObservations": len(merged),
                "NDates": len(daily),
                "MeanLossDifference": inf["Mean"],
                "CI_Low": inf["CI_Low"],
                "CI_High": inf["CI_High"],
                "PValue": inf["PValue"],
                "BootstrapReps": BOOTSTRAP_REPS,
            })

paired_all = pd.DataFrame(paired_rows)
paired_all.to_csv(
    TABLE_DIR / "Table_F04_Paired_Forecast_Inference_Block_Sensitivity.csv",
    index=False,
)

primary_inference = paired_all[
    (paired_all["BootstrapMethod"] == "MovingBlock")
    & (paired_all["BlockLength"] == PRIMARY_BLOCK)
].copy()

primary_inference["HolmP"] = np.nan
for (family, loss), idx in primary_inference.groupby(
    ["Family", "Loss"]
).groups.items():
    primary_inference.loc[idx, "HolmP"] = holm_adjust(
        primary_inference.loc[idx, "PValue"].to_numpy(float)
    )

primary_inference["SignificantRaw_5pct"] = (
    primary_inference["PValue"] < 0.05
)
primary_inference["SignificantHolm_5pct"] = (
    primary_inference["HolmP"] < 0.05
)
primary_inference["CIExcludesZero"] = (
    (primary_inference["CI_Low"] > 0)
    | (primary_inference["CI_High"] < 0)
)

primary_inference.to_csv(
    TABLE_DIR / "Table_F05_Primary_Paired_Inference_MBB22_Holm.csv",
    index=False,
)

# =============================================================================
# 9. RELIABILITY / CALIBRATION UNCERTAINTY
# =============================================================================

CALIBRATION_MODELS = [
    "XGBoost",
    "LSTM_Survival",
    "DynamicRidgeLogit",
    "DynamicHawkesGraphTransformer",
    "ExpandingHistorical",
]

reliability_rows = []
decomposition_rows = []

def assign_equal_count_bins(sub: pd.DataFrame, n_bins: int):
    x = sub.sort_values(
        ["Probability", "Date", "Sector"]
    ).reset_index(drop=True).copy()
    n = len(x)
    if n == 0:
        x["Bin"] = pd.Series(dtype=int)
        return x

    # Equal-count bins remain defined even when probabilities are tied.
    x["Bin"] = np.floor(
        np.arange(n) * n_bins / n
    ).astype(int)
    x["Bin"] = x["Bin"].clip(0, n_bins - 1)
    return x

for h in HORIZONS:
    for model in CALIBRATION_MODELS:
        sub = main_long[
            (main_long["Horizon"] == h)
            & (main_long["Model"] == model)
        ][["Date", "Sector", "Target", "Probability"]].dropna().copy()

        if len(sub) == 0:
            continue

        binned = assign_equal_count_bins(sub, RELIABILITY_BINS)
        ybar = float(binned["Target"].mean())
        total_n = len(binned)

        # Build fixed-bin reliability table.
        grouped = (
            binned.groupby("Bin", as_index=False)
            .agg(
                N=("Target", "size"),
                MeanPredicted=("Probability", "mean"),
                ObservedRate=("Target", "mean"),
            )
        )

        # Date x bin sufficient statistics for moving-block uncertainty.
        dates = np.array(sorted(binned["Date"].unique()))
        date_to_i = {pd.Timestamp(d): i for i, d in enumerate(dates)}
        B = int(binned["Bin"].max()) + 1

        count_mat = np.zeros((len(dates), B), dtype=float)
        ysum_mat = np.zeros((len(dates), B), dtype=float)
        psum_mat = np.zeros((len(dates), B), dtype=float)

        for row in binned.itertuples(index=False):
            i = date_to_i[pd.Timestamp(row.Date)]
            j = int(row.Bin)
            count_mat[i, j] += 1
            ysum_mat[i, j] += float(row.Target)
            psum_mat[i, j] += float(row.Probability)

        rng = np.random.default_rng(
            SEED + 200000 + 1000 * h + sum(map(ord, model))
        )
        boot_obs = np.full((BOOTSTRAP_REPS, B), np.nan)

        for r in range(BOOTSTRAP_REPS):
            idx = moving_block_indices(
                len(dates), PRIMARY_BLOCK, rng
            )
            c = count_mat[idx].sum(axis=0)
            ysum = ysum_mat[idx].sum(axis=0)
            valid = c > 0
            boot_obs[r, valid] = ysum[valid] / c[valid]

        ci_low = np.nanquantile(boot_obs, 0.025, axis=0)
        ci_high = np.nanquantile(boot_obs, 0.975, axis=0)

        for row in grouped.itertuples(index=False):
            reliability_rows.append({
                "Model": model,
                "Horizon": h,
                "Bin": int(row.Bin) + 1,
                "N": int(row.N),
                "MeanPredicted": float(row.MeanPredicted),
                "ObservedRate": float(row.ObservedRate),
                "ObservedRate_CI_Low_MBB22": float(ci_low[int(row.Bin)]),
                "ObservedRate_CI_High_MBB22": float(ci_high[int(row.Bin)]),
            })

        # Murphy decomposition using the same fixed equal-count bins.
        reliability = 0.0
        resolution = 0.0

        for row in grouped.itertuples(index=False):
            w = row.N / total_n
            reliability += w * (row.MeanPredicted - row.ObservedRate) ** 2
            resolution += w * (row.ObservedRate - ybar) ** 2

        uncertainty = ybar * (1 - ybar)
        decomposed_brier = reliability - resolution + uncertainty
        actual_brier = float(
            np.mean(brier_loss(binned["Target"], binned["Probability"]))
        )

        decomposition_rows.append({
            "Model": model,
            "Horizon": h,
            "N": total_n,
            "EventRate": ybar,
            "Reliability": reliability,
            "Resolution": resolution,
            "Uncertainty": uncertainty,
            "BrierFromBinnedDecomposition": decomposed_brier,
            "ActualBrier": actual_brier,
            "BinningApproximationDifference": decomposed_brier - actual_brier,
            "Bins": int(B),
        })

reliability_table = pd.DataFrame(reliability_rows)
reliability_table.to_csv(
    TABLE_DIR / "Table_F06_Reliability_with_MBB22_Uncertainty.csv",
    index=False,
)

decomposition_table = pd.DataFrame(decomposition_rows)
decomposition_table.to_csv(
    TABLE_DIR / "Table_F07_Brier_Decomposition.csv",
    index=False,
)

# Reliability figures: one distinct figure per horizon.
for h in HORIZONS:
    fig, ax = plt.subplots(figsize=(7.5, 6.5))

    for model in CALIBRATION_MODELS[:-1]:
        dd = reliability_table[
            (reliability_table["Horizon"] == h)
            & (reliability_table["Model"] == model)
        ].sort_values("Bin")
        if len(dd) == 0:
            continue

        lower = dd["ObservedRate"] - dd["ObservedRate_CI_Low_MBB22"]
        upper = dd["ObservedRate_CI_High_MBB22"] - dd["ObservedRate"]

        ax.errorbar(
            dd["MeanPredicted"],
            dd["ObservedRate"],
            yerr=np.vstack([lower, upper]),
            marker="o",
            linewidth=1.2,
            capsize=2,
            label=model,
        )

    max_axis = max(
        0.05,
        float(
            reliability_table[
                reliability_table["Horizon"] == h
            ][["MeanPredicted", "ObservedRate_CI_High_MBB22"]]
            .max()
            .max()
        ),
    )
    max_axis = min(1.0, max_axis * 1.08)
    ax.plot([0, max_axis], [0, max_axis], linestyle="--", linewidth=1)
    ax.set_xlim(0, max_axis)
    ax.set_ylim(0, max_axis)
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Observed tail-event rate")
    ax.set_title(f"Reliability with 95% MBB uncertainty — {h}-day horizon")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(
        FIG_DIR / f"Figure_F02_Reliability_H{h}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

# =============================================================================
# 10. DECISION / COST-SENSITIVE ANALYSIS
# =============================================================================

DECISION_MODELS = [
    "XGBoost",
    "LSTM_Survival",
    "DynamicRidgeLogit",
    "DynamicHawkesGraphTransformer",
    "ExpandingHistorical",
]

thresholds = np.round(np.arange(0.01, 0.501, 0.01), 2)
decision_rows = []

for h in HORIZONS:
    # Identical target support by construction.
    ref = main_long[
        (main_long["Horizon"] == h)
        & (main_long["Model"] == "ExpandingHistorical")
    ][["Date", "Sector", "Target"]]
    prevalence = float(ref["Target"].mean())
    n = len(ref)

    for model in DECISION_MODELS:
        dd = main_long[
            (main_long["Horizon"] == h)
            & (main_long["Model"] == model)
        ][["Date", "Sector", "Target", "Probability"]]

        for pt in thresholds:
            action = dd["Probability"].to_numpy(float) >= pt
            y = dd["Target"].to_numpy(float)

            tp = np.sum(action & (y == 1))
            fp = np.sum(action & (y == 0))

            nb = tp / n - fp / n * pt / (1 - pt)
            nb_all = prevalence - (1 - prevalence) * pt / (1 - pt)

            decision_rows.append({
                "Model": model,
                "Horizon": h,
                "Threshold": float(pt),
                "N": n,
                "EventRate": prevalence,
                "NetBenefit": float(nb),
                "TreatAllNetBenefit": float(nb_all),
                "TreatNoneNetBenefit": 0.0,
            })

decision_table = pd.DataFrame(decision_rows)
decision_table.to_csv(
    TABLE_DIR / "Table_F08_Decision_Curve_Net_Benefit.csv",
    index=False,
)

for h in HORIZONS:
    fig, ax = plt.subplots(figsize=(7.8, 6.2))

    for model in DECISION_MODELS[:-1]:
        dd = decision_table[
            (decision_table["Horizon"] == h)
            & (decision_table["Model"] == model)
        ]
        ax.plot(
            dd["Threshold"],
            dd["NetBenefit"],
            linewidth=1.3,
            label=model,
        )

    ref = decision_table[
        (decision_table["Horizon"] == h)
        & (decision_table["Model"] == "ExpandingHistorical")
    ]
    ax.plot(
        ref["Threshold"],
        ref["TreatAllNetBenefit"],
        linestyle="--",
        linewidth=1.0,
        label="Treat all",
    )
    ax.axhline(0.0, linestyle=":", linewidth=1.0, label="Treat none")
    ax.set_xlabel("Decision threshold")
    ax.set_ylabel("Net benefit")
    ax.set_title(f"Decision-curve analysis — {h}-day horizon")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(
        FIG_DIR / f"Figure_F03_DecisionCurve_H{h}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

# Cost-sensitive classification table.
FN_COST_RATIOS = [1, 2, 5, 10, 20, 50]
cost_rows = []

for h in HORIZONS:
    for ratio in FN_COST_RATIOS:
        threshold = 1.0 / (1.0 + ratio)

        for model in DECISION_MODELS:
            dd = main_long[
                (main_long["Horizon"] == h)
                & (main_long["Model"] == model)
            ][["Target", "Probability"]]
            y = dd["Target"].to_numpy(float)
            action = dd["Probability"].to_numpy(float) >= threshold

            fp = np.sum(action & (y == 0))
            fn = np.sum((~action) & (y == 1))
            total_cost = fp + ratio * fn
            mean_cost = total_cost / len(dd)

            cost_rows.append({
                "Model": model,
                "Horizon": h,
                "FalseNegative_to_FalsePositive_CostRatio": ratio,
                "BayesDecisionThreshold": threshold,
                "N": len(dd),
                "FalsePositives": int(fp),
                "FalseNegatives": int(fn),
                "MeanRealizedCost": float(mean_cost),
            })

cost_table = pd.DataFrame(cost_rows)
cost_table.to_csv(
    TABLE_DIR / "Table_F09_Cost_Sensitive_Classification.csv",
    index=False,
)

# =============================================================================
# 11. FORMAL H5 SECTOR x HORIZON HETEROGENEITY
# =============================================================================

def contribution_matrix(
    model_family: str,
    loss_name: str,
):
    """
    Positive contribution means the Full specification improves forecast loss:
        contribution = loss(NoGARCH) - loss(Full)
    """
    records = []

    for h in HORIZONS:
        target = f"EVT0025_Target_{h}"

        if model_family == "XGBoost":
            p_full = f"XGBoost__Full__P{h}"
            p_nog = f"XGBoost__NoGARCH__P{h}"
        elif model_family == "DynamicLogit":
            p_full = f"DynamicLogit__Full__P{h}"
            p_nog = f"DynamicLogit__NoGARCH__P{h}"
        else:
            raise ValueError(model_family)

        dd = p3_test[
            ["Date", "Sector", target, p_full, p_nog]
        ].copy()
        dd = dd[
            dd[target].notna()
            & dd[p_full].notna()
            & dd[p_nog].notna()
        ]

        y = dd[target].to_numpy(float)

        if loss_name == "Brier":
            full_loss = brier_loss(y, dd[p_full])
            nog_loss = brier_loss(y, dd[p_nog])
        elif loss_name == "LogScore":
            full_loss = log_loss_obs(y, dd[p_full])
            nog_loss = log_loss_obs(y, dd[p_nog])
        else:
            raise ValueError(loss_name)

        temp = dd[["Date", "Sector"]].copy()
        temp["Horizon"] = h
        temp["Contribution"] = nog_loss - full_loss
        records.append(temp)

    return pd.concat(records, ignore_index=True)

def omnibus_heterogeneity_test(
    long_df: pd.DataFrame,
    group_cols: list[str],
    reps: int,
    block: int,
    seed: int,
):
    """
    Tests equality of group means using a date-block bootstrap under the null.

    Statistic:
        Q = sum_g n_g (mean_g - grand_mean)^2

    The null bootstrap centers every group to the weighted grand mean, then
    resamples whole dates in moving blocks. Resampling dates preserves
    contemporaneous cross-sector and cross-horizon dependence.
    """
    d = long_df[["Date"] + group_cols + ["Contribution"]].dropna().copy()
    d["Group"] = d[group_cols].astype(str).agg("|".join, axis=1)

    groups = sorted(d["Group"].unique())
    dates = np.array(sorted(d["Date"].unique()))
    date_to_i = {pd.Timestamp(x): i for i, x in enumerate(dates)}
    group_to_j = {g: j for j, g in enumerate(groups)}

    mat = np.full((len(dates), len(groups)), np.nan)

    # There is at most one observation per Date/Sector/Horizon cell for the
    # full interaction. For pooled sector/horizon tests, duplicates are averaged.
    agg = (
        d.groupby(["Date", "Group"], as_index=False)["Contribution"]
        .mean()
    )

    for row in agg.itertuples(index=False):
        mat[date_to_i[pd.Timestamp(row.Date)], group_to_j[row.Group]] = (
            float(row.Contribution)
        )

    means = np.nanmean(mat, axis=0)
    counts = np.sum(np.isfinite(mat), axis=0)

    valid = np.isfinite(means) & (counts > 0)
    means = means[valid]
    counts = counts[valid]
    mat = mat[:, valid]
    groups_valid = np.array(groups, dtype=object)[valid]

    grand = float(np.sum(counts * means) / np.sum(counts))
    q_obs = float(np.sum(counts * (means - grand) ** 2))

    # Null-centre each group to the same grand mean.
    null_mat = mat.copy()
    for j in range(null_mat.shape[1]):
        ok = np.isfinite(null_mat[:, j])
        null_mat[ok, j] = null_mat[ok, j] - means[j] + grand

    rng = np.random.default_rng(seed)
    q_boot = np.empty(reps)

    for r in range(reps):
        idx = moving_block_indices(len(dates), block, rng)
        sample = null_mat[idx]
        boot_means = np.nanmean(sample, axis=0)
        boot_counts = np.sum(np.isfinite(sample), axis=0)

        ok = np.isfinite(boot_means) & (boot_counts > 0)
        boot_grand = float(
            np.sum(boot_counts[ok] * boot_means[ok])
            / np.sum(boot_counts[ok])
        )
        q_boot[r] = float(
            np.sum(
                boot_counts[ok]
                * (boot_means[ok] - boot_grand) ** 2
            )
        )

    p = (1 + np.sum(q_boot >= q_obs)) / (reps + 1)

    group_table = pd.DataFrame({
        "Group": groups_valid,
        "MeanContribution": means,
        "N": counts.astype(int),
    })

    return {
        "Statistic_Q": q_obs,
        "PValue": float(p),
        "Groups": len(groups_valid),
        "GrandMeanContribution": grand,
        "GroupTable": group_table,
    }

h5_cell_rows = []
h5_test_rows = []

for model_family in ["XGBoost", "DynamicLogit"]:
    for loss_name in ["Brier", "LogScore"]:
        cdf = contribution_matrix(model_family, loss_name)

        # Cell-specific means / CI / p versus zero.
        for (sector, h), sub in cdf.groupby(["Sector", "Horizon"]):
            daily = date_mean_frame(sub, "Contribution")
            inf = bootstrap_mean_inference(
                daily["Contribution"].to_numpy(float),
                method="MovingBlock",
                block=PRIMARY_BLOCK,
                reps=BOOTSTRAP_REPS,
                seed=(
                    SEED
                    + 300000
                    + int(h) * 100
                    + sum(map(ord, model_family + loss_name + sector))
                ),
            )
            h5_cell_rows.append({
                "ModelFamily": model_family,
                "Loss": loss_name,
                "Sector": sector,
                "Horizon": int(h),
                "Orientation": "NoGARCH loss - Full loss; positive favors GARCH information",
                "MeanContribution": inf["Mean"],
                "CI_Low": inf["CI_Low"],
                "CI_High": inf["CI_High"],
                "PValue": inf["PValue"],
                "NDates": len(daily),
            })

        # Formal omnibus 24-cell interaction heterogeneity.
        res = omnibus_heterogeneity_test(
            cdf,
            ["Sector", "Horizon"],
            reps=BOOTSTRAP_REPS,
            block=PRIMARY_BLOCK,
            seed=SEED + 400000 + sum(map(ord, model_family + loss_name)),
        )
        h5_test_rows.append({
            "ModelFamily": model_family,
            "Loss": loss_name,
            "Test": "Sector_by_Horizon_24Cell_Omnibus",
            "Statistic_Q": res["Statistic_Q"],
            "PValue": res["PValue"],
            "Groups": res["Groups"],
            "GrandMeanContribution": res["GrandMeanContribution"],
            "Bootstrap": f"Moving block; L={PRIMARY_BLOCK}; reps={BOOTSTRAP_REPS}",
        })

        # Sector-only heterogeneity pooling horizons within each date/sector.
        sector_df = (
            cdf.groupby(["Date", "Sector"], as_index=False)["Contribution"]
            .mean()
        )
        res = omnibus_heterogeneity_test(
            sector_df,
            ["Sector"],
            reps=BOOTSTRAP_REPS,
            block=PRIMARY_BLOCK,
            seed=SEED + 410000 + sum(map(ord, model_family + loss_name)),
        )
        h5_test_rows.append({
            "ModelFamily": model_family,
            "Loss": loss_name,
            "Test": "Sector_Omnibus",
            "Statistic_Q": res["Statistic_Q"],
            "PValue": res["PValue"],
            "Groups": res["Groups"],
            "GrandMeanContribution": res["GrandMeanContribution"],
            "Bootstrap": f"Moving block; L={PRIMARY_BLOCK}; reps={BOOTSTRAP_REPS}",
        })

        # Horizon-only heterogeneity pooling sectors within each date/horizon.
        horizon_df = (
            cdf.groupby(["Date", "Horizon"], as_index=False)["Contribution"]
            .mean()
        )
        res = omnibus_heterogeneity_test(
            horizon_df,
            ["Horizon"],
            reps=BOOTSTRAP_REPS,
            block=PRIMARY_BLOCK,
            seed=SEED + 420000 + sum(map(ord, model_family + loss_name)),
        )
        h5_test_rows.append({
            "ModelFamily": model_family,
            "Loss": loss_name,
            "Test": "Horizon_Omnibus",
            "Statistic_Q": res["Statistic_Q"],
            "PValue": res["PValue"],
            "Groups": res["Groups"],
            "GrandMeanContribution": res["GrandMeanContribution"],
            "Bootstrap": f"Moving block; L={PRIMARY_BLOCK}; reps={BOOTSTRAP_REPS}",
        })

h5_cells = pd.DataFrame(h5_cell_rows)
h5_cells["HolmP_within_Model_Loss"] = np.nan

for (model, loss), idx in h5_cells.groupby(
    ["ModelFamily", "Loss"]
).groups.items():
    h5_cells.loc[idx, "HolmP_within_Model_Loss"] = holm_adjust(
        h5_cells.loc[idx, "PValue"].to_numpy(float)
    )

h5_cells["SignificantHolm_5pct"] = (
    h5_cells["HolmP_within_Model_Loss"] < 0.05
)

h5_cells.to_csv(
    TABLE_DIR / "Table_F10_H5_Sector_Horizon_Cell_Contributions.csv",
    index=False,
)

h5_tests = pd.DataFrame(h5_test_rows)
h5_tests["HolmP_within_Model_Loss"] = np.nan
for (model, loss), idx in h5_tests.groupby(
    ["ModelFamily", "Loss"]
).groups.items():
    h5_tests.loc[idx, "HolmP_within_Model_Loss"] = holm_adjust(
        h5_tests.loc[idx, "PValue"].to_numpy(float)
    )
h5_tests["SignificantHolm_5pct"] = (
    h5_tests["HolmP_within_Model_Loss"] < 0.05
)

h5_tests.to_csv(
    TABLE_DIR / "Table_F11_H5_Formal_Heterogeneity_Tests.csv",
    index=False,
)

# H5 heatmaps: one figure per model family using Brier contribution.
for model_family in ["XGBoost", "DynamicLogit"]:
    pivot = (
        h5_cells[
            (h5_cells["ModelFamily"] == model_family)
            & (h5_cells["Loss"] == "Brier")
        ]
        .pivot(index="Sector", columns="Horizon", values="MeanContribution")
        .reindex(columns=HORIZONS)
    )

    fig, ax = plt.subplots(figsize=(7.5, 5.8))
    im = ax.imshow(pivot.to_numpy(), aspect="auto")
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(x) for x in pivot.columns])
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Forecast horizon (days)")
    ax.set_ylabel("Sector")
    ax.set_title(
        f"GJR-GARCH incremental Brier contribution — {model_family}\n"
        "(positive = Full model lower loss)"
    )
    fig.colorbar(im, ax=ax, label="NoGARCH Brier loss − Full Brier loss")
    fig.tight_layout()
    fig.savefig(
        FIG_DIR / f"Figure_F04_H5_GARCH_Contribution_{model_family}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close(fig)

# =============================================================================
# 12. RIDGE-LAMBDA GRID BOUNDARY AUDIT
# =============================================================================

logit_primary_tuning = p3_logit_tuning[
    (p3_logit_tuning["Definition"] == "EVT_0025")
    & (p3_logit_tuning["Variant"] == "Full")
].copy()

lambda_rows = []

for h, sub in logit_primary_tuning.groupby("Horizon"):
    sub = sub.sort_values("RidgeLambda").copy()
    best_idx = sub["TrainTuneLogScore"].idxmin()
    best = sub.loc[best_idx]
    max_lambda = float(sub["RidgeLambda"].max())

    lambda_rows.append({
        "Horizon": int(h),
        "SelectedLambda": float(best["RidgeLambda"]),
        "GridMinimum": float(sub["RidgeLambda"].min()),
        "GridMaximum": max_lambda,
        "SelectedAtUpperBoundary": bool(
            np.isclose(float(best["RidgeLambda"]), max_lambda)
        ),
        "BestTrainTuneLogScore": float(best["TrainTuneLogScore"]),
        "NextLargerLambdaAvailable": False
        if np.isclose(float(best["RidgeLambda"]), max_lambda)
        else True,
        "Interpretation": (
            "Selected penalty is the upper grid boundary; report transparently. "
            "No post-Test grid expansion is performed in this final inference script."
            if np.isclose(float(best["RidgeLambda"]), max_lambda)
            else "Selected penalty is interior to the pre-specified grid."
        ),
    })

lambda_audit = pd.DataFrame(lambda_rows)
lambda_audit.to_csv(
    TABLE_DIR / "Table_F12_Dynamic_Logit_Lambda_Boundary_Audit.csv",
    index=False,
)

# =============================================================================
# 13. SUMMARY FIGURE: MAIN BRIER SCORE
# =============================================================================

PLOT_MODELS = [
    "ExpandingHistorical",
    "XGBoost",
    "DynamicRidgeLogit",
    "LSTM_Survival",
    "TemporalTransformer_Survival",
    "DynamicHawkesGraphTransformer",
]

fig, ax = plt.subplots(figsize=(8.2, 6.3))

for model in PLOT_MODELS:
    dd = main_metrics[main_metrics["Model"] == model].sort_values("Horizon")
    ax.plot(
        dd["Horizon"],
        dd["Brier"],
        marker="o",
        linewidth=1.4,
        label=model,
    )

ax.set_xlabel("Forecast horizon (days)")
ax.set_ylabel("Brier score")
ax.set_title("Locked-Test probability forecast performance")
ax.set_xticks(HORIZONS)
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(
    FIG_DIR / "Figure_F01_Main_Test_Brier_by_Horizon.png",
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

# =============================================================================
# 14. FINAL INTERPRETIVE FLAGS (DATA-DRIVEN, NOT MANUSCRIPT CLAIMS)
# =============================================================================

interpretive_rows = []

# Best Brier model by horizon.
for h in HORIZONS:
    best = (
        main_metrics[main_metrics["Horizon"] == h]
        .sort_values("Brier")
        .iloc[0]
    )
    interpretive_rows.append({
        "Topic": f"BestBrier_H{h}",
        "Result": best["Model"],
        "Value": best["Brier"],
        "Rule": "Lowest locked-Test Brier score",
    })

# Graph evidence primary family.
graph_primary = primary_inference[
    (primary_inference["Family"] == "GraphAblation")
    & (primary_inference["Loss"] == "Brier")
]
for row in graph_primary.itertuples(index=False):
    interpretive_rows.append({
        "Topic": (
            f"Graph_{row.ModelA}_vs_{row.ModelB}_H{row.Horizon}"
        ),
        "Result": (
            "ModelA significantly lower loss after Holm"
            if row.SignificantHolm_5pct and row.MeanLossDifference < 0
            else "No Holm-adjusted superiority"
        ),
        "Value": row.MeanLossDifference,
        "Rule": "MBB22 paired Brier difference with Holm adjustment",
    })

# H5 omnibus.
for row in h5_tests[
    h5_tests["Test"] == "Sector_by_Horizon_24Cell_Omnibus"
].itertuples(index=False):
    interpretive_rows.append({
        "Topic": f"H5_{row.ModelFamily}_{row.Loss}",
        "Result": (
            "Formal heterogeneity detected"
            if row.SignificantHolm_5pct
            else "Formal heterogeneity not detected after Holm adjustment"
        ),
        "Value": row.PValue,
        "Rule": "24-cell sector-by-horizon MBB omnibus test",
    })

interpretive = pd.DataFrame(interpretive_rows)
interpretive.to_csv(
    TABLE_DIR / "Table_F13_Data_Driven_Interpretive_Flags.csv",
    index=False,
)

# =============================================================================
# 15. FINAL OUTPUT INTEGRITY
# =============================================================================

final_checks = []

def final_check(name, passed, detail=""):
    final_checks.append({
        "Check": name,
        "Passed": bool(passed),
        "Detail": str(detail),
    })

final_check(
    "Main Test prediction panel contains only Test observations",
    bool(
        main_long["Date"].min() >= pd.Timestamp(p2_meta["TestStart"])
    ),
    f"min date={main_long['Date'].min()}",
)
final_check(
    "All main probabilities finite and inside [0,1]",
    bool(
        np.isfinite(main_long["Probability"]).all()
        and (main_long["Probability"] >= 0).all()
        and (main_long["Probability"] <= 1).all()
    ),
)
final_check(
    "Dynamic ridge-logit included in main comparison",
    "DynamicRidgeLogit" in set(main_metrics["Model"]),
)
final_check(
    "Main table reports N/events/event rate at all horizons",
    set(HORIZONS).issubset(set(sample_table["Horizon"])),
)
final_check(
    "Paired inference includes all 10/22/44 moving-block lengths",
    set(BLOCK_LENGTHS).issubset(
        set(
            paired_all.loc[
                paired_all["BootstrapMethod"] == "MovingBlock",
                "BlockLength",
            ].astype(int)
        )
    ),
)
final_check(
    "Stationary bootstrap sensitivity present",
    bool((paired_all["BootstrapMethod"] == "Stationary").any()),
)
final_check(
    "Holm-adjusted primary inference produced",
    primary_inference["HolmP"].notna().all(),
)
final_check(
    "Reliability uncertainty bands produced",
    (
        reliability_table["ObservedRate_CI_Low_MBB22"].notna().all()
        and reliability_table["ObservedRate_CI_High_MBB22"].notna().all()
    ),
)
final_check(
    "Brier decomposition produced",
    len(decomposition_table) > 0,
)
final_check(
    "Decision/cost-sensitive outputs produced",
    len(decision_table) > 0 and len(cost_table) > 0,
)
final_check(
    "Formal H5 sector-by-horizon omnibus tests produced",
    bool(
        (
            h5_tests["Test"] == "Sector_by_Horizon_24Cell_Omnibus"
        ).sum() == 4
    ),
)
final_check(
    "No model retraining performed in final inference script",
    True,
    "Uses only frozen saved probabilities / Phase3 ablation probabilities.",
)

final_integrity = pd.DataFrame(final_checks)
final_integrity.to_csv(
    TABLE_DIR / "Table_F14_Final_Postprocessing_Integrity_Checks.csv",
    index=False,
)

if not final_integrity["Passed"].all():
    failed = final_integrity.loc[
        ~final_integrity["Passed"], "Check"
    ].tolist()
    raise RuntimeError(
        f"Final post-processing integrity checks failed: {failed}"
    )

# =============================================================================
# 16. METADATA / MANIFEST
# =============================================================================

metadata = {
    "Project": "Forecasting Sector Tail-Event Probabilities",
    "ScriptVersion": "1.0-reviewer-final-inference",
    "GeneratedAt": datetime.now().isoformat(),
    "Seed": SEED,
    "BootstrapReps": BOOTSTRAP_REPS,
    "MovingBlockLengths": BLOCK_LENGTHS,
    "PrimaryMovingBlockLength": PRIMARY_BLOCK,
    "StationaryBootstrapMeanBlock": STATIONARY_MEAN_BLOCK,
    "ReliabilityBins": RELIABILITY_BINS,
    "Phase2InputZip": PHASE2_ZIP.name,
    "Phase3InputZip": PHASE3_ZIP.name,
    "Phase2ScriptVersion": p2_meta.get("ScriptVersion"),
    "Phase3ScriptVersion": p3_meta.get("ScriptVersion"),
    "TestStart": p2_meta.get("TestStart"),
    "MainXGBoostSource": "Phase2.2 v2.3 main forecasting pipeline",
    "DynamicRidgeLogitSource": "Phase3 v3.2 parsimonious reviewer benchmark",
    "H5Definition": (
        "Heterogeneity in incremental GJR-GARCH forecast contribution, "
        "defined as NoGARCH loss minus Full loss across sector-horizon cells."
    ),
    "Multiplicity": (
        "Holm adjustment within pre-specified comparison family and loss; "
        "H5 cell tests adjusted within model family and loss."
    ),
    "NoModelRetraining": True,
}

with open(
    DATA_DIR / "final_inference_metadata.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(metadata, handle, indent=2)

def md5_file(path: Path):
    h = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_rows = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and ZIP_DIR not in path.parents:
        manifest_rows.append({
            "RelativePath": str(path.relative_to(OUTPUT_ROOT)),
            "Bytes": path.stat().st_size,
            "MD5": md5_file(path),
        })

manifest = pd.DataFrame(manifest_rows)
manifest.to_csv(
    DATA_DIR / "final_output_manifest_md5.csv",
    index=False,
)

# =============================================================================
# 17. ZIP + OPTIONAL COLAB DOWNLOAD
# =============================================================================

zip_path = ZIP_DIR / "Python_Final_Reviewer_Inference_v1_0_All_Outputs.zip"

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in sorted(OUTPUT_ROOT.rglob("*")):
        if (
            path.is_file()
            and path != zip_path
            and ZIP_DIR not in path.parents
        ):
            archive.write(
                path,
                arcname=str(path.relative_to(OUTPUT_ROOT)),
            )

log(f"Final output ZIP: {zip_path}")
log(f"Final tables={len(list(TABLE_DIR.glob('*.csv')))}; figures={len(list(FIG_DIR.glob('*.png')))}")
log("FINAL REVIEWER INFERENCE COMPLETE — ALL INTEGRITY CHECKS PASSED")

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass


[2026-09-12 18:49:13] ====================================================================================================
[2026-09-12 18:49:13] FINAL REVIEWER INFERENCE START
[2026-09-12 18:49:13] Python=3.13.15; platform=Linux-6.6.122+-x86_64-with-glibc2.39; seed=20260912
[2026-09-12 18:49:13] bootstrap reps=500; moving blocks=[10, 22, 44]; stationary mean block=22

Please upload BOTH frozen reviewer-revision ZIPs:
  1) Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip
  2) Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip



Saving Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip to Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip
Saving Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip to Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip
[2026-09-12 18:51:21] Accepted Phase2 ZIP: /content/Python_Phase2_2_ReviewerRevision_v2_3_All_Outputs.zip
[2026-09-12 18:51:21] Accepted Phase3 ZIP: /content/Python_Phase3_ReviewerRevision_v3_2_All_Outputs.zip
[2026-09-12 18:51:22] Phase2 rows=5,946; Phase3 rows=5,946
[2026-09-12 18:52:08] Final output ZIP: /content/Sectoral_Tail_Event_Final_Reviewer_Inference_v1_0/05_Zip/Python_Final_Reviewer_Inference_v1_0_All_Outputs.zip
[2026-09-12 18:52:08] Final tables=16; figures=11
[2026-09-12 18:52:08] FINAL REVIEWER INFERENCE COMPLETE — ALL INTEGRITY CHECKS PASSED


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>